In [ ]:
import datetime
# import numpy as np

import frads as fr
from pyenergyplus.dataset import ref_models, weather_files

In [ ]:
epmodel = fr.load_energyplus_model(ref_models["medium_office"]) #

In [ ]:
gs_sr2_ec01 = fr.create_glazing_system(
    name="sr2_ec01",
    layer_inputs=[
        fr.LayerInput("glaizng_2/glaizng_2___/sageglass-cooltonetm-sr2-7mm-lami-clear-tint.json"),
        fr.LayerInput("glaizngs/igsdb_product_14028.json"),
    ],
    gaps=[
        fr.Gap(
            gas=[fr.Gas("air", 0.1), fr.Gas("argon", 0.9)],
            thickness_m=0.0127,
        )
    ],
)

gs_sr2_ec02 = fr.create_glazing_system(
    name="sr2_ec02",
    layer_inputs=[
        fr.LayerInput("glaizng_2/glaizng_2___/sageglass-cooltonetm-sr2-7mm-lami-light-tint.json"),
        fr.LayerInput("glaizngs/igsdb_product_14028.json"),
    ],
    gaps=[
        fr.Gap(
            gas=[fr.Gas("air", 0.1), fr.Gas("argon", 0.9)],
            thickness_m=0.0127,
        )
    ],
)

gs_sr2_ec03 = fr.create_glazing_system(
    name="sr2_ec03",
    layer_inputs=[
        fr.LayerInput("glaizng_2/glaizng_2___/sageglass-cooltonetm-sr2-7mm-lami-medium-tint.json"),
        fr.LayerInput("glaizngs/igsdb_product_14028.json"),
    ],
    gaps=[
        fr.Gap(
            gas=[fr.Gas("air", 0.1), fr.Gas("argon", 0.9)],
            thickness_m=0.0127,
        )
    ],
)

gs_sr2_ec04 = fr.create_glazing_system(
    name="sr2_ec04",
    layer_inputs=[
        fr.LayerInput("glaizng_2/glaizng_2___/sageglass-cooltonetm-sr2-7mm-lami-full-tint.json"),
        fr.LayerInput("glaizngs/igsdb_product_14028.json"),
    ],
    gaps=[
        fr.Gap(
            gas=[fr.Gas("air", 0.1), fr.Gas("argon", 0.9)],
            thickness_m=0.0127,
        )
    ],
)



In [ ]:
epmodel.add_glazing_system(gs_sr2_ec01)
epmodel.add_glazing_system(gs_sr2_ec02)
epmodel.add_glazing_system(gs_sr2_ec03)
epmodel.add_glazing_system(gs_sr2_ec04)
epmodel.add_lighting(
    zone="Perimeter_mid_ZN_1",
    lighting_level=2231, # 
    replace=True
)
epmodel.add_lighting(
    zone="Perimeter_mid_ZN_2",
    lighting_level=2231, # 
    replace=True
)
epmodel.add_lighting(
    zone="Perimeter_mid_ZN_3",
    lighting_level=1412, # 
    replace=True
)
epmodel.add_lighting(
    zone="Perimeter_mid_ZN_4",
    lighting_level=1412, # 
    replace=True
)
epmodel.add_lighting(
    zone="Core_mid",
    lighting_level=10586, # 
    replace=True
)


In [ ]:
eps = fr.EnergyPlusSetup(
    epmodel, weather_files["usa_ca_san_francisco"], enable_radiance=True
)

In [ ]:
"""
EXPERT DATA COLLECTION POLICY v12 — SYMMETRIC closed-loop thermal + SDAR masks
==============================================================================

Changes from v11:
  SYMMETRIC THERMAL. v11 gave heating two state inputs (temp preheat + solar
  heat-cut) but cooling only one weak, easily-defeated input (solar precool).
  v12 makes cooling a first-class state-driven channel and fixes two failure
  modes:

    1) COOLING gets TWO state inputs, mirroring heating:
         - temp-driven: pre-occupancy pull-down (if zone already hot) +
           occupied drift-tightening (as zone climbs toward the cool SP),
         - solar-driven: anticipatory precool (sun x warm).

    2) COIL-STATE GATING on the solar trims (uses heating_active/cooling_active
       already computed each step): the heating solar-cut fires only when
       heating is the active/likely regime AND cooling is NOT active; the solar
       precool fires only when cooling is the active/likely regime. This stops
       the heating SP from wiggling (and generating phantom SDAR fires) on
       sunny summer afternoons when only the cooling coil is running.

    3) "ACTIVE COIL IS MASTER" deadband: in cooling regime the cooling SP moves
       freely and HEATING yields downward; otherwise heating moves and cooling
       yields upward. This stops the deadband from silently clamping the active
       channel's trim to a constant.
       (NOTE: a band set to exactly DEADBAND width still has no room — cooling
       then sits on the 22 °C actuator floor. Give occupied bands >= ~2-3 °C.)

  Two new per-profile knobs (fall back to THERMAL_DEFAULTS):
     cool_opt_gain    : °C pull-down per °C ABOVE occ_cool during pre-window
     cool_drift_gain  : °C tighten as zone climbs the top 2 °C toward cool SP

  Carried over from v11 / v10:
    - SDAR 19-dim action logging (Nie et al., ICLR 2025).
    - Action-level deadband on perimeter lighting.
    - Per-subsystem multi-rate updates (glazing/lighting/thermal).
    - Glare override never lightens; COIL_ACTIVE_EPS = 10 J; expert id logged.
    - Logged is_preconditioning uses the 1 h global; the thermal controller
      preheats over its own (possibly wider) opt_start_window_hrs.

  Action vector layout (all normalized to [-1, 1]):
    [0:4]   glazing tier (4 perimeter windows)
    [4:9]   lighting power (4 perim + 1 core)
    [9:14]  heating SP (4 perim + 1 core)
    [14:19] cooling SP (4 perim + 1 core)

Glazing mapping (Tvis):
  sr2_ec01 = clear-tint  (BRIGHTEST)
  sr2_ec02 = light-tint
  sr2_ec03 = medium-tint
  sr2_ec04 = full-tint   (DARKEST)
"""

import numpy as np


# ============================================================
# PICK YOUR EXPERT
# ============================================================

EXPERT_MODE = "smooth"   # "balanced" | "energy" | "comfort" | "visual" | "smooth"

EXPERT_ID_MAP = {
    "balanced": 0,
    "energy": 1,
    "comfort": 2,
    "visual": 3,
    "smooth": 4,
    "fast_responsive": 5,
    "slow_smooth": 6,
    "staggered_multirate": 7,
    "bad_energy_lazy": 8,
    "adaptive_balanced_good": 9
}

# ============================================================
# THERMAL-TRIM DEFAULTS
# ============================================================
# Any profile missing a knob falls back to these (read via prof.get(...)).
THERMAL_DEFAULTS = {
    "opt_start_window_hrs":  2.0,   # h before occupancy the preheat may act
    "opt_start_gain":        0.5,   # °C SP boost per °C below occ heat target
    "solar_ref":             600.0, # ext_irr (W/m²) at which solar term saturates
    "solar_heat_offset_max": 1.0,   # max °C to DROP heating SP under full sun
    "solar_cool_offset_max": 0.8,   # max °C to DROP cooling SP (precool, solar) under sun
    "cool_opt_gain":         0.5,   # °C SP pull-down per °C ABOVE occ_cool in pre-window
    "cool_drift_gain":       0.5,   # °C SP tighten as zone climbs top 2°C toward cool SP
}


# ============================================================
# PROFILES
# ============================================================
# Per-expert thermal-trim character:
#   opt_start_gain   high -> aggressive preheat (comfort); low -> cold arrivals
#   solar_heat_off   high -> cuts heating hard when sunny (savings); too high -> under-heats
#   solar_cool_off   high -> aggressive solar precool; zero -> lets zone overheat
#   cool_opt_gain    high -> aggressive early cooling if hot at open; zero -> lazy
#   cool_drift_gain  high -> tightens cooling as room warms; zero -> lets it drift
#   solar_ref        low  -> more sun-sensitive; high -> gentler response

EXPERT_PROFILES = {

    "smooth": {
        "occ_heat_sp": 21.0,   "occ_cool_sp": 24.0,
        "pre_heat_sp": 20.0,   "pre_cool_sp": 25.0,
        "sb_heat_sp":  16.0,   "sb_cool_sp":  28.0,
        "target_lux": 500.0,
        "lighting_deadband_lux": 50.0,
        "lighting_action_deadband_w": 50.0,
        "solar_cool_darken":  (200.0, 350.0, 500.0),
        "solar_cool_lighten":  (80.0, 230.0, 380.0),
        "solar_heat_darken":  (400.0, 600.0, 800.0),
        "solar_heat_lighten": (270.0, 470.0, 670.0),
        "solar_neutral_darken":  (300.0, 500.0, 700.0),
        "solar_neutral_lighten": (180.0, 380.0, 580.0),
        "wpi_glare_hard": 1000.0,
        "glare_state": "sr2_ec03",
        "action_hold_steps": 6,
        "glazing_update_steps": 6,
        "lighting_update_steps": 1,
        "thermal_update_steps": 1,
        "one_tier_at_a_time": True,
        # thermal trims — gentle gains + high solar_ref so setpoints move slowly
        "opt_start_window_hrs":  2.0,
        "opt_start_gain":        0.35,
        "solar_ref":             700.0,
        "solar_heat_offset_max": 0.8,
        "solar_cool_offset_max": 0.6,
        "cool_opt_gain":         0.35,
        "cool_drift_gain":       0.4,
    },
}


# ============================================================
# APPLY SELECTED PROFILE
# ============================================================

if EXPERT_MODE not in EXPERT_PROFILES:
    raise ValueError(f"Unknown EXPERT_MODE: {EXPERT_MODE!r}. "
                     f"Choose from {list(EXPERT_PROFILES.keys())}")

PROFILE = EXPERT_PROFILES[EXPERT_MODE]
EXPERT_ID = EXPERT_ID_MAP[EXPERT_MODE]
print(f"[v12 controller] EXPERT_MODE = '{EXPERT_MODE}'  (id={EXPERT_ID})")


# ============================================================
# CONFIGURATION
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1',
    'Perimeter_mid_ZN_2',
    'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4',
    'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
GLAZING_CLEAR_TO_DARK   = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
TIER_OF = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}

LIGHTING_POWER_RANGE = (0.0, 2500.0)
HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
DEADBAND = 1.0

# ---- deadband (comfort-band) override thermostat ----
# Zone temp INSIDE [COMFORT_BAND_LOW, COMFORT_BAND_HIGH] -> do nothing (band
# opened to the full actuator range so neither coil fires); OUTSIDE that band
# -> drive both setpoints hard to COMFORT_OVERRIDE_SP.
COMFORT_BAND_LOW    = 21.0
COMFORT_BAND_HIGH   = 24.0
COMFORT_OVERRIDE_SP = 22.0

LIGHTING_TO_LUX = 0.4   # shared luminous efficacy proxy (lux per W) — SAME for all zones

# Installed lighting Design Level (W) per zone — MUST match epmodel.add_lighting(...).
# These are the physical per-zone maxima; the perimeter expert clamps each zone to
# its OWN installed level (was a single global PERIM_MAX_POWER = 1500, now per-zone).
LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

TARGET_LUX                  = PROFILE["target_lux"]
LIGHTING_DEADBAND_LUX       = PROFILE["lighting_deadband_lux"]
LIGHTING_ACTION_DEADBAND_W  = PROFILE["lighting_action_deadband_w"]
CORE_OCC_POWER              = TARGET_LUX / LIGHTING_TO_LUX

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1   # used ONLY for the logged is_preconditioning feature

OCCUPIED_HEAT_SP     = PROFILE["occ_heat_sp"]
OCCUPIED_COOL_SP     = PROFILE["occ_cool_sp"]
PRECONDITION_HEAT_SP = PROFILE["pre_heat_sp"]
PRECONDITION_COOL_SP = PROFILE["pre_cool_sp"]
SETBACK_HEAT_SP      = PROFILE["sb_heat_sp"]
SETBACK_COOL_SP      = PROFILE["sb_cool_sp"]

SOLAR_COOL_DARKEN     = PROFILE["solar_cool_darken"]
SOLAR_COOL_LIGHTEN    = PROFILE["solar_cool_lighten"]
SOLAR_HEAT_DARKEN     = PROFILE["solar_heat_darken"]
SOLAR_HEAT_LIGHTEN    = PROFILE["solar_heat_lighten"]
SOLAR_NEUTRAL_DARKEN  = PROFILE["solar_neutral_darken"]
SOLAR_NEUTRAL_LIGHTEN = PROFILE["solar_neutral_lighten"]

WPI_GLARE_HARD     = PROFILE["wpi_glare_hard"]
GLARE_STATE        = PROFILE["glare_state"]
ACTION_HOLD_STEPS  = PROFILE["action_hold_steps"]
GLAZING_UPDATE_STEPS = PROFILE.get("glazing_update_steps", PROFILE["action_hold_steps"])
LIGHTING_UPDATE_STEPS = PROFILE.get("lighting_update_steps", 1)
THERMAL_UPDATE_STEPS = PROFILE.get("thermal_update_steps", 1)
ONE_TIER_AT_A_TIME = PROFILE["one_tier_at_a_time"]

# Thermal-trim knobs (fall back to THERMAL_DEFAULTS)
OPT_START_WINDOW_HRS  = PROFILE.get("opt_start_window_hrs",  THERMAL_DEFAULTS["opt_start_window_hrs"])
OPT_START_GAIN        = PROFILE.get("opt_start_gain",        THERMAL_DEFAULTS["opt_start_gain"])
SOLAR_REF             = PROFILE.get("solar_ref",             THERMAL_DEFAULTS["solar_ref"])
SOLAR_HEAT_OFFSET_MAX = PROFILE.get("solar_heat_offset_max", THERMAL_DEFAULTS["solar_heat_offset_max"])
SOLAR_COOL_OFFSET_MAX = PROFILE.get("solar_cool_offset_max", THERMAL_DEFAULTS["solar_cool_offset_max"])
COOL_OPT_GAIN         = PROFILE.get("cool_opt_gain",         THERMAL_DEFAULTS["cool_opt_gain"])
COOL_DRIFT_GAIN       = PROFILE.get("cool_drift_gain",       THERMAL_DEFAULTS["cool_drift_gain"])

# Coil-active epsilon: ignore residual meter noise smaller than this (J/step)
COIL_ACTIVE_EPS = 10.0


# ============================================================
# SDAR ACTION VECTOR LAYOUT
# ============================================================
ACTION_DIM   = 19
GLAZE_SLICE  = slice(0, 4)
LIGHT_SLICE  = slice(4, 9)
HEAT_SLICE   = slice(9, 14)
COOL_SLICE   = slice(14, 19)

GLAZE_RANGE  = (0.0, 3.0)              # tier index domain
# Lighting is normalized PER ZONE by its own installed max (zones no longer share
# one max power). See LIGHT_RANGE_PER_ZONE below — that is what encoding uses.
# LIGHT_RANGE is kept only as a nominal documentation value.
LIGHT_RANGE  = (0.0, 2500.0)           # W (nominal; per-zone ranges used for encoding)
HEAT_RANGE   = (16.0, 24.0)            # °C
COOL_RANGE   = (22.0, 28.0)            # °C

WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())
ZONE_ORDER   = list(ZONES_MID)

# Per-zone lighting normalization: each lighting dim maps [0, installed_W] -> [-1, 1]
# using THAT zone's own installed power (so every lighting dim uses its full range).
LIGHT_RANGE_PER_ZONE = {zone: (0.0, LIGHTING_INSTALLED_W[zone]) for zone in ZONE_ORDER}

ACTION_MASK_VALUE = -2.0   # ξ in SDAR Eq. (2), Nie et al. 2025
MASK_EPS = 1e-3            # normalized-space equality tolerance

ACTION_LAYOUT = {
    "glazing":  (GLAZE_SLICE, WINDOW_ORDER),
    "lighting": (LIGHT_SLICE, ZONE_ORDER),
    "heating":  (HEAT_SLICE,  ZONE_ORDER),
    "cooling":  (COOL_SLICE,  ZONE_ORDER),
    "ranges":   {"glaze": GLAZE_RANGE, "light": LIGHT_RANGE_PER_ZONE,
                 "heat":  HEAT_RANGE,  "cool":  COOL_RANGE},
    "mask_value": ACTION_MASK_VALUE,
    "dim": ACTION_DIM,
}


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
    'expert_id': [],
}

# SDAR per-step logs
action_vector_data = {
    "action":         [],   # a       (19-d, normalized to [-1,1])
    "prev_action":    [],   # a⁻      (19-d, normalized)
    "selection_mask": [],   # b ∈ {0,1}^19   (1 = act/changed, 0 = repeat)
    "action_mix":     [],   # a_mix = (1-b)⊙a⁻ + b⊙ξ   (SDAR Eq. 2)
}
output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}


# ============================================================
# POLICY MEMORY
# ============================================================

policy_memory = {
    "expert_mode": EXPERT_MODE,
    "expert_id": EXPERT_ID,
    "step": 0,
    "glazing": {zone: "sr2_ec01" for zone in WINDOW_ZONES_MID},
    "lighting": {zone: 0.0 for zone in ZONES_MID},
    "heating": {zone: SETBACK_HEAT_SP for zone in ZONES_MID},
    "cooling": {zone: SETBACK_COOL_SP for zone in ZONES_MID},
    "cooling_active": False,
    "heating_active": False,
    "prev_action_vec": None,   # SDAR a⁻ (set after first callback)
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def _norm(x, lo, hi):
    """Affine map from [lo, hi] -> [-1, 1]."""
    return 2.0 * (x - lo) / (hi - lo) - 1.0


def encode_action_vec(glaze, light, heat, cool):
    """Pack expert per-zone actions into a 19-dim normalized vector in [-1, 1]."""
    v = np.zeros(ACTION_DIM, dtype=np.float32)
    for i, zone in enumerate(WINDOW_ORDER):
        v[i] = _norm(float(TIER_OF[glaze[zone]]), *GLAZE_RANGE)
    for i, zone in enumerate(ZONE_ORDER):
        v[4 + i]  = _norm(light[zone], *LIGHT_RANGE_PER_ZONE[zone])
        v[9 + i]  = _norm(heat[zone],  *HEAT_RANGE)
        v[14 + i] = _norm(cool[zone],  *COOL_RANGE)
    return v


# ============================================================
# EXPERT RULES
# ============================================================

def choose_glazing_expert(occupied, cooling_active, heating_active,
                          ext_irr, wpi, current_glazing):
    if not occupied:
        return 'sr2_ec01'

    # Glare override — can only darken (never lighten during glare)
    if wpi >= WPI_GLARE_HARD:
        current_tier = TIER_OF.get(current_glazing, 0)
        glare_tier = TIER_OF[GLARE_STATE]
        return AVAILABLE_GLAZING_STATES[max(current_tier, glare_tier)]

    if cooling_active:
        darken, lighten = SOLAR_COOL_DARKEN, SOLAR_COOL_LIGHTEN
    elif heating_active:
        darken, lighten = SOLAR_HEAT_DARKEN, SOLAR_HEAT_LIGHTEN
    else:
        darken, lighten = SOLAR_NEUTRAL_DARKEN, SOLAR_NEUTRAL_LIGHTEN

    current_tier = TIER_OF.get(current_glazing, 0)

    # Darken?
    target = current_tier
    while target < 3 and ext_irr > darken[target]:
        target += 1
    if target > current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier + 1]
        return AVAILABLE_GLAZING_STATES[target]

    # Lighten?
    target = current_tier
    while target > 0 and ext_irr < lighten[target - 1]:
        target -= 1
    if target < current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier - 1]
        return AVAILABLE_GLAZING_STATES[target]

    return current_glazing


def choose_perimeter_lighting_expert(occupied, current_wpi, zone_max_w,
                                     prev_power=0.0,
                                     action_deadband_w=0.0):
    if not occupied:
        return 0.0
    shortfall_lux = TARGET_LUX - current_wpi
    if shortfall_lux < LIGHTING_DEADBAND_LUX:
        return 0.0   # lux deadband always wins — snap to 0
    desired_power = clamp(shortfall_lux / LIGHTING_TO_LUX,
                          0.0, zone_max_w)   # per-zone installed cap (was PERIM_MAX_POWER)
    if action_deadband_w > 0.0 \
       and abs(desired_power - prev_power) < action_deadband_w:
        return prev_power
    return desired_power


def choose_core_lighting_expert(occupied):
    if not occupied:
        return 0.0
    return CORE_OCC_POWER


def choose_hvac_zone(dt, zone_temp, zone_ext_irr,
                     heating_active=False, cooling_active=False,
                     has_window=True):
    """
    DEADBAND (comfort-band) override thermostat.

      - Zone temp INSIDE [COMFORT_BAND_LOW, COMFORT_BAND_HIGH] -> DO NOTHING:
        open the band to the full actuator range (heating floor / cooling
        ceiling) so neither coil fires.
      - Zone temp OUTSIDE that band -> drive BOTH setpoints to
        COMFORT_OVERRIDE_SP (heating SP = cooling SP = 22), pulling the zone
        back toward 22.

    Everything else in the smooth expert (glazing, lighting, multi-rate
    dispatch, SDAR logging) is unchanged; only the thermal setpoint rule
    differs. The signature is kept identical so call sites need no changes; the
    schedule / solar / coil-state arguments are unused here.
    """
    if COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH:
        # inside comfort band -> no HVAC action (wide band)
        h = HEATING_SETPOINT_RANGE[0]   # 16.0
        c = COOLING_SETPOINT_RANGE[1]   # 28.0
    else:
        # outside comfort band -> hard override to 22 / 22
        h = COMFORT_OVERRIDE_SP
        c = COMFORT_OVERRIDE_SP
    return h, c


# ============================================================
# DECISION DISPATCH — per-subsystem multi-rate updates
# (glazing / lighting / thermal each gated by its own update period;
#  on a non-update step the subsystem repeats its previously applied value)
# THERMAL IS PER-ZONE (state-dependent) and gated on coil state.
# ============================================================

def choose_expert_actions(current_obs, dt):
    policy_memory["step"] += 1
    update_glazing  = (policy_memory["step"] % GLAZING_UPDATE_STEPS == 0)
    update_lighting = (policy_memory["step"] % LIGHTING_UPDATE_STEPS == 0)
    update_thermal  = (policy_memory["step"] % THERMAL_UPDATE_STEPS == 0)

    occupied = is_occupied(dt)
    cooling_active = policy_memory["cooling_active"]
    heating_active = policy_memory["heating_active"]

    selected_g, selected_l, selected_h, selected_c = {}, {}, {}, {}

    for zone, _window in WINDOW_ZONES_MID.items():
        wpi = current_obs["wpi"][zone]
        ext_irr = current_obs["ext_irr"][zone]
        current_g = policy_memory["glazing"][zone]

        # Glazing — only on glazing-update cycle
        if update_glazing:
            g = choose_glazing_expert(
                occupied=occupied,
                cooling_active=cooling_active,
                heating_active=heating_active,
                ext_irr=ext_irr,
                wpi=wpi,
                current_glazing=current_g,
            )
        else:
            g = current_g
        selected_g[zone] = g

        # Lighting — only on lighting-update cycle, with action-level deadband
        if update_lighting:
            target_p = choose_perimeter_lighting_expert(
                occupied=occupied,
                current_wpi=wpi,
                zone_max_w=LIGHTING_INSTALLED_W[zone],   # per-zone installed cap
                prev_power=policy_memory["lighting"][zone],
                action_deadband_w=LIGHTING_ACTION_DEADBAND_W,
            )
            selected_l[zone] = clamp(target_p, *LIGHTING_POWER_RANGE)
        else:
            selected_l[zone] = policy_memory["lighting"][zone]

        # Thermal — SYMMETRIC closed-loop, per zone, gated on coil state
        if update_thermal:
            h_z, c_z = choose_hvac_zone(
                dt,
                zone_temp=current_obs["temp"][zone],
                zone_ext_irr=ext_irr,
                heating_active=heating_active,
                cooling_active=cooling_active,
                has_window=True,
            )
            selected_h[zone] = h_z
            selected_c[zone] = c_z
        else:
            selected_h[zone] = policy_memory["heating"][zone]
            selected_c[zone] = policy_memory["cooling"][zone]

    core_zone = "Core_mid"

    # Core lighting — only on lighting-update cycle
    if update_lighting:
        selected_l[core_zone] = clamp(choose_core_lighting_expert(occupied),
                                      *LIGHTING_POWER_RANGE)
    else:
        selected_l[core_zone] = policy_memory["lighting"][core_zone]

    # Core thermal — closed-loop but no window (solar term off; optimum start
    # and drift-tightening still act on core temperature)
    if update_thermal:
        h_c, c_c = choose_hvac_zone(
            dt,
            zone_temp=current_obs["temp"][core_zone],
            zone_ext_irr=0.0,
            heating_active=heating_active,
            cooling_active=cooling_active,
            has_window=False,
        )
        selected_h[core_zone] = h_c
        selected_c[core_zone] = c_c
    else:
        selected_h[core_zone] = policy_memory["heating"][core_zone]
        selected_c[core_zone] = policy_memory["cooling"][core_zone]

    for zone in WINDOW_ZONES_MID:
        policy_memory["glazing"][zone] = selected_g[zone]
    for zone in ZONES_MID:
        policy_memory["lighting"][zone] = selected_l[zone]
        policy_memory["heating"][zone] = selected_h[zone]
        policy_memory["cooling"][zone] = selected_c[zone]

    return selected_g, selected_l, selected_h, selected_c


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    dt = eps.get_datetime()

    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = policy_memory["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec  = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec    = get_meter_value_cached(state, "Electricity:HVAC")   # new HVAC end-use meter (J/step)

    # Coil-active flags with epsilon
    policy_memory["cooling_active"] = (cooling_elec > COIL_ACTIVE_EPS)
    policy_memory["heating_active"] = (heating_elec > COIL_ACTIVE_EPS)

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)
    temporal_data['expert_id'].append(EXPERT_ID)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
            'Core_mid': 0.0,  # core has no window
        },
        # energy obs — only the correctly-scoped OUTPUT VARIABLES (no meters):
        #   controlled-zone lighting rate (sum of the 5 zones' Lights Electricity Rate)
        #   + facility HVAC electricity demand rate. Both in W.
        "energy": {
            'lighting_rate': (light_mid_1 + light_mid_2 + light_mid_3
                              + light_mid_4 + light_core_mid),   # W, sum of 5 controlled zones
            'hvac_demand_rate': hvac_electricity_demand_rate,    # W, facility HVAC output var
        },
        "wpi": current_wpi,
        "temporal": tf,
    }

    sel_g, sel_l, sel_h, sel_c = choose_expert_actions(current_obs, dt)

    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=sel_g[zone])
        eps.actuate_lighting_power(light=zone, value=sel_l[zone])
        eps.actuate_heating_setpoint(zone=zone, value=sel_h[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=sel_c[zone])

        glazing_state_data[zone].append(sel_g[zone])
        lighting_power_data[zone].append(sel_l[zone])
        heating_setpoint_data[zone].append(sel_h[zone])
        cooling_setpoint_data[zone].append(sel_c[zone])

    eps.actuate_lighting_power(light="Core_mid", value=sel_l['Core_mid'])
    eps.actuate_heating_setpoint(zone="Core_mid", value=sel_h['Core_mid'])
    eps.actuate_cooling_setpoint(zone="Core_mid", value=sel_c['Core_mid'])

    lighting_power_data['Core_mid'].append(sel_l['Core_mid'])
    heating_setpoint_data['Core_mid'].append(sel_h['Core_mid'])
    cooling_setpoint_data['Core_mid'].append(sel_c['Core_mid'])

    # ---- SDAR action-mask logging ----------------------------------
    a_vec = encode_action_vec(sel_g, sel_l, sel_h, sel_c)
    prev_a_vec = policy_memory["prev_action_vec"]

    if prev_a_vec is None:
        # First step: force all-act (SDAR Algo 1, line 3-4)
        b_mask = np.ones(ACTION_DIM, dtype=np.float32)
        prev_for_log = a_vec.copy()
    else:
        b_mask = (np.abs(a_vec - prev_a_vec) > MASK_EPS).astype(np.float32)
        prev_for_log = prev_a_vec.copy()

    a_mix = ((1.0 - b_mask) * prev_for_log
             + b_mask * ACTION_MASK_VALUE).astype(np.float32)

    action_vector_data["action"].append(a_vec.copy())
    action_vector_data["prev_action"].append(prev_for_log)
    action_vector_data["selection_mask"].append(b_mask)
    action_vector_data["action_mix"].append(a_mix)

    policy_memory["prev_action_vec"] = a_vec

In [ ]:
"""
EXPERT DATA COLLECTION POLICY v12 — smooth expert + CALLBACK-SIDE EXPLORATION
============================================================================

Deadband-override smooth expert with three exploration systems added INSIDE the
controller (so noisy actions reach EnergyPlus and the resulting next-state /
reward correspond to the executed action):

  expert action -> exploration -> actuation -> logging

  1) Safe local epsilon-greedy GLAZING (doc: "Safe epsilon-greedy glazing"):
     - applied only when glazing is scheduled to update (hourly at 6 steps/h);
     - epsilon on the JOINT decision, randomizing one (usually) or two windows;
     - respects one-tier-at-a-time and the glare rule (never lightens in glare).

  2) Dimming-space LIGHTING noise (doc: "Lighting exploration in dimming space"):
     - per-zone Gaussian noise in dimming fraction (sigma * nominal power);
     - only while occupied and on a lighting-update step;
     - bounded so artificial light keeps total < LIGHTING_SAFE_MAX_LUX.

  3) Per-zone stateful mixed THERMAL policy (doc: "Mixed thermal controller"):
     - outside 21-24: recovery (22/22) or occasionally noisy recovery;
     - inside 21-24: idle (16/28) / random-inactive band / active target;
     - each zone holds its chosen behavior for THERMAL_POLICY_HOLD_STEPS;
     - leaving the band immediately discards the held exploratory behavior.

policy_memory is updated with the EXECUTED (noisy) actions. The SDAR mask/prev/
mix use the executed action (as before). A separate `exploration_data` log keeps
the expert (clean) action, executed action, and exploration masks.

Action vector layout (all normalized to [-1, 1]):
  [0:4] glazing tier | [4:9] lighting | [9:14] heating SP | [14:19] cooling SP
"""

import numpy as np


# ============================================================
# PICK YOUR EXPERT
# ============================================================

EXPERT_MODE = "smooth"

EXPERT_ID_MAP = {
    "balanced": 0, "energy": 1, "comfort": 2, "visual": 3, "smooth": 4,
    "fast_responsive": 5, "slow_smooth": 6, "staggered_multirate": 7,
    "bad_energy_lazy": 8, "adaptive_balanced_good": 9,
}

# ============================================================
# THERMAL-TRIM DEFAULTS
# ============================================================
THERMAL_DEFAULTS = {
    "opt_start_window_hrs":  2.0,
    "opt_start_gain":        0.5,
    "solar_ref":             600.0,
    "solar_heat_offset_max": 1.0,
    "solar_cool_offset_max": 0.8,
    "cool_opt_gain":         0.5,
    "cool_drift_gain":       0.5,
}


# ============================================================
# PROFILES
# ============================================================
EXPERT_PROFILES = {
    "smooth": {
        "occ_heat_sp": 21.0,   "occ_cool_sp": 24.0,
        "pre_heat_sp": 20.0,   "pre_cool_sp": 25.0,
        "sb_heat_sp":  16.0,   "sb_cool_sp":  28.0,
        "target_lux": 500.0,
        "lighting_deadband_lux": 50.0,
        "lighting_action_deadband_w": 50.0,
        "solar_cool_darken":  (200.0, 350.0, 500.0),
        "solar_cool_lighten":  (80.0, 230.0, 380.0),
        "solar_heat_darken":  (400.0, 600.0, 800.0),
        "solar_heat_lighten": (270.0, 470.0, 670.0),
        "solar_neutral_darken":  (300.0, 500.0, 700.0),
        "solar_neutral_lighten": (180.0, 380.0, 580.0),
        "wpi_glare_hard": 1000.0,
        "glare_state": "sr2_ec03",
        "action_hold_steps": 6,
        "glazing_update_steps": 6,
        "lighting_update_steps": 1,
        "thermal_update_steps": 1,
        "one_tier_at_a_time": True,
        "opt_start_window_hrs":  2.0,
        "opt_start_gain":        0.35,
        "solar_ref":             700.0,
        "solar_heat_offset_max": 0.8,
        "solar_cool_offset_max": 0.6,
        "cool_opt_gain":         0.35,
        "cool_drift_gain":       0.4,
    },
}


# ============================================================
# APPLY SELECTED PROFILE
# ============================================================

if EXPERT_MODE not in EXPERT_PROFILES:
    raise ValueError(f"Unknown EXPERT_MODE: {EXPERT_MODE!r}. "
                     f"Choose from {list(EXPERT_PROFILES.keys())}")

PROFILE = EXPERT_PROFILES[EXPERT_MODE]
EXPERT_ID = EXPERT_ID_MAP[EXPERT_MODE]
print(f"[v12 controller] EXPERT_MODE = '{EXPERT_MODE}'  (id={EXPERT_ID})")


# ============================================================
# CONFIGURATION
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
GLAZING_CLEAR_TO_DARK   = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
TIER_OF = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}

LIGHTING_POWER_RANGE = (0.0, 2500.0)
HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
DEADBAND = 1.0

# ---- deadband (comfort-band) override thermostat ----
COMFORT_BAND_LOW    = 21.0
COMFORT_BAND_HIGH   = 24.0
COMFORT_OVERRIDE_SP = 22.0

LIGHTING_TO_LUX = 0.4

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

TARGET_LUX                  = PROFILE["target_lux"]
LIGHTING_DEADBAND_LUX       = PROFILE["lighting_deadband_lux"]
LIGHTING_ACTION_DEADBAND_W  = PROFILE["lighting_action_deadband_w"]
CORE_OCC_POWER              = TARGET_LUX / LIGHTING_TO_LUX

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

OCCUPIED_HEAT_SP     = PROFILE["occ_heat_sp"]
OCCUPIED_COOL_SP     = PROFILE["occ_cool_sp"]
PRECONDITION_HEAT_SP = PROFILE["pre_heat_sp"]
PRECONDITION_COOL_SP = PROFILE["pre_cool_sp"]
SETBACK_HEAT_SP      = PROFILE["sb_heat_sp"]
SETBACK_COOL_SP      = PROFILE["sb_cool_sp"]

SOLAR_COOL_DARKEN     = PROFILE["solar_cool_darken"]
SOLAR_COOL_LIGHTEN    = PROFILE["solar_cool_lighten"]
SOLAR_HEAT_DARKEN     = PROFILE["solar_heat_darken"]
SOLAR_HEAT_LIGHTEN    = PROFILE["solar_heat_lighten"]
SOLAR_NEUTRAL_DARKEN  = PROFILE["solar_neutral_darken"]
SOLAR_NEUTRAL_LIGHTEN = PROFILE["solar_neutral_lighten"]

WPI_GLARE_HARD     = PROFILE["wpi_glare_hard"]
GLARE_STATE        = PROFILE["glare_state"]
ACTION_HOLD_STEPS  = PROFILE["action_hold_steps"]
GLAZING_UPDATE_STEPS = PROFILE.get("glazing_update_steps", PROFILE["action_hold_steps"])
LIGHTING_UPDATE_STEPS = PROFILE.get("lighting_update_steps", 1)
THERMAL_UPDATE_STEPS = PROFILE.get("thermal_update_steps", 1)
ONE_TIER_AT_A_TIME = PROFILE["one_tier_at_a_time"]

OPT_START_WINDOW_HRS  = PROFILE.get("opt_start_window_hrs",  THERMAL_DEFAULTS["opt_start_window_hrs"])
OPT_START_GAIN        = PROFILE.get("opt_start_gain",        THERMAL_DEFAULTS["opt_start_gain"])
SOLAR_REF             = PROFILE.get("solar_ref",             THERMAL_DEFAULTS["solar_ref"])
SOLAR_HEAT_OFFSET_MAX = PROFILE.get("solar_heat_offset_max", THERMAL_DEFAULTS["solar_heat_offset_max"])
SOLAR_COOL_OFFSET_MAX = PROFILE.get("solar_cool_offset_max", THERMAL_DEFAULTS["solar_cool_offset_max"])
COOL_OPT_GAIN         = PROFILE.get("cool_opt_gain",         THERMAL_DEFAULTS["cool_opt_gain"])
COOL_DRIFT_GAIN       = PROFILE.get("cool_drift_gain",       THERMAL_DEFAULTS["cool_drift_gain"])

COIL_ACTIVE_EPS = 10.0


# ============================================================
# SDAR ACTION VECTOR LAYOUT
# ============================================================
ACTION_DIM   = 19
GLAZE_SLICE  = slice(0, 4)
LIGHT_SLICE  = slice(4, 9)
HEAT_SLICE   = slice(9, 14)
COOL_SLICE   = slice(14, 19)

GLAZE_RANGE  = (0.0, 3.0)
LIGHT_RANGE  = (0.0, 2500.0)
HEAT_RANGE   = (16.0, 24.0)
COOL_RANGE   = (22.0, 28.0)

WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())
ZONE_ORDER   = list(ZONES_MID)

LIGHT_RANGE_PER_ZONE = {zone: (0.0, LIGHTING_INSTALLED_W[zone]) for zone in ZONE_ORDER}

ACTION_MASK_VALUE = -2.0
MASK_EPS = 1e-3

ACTION_LAYOUT = {
    "glazing":  (GLAZE_SLICE, WINDOW_ORDER),
    "lighting": (LIGHT_SLICE, ZONE_ORDER),
    "heating":  (HEAT_SLICE,  ZONE_ORDER),
    "cooling":  (COOL_SLICE,  ZONE_ORDER),
    "ranges":   {"glaze": GLAZE_RANGE, "light": LIGHT_RANGE_PER_ZONE,
                 "heat":  HEAT_RANGE,  "cool":  COOL_RANGE},
    "mask_value": ACTION_MASK_VALUE,
    "dim": ACTION_DIM,
}


# ============================================================
# EXPLORATION CONFIG
# ============================================================
EXPLORATION_ENABLED = True
EXPLORATION_SEED = 10_000 + EXPERT_ID

# Glazing: epsilon on the JOINT decision, applied only on glazing-update steps.
GLAZING_EPSILON = 0.10

# Lighting: noise as fraction of installed power, occupied + lighting-update only.
LIGHTING_NOISE_PROB = 0.30
LIGHTING_NOISE_STD_FRAC = 0.05
LIGHTING_SAFE_MAX_LUX = 1000.0

# Thermal: per-zone stateful mixed policy.
THERMAL_POLICY_HOLD_STEPS = 6   # one hour at six steps/hour

INSIDE_IDLE_PROB = 0.70
INSIDE_RANDOM_BAND_PROB = 0.20
INSIDE_ACTIVE_TARGET_PROB = 0.10   # inside probs must sum to 1

OUTSIDE_NOISY_TARGET_PROB = 0.20
OUTSIDE_TARGET_NOISE_STD_C = 0.50

THERMAL_TARGET_SP = 22.0
THERMAL_RANDOM_TARGET_LOW = 21.0
THERMAL_RANDOM_TARGET_HIGH = 24.0
THERMAL_INACTIVE_MARGIN_C = 0.25

rng = np.random.default_rng(EXPLORATION_SEED)


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
    'expert_id': [],
}

# SDAR per-step logs (action = EXECUTED / noisy action)
action_vector_data = {
    "action":         [],
    "prev_action":    [],
    "selection_mask": [],
    "action_mix":     [],
}

# Exploration log: expert (clean) vs executed (noisy) + which dims explored.
exploration_data = {
    "expert_action": [],
    "executed_action": [],
    "exploration_mask": [],
    "glazing_explored": [],
    "lighting_explored": [],
    "thermal_mode": [],
    "epsilon": [],
    "seed": EXPLORATION_SEED,
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}


# ============================================================
# POLICY MEMORY
# ============================================================

policy_memory = {
    "expert_mode": EXPERT_MODE,
    "expert_id": EXPERT_ID,
    "step": 0,
    "glazing": {zone: "sr2_ec01" for zone in WINDOW_ZONES_MID},
    "lighting": {zone: 0.0 for zone in ZONES_MID},
    "heating": {zone: SETBACK_HEAT_SP for zone in ZONES_MID},
    "cooling": {zone: SETBACK_COOL_SP for zone in ZONES_MID},
    "cooling_active": False,
    "heating_active": False,
    "prev_action_vec": None,
    # per-zone thermal exploration state (held behavior)
    "thermal_exploration": {
        zone: {
            "heating_sp": HEATING_SETPOINT_RANGE[0],
            "cooling_sp": COOLING_SETPOINT_RANGE[1],
            "mode": "idle",
            "remaining": 0,
        }
        for zone in ZONES_MID
    },
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def _norm(x, lo, hi):
    """Affine map from [lo, hi] -> [-1, 1]."""
    return 2.0 * (x - lo) / (hi - lo) - 1.0


def encode_action_vec(glaze, light, heat, cool):
    """Pack expert per-zone actions into a 19-dim normalized vector in [-1, 1]."""
    v = np.zeros(ACTION_DIM, dtype=np.float32)
    for i, zone in enumerate(WINDOW_ORDER):
        v[i] = _norm(float(TIER_OF[glaze[zone]]), *GLAZE_RANGE)
    for i, zone in enumerate(ZONE_ORDER):
        v[4 + i]  = _norm(light[zone], *LIGHT_RANGE_PER_ZONE[zone])
        v[9 + i]  = _norm(heat[zone],  *HEAT_RANGE)
        v[14 + i] = _norm(cool[zone],  *COOL_RANGE)
    return v


# ============================================================
# EXPERT RULES
# ============================================================

def choose_glazing_expert(occupied, cooling_active, heating_active,
                          ext_irr, wpi, current_glazing):
    if not occupied:
        return 'sr2_ec01'

    if wpi >= WPI_GLARE_HARD:
        current_tier = TIER_OF.get(current_glazing, 0)
        glare_tier = TIER_OF[GLARE_STATE]
        return AVAILABLE_GLAZING_STATES[max(current_tier, glare_tier)]

    if cooling_active:
        darken, lighten = SOLAR_COOL_DARKEN, SOLAR_COOL_LIGHTEN
    elif heating_active:
        darken, lighten = SOLAR_HEAT_DARKEN, SOLAR_HEAT_LIGHTEN
    else:
        darken, lighten = SOLAR_NEUTRAL_DARKEN, SOLAR_NEUTRAL_LIGHTEN

    current_tier = TIER_OF.get(current_glazing, 0)

    target = current_tier
    while target < 3 and ext_irr > darken[target]:
        target += 1
    if target > current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier + 1]
        return AVAILABLE_GLAZING_STATES[target]

    target = current_tier
    while target > 0 and ext_irr < lighten[target - 1]:
        target -= 1
    if target < current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier - 1]
        return AVAILABLE_GLAZING_STATES[target]

    return current_glazing


def choose_perimeter_lighting_expert(occupied, current_wpi, zone_max_w,
                                     prev_power=0.0,
                                     action_deadband_w=0.0):
    if not occupied:
        return 0.0
    shortfall_lux = TARGET_LUX - current_wpi
    if shortfall_lux < LIGHTING_DEADBAND_LUX:
        return 0.0
    desired_power = clamp(shortfall_lux / LIGHTING_TO_LUX, 0.0, zone_max_w)
    if action_deadband_w > 0.0 \
       and abs(desired_power - prev_power) < action_deadband_w:
        return prev_power
    return desired_power


def choose_core_lighting_expert(occupied):
    if not occupied:
        return 0.0
    return CORE_OCC_POWER


def choose_hvac_zone(dt, zone_temp, zone_ext_irr,
                     heating_active=False, cooling_active=False,
                     has_window=True):
    """
    DEADBAND (comfort-band) override thermostat (deterministic expert).
      - INSIDE [21,24] -> 16/28 (no coil fires).
      - OUTSIDE       -> 22/22 (recover toward 22).
    """
    if COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH:
        h = HEATING_SETPOINT_RANGE[0]
        c = COOLING_SETPOINT_RANGE[1]
    else:
        h = COMFORT_OVERRIDE_SP
        c = COMFORT_OVERRIDE_SP
    return h, c


# ============================================================
# EXPLORATION — glazing (safe local epsilon-greedy)
# ============================================================

def apply_glazing_exploration(selected_g, current_obs, update_glazing):
    """
    With prob GLAZING_EPSILON (on glazing-update steps only), modify one or two
    glazing zones. Respects the update period, one-tier-at-a-time transitions,
    and the glare rule (never lightens during glare).
    """
    explored = {zone: False for zone in WINDOW_ORDER}

    if (not EXPLORATION_ENABLED
            or not update_glazing
            or rng.random() >= GLAZING_EPSILON):
        return selected_g, explored

    n_zones = 1 if rng.random() < 0.8 else 2
    zones_to_explore = rng.choice(WINDOW_ORDER, size=n_zones, replace=False)

    result = selected_g.copy()
    for zone in zones_to_explore:
        current_tier = TIER_OF[policy_memory["glazing"][zone]]
        expert_tier = TIER_OF[selected_g[zone]]

        candidate_tiers = [
            tier for tier in (current_tier - 1, current_tier, current_tier + 1)
            if 0 <= tier <= 3
        ]

        if current_obs["wpi"][zone] >= WPI_GLARE_HARD:
            minimum_safe_tier = max(current_tier, TIER_OF[GLARE_STATE])
            candidate_tiers = [t for t in candidate_tiers if t >= minimum_safe_tier]

        alternatives = [t for t in candidate_tiers if t != expert_tier]
        if alternatives:
            chosen_tier = int(rng.choice(alternatives))
            result[zone] = AVAILABLE_GLAZING_STATES[chosen_tier]
            explored[zone] = True

    return result, explored


# ============================================================
# EXPLORATION — lighting (bounded Gaussian in dimming space)
# ============================================================

def apply_lighting_exploration(selected_l, current_obs, occupied, update_lighting):
    """Add bounded Gaussian noise in dimming-fraction space (occupied only)."""
    explored = {zone: False for zone in ZONE_ORDER}

    if (not EXPLORATION_ENABLED or not update_lighting or not occupied):
        return selected_l, explored

    result = selected_l.copy()
    for zone in ZONE_ORDER:
        if rng.random() >= LIGHTING_NOISE_PROB:
            continue

        nominal_w = LIGHTING_INSTALLED_W[zone]
        expert_dimming = np.clip(selected_l[zone] / nominal_w, 0.0, 1.0)
        noisy_dimming = np.clip(
            expert_dimming + rng.normal(0.0, LIGHTING_NOISE_STD_FRAC), 0.0, 1.0)
        noisy_power_w = noisy_dimming * nominal_w

        maximum_power_w = min(nominal_w, LIGHTING_POWER_RANGE[1])
        if zone in WINDOW_ZONES_MID:
            daylight_lux = current_obs["wpi"][zone]
            maximum_power_by_lux = max(
                0.0, (LIGHTING_SAFE_MAX_LUX - daylight_lux) / LIGHTING_TO_LUX)
            maximum_power_w = min(maximum_power_w, maximum_power_by_lux)

        result[zone] = clamp(noisy_power_w, 0.0, maximum_power_w)
        explored[zone] = not np.isclose(result[zone], selected_l[zone])

    return result, explored


# ============================================================
# EXPLORATION — per-zone mixed thermal policy
# ============================================================

def choose_exploratory_hvac_zone(zone, zone_temp):
    """
    Per-zone stateful mixed thermal behavior.
      OUTSIDE 21-24 : recovery (22/22) or occasional noisy recovery.
      INSIDE  21-24 : idle (16/28) / random-inactive band / active target,
                      held for THERMAL_POLICY_HOLD_STEPS.
    Leaving the band immediately discards any held exploratory behavior.
    """
    memory = policy_memory["thermal_exploration"][zone]
    inside_comfort = COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH

    # ---- outside comfort: recovery takes precedence ----
    if not inside_comfort:
        memory["remaining"] = 0
        if EXPLORATION_ENABLED and rng.random() < OUTSIDE_NOISY_TARGET_PROB:
            target = THERMAL_TARGET_SP + rng.normal(0.0, OUTSIDE_TARGET_NOISE_STD_C)
            target = clamp(target, THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
            mode = "noisy_recovery"
        else:
            target = THERMAL_TARGET_SP
            mode = "expert_recovery"
        memory["heating_sp"] = target
        memory["cooling_sp"] = target
        memory["mode"] = mode
        return target, target, mode

    # ---- inside comfort: hold prior exploratory behavior ----
    if memory["remaining"] > 0:
        memory["remaining"] -= 1
        return memory["heating_sp"], memory["cooling_sp"], memory["mode"]

    # exploration disabled -> plain idle expert inside the band
    if not EXPLORATION_ENABLED:
        memory["heating_sp"] = HEATING_SETPOINT_RANGE[0]
        memory["cooling_sp"] = COOLING_SETPOINT_RANGE[1]
        memory["mode"] = "idle"
        memory["remaining"] = 0
        return HEATING_SETPOINT_RANGE[0], COOLING_SETPOINT_RANGE[1], "idle"

    # select a new inside-band behavior
    sample = rng.random()
    if sample < INSIDE_IDLE_PROB:
        heating_sp = HEATING_SETPOINT_RANGE[0]   # 16
        cooling_sp = COOLING_SETPOINT_RANGE[1]   # 28
        mode = "idle"
    elif sample < (INSIDE_IDLE_PROB + INSIDE_RANDOM_BAND_PROB):
        maximum_heating_sp = min(zone_temp - THERMAL_INACTIVE_MARGIN_C,
                                 HEATING_SETPOINT_RANGE[1])
        minimum_cooling_sp = max(zone_temp + THERMAL_INACTIVE_MARGIN_C,
                                 COOLING_SETPOINT_RANGE[0])
        heating_sp = rng.uniform(HEATING_SETPOINT_RANGE[0], maximum_heating_sp)
        cooling_sp = rng.uniform(minimum_cooling_sp, COOLING_SETPOINT_RANGE[1])
        mode = "random_inactive_band"
    else:
        target = rng.uniform(THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
        heating_sp = target
        cooling_sp = target
        mode = "active_exploration"

    memory["heating_sp"] = float(heating_sp)
    memory["cooling_sp"] = float(cooling_sp)
    memory["mode"] = mode
    memory["remaining"] = THERMAL_POLICY_HOLD_STEPS - 1
    return float(heating_sp), float(cooling_sp), mode


# ============================================================
# DECISION DISPATCH — expert action -> exploration -> memory update
# ============================================================

def choose_expert_actions(current_obs, dt):
    policy_memory["step"] += 1
    update_glazing  = (policy_memory["step"] % GLAZING_UPDATE_STEPS == 0)
    update_lighting = (policy_memory["step"] % LIGHTING_UPDATE_STEPS == 0)
    update_thermal  = (policy_memory["step"] % THERMAL_UPDATE_STEPS == 0)

    occupied = is_occupied(dt)
    cooling_active = policy_memory["cooling_active"]
    heating_active = policy_memory["heating_active"]

    selected_g, selected_l, selected_h, selected_c = {}, {}, {}, {}
    thermal_mode = {}

    for zone, _window in WINDOW_ZONES_MID.items():
        wpi = current_obs["wpi"][zone]
        ext_irr = current_obs["ext_irr"][zone]
        current_g = policy_memory["glazing"][zone]

        if update_glazing:
            g = choose_glazing_expert(
                occupied=occupied,
                cooling_active=cooling_active,
                heating_active=heating_active,
                ext_irr=ext_irr,
                wpi=wpi,
                current_glazing=current_g,
            )
        else:
            g = current_g
        selected_g[zone] = g

        if update_lighting:
            target_p = choose_perimeter_lighting_expert(
                occupied=occupied,
                current_wpi=wpi,
                zone_max_w=LIGHTING_INSTALLED_W[zone],
                prev_power=policy_memory["lighting"][zone],
                action_deadband_w=LIGHTING_ACTION_DEADBAND_W,
            )
            selected_l[zone] = clamp(target_p, *LIGHTING_POWER_RANGE)
        else:
            selected_l[zone] = policy_memory["lighting"][zone]

        # Thermal — per-zone mixed exploratory policy (expert idle/recovery
        # is the majority branch inside/outside the band).
        if update_thermal:
            h_z, c_z, m_z = choose_exploratory_hvac_zone(
                zone=zone, zone_temp=current_obs["temp"][zone])
            selected_h[zone] = h_z
            selected_c[zone] = c_z
            thermal_mode[zone] = m_z
        else:
            selected_h[zone] = policy_memory["heating"][zone]
            selected_c[zone] = policy_memory["cooling"][zone]
            thermal_mode[zone] = policy_memory["thermal_exploration"][zone]["mode"]

    core_zone = "Core_mid"

    if update_lighting:
        selected_l[core_zone] = clamp(choose_core_lighting_expert(occupied),
                                      *LIGHTING_POWER_RANGE)
    else:
        selected_l[core_zone] = policy_memory["lighting"][core_zone]

    if update_thermal:
        h_c, c_c, m_c = choose_exploratory_hvac_zone(
            zone=core_zone, zone_temp=current_obs["temp"][core_zone])
        selected_h[core_zone] = h_c
        selected_c[core_zone] = c_c
        thermal_mode[core_zone] = m_c
    else:
        selected_h[core_zone] = policy_memory["heating"][core_zone]
        selected_c[core_zone] = policy_memory["cooling"][core_zone]
        thermal_mode[core_zone] = policy_memory["thermal_exploration"][core_zone]["mode"]

    # ---- preserve deterministic expert decision (glazing/lighting) ----
    expert_g = selected_g.copy()
    expert_l = selected_l.copy()
    expert_h = selected_h.copy()
    expert_c = selected_c.copy()

    # ---- apply glazing + lighting exploration ----
    selected_g, glazing_explored = apply_glazing_exploration(
        selected_g=selected_g, current_obs=current_obs, update_glazing=update_glazing)
    selected_l, lighting_explored = apply_lighting_exploration(
        selected_l=selected_l, current_obs=current_obs,
        occupied=occupied, update_lighting=update_lighting)

    # ---- update memory using EXECUTED (noisy) actions ----
    for zone in WINDOW_ZONES_MID:
        policy_memory["glazing"][zone] = selected_g[zone]
    for zone in ZONES_MID:
        policy_memory["lighting"][zone] = selected_l[zone]
        policy_memory["heating"][zone] = selected_h[zone]
        policy_memory["cooling"][zone] = selected_c[zone]

    # ---- exploration logging (expert vs executed) ----
    expert_vec = encode_action_vec(expert_g, expert_l, expert_h, expert_c)
    executed_vec = encode_action_vec(selected_g, selected_l, selected_h, selected_c)
    exploration_mask = (np.abs(executed_vec - expert_vec) > MASK_EPS).astype(np.float32)

    exploration_data["expert_action"].append(expert_vec)
    exploration_data["executed_action"].append(executed_vec)
    exploration_data["exploration_mask"].append(exploration_mask)
    exploration_data["glazing_explored"].append(glazing_explored)
    exploration_data["lighting_explored"].append(lighting_explored)
    exploration_data["thermal_mode"].append(dict(thermal_mode))
    exploration_data["epsilon"].append(GLAZING_EPSILON)

    return selected_g, selected_l, selected_h, selected_c


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    dt = eps.get_datetime()

    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = policy_memory["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec  = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec    = get_meter_value_cached(state, "Electricity:HVAC")

    policy_memory["cooling_active"] = (cooling_elec > COIL_ACTIVE_EPS)
    policy_memory["heating_active"] = (heating_elec > COIL_ACTIVE_EPS)

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)
    temporal_data['expert_id'].append(EXPERT_ID)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
            'Core_mid': 0.0,
        },
        "energy": {
            'lighting_rate': (light_mid_1 + light_mid_2 + light_mid_3
                              + light_mid_4 + light_core_mid),
            'hvac_demand_rate': hvac_electricity_demand_rate,
        },
        "wpi": current_wpi,
        "temporal": tf,
    }

    sel_g, sel_l, sel_h, sel_c = choose_expert_actions(current_obs, dt)

    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=sel_g[zone])
        eps.actuate_lighting_power(light=zone, value=sel_l[zone])
        eps.actuate_heating_setpoint(zone=zone, value=sel_h[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=sel_c[zone])

        glazing_state_data[zone].append(sel_g[zone])
        lighting_power_data[zone].append(sel_l[zone])
        heating_setpoint_data[zone].append(sel_h[zone])
        cooling_setpoint_data[zone].append(sel_c[zone])

    eps.actuate_lighting_power(light="CNOISEore_mid", value=sel_l['Core_mid'])
    eps.actuate_heating_setpoint(zone="Core_mid", value=sel_h['Core_mid'])
    eps.actuate_cooling_setpoint(zone="Core_mid", value=sel_c['Core_mid'])

    lighting_power_data['Core_mid'].append(sel_l['Core_mid'])
    heating_setpoint_data['Core_mid'].append(sel_h['Core_mid'])
    cooling_setpoint_data['Core_mid'].append(sel_c['Core_mid'])

    # ---- SDAR action-mask logging (uses EXECUTED action) ----
    a_vec = encode_action_vec(sel_g, sel_l, sel_h, sel_c)
    prev_a_vec = policy_memory["prev_action_vec"]

    if prev_a_vec is None:
        b_mask = np.ones(ACTION_DIM, dtype=np.float32)
        prev_for_log = a_vec.copy()
    else:
        b_mask = (np.abs(a_vec - prev_a_vec) > MASK_EPS).astype(np.float32)
        prev_for_log = prev_a_vec.copy()

    a_mix = ((1.0 - b_mask) * prev_for_log
             + b_mask * ACTION_MASK_VALUE).astype(np.float32)

    action_vector_data["action"].append(a_vec.copy())
    action_vector_data["prev_action"].append(prev_for_log)
    action_vector_data["selection_mask"].append(b_mask)
    action_vector_data["action_mix"].append(a_mix)

    policy_memory["prev_action_vec"] = a_vec

In [ ]:
"""
EXPERT DATA COLLECTION POLICY v12 — smooth expert + CALLBACK-SIDE EXPLORATION
============================================================================

Deadband-override smooth expert with three exploration systems added INSIDE the
controller (so noisy actions reach EnergyPlus and the resulting next-state /
reward correspond to the executed action):

  expert action -> exploration -> actuation -> logging

  1) Safe local epsilon-greedy GLAZING (doc: "Safe epsilon-greedy glazing"):
     - applied only when glazing is scheduled to update (hourly at 6 steps/h);
     - epsilon on the JOINT decision, randomizing one (usually) or two windows;
     - respects one-tier-at-a-time and the glare rule (never lightens in glare).

  2) Dimming-space LIGHTING noise (doc: "Lighting exploration in dimming space"):
     - per-zone Gaussian noise in dimming fraction (sigma * nominal power);
     - only while occupied and on a lighting-update step;
     - bounded so artificial light keeps total < LIGHTING_SAFE_MAX_LUX.

  3) Per-zone stateful mixed THERMAL policy (doc: "Mixed thermal controller"):
     - outside 21-24: recovery (22/22) or occasionally noisy recovery;
     - inside 21-24: idle (16/28) / random-inactive band / active target;
     - each zone holds its chosen behavior for THERMAL_POLICY_HOLD_STEPS;
     - leaving the band immediately discards the held exploratory behavior.

policy_memory is updated with the EXECUTED (noisy) actions. The SDAR mask/prev/
mix use the executed action (as before). A separate `exploration_data` log keeps
the expert (clean) action, executed action, and exploration masks.

Action vector layout (all normalized to [-1, 1]):
  [0:4] glazing tier | [4:9] lighting | [9:14] heating SP | [14:19] cooling SP
"""

import numpy as np


# ============================================================
# PICK YOUR EXPERT
# ============================================================

EXPERT_MODE = "smooth"

EXPERT_ID_MAP = {
    "balanced": 0, "energy": 1, "comfort": 2, "visual": 3, "smooth": 4,
    "fast_responsive": 5, "slow_smooth": 6, "staggered_multirate": 7,
    "bad_energy_lazy": 8, "adaptive_balanced_good": 9,
}

# ============================================================
# THERMAL-TRIM DEFAULTS
# ============================================================
THERMAL_DEFAULTS = {
    "opt_start_window_hrs":  2.0,
    "opt_start_gain":        0.5,
    "solar_ref":             600.0,
    "solar_heat_offset_max": 1.0,
    "solar_cool_offset_max": 0.8,
    "cool_opt_gain":         0.5,
    "cool_drift_gain":       0.5,
}


# ============================================================
# PROFILES
# ============================================================
EXPERT_PROFILES = {
    "smooth": {
        "occ_heat_sp": 21.0,   "occ_cool_sp": 24.0,
        "pre_heat_sp": 20.0,   "pre_cool_sp": 25.0,
        "sb_heat_sp":  16.0,   "sb_cool_sp":  28.0,
        "target_lux": 500.0,
        "lighting_deadband_lux": 50.0,
        "lighting_action_deadband_w": 50.0,
        "solar_cool_darken":  (200.0, 350.0, 500.0),
        "solar_cool_lighten":  (80.0, 230.0, 380.0),
        "solar_heat_darken":  (400.0, 600.0, 800.0),
        "solar_heat_lighten": (270.0, 470.0, 670.0),
        "solar_neutral_darken":  (300.0, 500.0, 700.0),
        "solar_neutral_lighten": (180.0, 380.0, 580.0),
        "wpi_glare_hard": 1000.0,
        "glare_state": "sr2_ec03",
        "action_hold_steps": 6,
        "glazing_update_steps": 6,
        "lighting_update_steps": 1,
        "thermal_update_steps": 1,
        "one_tier_at_a_time": True,
        "opt_start_window_hrs":  2.0,
        "opt_start_gain":        0.35,
        "solar_ref":             700.0,
        "solar_heat_offset_max": 0.8,
        "solar_cool_offset_max": 0.6,
        "cool_opt_gain":         0.35,
        "cool_drift_gain":       0.4,
    },
}


# ============================================================
# APPLY SELECTED PROFILE
# ============================================================

if EXPERT_MODE not in EXPERT_PROFILES:
    raise ValueError(f"Unknown EXPERT_MODE: {EXPERT_MODE!r}. "
                     f"Choose from {list(EXPERT_PROFILES.keys())}")

PROFILE = EXPERT_PROFILES[EXPERT_MODE]
EXPERT_ID = EXPERT_ID_MAP[EXPERT_MODE]
print(f"[v12 controller] EXPERT_MODE = '{EXPERT_MODE}'  (id={EXPERT_ID})")


# ============================================================
# CONFIGURATION
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
GLAZING_CLEAR_TO_DARK   = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
TIER_OF = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}

LIGHTING_POWER_RANGE = (0.0, 2500.0)
HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
DEADBAND = 1.0

# ---- deadband (comfort-band) override thermostat ----
COMFORT_BAND_LOW    = 21.0
COMFORT_BAND_HIGH   = 24.0
COMFORT_OVERRIDE_SP = 22.0

LIGHTING_TO_LUX = 0.4

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

TARGET_LUX                  = PROFILE["target_lux"]
LIGHTING_DEADBAND_LUX       = PROFILE["lighting_deadband_lux"]
LIGHTING_ACTION_DEADBAND_W  = PROFILE["lighting_action_deadband_w"]
CORE_OCC_POWER              = TARGET_LUX / LIGHTING_TO_LUX

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

OCCUPIED_HEAT_SP     = PROFILE["occ_heat_sp"]
OCCUPIED_COOL_SP     = PROFILE["occ_cool_sp"]
PRECONDITION_HEAT_SP = PROFILE["pre_heat_sp"]
PRECONDITION_COOL_SP = PROFILE["pre_cool_sp"]
SETBACK_HEAT_SP      = PROFILE["sb_heat_sp"]
SETBACK_COOL_SP      = PROFILE["sb_cool_sp"]

SOLAR_COOL_DARKEN     = PROFILE["solar_cool_darken"]
SOLAR_COOL_LIGHTEN    = PROFILE["solar_cool_lighten"]
SOLAR_HEAT_DARKEN     = PROFILE["solar_heat_darken"]
SOLAR_HEAT_LIGHTEN    = PROFILE["solar_heat_lighten"]
SOLAR_NEUTRAL_DARKEN  = PROFILE["solar_neutral_darken"]
SOLAR_NEUTRAL_LIGHTEN = PROFILE["solar_neutral_lighten"]

WPI_GLARE_HARD     = PROFILE["wpi_glare_hard"]
GLARE_STATE        = PROFILE["glare_state"]
ACTION_HOLD_STEPS  = PROFILE["action_hold_steps"]
GLAZING_UPDATE_STEPS = PROFILE.get("glazing_update_steps", PROFILE["action_hold_steps"])
LIGHTING_UPDATE_STEPS = PROFILE.get("lighting_update_steps", 1)
THERMAL_UPDATE_STEPS = PROFILE.get("thermal_update_steps", 1)
ONE_TIER_AT_A_TIME = PROFILE["one_tier_at_a_time"]

OPT_START_WINDOW_HRS  = PROFILE.get("opt_start_window_hrs",  THERMAL_DEFAULTS["opt_start_window_hrs"])
OPT_START_GAIN        = PROFILE.get("opt_start_gain",        THERMAL_DEFAULTS["opt_start_gain"])
SOLAR_REF             = PROFILE.get("solar_ref",             THERMAL_DEFAULTS["solar_ref"])
SOLAR_HEAT_OFFSET_MAX = PROFILE.get("solar_heat_offset_max", THERMAL_DEFAULTS["solar_heat_offset_max"])
SOLAR_COOL_OFFSET_MAX = PROFILE.get("solar_cool_offset_max", THERMAL_DEFAULTS["solar_cool_offset_max"])
COOL_OPT_GAIN         = PROFILE.get("cool_opt_gain",         THERMAL_DEFAULTS["cool_opt_gain"])
COOL_DRIFT_GAIN       = PROFILE.get("cool_drift_gain",       THERMAL_DEFAULTS["cool_drift_gain"])

COIL_ACTIVE_EPS = 10.0


# ============================================================
# SDAR ACTION VECTOR LAYOUT
# ============================================================
ACTION_DIM   = 19
GLAZE_SLICE  = slice(0, 4)
LIGHT_SLICE  = slice(4, 9)
HEAT_SLICE   = slice(9, 14)
COOL_SLICE   = slice(14, 19)

GLAZE_RANGE  = (0.0, 3.0)
LIGHT_RANGE  = (0.0, 2500.0)
HEAT_RANGE   = (16.0, 24.0)
COOL_RANGE   = (22.0, 28.0)

WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())
ZONE_ORDER   = list(ZONES_MID)

LIGHT_RANGE_PER_ZONE = {zone: (0.0, LIGHTING_INSTALLED_W[zone]) for zone in ZONE_ORDER}

ACTION_MASK_VALUE = -2.0
MASK_EPS = 1e-3

ACTION_LAYOUT = {
    "glazing":  (GLAZE_SLICE, WINDOW_ORDER),
    "lighting": (LIGHT_SLICE, ZONE_ORDER),
    "heating":  (HEAT_SLICE,  ZONE_ORDER),
    "cooling":  (COOL_SLICE,  ZONE_ORDER),
    "ranges":   {"glaze": GLAZE_RANGE, "light": LIGHT_RANGE_PER_ZONE,
                 "heat":  HEAT_RANGE,  "cool":  COOL_RANGE},
    "mask_value": ACTION_MASK_VALUE,
    "dim": ACTION_DIM,
}


# ============================================================
# EXPLORATION CONFIG
# ============================================================
EXPLORATION_ENABLED = True
EXPLORATION_SEED = 10_000 + EXPERT_ID

# Glazing: epsilon on the JOINT decision, applied only on glazing-update steps.
GLAZING_EPSILON = 0.10

# Lighting: noise as fraction of installed power, occupied + lighting-update only.
LIGHTING_NOISE_PROB = 0.30
LIGHTING_NOISE_STD_FRAC = 0.05
LIGHTING_SAFE_MAX_LUX = 1000.0

# Thermal: per-zone stateful mixed policy.
THERMAL_POLICY_HOLD_STEPS = 6   # one hour at six steps/hour

INSIDE_IDLE_PROB = 0.70
INSIDE_RANDOM_BAND_PROB = 0.20
INSIDE_ACTIVE_TARGET_PROB = 0.10   # inside probs must sum to 1

OUTSIDE_NOISY_TARGET_PROB = 0.20
OUTSIDE_TARGET_NOISE_STD_C = 0.50

THERMAL_TARGET_SP = 20.0

# Exploratory single-target setpoints must be valid for BOTH coils. Use the
# common range of the comfort band and the heating/cooling actuator ranges so a
# target never normalizes outside [-1, 1] (e.g. cooling SP < 22 -> < -1).
THERMAL_RANDOM_TARGET_LOW = max(
    COMFORT_BAND_LOW,
    HEATING_SETPOINT_RANGE[0],
    COOLING_SETPOINT_RANGE[0],
)
THERMAL_RANDOM_TARGET_HIGH = min(
    COMFORT_BAND_HIGH,
    HEATING_SETPOINT_RANGE[1],
    COOLING_SETPOINT_RANGE[1],
)
if THERMAL_RANDOM_TARGET_LOW > THERMAL_RANDOM_TARGET_HIGH:
    raise ValueError(
        "No common valid thermal target exists for heating and cooling.")

THERMAL_INACTIVE_MARGIN_C = 0.25

rng = np.random.default_rng(EXPLORATION_SEED)


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
    'expert_id': [],
}

# SDAR per-step logs (action = EXECUTED / noisy action)
action_vector_data = {
    "action":         [],
    "prev_action":    [],
    "selection_mask": [],
    "action_mix":     [],
}

# Exploration log: expert (clean) vs executed (noisy) + which dims explored.
exploration_data = {
    "expert_action": [],
    "executed_action": [],
    "exploration_mask": [],
    "glazing_explored": [],
    "lighting_explored": [],
    "thermal_mode": [],
    "epsilon": [],
    "seed": EXPLORATION_SEED,
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}


# ============================================================
# POLICY MEMORY
# ============================================================

policy_memory = {
    "expert_mode": EXPERT_MODE,
    "expert_id": EXPERT_ID,
    "step": 0,
    "glazing": {zone: "sr2_ec01" for zone in WINDOW_ZONES_MID},
    "lighting": {zone: 0.0 for zone in ZONES_MID},
    "heating": {zone: SETBACK_HEAT_SP for zone in ZONES_MID},
    "cooling": {zone: SETBACK_COOL_SP for zone in ZONES_MID},
    "cooling_active": False,
    "heating_active": False,
    "prev_action_vec": None,
    # per-zone thermal exploration state (held behavior)
    "thermal_exploration": {
        zone: {
            "heating_sp": HEATING_SETPOINT_RANGE[0],
            "cooling_sp": COOLING_SETPOINT_RANGE[1],
            "mode": "idle",
            "remaining": 0,
        }
        for zone in ZONES_MID
    },
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def _norm(x, lo, hi):
    """Affine map from [lo, hi] -> [-1, 1]."""
    return 2.0 * (x - lo) / (hi - lo) - 1.0


def encode_action_vec(glaze, light, heat, cool):
    """Pack expert per-zone actions into a 19-dim normalized vector in [-1, 1]."""
    v = np.zeros(ACTION_DIM, dtype=np.float32)
    for i, zone in enumerate(WINDOW_ORDER):
        v[i] = _norm(float(TIER_OF[glaze[zone]]), *GLAZE_RANGE)
    for i, zone in enumerate(ZONE_ORDER):
        v[4 + i]  = _norm(light[zone], *LIGHT_RANGE_PER_ZONE[zone])
        v[9 + i]  = _norm(heat[zone],  *HEAT_RANGE)
        v[14 + i] = _norm(cool[zone],  *COOL_RANGE)
    return v


# ============================================================
# EXPERT RULES
# ============================================================

def choose_glazing_expert(occupied, cooling_active, heating_active,
                          ext_irr, wpi, current_glazing):
    if not occupied:
        return 'sr2_ec01'

    if wpi >= WPI_GLARE_HARD:
        current_tier = TIER_OF.get(current_glazing, 0)
        glare_tier = TIER_OF[GLARE_STATE]
        return AVAILABLE_GLAZING_STATES[max(current_tier, glare_tier)]

    if cooling_active:
        darken, lighten = SOLAR_COOL_DARKEN, SOLAR_COOL_LIGHTEN
    elif heating_active:
        darken, lighten = SOLAR_HEAT_DARKEN, SOLAR_HEAT_LIGHTEN
    else:
        darken, lighten = SOLAR_NEUTRAL_DARKEN, SOLAR_NEUTRAL_LIGHTEN

    current_tier = TIER_OF.get(current_glazing, 0)

    target = current_tier
    while target < 3 and ext_irr > darken[target]:
        target += 1
    if target > current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier + 1]
        return AVAILABLE_GLAZING_STATES[target]

    target = current_tier
    while target > 0 and ext_irr < lighten[target - 1]:
        target -= 1
    if target < current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier - 1]
        return AVAILABLE_GLAZING_STATES[target]

    return current_glazing


def choose_perimeter_lighting_expert(occupied, current_wpi, zone_max_w,
                                     prev_power=0.0,
                                     action_deadband_w=0.0):
    if not occupied:
        return 0.0
    shortfall_lux = TARGET_LUX - current_wpi
    if shortfall_lux < LIGHTING_DEADBAND_LUX:
        return 0.0
    desired_power = clamp(shortfall_lux / LIGHTING_TO_LUX, 0.0, zone_max_w)
    if action_deadband_w > 0.0 \
       and abs(desired_power - prev_power) < action_deadband_w:
        return prev_power
    return desired_power


def choose_core_lighting_expert(occupied):
    if not occupied:
        return 0.0
    return CORE_OCC_POWER


def choose_hvac_zone(dt, zone_temp, zone_ext_irr,
                     heating_active=False, cooling_active=False,
                     has_window=True):
    """
    DEADBAND (comfort-band) override thermostat (deterministic expert).
      - INSIDE [21,24] -> 16/28 (no coil fires).
      - OUTSIDE       -> 22/22 (recover toward 22).
    """
    if COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH:
        h = HEATING_SETPOINT_RANGE[0]
        c = COOLING_SETPOINT_RANGE[1]
    else:
        h = COMFORT_OVERRIDE_SP
        c = COMFORT_OVERRIDE_SP
    return h, c


# ============================================================
# EXPLORATION — glazing (safe local epsilon-greedy)
# ============================================================

def apply_glazing_exploration(selected_g, current_obs, update_glazing):
    """
    With prob GLAZING_EPSILON (on glazing-update steps only), modify one or two
    glazing zones. Respects the update period, one-tier-at-a-time transitions,
    and the glare rule (never lightens during glare).
    """
    explored = {zone: False for zone in WINDOW_ORDER}

    if (not EXPLORATION_ENABLED
            or not update_glazing
            or rng.random() >= GLAZING_EPSILON):
        return selected_g, explored

    n_zones = 1 if rng.random() < 0.8 else 2
    zones_to_explore = rng.choice(WINDOW_ORDER, size=n_zones, replace=False)

    result = selected_g.copy()
    for zone in zones_to_explore:
        current_tier = TIER_OF[policy_memory["glazing"][zone]]
        expert_tier = TIER_OF[selected_g[zone]]

        candidate_tiers = [
            tier for tier in (current_tier - 1, current_tier, current_tier + 1)
            if 0 <= tier <= 3
        ]

        if current_obs["wpi"][zone] >= WPI_GLARE_HARD:
            minimum_safe_tier = max(current_tier, TIER_OF[GLARE_STATE])
            candidate_tiers = [t for t in candidate_tiers if t >= minimum_safe_tier]

        alternatives = [t for t in candidate_tiers if t != expert_tier]
        if alternatives:
            chosen_tier = int(rng.choice(alternatives))
            result[zone] = AVAILABLE_GLAZING_STATES[chosen_tier]
            explored[zone] = True

    return result, explored


# ============================================================
# EXPLORATION — lighting (bounded Gaussian in dimming space)
# ============================================================

def apply_lighting_exploration(selected_l, current_obs, occupied, update_lighting):
    """Add bounded Gaussian noise in dimming-fraction space (occupied only)."""
    explored = {zone: False for zone in ZONE_ORDER}

    if (not EXPLORATION_ENABLED or not update_lighting or not occupied):
        return selected_l, explored

    result = selected_l.copy()
    for zone in ZONE_ORDER:
        if rng.random() >= LIGHTING_NOISE_PROB:
            continue

        nominal_w = LIGHTING_INSTALLED_W[zone]
        expert_dimming = np.clip(selected_l[zone] / nominal_w, 0.0, 1.0)
        noisy_dimming = np.clip(
            expert_dimming + rng.normal(0.0, LIGHTING_NOISE_STD_FRAC), 0.0, 1.0)
        noisy_power_w = noisy_dimming * nominal_w

        maximum_power_w = min(nominal_w, LIGHTING_POWER_RANGE[1])
        if zone in WINDOW_ZONES_MID:
            daylight_lux = current_obs["wpi"][zone]
            maximum_power_by_lux = max(
                0.0, (LIGHTING_SAFE_MAX_LUX - daylight_lux) / LIGHTING_TO_LUX)
            maximum_power_w = min(maximum_power_w, maximum_power_by_lux)

        result[zone] = clamp(noisy_power_w, 0.0, maximum_power_w)
        explored[zone] = not np.isclose(result[zone], selected_l[zone])

    return result, explored


# ============================================================
# EXPLORATION — per-zone mixed thermal policy
# ============================================================

def choose_exploratory_hvac_zone(zone, zone_temp):
    """
    Per-zone stateful mixed thermal behavior.
      OUTSIDE 21-24 : recovery (22/22) or occasional noisy recovery.
      INSIDE  21-24 : idle (16/28) / random-inactive band / active target,
                      held for THERMAL_POLICY_HOLD_STEPS.
    Leaving the band immediately discards any held exploratory behavior.
    """
    memory = policy_memory["thermal_exploration"][zone]
    inside_comfort = COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH

    # ---- outside comfort: recovery takes precedence ----
    if not inside_comfort:
        memory["remaining"] = 0
        if EXPLORATION_ENABLED and rng.random() < OUTSIDE_NOISY_TARGET_PROB:
            target = THERMAL_TARGET_SP + rng.normal(0.0, OUTSIDE_TARGET_NOISE_STD_C)
            target = clamp(target, THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
            mode = "noisy_recovery"
        else:
            target = THERMAL_TARGET_SP
            mode = "expert_recovery"
        memory["heating_sp"] = target
        memory["cooling_sp"] = target
        memory["mode"] = mode
        return target, target, mode

    # ---- inside comfort: hold prior exploratory behavior ----
    if memory["remaining"] > 0:
        memory["remaining"] -= 1
        return memory["heating_sp"], memory["cooling_sp"], memory["mode"]

    # exploration disabled -> plain idle expert inside the band
    if not EXPLORATION_ENABLED:
        memory["heating_sp"] = HEATING_SETPOINT_RANGE[0]
        memory["cooling_sp"] = COOLING_SETPOINT_RANGE[1]
        memory["mode"] = "idle"
        memory["remaining"] = 0
        return HEATING_SETPOINT_RANGE[0], COOLING_SETPOINT_RANGE[1], "idle"

    # select a new inside-band behavior
    sample = rng.random()
    if sample < INSIDE_IDLE_PROB:
        heating_sp = HEATING_SETPOINT_RANGE[0]   # 16
        cooling_sp = COOLING_SETPOINT_RANGE[1]   # 28
        mode = "idle"
    elif sample < (INSIDE_IDLE_PROB + INSIDE_RANDOM_BAND_PROB):
        maximum_heating_sp = min(zone_temp - THERMAL_INACTIVE_MARGIN_C,
                                 HEATING_SETPOINT_RANGE[1])
        minimum_cooling_sp = max(zone_temp + THERMAL_INACTIVE_MARGIN_C,
                                 COOLING_SETPOINT_RANGE[0])
        heating_sp = rng.uniform(HEATING_SETPOINT_RANGE[0], maximum_heating_sp)
        cooling_sp = rng.uniform(minimum_cooling_sp, COOLING_SETPOINT_RANGE[1])
        mode = "random_inactive_band"
    else:
        target = rng.uniform(THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
        heating_sp = target
        cooling_sp = target
        mode = "active_exploration"

    memory["heating_sp"] = float(heating_sp)
    memory["cooling_sp"] = float(cooling_sp)
    memory["mode"] = mode
    memory["remaining"] = THERMAL_POLICY_HOLD_STEPS - 1
    return float(heating_sp), float(cooling_sp), mode


# ============================================================
# DECISION DISPATCH — expert action -> exploration -> memory update
# ============================================================

def choose_expert_actions(current_obs, dt):
    policy_memory["step"] += 1
    update_glazing  = (policy_memory["step"] % GLAZING_UPDATE_STEPS == 0)
    update_lighting = (policy_memory["step"] % LIGHTING_UPDATE_STEPS == 0)
    update_thermal  = (policy_memory["step"] % THERMAL_UPDATE_STEPS == 0)

    occupied = is_occupied(dt)
    cooling_active = policy_memory["cooling_active"]
    heating_active = policy_memory["heating_active"]

    selected_g, selected_l, selected_h, selected_c = {}, {}, {}, {}
    base_h, base_c = {}, {}   # deterministic thermal expert (for exploration mask)
    thermal_mode = {}

    for zone, _window in WINDOW_ZONES_MID.items():
        wpi = current_obs["wpi"][zone]
        ext_irr = current_obs["ext_irr"][zone]
        current_g = policy_memory["glazing"][zone]

        if update_glazing:
            g = choose_glazing_expert(
                occupied=occupied,
                cooling_active=cooling_active,
                heating_active=heating_active,
                ext_irr=ext_irr,
                wpi=wpi,
                current_glazing=current_g,
            )
        else:
            g = current_g
        selected_g[zone] = g

        if update_lighting:
            target_p = choose_perimeter_lighting_expert(
                occupied=occupied,
                current_wpi=wpi,
                zone_max_w=LIGHTING_INSTALLED_W[zone],
                prev_power=policy_memory["lighting"][zone],
                action_deadband_w=LIGHTING_ACTION_DEADBAND_W,
            )
            selected_l[zone] = clamp(target_p, *LIGHTING_POWER_RANGE)
        else:
            selected_l[zone] = policy_memory["lighting"][zone]

        # Deterministic thermal expert (logging baseline for the mask)
        h_expert, c_expert = choose_hvac_zone(
            dt=dt,
            zone_temp=current_obs["temp"][zone],
            zone_ext_irr=current_obs["ext_irr"][zone],
            heating_active=heating_active,
            cooling_active=cooling_active,
            has_window=True,
        )
        base_h[zone] = h_expert
        base_c[zone] = c_expert

        # Thermal — per-zone mixed exploratory policy (actually executed).
        if update_thermal:
            h_z, c_z, m_z = choose_exploratory_hvac_zone(
                zone=zone, zone_temp=current_obs["temp"][zone])
            selected_h[zone] = h_z
            selected_c[zone] = c_z
            thermal_mode[zone] = m_z
        else:
            selected_h[zone] = policy_memory["heating"][zone]
            selected_c[zone] = policy_memory["cooling"][zone]
            thermal_mode[zone] = policy_memory["thermal_exploration"][zone]["mode"]

    core_zone = "Core_mid"

    if update_lighting:
        selected_l[core_zone] = clamp(choose_core_lighting_expert(occupied),
                                      *LIGHTING_POWER_RANGE)
    else:
        selected_l[core_zone] = policy_memory["lighting"][core_zone]

    # Deterministic thermal expert for the core (logging baseline)
    h_expert_c, c_expert_c = choose_hvac_zone(
        dt=dt,
        zone_temp=current_obs["temp"][core_zone],
        zone_ext_irr=0.0,
        heating_active=heating_active,
        cooling_active=cooling_active,
        has_window=False,
    )
    base_h[core_zone] = h_expert_c
    base_c[core_zone] = c_expert_c

    if update_thermal:
        h_c, c_c, m_c = choose_exploratory_hvac_zone(
            zone=core_zone, zone_temp=current_obs["temp"][core_zone])
        selected_h[core_zone] = h_c
        selected_c[core_zone] = c_c
        thermal_mode[core_zone] = m_c
    else:
        selected_h[core_zone] = policy_memory["heating"][core_zone]
        selected_c[core_zone] = policy_memory["cooling"][core_zone]
        thermal_mode[core_zone] = policy_memory["thermal_exploration"][core_zone]["mode"]

    # ---- preserve deterministic expert decision ----
    # glazing/lighting: pre-exploration values (exploration applied just below);
    # thermal: the deterministic base_h/base_c (selected_h/c already hold the
    # EXECUTED mixed-policy setpoints, so using them would zero the thermal mask).
    expert_g = selected_g.copy()
    expert_l = selected_l.copy()
    expert_h = base_h.copy()
    expert_c = base_c.copy()

    # ---- apply glazing + lighting exploration ----
    selected_g, glazing_explored = apply_glazing_exploration(
        selected_g=selected_g, current_obs=current_obs, update_glazing=update_glazing)
    selected_l, lighting_explored = apply_lighting_exploration(
        selected_l=selected_l, current_obs=current_obs,
        occupied=occupied, update_lighting=update_lighting)

    # ---- update memory using EXECUTED (noisy) actions ----
    for zone in WINDOW_ZONES_MID:
        policy_memory["glazing"][zone] = selected_g[zone]
    for zone in ZONES_MID:
        policy_memory["lighting"][zone] = selected_l[zone]
        policy_memory["heating"][zone] = selected_h[zone]
        policy_memory["cooling"][zone] = selected_c[zone]

    # ---- exploration logging (expert vs executed) ----
    expert_vec = encode_action_vec(expert_g, expert_l, expert_h, expert_c)
    executed_vec = encode_action_vec(selected_g, selected_l, selected_h, selected_c)
    exploration_mask = (np.abs(executed_vec - expert_vec) > MASK_EPS).astype(np.float32)

    exploration_data["expert_action"].append(expert_vec)
    exploration_data["executed_action"].append(executed_vec)
    exploration_data["exploration_mask"].append(exploration_mask)
    exploration_data["glazing_explored"].append(glazing_explored)
    exploration_data["lighting_explored"].append(lighting_explored)
    exploration_data["thermal_mode"].append(dict(thermal_mode))
    exploration_data["epsilon"].append(GLAZING_EPSILON)

    return selected_g, selected_l, selected_h, selected_c


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    dt = eps.get_datetime()

    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = policy_memory["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec  = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec    = get_meter_value_cached(state, "Electricity:HVAC")

    policy_memory["cooling_active"] = (cooling_elec > COIL_ACTIVE_EPS)
    policy_memory["heating_active"] = (heating_elec > COIL_ACTIVE_EPS)

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)
    temporal_data['expert_id'].append(EXPERT_ID)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
            'Core_mid': 0.0,
        },
        "energy": {
            'lighting_rate': (light_mid_1 + light_mid_2 + light_mid_3
                              + light_mid_4 + light_core_mid),
            'hvac_demand_rate': hvac_electricity_demand_rate,
        },
        "wpi": current_wpi,
        "temporal": tf,
    }

    sel_g, sel_l, sel_h, sel_c = choose_expert_actions(current_obs, dt)

    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=sel_g[zone])
        eps.actuate_lighting_power(light=zone, value=sel_l[zone])
        eps.actuate_heating_setpoint(zone=zone, value=sel_h[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=sel_c[zone])

        glazing_state_data[zone].append(sel_g[zone])
        lighting_power_data[zone].append(sel_l[zone])
        heating_setpoint_data[zone].append(sel_h[zone])
        cooling_setpoint_data[zone].append(sel_c[zone])

    eps.actuate_lighting_power(light="Core_mid", value=sel_l['Core_mid'])
    eps.actuate_heating_setpoint(zone="Core_mid", value=sel_h['Core_mid'])
    eps.actuate_cooling_setpoint(zone="Core_mid", value=sel_c['Core_mid'])

    lighting_power_data['Core_mid'].append(sel_l['Core_mid'])
    heating_setpoint_data['Core_mid'].append(sel_h['Core_mid'])
    cooling_setpoint_data['Core_mid'].append(sel_c['Core_mid'])

    # ---- SDAR action-mask logging (uses EXECUTED action) ----
    a_vec = encode_action_vec(sel_g, sel_l, sel_h, sel_c)
    prev_a_vec = policy_memory["prev_action_vec"]

    if prev_a_vec is None:
        b_mask = np.ones(ACTION_DIM, dtype=np.float32)
        prev_for_log = a_vec.copy()
    else:
        b_mask = (np.abs(a_vec - prev_a_vec) > MASK_EPS).astype(np.float32)
        prev_for_log = prev_a_vec.copy()

    a_mix = ((1.0 - b_mask) * prev_for_log
             + b_mask * ACTION_MASK_VALUE).astype(np.float32)

    action_vector_data["action"].append(a_vec.copy())
    action_vector_data["prev_action"].append(prev_for_log)
    action_vector_data["selection_mask"].append(b_mask)
    action_vector_data["action_mix"].append(a_mix)

    policy_memory["prev_action_vec"] = a_vec

In [ ]:
"""
EXPERT DATA COLLECTION POLICY v12 — smooth expert + CALLBACK-SIDE EXPLORATION
============================================================================

Deadband-override smooth expert with three exploration systems added INSIDE the
controller (so noisy actions reach EnergyPlus and the resulting next-state /
reward correspond to the executed action):

  expert action -> exploration -> actuation -> logging

  1) Safe local epsilon-greedy GLAZING (doc: "Safe epsilon-greedy glazing"):
     - applied only when glazing is scheduled to update (hourly at 6 steps/h);
     - epsilon on the JOINT decision, randomizing one (usually) or two windows;
     - respects one-tier-at-a-time and the glare rule (never lightens in glare).

  2) Dimming-space LIGHTING noise (doc: "Lighting exploration in dimming space"):
     - per-zone Gaussian noise in dimming fraction (sigma * nominal power);
     - only while occupied and on a lighting-update step;
     - bounded so artificial light keeps total < LIGHTING_SAFE_MAX_LUX.

  3) Per-zone stateful mixed THERMAL policy (doc: "Mixed thermal controller"):
     - outside 21-24: recovery (22/22) or occasionally noisy recovery;
     - inside 21-24: idle (16/28) / random-inactive band / active target;
     - each zone holds its chosen behavior for THERMAL_POLICY_HOLD_STEPS;
     - leaving the band immediately discards the held exploratory behavior.

policy_memory is updated with the EXECUTED (noisy) actions. The SDAR mask/prev/
mix use the executed action (as before). A separate `exploration_data` log keeps
the expert (clean) action, executed action, and exploration masks.

Action vector layout (all normalized to [-1, 1]):
  [0:4] glazing tier | [4:9] lighting | [9:14] heating SP | [14:19] cooling SP
"""

import numpy as np


# ============================================================
# PICK YOUR EXPERT
# ============================================================

EXPERT_MODE = "smooth"

EXPERT_ID_MAP = {
    "balanced": 0, "energy": 1, "comfort": 2, "visual": 3, "smooth": 4,
    "fast_responsive": 5, "slow_smooth": 6, "staggered_multirate": 7,
    "bad_energy_lazy": 8, "adaptive_balanced_good": 9,
}

# ============================================================
# THERMAL-TRIM DEFAULTS
# ============================================================
THERMAL_DEFAULTS = {
    "opt_start_window_hrs":  2.0,
    "opt_start_gain":        0.5,
    "solar_ref":             600.0,
    "solar_heat_offset_max": 1.0,
    "solar_cool_offset_max": 0.8,
    "cool_opt_gain":         0.5,
    "cool_drift_gain":       0.5,
}


# ============================================================
# PROFILES
# ============================================================
EXPERT_PROFILES = {
    "smooth": {
        "occ_heat_sp": 21.0,   "occ_cool_sp": 24.0,
        "pre_heat_sp": 20.0,   "pre_cool_sp": 25.0,
        "sb_heat_sp":  16.0,   "sb_cool_sp":  28.0,
        "target_lux": 500.0,
        "lighting_deadband_lux": 50.0,
        "lighting_action_deadband_w": 50.0,
        "solar_cool_darken":  (200.0, 350.0, 500.0),
        "solar_cool_lighten":  (80.0, 230.0, 380.0),
        "solar_heat_darken":  (400.0, 600.0, 800.0),
        "solar_heat_lighten": (270.0, 470.0, 670.0),
        "solar_neutral_darken":  (300.0, 500.0, 700.0),
        "solar_neutral_lighten": (180.0, 380.0, 580.0),
        "wpi_glare_hard": 1000.0,
        "glare_state": "sr2_ec03",
        "action_hold_steps": 6,
        "glazing_update_steps": 6,
        "lighting_update_steps": 1,
        "thermal_update_steps": 1,
        "one_tier_at_a_time": True,
        "opt_start_window_hrs":  2.0,
        "opt_start_gain":        0.35,
        "solar_ref":             700.0,
        "solar_heat_offset_max": 0.8,
        "solar_cool_offset_max": 0.6,
        "cool_opt_gain":         0.35,
        "cool_drift_gain":       0.4,
    },
}


# ============================================================
# APPLY SELECTED PROFILE
# ============================================================

if EXPERT_MODE not in EXPERT_PROFILES:
    raise ValueError(f"Unknown EXPERT_MODE: {EXPERT_MODE!r}. "
                     f"Choose from {list(EXPERT_PROFILES.keys())}")

PROFILE = EXPERT_PROFILES[EXPERT_MODE]
EXPERT_ID = EXPERT_ID_MAP[EXPERT_MODE]
print(f"[v12 controller] EXPERT_MODE = '{EXPERT_MODE}'  (id={EXPERT_ID})")


# ============================================================
# CONFIGURATION
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
GLAZING_CLEAR_TO_DARK   = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']
TIER_OF = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}

LIGHTING_POWER_RANGE = (0.0, 2500.0)
HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
DEADBAND = 1.0

# ---- deadband (comfort-band) override thermostat ----
COMFORT_BAND_LOW    = 22.0
COMFORT_BAND_HIGH   = 25.0
COMFORT_OVERRIDE_SP = 24.0

LIGHTING_TO_LUX = 0.4

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

TARGET_LUX                  = PROFILE["target_lux"]
LIGHTING_DEADBAND_LUX       = PROFILE["lighting_deadband_lux"]
LIGHTING_ACTION_DEADBAND_W  = PROFILE["lighting_action_deadband_w"]
CORE_OCC_POWER              = TARGET_LUX / LIGHTING_TO_LUX

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

OCCUPIED_HEAT_SP     = PROFILE["occ_heat_sp"]
OCCUPIED_COOL_SP     = PROFILE["occ_cool_sp"]
PRECONDITION_HEAT_SP = PROFILE["pre_heat_sp"]
PRECONDITION_COOL_SP = PROFILE["pre_cool_sp"]
SETBACK_HEAT_SP      = PROFILE["sb_heat_sp"]
SETBACK_COOL_SP      = PROFILE["sb_cool_sp"]

SOLAR_COOL_DARKEN     = PROFILE["solar_cool_darken"]
SOLAR_COOL_LIGHTEN    = PROFILE["solar_cool_lighten"]
SOLAR_HEAT_DARKEN     = PROFILE["solar_heat_darken"]
SOLAR_HEAT_LIGHTEN    = PROFILE["solar_heat_lighten"]
SOLAR_NEUTRAL_DARKEN  = PROFILE["solar_neutral_darken"]
SOLAR_NEUTRAL_LIGHTEN = PROFILE["solar_neutral_lighten"]

WPI_GLARE_HARD     = PROFILE["wpi_glare_hard"]
GLARE_STATE        = PROFILE["glare_state"]
ACTION_HOLD_STEPS  = PROFILE["action_hold_steps"]
GLAZING_UPDATE_STEPS = PROFILE.get("glazing_update_steps", PROFILE["action_hold_steps"])
LIGHTING_UPDATE_STEPS = PROFILE.get("lighting_update_steps", 1)
THERMAL_UPDATE_STEPS = PROFILE.get("thermal_update_steps", 1)
ONE_TIER_AT_A_TIME = PROFILE["one_tier_at_a_time"]

OPT_START_WINDOW_HRS  = PROFILE.get("opt_start_window_hrs",  THERMAL_DEFAULTS["opt_start_window_hrs"])
OPT_START_GAIN        = PROFILE.get("opt_start_gain",        THERMAL_DEFAULTS["opt_start_gain"])
SOLAR_REF             = PROFILE.get("solar_ref",             THERMAL_DEFAULTS["solar_ref"])
SOLAR_HEAT_OFFSET_MAX = PROFILE.get("solar_heat_offset_max", THERMAL_DEFAULTS["solar_heat_offset_max"])
SOLAR_COOL_OFFSET_MAX = PROFILE.get("solar_cool_offset_max", THERMAL_DEFAULTS["solar_cool_offset_max"])
COOL_OPT_GAIN         = PROFILE.get("cool_opt_gain",         THERMAL_DEFAULTS["cool_opt_gain"])
COOL_DRIFT_GAIN       = PROFILE.get("cool_drift_gain",       THERMAL_DEFAULTS["cool_drift_gain"])

COIL_ACTIVE_EPS = 10.0


# ============================================================
# SDAR ACTION VECTOR LAYOUT
# ============================================================
ACTION_DIM   = 19
GLAZE_SLICE  = slice(0, 4)
LIGHT_SLICE  = slice(4, 9)
HEAT_SLICE   = slice(9, 14)
COOL_SLICE   = slice(14, 19)

GLAZE_RANGE  = (0.0, 3.0)
LIGHT_RANGE  = (0.0, 2500.0)
HEAT_RANGE   = (16.0, 24.0)
COOL_RANGE   = (22.0, 28.0)

WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())
ZONE_ORDER   = list(ZONES_MID)

LIGHT_RANGE_PER_ZONE = {zone: (0.0, LIGHTING_INSTALLED_W[zone]) for zone in ZONE_ORDER}

ACTION_MASK_VALUE = -2.0
MASK_EPS = 1e-3

ACTION_LAYOUT = {
    "glazing":  (GLAZE_SLICE, WINDOW_ORDER),
    "lighting": (LIGHT_SLICE, ZONE_ORDER),
    "heating":  (HEAT_SLICE,  ZONE_ORDER),
    "cooling":  (COOL_SLICE,  ZONE_ORDER),
    "ranges":   {"glaze": GLAZE_RANGE, "light": LIGHT_RANGE_PER_ZONE,
                 "heat":  HEAT_RANGE,  "cool":  COOL_RANGE},
    "mask_value": ACTION_MASK_VALUE,
    "dim": ACTION_DIM,
}


# ============================================================
# EXPLORATION CONFIG
# ============================================================

EPISODE_ID = 13                 # change for every rollout
NOISE_PROFILE = "high"       # "clean", "low", "medium", "high"

EXPLORATION_ENABLED = NOISE_PROFILE != "clean"

NOISE_PROFILES = {
    "clean": {
        "glazing_epsilon": 0.00,
        "lighting_noise_prob": 0.00,
        "lighting_noise_std_frac": 0.00,
        "inside_idle_prob": 1.00,
        "inside_random_band_prob": 0.00,
        "inside_active_target_prob": 0.00,
        "outside_noisy_target_prob": 0.00,
        "outside_target_noise_std_c": 0.00,
    },
    "low": {
        "glazing_epsilon": 0.05,
        "lighting_noise_prob": 0.15,
        "lighting_noise_std_frac": 0.03,
        "inside_idle_prob": 0.85,
        "inside_random_band_prob": 0.10,
        "inside_active_target_prob": 0.05,
        "outside_noisy_target_prob": 0.10,
        "outside_target_noise_std_c": 0.25,
    },
    "medium": {
        "glazing_epsilon": 0.10,
        "lighting_noise_prob": 0.30,
        "lighting_noise_std_frac": 0.05,
        "inside_idle_prob": 0.70,
        "inside_random_band_prob": 0.20,
        "inside_active_target_prob": 0.10,
        "outside_noisy_target_prob": 0.20,
        "outside_target_noise_std_c": 0.50,
    },
    "high": {
        "glazing_epsilon": 0.20,
        "lighting_noise_prob": 0.40,
        "lighting_noise_std_frac": 0.08,
        "inside_idle_prob": 0.55,
        "inside_random_band_prob": 0.30,
        "inside_active_target_prob": 0.15,
        "outside_noisy_target_prob": 0.30,
        "outside_target_noise_std_c": 0.70,
    },
}

P = NOISE_PROFILES[NOISE_PROFILE]

GLAZING_EPSILON = P["glazing_epsilon"]

LIGHTING_NOISE_PROB = P["lighting_noise_prob"]
LIGHTING_NOISE_STD_FRAC = P["lighting_noise_std_frac"]
LIGHTING_SAFE_MAX_LUX = 1000.0

INSIDE_IDLE_PROB = P["inside_idle_prob"]
INSIDE_RANDOM_BAND_PROB = P["inside_random_band_prob"]
INSIDE_ACTIVE_TARGET_PROB = P["inside_active_target_prob"]

OUTSIDE_NOISY_TARGET_PROB = P["outside_noisy_target_prob"]
OUTSIDE_TARGET_NOISE_STD_C = P["outside_target_noise_std_c"]

THERMAL_POLICY_HOLD_STEPS = 6

# SDAR temporal exploration: probability of repeating the previous action
SDAR_GLAZING_REPEAT_PROB = .10
SDAR_LIGHTING_REPEAT_PROB = .10
SDAR_THERMAL_REPEAT_PROB = .05

THERMAL_TARGET_SP = 22.0
# Keep THERMAL_RANDOM_TARGET_LOW = 22.0: the cooling action range begins at
# 22 C, so a 21 C target would normalize the cooling action below -1.
THERMAL_RANDOM_TARGET_LOW = 22.0
THERMAL_RANDOM_TARGET_HIGH = 24.0
THERMAL_INACTIVE_MARGIN_C = 0.25

# Unique deterministic seed for every expert and episode
EXPLORATION_SEED = (
    10_000
    + 1_000 * EXPERT_ID
    + EPISODE_ID+ 42142142142
)

rng = np.random.default_rng(EXPLORATION_SEED)

# Validate thermal probabilities
inside_probability_sum = (
    INSIDE_IDLE_PROB
    + INSIDE_RANDOM_BAND_PROB
    + INSIDE_ACTIVE_TARGET_PROB
)

if not np.isclose(inside_probability_sum, 1.0):
    raise ValueError(
        "Thermal inside-band probabilities must sum to 1. "
        f"Current sum: {inside_probability_sum}"
    )


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
    'expert_id': [],
}

# SDAR per-step logs (action = EXECUTED / noisy action)
action_vector_data = {
    "action":         [],
    "prev_action":    [],
    "selection_mask": [],
    "action_mix":     [],
}

# Exploration log: expert (clean) vs executed (noisy) + which dims explored.
exploration_data = {
    "expert_action": [],
    "executed_action": [],
    "exploration_mask": [],
    "glazing_explored": [],
    "lighting_explored": [],
    "thermal_mode": [],
    "epsilon": [],
    "sdar_forced_repeat": [],
    "seed": EXPLORATION_SEED,
    "episode_id": EPISODE_ID,
    "noise_profile": NOISE_PROFILE,
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}


# ============================================================
# POLICY MEMORY
# ============================================================

policy_memory = {
    "expert_mode": EXPERT_MODE,
    "expert_id": EXPERT_ID,
    "step": 0,
    "glazing": {zone: "sr2_ec01" for zone in WINDOW_ZONES_MID},
    "lighting": {zone: 0.0 for zone in ZONES_MID},
    "heating": {zone: SETBACK_HEAT_SP for zone in ZONES_MID},
    "cooling": {zone: SETBACK_COOL_SP for zone in ZONES_MID},
    "cooling_active": False,
    "heating_active": False,
    "prev_action_vec": None,
    # per-zone thermal exploration state (held behavior)
    "thermal_exploration": {
        zone: {
            "heating_sp": HEATING_SETPOINT_RANGE[0],
            "cooling_sp": COOLING_SETPOINT_RANGE[1],
            "mode": "idle",
            "remaining": 0,
        }
        for zone in ZONES_MID
    },
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def _norm(x, lo, hi):
    """Affine map from [lo, hi] -> [-1, 1]."""
    return 2.0 * (x - lo) / (hi - lo) - 1.0


def encode_action_vec(glaze, light, heat, cool):
    """Pack expert per-zone actions into a 19-dim normalized vector in [-1, 1]."""
    v = np.zeros(ACTION_DIM, dtype=np.float32)
    for i, zone in enumerate(WINDOW_ORDER):
        v[i] = _norm(float(TIER_OF[glaze[zone]]), *GLAZE_RANGE)
    for i, zone in enumerate(ZONE_ORDER):
        v[4 + i]  = _norm(light[zone], *LIGHT_RANGE_PER_ZONE[zone])
        v[9 + i]  = _norm(heat[zone],  *HEAT_RANGE)
        v[14 + i] = _norm(cool[zone],  *COOL_RANGE)
    return v


# ============================================================
# EXPERT RULES
# ============================================================

def choose_glazing_expert(occupied, cooling_active, heating_active,
                          ext_irr, wpi, current_glazing):
    if not occupied:
        return 'sr2_ec01'

    if wpi >= WPI_GLARE_HARD:
        current_tier = TIER_OF.get(current_glazing, 0)
        glare_tier = TIER_OF[GLARE_STATE]
        return AVAILABLE_GLAZING_STATES[max(current_tier, glare_tier)]

    if cooling_active:
        darken, lighten = SOLAR_COOL_DARKEN, SOLAR_COOL_LIGHTEN
    elif heating_active:
        darken, lighten = SOLAR_HEAT_DARKEN, SOLAR_HEAT_LIGHTEN
    else:
        darken, lighten = SOLAR_NEUTRAL_DARKEN, SOLAR_NEUTRAL_LIGHTEN

    current_tier = TIER_OF.get(current_glazing, 0)

    target = current_tier
    while target < 3 and ext_irr > darken[target]:
        target += 1
    if target > current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier + 1]
        return AVAILABLE_GLAZING_STATES[target]

    target = current_tier
    while target > 0 and ext_irr < lighten[target - 1]:
        target -= 1
    if target < current_tier:
        if ONE_TIER_AT_A_TIME:
            return AVAILABLE_GLAZING_STATES[current_tier - 1]
        return AVAILABLE_GLAZING_STATES[target]

    return current_glazing


def choose_perimeter_lighting_expert(occupied, current_wpi, zone_max_w,
                                     prev_power=0.0,
                                     action_deadband_w=0.0):
    if not occupied:
        return 0.0
    shortfall_lux = TARGET_LUX - current_wpi
    if shortfall_lux < LIGHTING_DEADBAND_LUX:
        return 0.0
    desired_power = clamp(shortfall_lux / LIGHTING_TO_LUX, 0.0, zone_max_w)
    if action_deadband_w > 0.0 \
       and abs(desired_power - prev_power) < action_deadband_w:
        return prev_power
    return desired_power


def choose_core_lighting_expert(occupied):
    if not occupied:
        return 0.0
    return CORE_OCC_POWER


def choose_hvac_zone(dt, zone_temp, zone_ext_irr,
                     heating_active=False, cooling_active=False,
                     has_window=True):
    """
    DEADBAND (comfort-band) override thermostat (deterministic expert).
      - INSIDE [21,24] -> 16/28 (no coil fires).
      - OUTSIDE       -> 22/22 (recover toward 22).
    """
    if COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH:
        h = HEATING_SETPOINT_RANGE[0]
        c = COOLING_SETPOINT_RANGE[1]
    else:
        h = COMFORT_OVERRIDE_SP
        c = COMFORT_OVERRIDE_SP
    return h, c


# ============================================================
# EXPLORATION — glazing (safe local epsilon-greedy)
# ============================================================

def apply_glazing_exploration(selected_g, current_obs, update_glazing):
    """
    With prob GLAZING_EPSILON (on glazing-update steps only), modify one or two
    glazing zones. Respects the update period, one-tier-at-a-time transitions,
    and the glare rule (never lightens during glare).
    """
    explored = {zone: False for zone in WINDOW_ORDER}

    if (not EXPLORATION_ENABLED
            or not update_glazing
            or rng.random() >= GLAZING_EPSILON):
        return selected_g, explored

    n_zones = 1 if rng.random() < 0.8 else 2
    zones_to_explore = rng.choice(WINDOW_ORDER, size=n_zones, replace=False)

    result = selected_g.copy()
    for zone in zones_to_explore:
        current_tier = TIER_OF[policy_memory["glazing"][zone]]
        expert_tier = TIER_OF[selected_g[zone]]

        candidate_tiers = [
            tier for tier in (current_tier - 1, current_tier, current_tier + 1)
            if 0 <= tier <= 3
        ]

        if current_obs["wpi"][zone] >= WPI_GLARE_HARD:
            minimum_safe_tier = max(current_tier, TIER_OF[GLARE_STATE])
            candidate_tiers = [t for t in candidate_tiers if t >= minimum_safe_tier]

        alternatives = [t for t in candidate_tiers if t != expert_tier]
        if alternatives:
            chosen_tier = int(rng.choice(alternatives))
            result[zone] = AVAILABLE_GLAZING_STATES[chosen_tier]
            explored[zone] = True

    return result, explored


# ============================================================
# EXPLORATION — lighting (bounded Gaussian in dimming space)
# ============================================================

def apply_lighting_exploration(selected_l, current_obs, occupied, update_lighting):
    """Add bounded Gaussian noise in dimming-fraction space (occupied only)."""
    explored = {zone: False for zone in ZONE_ORDER}

    if (not EXPLORATION_ENABLED or not update_lighting or not occupied):
        return selected_l, explored

    result = selected_l.copy()
    for zone in ZONE_ORDER:
        if rng.random() >= LIGHTING_NOISE_PROB:
            continue

        nominal_w = LIGHTING_INSTALLED_W[zone]
        expert_dimming = np.clip(selected_l[zone] / nominal_w, 0.0, 1.0)
        noisy_dimming = np.clip(
            expert_dimming + rng.normal(0.0, LIGHTING_NOISE_STD_FRAC), 0.0, 1.0)
        noisy_power_w = noisy_dimming * nominal_w

        maximum_power_w = min(nominal_w, LIGHTING_POWER_RANGE[1])
        if zone in WINDOW_ZONES_MID:
            daylight_lux = current_obs["wpi"][zone]
            maximum_power_by_lux = max(
                0.0, (LIGHTING_SAFE_MAX_LUX - daylight_lux) / LIGHTING_TO_LUX)
            maximum_power_w = min(maximum_power_w, maximum_power_by_lux)

        result[zone] = clamp(noisy_power_w, 0.0, maximum_power_w)
        explored[zone] = not np.isclose(result[zone], selected_l[zone])

    return result, explored


# ============================================================
# EXPLORATION — per-zone mixed thermal policy
# ============================================================

def choose_exploratory_hvac_zone(zone, zone_temp):
    """
    Per-zone stateful mixed thermal behavior.
      OUTSIDE 21-24 : recovery (22/22) or occasional noisy recovery.
      INSIDE  21-24 : idle (16/28) / random-inactive band / active target,
                      held for THERMAL_POLICY_HOLD_STEPS.
    Leaving the band immediately discards any held exploratory behavior.
    """
    memory = policy_memory["thermal_exploration"][zone]
    inside_comfort = COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH

    # ---- outside comfort: recovery takes precedence ----
    if not inside_comfort:
        memory["remaining"] = 0
        if EXPLORATION_ENABLED and rng.random() < OUTSIDE_NOISY_TARGET_PROB:
            target = THERMAL_TARGET_SP + rng.normal(0.0, OUTSIDE_TARGET_NOISE_STD_C)
            target = clamp(target, THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
            mode = "noisy_recovery"
        else:
            target = THERMAL_TARGET_SP
            mode = "expert_recovery"
        memory["heating_sp"] = target
        memory["cooling_sp"] = target
        memory["mode"] = mode
        return target, target, mode

    # ---- inside comfort: hold prior exploratory behavior ----
    if memory["remaining"] > 0:
        memory["remaining"] -= 1
        return memory["heating_sp"], memory["cooling_sp"], memory["mode"]

    # exploration disabled -> plain idle expert inside the band
    if not EXPLORATION_ENABLED:
        memory["heating_sp"] = HEATING_SETPOINT_RANGE[0]
        memory["cooling_sp"] = COOLING_SETPOINT_RANGE[1]
        memory["mode"] = "idle"
        memory["remaining"] = 0
        return HEATING_SETPOINT_RANGE[0], COOLING_SETPOINT_RANGE[1], "idle"

    # select a new inside-band behavior
    sample = rng.random()
    if sample < INSIDE_IDLE_PROB:
        heating_sp = HEATING_SETPOINT_RANGE[0]   # 16
        cooling_sp = COOLING_SETPOINT_RANGE[1]   # 28
        mode = "idle"
    elif sample < (INSIDE_IDLE_PROB + INSIDE_RANDOM_BAND_PROB):
        maximum_heating_sp = min(zone_temp - THERMAL_INACTIVE_MARGIN_C,
                                 HEATING_SETPOINT_RANGE[1])
        minimum_cooling_sp = max(zone_temp + THERMAL_INACTIVE_MARGIN_C,
                                 COOLING_SETPOINT_RANGE[0])
        heating_sp = rng.uniform(HEATING_SETPOINT_RANGE[0], maximum_heating_sp)
        cooling_sp = rng.uniform(minimum_cooling_sp, COOLING_SETPOINT_RANGE[1])
        mode = "random_inactive_band"
    else:
        target = rng.uniform(THERMAL_RANDOM_TARGET_LOW, THERMAL_RANDOM_TARGET_HIGH)
        heating_sp = target
        cooling_sp = target
        mode = "active_exploration"

    memory["heating_sp"] = float(heating_sp)
    memory["cooling_sp"] = float(cooling_sp)
    memory["mode"] = mode
    memory["remaining"] = THERMAL_POLICY_HOLD_STEPS - 1
    return float(heating_sp), float(cooling_sp), mode


def apply_sdar_repeat_exploration(
    selected_g,
    selected_l,
    selected_h,
    selected_c,
    current_obs,
    thermal_mode,
    thermal_mem_snapshot,
):
    """
    Occasionally repeat the previous executed action.

    This creates genuine SDAR b=0 samples. Current action-value noise creates
    b=1 samples, so together they improve SDAR update/repeat coverage.

    When a THERMAL repeat fires, the exploratory thermal call has already mutated
    policy_memory["thermal_exploration"][zone] (remaining/mode/held setpoints) and
    set thermal_mode[zone] to the NEW, now-unexecuted policy. So on a thermal
    repeat we also restore that zone's thermal memory to its pre-call snapshot and
    report thermal_mode = "sdar_repeat", keeping mode/memory consistent with what
    actually executed.
    """
    forced_repeat_mask = np.zeros(ACTION_DIM, dtype=np.float32)

    # No previous action exists on the first timestep
    if not EXPLORATION_ENABLED or policy_memory["prev_action_vec"] is None:
        return (
            selected_g,
            selected_l,
            selected_h,
            selected_c,
            forced_repeat_mask,
        )

    # Glazing: independent decision for each perimeter zone
    for i, zone in enumerate(WINDOW_ORDER):
        if rng.random() < SDAR_GLAZING_REPEAT_PROB:
            selected_g[zone] = policy_memory["glazing"][zone]
            forced_repeat_mask[i] = 1.0

    # Lighting: independent decision for all five zones
    for i, zone in enumerate(ZONE_ORDER):
        if rng.random() < SDAR_LIGHTING_REPEAT_PROB:
            selected_l[zone] = policy_memory["lighting"][zone]
            forced_repeat_mask[4 + i] = 1.0

    # Thermal: repeat heating and cooling jointly for each zone.
    # Only allow this inside the comfort band; recovery is never blocked.
    for i, zone in enumerate(ZONE_ORDER):
        zone_temp = current_obs["temp"][zone]
        inside_comfort = COMFORT_BAND_LOW <= zone_temp <= COMFORT_BAND_HIGH

        if inside_comfort and rng.random() < SDAR_THERMAL_REPEAT_PROB:
            selected_h[zone] = policy_memory["heating"][zone]
            selected_c[zone] = policy_memory["cooling"][zone]

            forced_repeat_mask[9 + i] = 1.0
            forced_repeat_mask[14 + i] = 1.0

            # restore the thermal memory that the exploratory call mutated, and
            # report the repeat as its own mode (consistent with what executed).
            policy_memory["thermal_exploration"][zone] = dict(
                thermal_mem_snapshot[zone])
            thermal_mode[zone] = "sdar_repeat"

    return (
        selected_g,
        selected_l,
        selected_h,
        selected_c,
        forced_repeat_mask,
    )


# ============================================================
# DECISION DISPATCH — expert action -> exploration -> memory update
# ============================================================

def choose_expert_actions(current_obs, dt):
    policy_memory["step"] += 1
    update_glazing  = (policy_memory["step"] % GLAZING_UPDATE_STEPS == 0)
    update_lighting = (policy_memory["step"] % LIGHTING_UPDATE_STEPS == 0)
    update_thermal  = (policy_memory["step"] % THERMAL_UPDATE_STEPS == 0)

    occupied = is_occupied(dt)
    cooling_active = policy_memory["cooling_active"]
    heating_active = policy_memory["heating_active"]

    selected_g, selected_l, selected_h, selected_c = {}, {}, {}, {}
    base_h, base_c = {}, {}   # deterministic thermal expert (for exploration mask)
    thermal_mode = {}
    thermal_mem_snapshot = {}  # pre-call thermal_exploration memory (for SDAR repeat restore)

    for zone, _window in WINDOW_ZONES_MID.items():
        wpi = current_obs["wpi"][zone]
        ext_irr = current_obs["ext_irr"][zone]
        current_g = policy_memory["glazing"][zone]

        if update_glazing:
            g = choose_glazing_expert(
                occupied=occupied,
                cooling_active=cooling_active,
                heating_active=heating_active,
                ext_irr=ext_irr,
                wpi=wpi,
                current_glazing=current_g,
            )
        else:
            g = current_g
        selected_g[zone] = g

        if update_lighting:
            target_p = choose_perimeter_lighting_expert(
                occupied=occupied,
                current_wpi=wpi,
                zone_max_w=LIGHTING_INSTALLED_W[zone],
                prev_power=policy_memory["lighting"][zone],
                action_deadband_w=LIGHTING_ACTION_DEADBAND_W,
            )
            selected_l[zone] = clamp(target_p, *LIGHTING_POWER_RANGE)
        else:
            selected_l[zone] = policy_memory["lighting"][zone]

        # Deterministic thermal expert (logging baseline for the mask)
        h_expert, c_expert = choose_hvac_zone(
            dt=dt,
            zone_temp=current_obs["temp"][zone],
            zone_ext_irr=current_obs["ext_irr"][zone],
            heating_active=heating_active,
            cooling_active=cooling_active,
            has_window=True,
        )
        base_h[zone] = h_expert
        base_c[zone] = c_expert

        # Thermal — per-zone mixed exploratory policy (actually executed).
        # Snapshot the pre-call thermal memory so a later SDAR forced-repeat can
        # restore it (choose_exploratory_hvac_zone mutates it in place).
        thermal_mem_snapshot[zone] = dict(policy_memory["thermal_exploration"][zone])
        if update_thermal:
            h_z, c_z, m_z = choose_exploratory_hvac_zone(
                zone=zone, zone_temp=current_obs["temp"][zone])
            selected_h[zone] = h_z
            selected_c[zone] = c_z
            thermal_mode[zone] = m_z
        else:
            selected_h[zone] = policy_memory["heating"][zone]
            selected_c[zone] = policy_memory["cooling"][zone]
            thermal_mode[zone] = policy_memory["thermal_exploration"][zone]["mode"]

    core_zone = "Core_mid"

    if update_lighting:
        selected_l[core_zone] = clamp(choose_core_lighting_expert(occupied),
                                      *LIGHTING_POWER_RANGE)
    else:
        selected_l[core_zone] = policy_memory["lighting"][core_zone]

    # Deterministic thermal expert for the core (logging baseline)
    h_expert_c, c_expert_c = choose_hvac_zone(
        dt=dt,
        zone_temp=current_obs["temp"][core_zone],
        zone_ext_irr=0.0,
        heating_active=heating_active,
        cooling_active=cooling_active,
        has_window=False,
    )
    base_h[core_zone] = h_expert_c
    base_c[core_zone] = c_expert_c

    thermal_mem_snapshot[core_zone] = dict(policy_memory["thermal_exploration"][core_zone])
    if update_thermal:
        h_c, c_c, m_c = choose_exploratory_hvac_zone(
            zone=core_zone, zone_temp=current_obs["temp"][core_zone])
        selected_h[core_zone] = h_c
        selected_c[core_zone] = c_c
        thermal_mode[core_zone] = m_c
    else:
        selected_h[core_zone] = policy_memory["heating"][core_zone]
        selected_c[core_zone] = policy_memory["cooling"][core_zone]
        thermal_mode[core_zone] = policy_memory["thermal_exploration"][core_zone]["mode"]

    # ---- preserve deterministic expert decision ----
    # glazing/lighting: pre-exploration values (exploration applied just below);
    # thermal: the deterministic base_h/base_c (selected_h/c already hold the
    # EXECUTED mixed-policy setpoints, so using them would zero the thermal mask).
    expert_g = selected_g.copy()
    expert_l = selected_l.copy()
    expert_h = base_h.copy()
    expert_c = base_c.copy()

    # ---- apply glazing + lighting exploration ----
    selected_g, glazing_explored = apply_glazing_exploration(
        selected_g=selected_g, current_obs=current_obs, update_glazing=update_glazing)
    selected_l, lighting_explored = apply_lighting_exploration(
        selected_l=selected_l, current_obs=current_obs,
        occupied=occupied, update_lighting=update_lighting)

    # ---- SDAR temporal-repeat exploration (forces genuine b=0 samples) ----
    # Reads policy_memory (the PREVIOUS executed action) before it is updated
    # below, so a forced repeat copies last step's executed value.
    (
        selected_g,
        selected_l,
        selected_h,
        selected_c,
        sdar_forced_repeat,
    ) = apply_sdar_repeat_exploration(
        selected_g,
        selected_l,
        selected_h,
        selected_c,
        current_obs,
        thermal_mode,
        thermal_mem_snapshot,
    )

    # ---- update memory using EXECUTED (noisy) actions ----
    for zone in WINDOW_ZONES_MID:
        policy_memory["glazing"][zone] = selected_g[zone]
    for zone in ZONES_MID:
        policy_memory["lighting"][zone] = selected_l[zone]
        policy_memory["heating"][zone] = selected_h[zone]
        policy_memory["cooling"][zone] = selected_c[zone]

    # ---- exploration logging (expert vs executed) ----
    expert_vec = encode_action_vec(expert_g, expert_l, expert_h, expert_c)
    executed_vec = encode_action_vec(selected_g, selected_l, selected_h, selected_c)
    exploration_mask = (np.abs(executed_vec - expert_vec) > MASK_EPS).astype(np.float32)

    exploration_data["expert_action"].append(expert_vec)
    exploration_data["executed_action"].append(executed_vec)
    exploration_data["exploration_mask"].append(exploration_mask)
    exploration_data["glazing_explored"].append(glazing_explored)
    exploration_data["lighting_explored"].append(lighting_explored)
    exploration_data["thermal_mode"].append(dict(thermal_mode))
    exploration_data["epsilon"].append(GLAZING_EPSILON)
    exploration_data["sdar_forced_repeat"].append(sdar_forced_repeat.copy())

    return selected_g, selected_l, selected_h, selected_c


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    dt = eps.get_datetime()

    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = policy_memory["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec  = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec    = get_meter_value_cached(state, "Electricity:HVAC")

    policy_memory["cooling_active"] = (cooling_elec > COIL_ACTIVE_EPS)
    policy_memory["heating_active"] = (heating_elec > COIL_ACTIVE_EPS)

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)
    temporal_data['expert_id'].append(EXPERT_ID)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
            'Core_mid': 0.0,
        },
        "energy": {
            'lighting_rate': (light_mid_1 + light_mid_2 + light_mid_3
                              + light_mid_4 + light_core_mid),
            'hvac_demand_rate': hvac_electricity_demand_rate,
        },
        "wpi": current_wpi,
        "temporal": tf,
    }

    sel_g, sel_l, sel_h, sel_c = choose_expert_actions(current_obs, dt)

    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=sel_g[zone])
        eps.actuate_lighting_power(light=zone, value=sel_l[zone])
        eps.actuate_heating_setpoint(zone=zone, value=sel_h[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=sel_c[zone])

        glazing_state_data[zone].append(sel_g[zone])
        lighting_power_data[zone].append(sel_l[zone])
        heating_setpoint_data[zone].append(sel_h[zone])
        cooling_setpoint_data[zone].append(sel_c[zone])

    eps.actuate_lighting_power(light="Core_mid", value=sel_l['Core_mid'])
    eps.actuate_heating_setpoint(zone="Core_mid", value=sel_h['Core_mid'])
    eps.actuate_cooling_setpoint(zone="Core_mid", value=sel_c['Core_mid'])

    lighting_power_data['Core_mid'].append(sel_l['Core_mid'])
    heating_setpoint_data['Core_mid'].append(sel_h['Core_mid'])
    cooling_setpoint_data['Core_mid'].append(sel_c['Core_mid'])

    # ---- SDAR action-mask logging (uses EXECUTED action) ----
    a_vec = encode_action_vec(sel_g, sel_l, sel_h, sel_c)
    prev_a_vec = policy_memory["prev_action_vec"]

    if prev_a_vec is None:
        b_mask = np.ones(ACTION_DIM, dtype=np.float32)
        prev_for_log = a_vec.copy()
    else:
        b_mask = (np.abs(a_vec - prev_a_vec) > MASK_EPS).astype(np.float32)
        prev_for_log = prev_a_vec.copy()

    a_mix = ((1.0 - b_mask) * prev_for_log
             + b_mask * ACTION_MASK_VALUE).astype(np.float32)

    action_vector_data["action"].append(a_vec.copy())
    action_vector_data["prev_action"].append(prev_for_log)
    action_vector_data["selection_mask"].append(b_mask)
    action_vector_data["action_mix"].append(a_mix)

    policy_memory["prev_action_vec"] = a_vec

In [ ]:
"""
EVALUATION CALLBACK — run a TRAINED SDAR-IQL agent in EnergyPlus
================================================================

This is the evaluation counterpart of the v12 expert-collection controller.
Instead of running the expert rules + exploration, each step it:

    read EnergyPlus  ->  build 29-dim RAW observation  ->  agent.get_action()
                     ->  actuate the agent's physical action  ->  log

All the machine-learning decode (obs normalization, GRU windowing, SDAR mask
persistence / action-mix, glazing decode, per-zone lighting decode) lives inside
`load_and_act` from the training script. We deliberately do NOT re-implement any
of it here, so the rollout uses exactly the transforms the checkpoint was trained
with. This callback only assembles the raw observation and forwards the action.

The agent is stateful across the episode (it keeps its own SEQ_LEN observation
window and previous-action vector, and the selector decides which dims to change
each step). So we call `get_action` once per timestep and let it manage SDAR
persistence internally — there is no mask/prev/mix bookkeeping to do here.

------------------------------------------------------------------------------
TWO ASSUMPTIONS THAT MUST MATCH YOUR DATASET BUILDER (verify these):

1. OBSERVATION LAYOUT / ORDER. The 29-dim vector is built in the exact order the
   training obs used (see build_observation and the training script's header):
       [0:5]   zone temps        in ZONE_ORDER
       [5:10]  lighting rates    in ZONE_ORDER   (sensor "Lights Electricity Rate", W)
       [10:14] transmitted solar in WINDOW_ORDER (W)
       [14:18] exterior irradiance in WINDOW_ORDER (W/m^2)
       [18:22] WPI               in WINDOW_ORDER (lux)
       [22]    facility HVAC electricity demand rate (W)
       [23:27] hour_sin, hour_cos, doy_sin, doy_cos
       [27:29] is_occupied, is_preconditioning
   If your NPZ builder used a different zone/window order, change ZONE_ORDER /
   WINDOW_ORDER below to match — otherwise the observation is permuted.

2. RAW UNITS. The NPZ "observations" must be RAW physical values in the units
   above (not pre-scaled). The checkpoint's stored Normalizer z-scores them; if
   your builder applied any extra scaling before saving, replicate it here.
------------------------------------------------------------------------------
"""

import numpy as np

from sdar_iql_train import load_and_act, BEST_CKPT_PATH


# ============================================================
# EVAL CONFIG
# ============================================================

# Checkpoint to evaluate. Point this at the run you want to roll out
# (e.g. the BEST or FINAL checkpoint from training).
EVAL_CHECKPOINT = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_checkpoint_epoch100.pt"

# Deterministic policy for evaluation: selector thresholds at 0.5, actor uses
# argmax glazing + mean continuous action. Set False for stochastic rollouts.
DETERMINISTIC = True

OBS_DIM = 29


# ============================================================
# BUILDING CONFIG (mirrors the v12 controller)
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']

# Order used to lay out the observation vector. MUST match the dataset builder.
ZONE_ORDER = list(ZONES_MID)                    # 5 zones (temps, lights, heat, cool)
WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())    # 4 windows (solar, ext_irr, wpi, glazing)

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

# Occupancy schedule (only used to build the two occupancy obs flags).
OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

# Initial glazing in effect before the first agent action (for the first WPI).
INITIAL_GLAZING = 'sr2_ec01'


# ============================================================
# AGENT (lazy-loaded on first callback so the checkpoint only
# needs to exist when the simulation runs, not at import time)
# ============================================================

_AGENT = {"get_action": None, "reset": None, "loaded_from": None}


def setup(checkpoint_path=None):
    """Load the trained agent and reset its internal SDAR/GRU state. Called
    automatically on the first callback; call it explicitly between episodes."""
    path = checkpoint_path if checkpoint_path is not None else EVAL_CHECKPOINT
    get_action, reset = load_and_act(path)
    reset()
    _AGENT["get_action"] = get_action
    _AGENT["reset"] = reset
    _AGENT["loaded_from"] = str(path)
    eval_state["step"] = 0
    eval_state["glazing"] = {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID}
    print(f"[eval callback] agent loaded from {path}  (deterministic={DETERMINISTIC})")
    return get_action, reset


# ============================================================
# EVAL STATE
# ============================================================

eval_state = {
    "step": 0,
    # glazing currently in effect (set last step); used for this step's WPI.
    "glazing": {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID},
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None,
}


# ============================================================
# DATA STORAGE (same keys as the collection controller so your
# existing analysis/serialization keeps working)
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}

# Agent-specific evaluation log.
agent_data = {
    "observation": [],            # 29-dim raw obs fed to the agent
    "selection_mask": [],         # which of the 19 action dims the agent changed
    "executed_action_norm": [],   # normalized executed action returned by the agent
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def build_observation(current_obs, tf, hvac_demand_rate):
    """Assemble the 29-dim RAW observation in the training layout/order."""
    obs = np.zeros(OBS_DIM, dtype=np.float32)

    for i, zone in enumerate(ZONE_ORDER):
        obs[0 + i] = current_obs["temp"][zone]
    for i, zone in enumerate(ZONE_ORDER):
        obs[5 + i] = current_obs["light_rate"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[10 + i] = current_obs["solar"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[14 + i] = current_obs["ext_irr"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[18 + i] = current_obs["wpi"][zone]

    obs[22] = hvac_demand_rate

    obs[23] = tf["hour_sin"]
    obs[24] = tf["hour_cos"]
    obs[25] = tf["doy_sin"]
    obs[26] = tf["doy_cos"]
    obs[27] = float(tf["is_occupied"])
    obs[28] = float(tf["is_preconditioning"])

    return obs


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    if _AGENT["get_action"] is None:
        setup()

    dt = eps.get_datetime()

    # ---- read sensors (identical variables to the collection controller) ----
    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    # ---- WPI under the glazing currently in effect (set last step) ----
    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = eval_state["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    # ---- meters (logged for energy analysis) ----
    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec = get_meter_value_cached(state, "Electricity:HVAC")

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    # ---- temporal features + logging ----
    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    # ---- assemble observation dict ----
    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
        },
        "wpi": current_wpi,
    }

    raw_obs = build_observation(current_obs, tf, hvac_electricity_demand_rate)

    # ---- query the trained agent ----
    # force_first_update on the very first step mirrors the collection convention
    # (first transition has b=1 on all dims), so the agent commits an action
    # instead of persisting its zero-init defaults.
    first_step = (eval_state["step"] == 0)
    result = _AGENT["get_action"](
        raw_obs,
        deterministic=DETERMINISTIC,
        force_first_update=first_step,
    )
    eval_state["step"] += 1

    # ---- map agent output (lists in ZONE_ORDER / WINDOW_ORDER) to per-zone ----
    glz_by_zone = {WINDOW_ORDER[i]: result["glazing"][i] for i in range(len(WINDOW_ORDER))}
    light_by_zone = {ZONE_ORDER[i]: float(result["lighting_power"][i]) for i in range(len(ZONE_ORDER))}
    heat_by_zone = {ZONE_ORDER[i]: float(result["heating_sp"][i]) for i in range(len(ZONE_ORDER))}
    cool_by_zone = {ZONE_ORDER[i]: float(result["cooling_sp"][i]) for i in range(len(ZONE_ORDER))}

    # ---- actuate (perimeter zones: glazing + lighting + setpoints) ----
    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=glz_by_zone[zone])
        eps.actuate_lighting_power(light=zone, value=light_by_zone[zone])
        eps.actuate_heating_setpoint(zone=zone, value=heat_by_zone[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=cool_by_zone[zone])

        glazing_state_data[zone].append(glz_by_zone[zone])
        lighting_power_data[zone].append(light_by_zone[zone])
        heating_setpoint_data[zone].append(heat_by_zone[zone])
        cooling_setpoint_data[zone].append(cool_by_zone[zone])

    # ---- actuate core (no window) ----
    core_zone = "Core_mid"
    eps.actuate_lighting_power(light=core_zone, value=light_by_zone[core_zone])
    eps.actuate_heating_setpoint(zone=core_zone, value=heat_by_zone[core_zone])
    eps.actuate_cooling_setpoint(zone=core_zone, value=cool_by_zone[core_zone])

    lighting_power_data[core_zone].append(light_by_zone[core_zone])
    heating_setpoint_data[core_zone].append(heat_by_zone[core_zone])
    cooling_setpoint_data[core_zone].append(cool_by_zone[core_zone])

    # ---- glazing now in effect for next step's WPI ----
    for zone in WINDOW_ZONES_MID:
        eval_state["glazing"][zone] = glz_by_zone[zone]

    # ---- agent log ----
    agent_data["observation"].append(raw_obs.copy())
    agent_data["selection_mask"].append(np.asarray(result["selection_mask"], dtype=np.float32))
    agent_data["executed_action_norm"].append(
        np.asarray(result["executed_action_norm"], dtype=np.float32))

In [ ]:
"""
EVALUATION CALLBACK — trained SDAR-IQL agent in EnergyPlus  (PATCHED)
=====================================================================

Changes vs the previous version:
  * setup() requests the unoccupied-lights-off safeguard explicitly.
  * agent_data logs the policy action, the executed (safeguarded) action, both
    masks, and per-step safeguard flags, so the rerun can be audited.
  * a small end-of-run assertion helper (`verify_safeguard`) checks the
    invariant directly from the logs.

Nothing else changed: observation assembly, actuation, and sensor logging are
identical, so this rollout is comparable to the previous one except for the
safeguard.
"""

import numpy as np

from sdar_iql_train_3 import load_and_act, BEST_CKPT_PATH


# ============================================================
# EVAL CONFIG
# ============================================================

EVAL_CHECKPOINT = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3_checkpoint_epoch500.pt"

DETERMINISTIC = True

# Hard operational constraint: no lighting in an unoccupied building.
ENFORCE_UNOCCUPIED_LIGHTS_OFF = True

# Keep the agent's internal prev_action in sync with the deadband-corrected
# cooling setpoint that is actually actuated. Set False to isolate the
# lighting fix when comparing against the previous rollout.
SYNC_PREV_WITH_DEADBAND = True

OBS_DIM = 29


# ============================================================
# BUILDING CONFIG (mirrors the v12 controller)
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']

ZONE_ORDER = list(ZONES_MID)
WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

INITIAL_GLAZING = 'sr2_ec01'


# ============================================================
# AGENT
# ============================================================

_AGENT = {"get_action": None, "reset": None, "loaded_from": None}


def setup(checkpoint_path=None):
    """Load the trained agent and reset its internal SDAR/GRU state."""
    path = checkpoint_path if checkpoint_path is not None else EVAL_CHECKPOINT
    get_action, reset = load_and_act(
        path,
        enforce_unoccupied_lights_off=ENFORCE_UNOCCUPIED_LIGHTS_OFF,
        sync_prev_with_deadband=SYNC_PREV_WITH_DEADBAND,
    )
    reset()
    _AGENT["get_action"] = get_action
    _AGENT["reset"] = reset
    _AGENT["loaded_from"] = str(path)

    eval_state["step"] = 0
    eval_state["glazing"] = {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID}

    for k in agent_data:
        if isinstance(agent_data[k], list):
            agent_data[k].clear()

    print(f"[eval callback] agent loaded from {path}  (deterministic={DETERMINISTIC})")
    print(f"[eval callback] unoccupied lights-off safeguard = {ENFORCE_UNOCCUPIED_LIGHTS_OFF}")
    return get_action, reset


# ============================================================
# EVAL STATE
# ============================================================

eval_state = {
    "step": 0,
    "glazing": {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID},
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None,
}


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}

agent_data = {
    "observation": [],
    "selection_logits": [],
    "selection_probability": [],
    "selection_mask": [],              # policy mask b
    "executed_selection_mask": [],     # mask consistent with executed action
    "policy_executed_action_norm": [], # before safeguard
    "executed_action_norm": [],        # after safeguard (actuated)
    "override_dims": [],               # per-dim: safeguard/deadband changed it
    "lighting_safety_active": [],
    "lighting_safety_override": [],
    "deadband_applied": [],
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def build_observation(current_obs, tf, hvac_demand_rate):
    """Assemble the 29-dim RAW observation in the training layout/order."""
    obs = np.zeros(OBS_DIM, dtype=np.float32)

    for i, zone in enumerate(ZONE_ORDER):
        obs[0 + i] = current_obs["temp"][zone]
    for i, zone in enumerate(ZONE_ORDER):
        obs[5 + i] = current_obs["light_rate"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[10 + i] = current_obs["solar"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[14 + i] = current_obs["ext_irr"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[18 + i] = current_obs["wpi"][zone]

    obs[22] = hvac_demand_rate

    obs[23] = tf["hour_sin"]
    obs[24] = tf["hour_cos"]
    obs[25] = tf["doy_sin"]
    obs[26] = tf["doy_cos"]
    obs[27] = float(tf["is_occupied"])
    obs[28] = float(tf["is_preconditioning"])

    return obs


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    if _AGENT["get_action"] is None:
        setup()

    dt = eps.get_datetime()

    # ---- read sensors ----
    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    # ---- WPI under the glazing currently in effect ----
    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = eval_state["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    # ---- meters ----
    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec = get_meter_value_cached(state, "Electricity:HVAC")

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    # ---- temporal + sensor logging ----
    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
        },
        "wpi": current_wpi,
    }

    raw_obs = build_observation(current_obs, tf, hvac_electricity_demand_rate)

    # ---- query the trained agent ----
    first_step = (eval_state["step"] == 0)
    result = _AGENT["get_action"](
        raw_obs,
        deterministic=DETERMINISTIC,
        force_first_update=first_step,
    )
    eval_state["step"] += 1

    glz_by_zone = {WINDOW_ORDER[i]: result["glazing"][i] for i in range(len(WINDOW_ORDER))}
    light_by_zone = {ZONE_ORDER[i]: float(result["lighting_power"][i]) for i in range(len(ZONE_ORDER))}
    heat_by_zone = {ZONE_ORDER[i]: float(result["heating_sp"][i]) for i in range(len(ZONE_ORDER))}
    cool_by_zone = {ZONE_ORDER[i]: float(result["cooling_sp"][i]) for i in range(len(ZONE_ORDER))}

    # ---- actuate perimeter zones ----
    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=glz_by_zone[zone])
        eps.actuate_lighting_power(light=zone, value=light_by_zone[zone])
        eps.actuate_heating_setpoint(zone=zone, value=heat_by_zone[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=cool_by_zone[zone])

        glazing_state_data[zone].append(glz_by_zone[zone])
        lighting_power_data[zone].append(light_by_zone[zone])
        heating_setpoint_data[zone].append(heat_by_zone[zone])
        cooling_setpoint_data[zone].append(cool_by_zone[zone])

    # ---- actuate core ----
    core_zone = "Core_mid"
    eps.actuate_lighting_power(light=core_zone, value=light_by_zone[core_zone])
    eps.actuate_heating_setpoint(zone=core_zone, value=heat_by_zone[core_zone])
    eps.actuate_cooling_setpoint(zone=core_zone, value=cool_by_zone[core_zone])

    lighting_power_data[core_zone].append(light_by_zone[core_zone])
    heating_setpoint_data[core_zone].append(heat_by_zone[core_zone])
    cooling_setpoint_data[core_zone].append(cool_by_zone[core_zone])

    for zone in WINDOW_ZONES_MID:
        eval_state["glazing"][zone] = glz_by_zone[zone]

    # ---- agent log ----
    agent_data["observation"].append(raw_obs.copy())
    agent_data["selection_mask"].append(
        np.asarray(result["selection_mask"], dtype=np.float32))
    agent_data["executed_selection_mask"].append(
        np.asarray(result["executed_selection_mask"], dtype=np.float32))
    agent_data["policy_executed_action_norm"].append(
        np.asarray(result["policy_executed_action_norm"], dtype=np.float32))
    agent_data["executed_action_norm"].append(
        np.asarray(result["executed_action_norm"], dtype=np.float32))
    agent_data["override_dims"].append(
        np.asarray(result["override_dims"], dtype=np.float32))
    agent_data["deadband_applied"].append(
        np.asarray(result["deadband_applied"], dtype=np.float32))
    agent_data["lighting_safety_active"].append(bool(result["lighting_safety_active"]))
    agent_data["lighting_safety_override"].append(bool(result["lighting_safety_override"]))

    agent_data["observation"].append(
    raw_obs.copy()
)

    agent_data["selection_logits"].append(
    np.asarray(
        result["selection_logits"],
        dtype=np.float32,
    ).copy()
)

    agent_data["selection_probability"].append(
    np.asarray(
        result["selection_probability"],
        dtype=np.float32,
    ).copy()
)

    agent_data["selection_mask"].append(
    np.asarray(
        result["selection_mask"],
        dtype=np.float32,
    ).copy()
)


# ============================================================
# POST-RUN VERIFICATION
# ============================================================

def verify_safeguard(verbose=True):
    """Check the invariant directly from the logs. Call after the run."""
    obs = np.array(agent_data["observation"])
    exe = np.array(agent_data["executed_action_norm"])
    pol = np.array(agent_data["policy_executed_action_norm"])
    unocc = obs[:, 27] < 0.5

    light_exe = exe[unocc, 4:9]
    light_pol = pol[unocc, 4:9]
    ok = np.allclose(light_exe, -1.0, atol=1e-6)

    # commanded watts from the actuation log
    lp = np.array([lighting_power_data[z] for z in ZONE_ORDER]).T
    lp_unocc = lp[unocc]
    kwh_unocc = lp_unocc.sum() * (1 / 6) / 1000.0
    kwh_total = lp.sum() * (1 / 6) / 1000.0

    # what the raw policy WOULD have done (counterfactual saving)
    would_w = 0.5 * (light_pol + 1.0)
    installed = np.array([LIGHTING_INSTALLED_W[z] for z in ZONE_ORDER])
    kwh_prevented = (would_w * installed).sum() * (1 / 6) / 1000.0

    if verbose:
        print("=== safeguard verification ===")
        print(f"  unoccupied steps                : {unocc.sum()}")
        print(f"  executed lighting == -1 (0 W)   : {ok}")
        print(f"  override fired on               : {int(np.sum(agent_data['lighting_safety_override']))} steps")
        print(f"  unoccupied lighting energy      : {kwh_unocc:,.1f} kWh   (target 0.0)")
        print(f"  total lighting energy           : {kwh_total:,.1f} kWh")
        print(f"  energy the policy WOULD have used unoccupied: {kwh_prevented:,.1f} kWh")
    return ok, kwh_unocc, kwh_prevented

In [ ]:
"""
EVALUATION CALLBACK â€” trained SDAR-IQL agent in EnergyPlus  (PATCHED)
=====================================================================

Changes vs the previous version:
  * setup() requests the unoccupied-lights-off safeguard explicitly.
  * the learned Bernoulli selector is sampled with a fixed seed, while the
    glazing, lighting, and HVAC proposal actor remains deterministic.
  * agent_data logs the policy action, the executed (safeguarded) action, both
    masks, selector logits/probabilities, and per-step safeguard flags exactly
    once per timestep, so the rerun can be audited.
  * a small end-of-run assertion helper (`verify_safeguard`) checks the
    invariant directly from the logs.

Observation assembly, actuation, and sensor logging are otherwise unchanged.
This rollout intentionally differs in both safeguard execution and selector
realization, and the selector seed makes the result reproducible.
"""

import numpy as np

from sdar_iql_train_3_updated import load_and_act, BEST_CKPT_PATH


# ============================================================
# EVAL CONFIG
# ============================================================

EVAL_CHECKPOINT = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3_checkpoint_epoch100.pt"

# Sample the learned Bernoulli update selector, but keep the proposed
# glazing/lighting/HVAC actions deterministic.
SELECTOR_MODE = "sample"
PROPOSAL_DETERMINISTIC = True
SELECTOR_SEED = 20260728

# Hard operational constraint: no lighting in an unoccupied building.
ENFORCE_UNOCCUPIED_LIGHTS_OFF = True

# Keep the agent's internal prev_action in sync with the deadband-corrected
# cooling setpoint that is actually actuated. Set False to isolate the
# lighting fix when comparing against the previous rollout.
SYNC_PREV_WITH_DEADBAND = True

OBS_DIM = 29


# ============================================================
# BUILDING CONFIG (mirrors the v12 controller)
# ============================================================

ZONES_MID = [
    'Perimeter_mid_ZN_1', 'Perimeter_mid_ZN_2', 'Perimeter_mid_ZN_3',
    'Perimeter_mid_ZN_4', 'Core_mid',
]

WINDOW_ZONES_MID = {
    'Perimeter_mid_ZN_1': 'Perimeter_mid_ZN_1_Wall_South_Window',
    'Perimeter_mid_ZN_2': 'Perimeter_mid_ZN_2_Wall_East_Window',
    'Perimeter_mid_ZN_3': 'Perimeter_mid_ZN_3_Wall_North_Window',
    'Perimeter_mid_ZN_4': 'Perimeter_mid_ZN_4_Wall_West_Window',
}

AVAILABLE_GLAZING_STATES = ['sr2_ec01', 'sr2_ec02', 'sr2_ec03', 'sr2_ec04']

ZONE_ORDER = list(ZONES_MID)
WINDOW_ORDER = list(WINDOW_ZONES_MID.keys())

LIGHTING_INSTALLED_W = {
    'Perimeter_mid_ZN_1': 2231.0,
    'Perimeter_mid_ZN_2': 2231.0,
    'Perimeter_mid_ZN_3': 1412.0,
    'Perimeter_mid_ZN_4': 1412.0,
    'Core_mid':          10586.0,
}

OCCUPIED_START = 8
OCCUPIED_END = 16
PRECONDITION_LEAD_HRS = 1

INITIAL_GLAZING = 'sr2_ec01'


# ============================================================
# AGENT
# ============================================================

_AGENT = {"get_action": None, "reset": None, "loaded_from": None}


def setup(checkpoint_path=None):
    """Load the trained agent and reset its internal SDAR/GRU state."""
    path = checkpoint_path if checkpoint_path is not None else EVAL_CHECKPOINT
    get_action, reset = load_and_act(
        path,
        enforce_unoccupied_lights_off=ENFORCE_UNOCCUPIED_LIGHTS_OFF,
        sync_prev_with_deadband=SYNC_PREV_WITH_DEADBAND,
        selector_mode=SELECTOR_MODE,
        proposal_deterministic=PROPOSAL_DETERMINISTIC,
        selector_seed=SELECTOR_SEED,
    )
    reset()
    _AGENT["get_action"] = get_action
    _AGENT["reset"] = reset
    _AGENT["loaded_from"] = str(path)

    eval_state["step"] = 0
    eval_state["glazing"] = {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID}

    for k in agent_data:
        if isinstance(agent_data[k], list):
            agent_data[k].clear()

    print(
        f"[eval callback] agent loaded from {path}  "
        f"(selector_mode={SELECTOR_MODE}, "
        f"proposal_deterministic={PROPOSAL_DETERMINISTIC}, "
        f"selector_seed={SELECTOR_SEED})"
    )
    print(f"[eval callback] unoccupied lights-off safeguard = {ENFORCE_UNOCCUPIED_LIGHTS_OFF}")
    return get_action, reset


# ============================================================
# EVAL STATE
# ============================================================

eval_state = {
    "step": 0,
    "glazing": {z: INITIAL_GLAZING for z in WINDOW_ZONES_MID},
}

meter_handles = {
    "Heating:Electricity": None,
    "Cooling:Electricity": None,
    "InteriorLights:Electricity": None,
    "Electricity:HVAC": None,
}


# ============================================================
# DATA STORAGE
# ============================================================

lighting_data = {zone: [] for zone in ZONES_MID}
temperature_data = {zone: [] for zone in ZONES_MID}
solar_data = {zone: [] for zone in WINDOW_ZONES_MID}
ext_irr_data = {zone: [] for zone in WINDOW_ZONES_MID}
wpi_data = {zone: [] for zone in WINDOW_ZONES_MID}

glazing_state_data = {zone: [] for zone in WINDOW_ZONES_MID}
lighting_power_data = {zone: [] for zone in ZONES_MID}
heating_setpoint_data = {zone: [] for zone in ZONES_MID}
cooling_setpoint_data = {zone: [] for zone in ZONES_MID}

meter_data = {
    'Heating:Electricity': [],
    'Cooling:Electricity': [],
    'InteriorLights:Electricity': [],
    'Electricity:HVAC': [],
}

temporal_data = {
    'hour': [], 'minute': [], 'day_of_year': [], 'weekday': [],
    'hour_sin': [], 'hour_cos': [], 'doy_sin': [], 'doy_cos': [],
    'is_occupied': [], 'is_preconditioning': [],
}

output_variable_data = {
    'HVAC_electricity_demand_rate': [],
}

agent_data = {
    "observation": [],
    "selection_logits": [],
    "selection_probability": [],
    "selection_mask": [],              # policy mask b
    "executed_selection_mask": [],     # mask consistent with executed action
    "policy_executed_action_norm": [], # before safeguard
    "executed_action_norm": [],        # after safeguard (actuated)
    "override_dims": [],               # per-dim: safeguard/deadband changed it
    "lighting_safety_active": [],
    "lighting_safety_override": [],
    "deadband_applied": [],
}


# ============================================================
# HELPERS
# ============================================================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def get_meter_value_cached(state, meter_name):
    if meter_handles[meter_name] is None:
        meter_handles[meter_name] = eps.api.exchange.get_meter_handle(state, meter_name)
    return eps.api.exchange.get_meter_value(state, meter_handles[meter_name])


def is_occupied(dt):
    return OCCUPIED_START <= dt.hour < OCCUPIED_END


def is_preconditioning(dt):
    return (OCCUPIED_START - PRECONDITION_LEAD_HRS) <= dt.hour < OCCUPIED_START


def compute_temporal_features(dt):
    hour = dt.hour
    minute = dt.minute
    doy = dt.timetuple().tm_yday
    weekday = dt.weekday()

    tod_frac = (hour + minute / 60.0) / 24.0
    hour_sin = float(np.sin(2.0 * np.pi * tod_frac))
    hour_cos = float(np.cos(2.0 * np.pi * tod_frac))

    doy_frac = (doy - 1) / 365.0
    doy_sin = float(np.sin(2.0 * np.pi * doy_frac))
    doy_cos = float(np.cos(2.0 * np.pi * doy_frac))

    return {
        'hour': hour, 'minute': minute, 'day_of_year': doy, 'weekday': weekday,
        'hour_sin': hour_sin, 'hour_cos': hour_cos,
        'doy_sin': doy_sin, 'doy_cos': doy_cos,
        'is_occupied': bool(is_occupied(dt)),
        'is_preconditioning': bool(is_preconditioning(dt)),
    }


def build_observation(current_obs, tf, hvac_demand_rate):
    """Assemble the 29-dim RAW observation in the training layout/order."""
    obs = np.zeros(OBS_DIM, dtype=np.float32)

    for i, zone in enumerate(ZONE_ORDER):
        obs[0 + i] = current_obs["temp"][zone]
    for i, zone in enumerate(ZONE_ORDER):
        obs[5 + i] = current_obs["light_rate"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[10 + i] = current_obs["solar"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[14 + i] = current_obs["ext_irr"][zone]
    for i, zone in enumerate(WINDOW_ORDER):
        obs[18 + i] = current_obs["wpi"][zone]

    obs[22] = hvac_demand_rate

    obs[23] = tf["hour_sin"]
    obs[24] = tf["hour_cos"]
    obs[25] = tf["doy_sin"]
    obs[26] = tf["doy_cos"]
    obs[27] = float(tf["is_occupied"])
    obs[28] = float(tf["is_preconditioning"])

    return obs


# ============================================================
# CALLBACK
# ============================================================

def callback_func(state):
    if not eps.api.exchange.api_data_fully_ready(state):
        return
    if eps.api.exchange.warmup_flag(state):
        return

    if _AGENT["get_action"] is None:
        setup()

    dt = eps.get_datetime()

    # ---- read sensors ----
    light_mid_1 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_1")
    temp_mid_1 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_1")
    solar_mid_1 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_1_Wall_South_Window")
    ext_irr_mid_1 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_1_Wall_South_Window")

    light_mid_2 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_2")
    temp_mid_2 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_2")
    solar_mid_2 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_2_Wall_East_Window")
    ext_irr_mid_2 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_2_Wall_East_Window")

    light_mid_3 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_3")
    temp_mid_3 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_3")
    solar_mid_3 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_3_Wall_North_Window")
    ext_irr_mid_3 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_3_Wall_North_Window")

    light_mid_4 = eps.get_variable_value(name="Lights Electricity Rate", key="Perimeter_mid_ZN_4")
    temp_mid_4 = eps.get_variable_value(name="Zone Mean Air Temperature", key="Perimeter_mid_ZN_4")
    solar_mid_4 = eps.get_variable_value(
        name="Surface Window Transmitted Solar Radiation Rate",
        key="Perimeter_mid_ZN_4_Wall_West_Window")
    ext_irr_mid_4 = eps.get_variable_value(
        name="Surface Outside Face Incident Solar Radiation Rate per Area",
        key="Perimeter_mid_ZN_4_Wall_West_Window")

    light_core_mid = eps.get_variable_value(name="Lights Electricity Rate", key="Core_mid")
    temp_core_mid = eps.get_variable_value(name="Zone Mean Air Temperature", key="Core_mid")
    hvac_electricity_demand_rate = eps.get_variable_value(
        name="Facility Total HVAC Electricity Demand Rate", key="Whole Building")

    # ---- WPI under the glazing currently in effect ----
    current_wpi = {}
    for zone, window in WINDOW_ZONES_MID.items():
        g_now = eval_state["glazing"][zone]
        current_wpi[zone] = float(
            eps.calculate_wpi(zone=zone, cfs_name={window: g_now}).mean()
        )

    # ---- meters ----
    heating_elec = get_meter_value_cached(state, "Heating:Electricity")
    cooling_elec = get_meter_value_cached(state, "Cooling:Electricity")
    lights_elec = get_meter_value_cached(state, "InteriorLights:Electricity")
    hvac_elec = get_meter_value_cached(state, "Electricity:HVAC")

    meter_data['Heating:Electricity'].append(heating_elec)
    meter_data['Cooling:Electricity'].append(cooling_elec)
    meter_data['InteriorLights:Electricity'].append(lights_elec)
    meter_data['Electricity:HVAC'].append(hvac_elec)
    output_variable_data['HVAC_electricity_demand_rate'].append(hvac_electricity_demand_rate)

    # ---- temporal + sensor logging ----
    tf = compute_temporal_features(dt)
    for k, v in tf.items():
        temporal_data[k].append(v)

    lighting_data['Perimeter_mid_ZN_1'].append(light_mid_1)
    lighting_data['Perimeter_mid_ZN_2'].append(light_mid_2)
    lighting_data['Perimeter_mid_ZN_3'].append(light_mid_3)
    lighting_data['Perimeter_mid_ZN_4'].append(light_mid_4)
    lighting_data['Core_mid'].append(light_core_mid)

    temperature_data['Perimeter_mid_ZN_1'].append(temp_mid_1)
    temperature_data['Perimeter_mid_ZN_2'].append(temp_mid_2)
    temperature_data['Perimeter_mid_ZN_3'].append(temp_mid_3)
    temperature_data['Perimeter_mid_ZN_4'].append(temp_mid_4)
    temperature_data['Core_mid'].append(temp_core_mid)

    solar_data['Perimeter_mid_ZN_1'].append(solar_mid_1)
    solar_data['Perimeter_mid_ZN_2'].append(solar_mid_2)
    solar_data['Perimeter_mid_ZN_3'].append(solar_mid_3)
    solar_data['Perimeter_mid_ZN_4'].append(solar_mid_4)

    ext_irr_data['Perimeter_mid_ZN_1'].append(ext_irr_mid_1)
    ext_irr_data['Perimeter_mid_ZN_2'].append(ext_irr_mid_2)
    ext_irr_data['Perimeter_mid_ZN_3'].append(ext_irr_mid_3)
    ext_irr_data['Perimeter_mid_ZN_4'].append(ext_irr_mid_4)

    wpi_data['Perimeter_mid_ZN_1'].append(current_wpi['Perimeter_mid_ZN_1'])
    wpi_data['Perimeter_mid_ZN_2'].append(current_wpi['Perimeter_mid_ZN_2'])
    wpi_data['Perimeter_mid_ZN_3'].append(current_wpi['Perimeter_mid_ZN_3'])
    wpi_data['Perimeter_mid_ZN_4'].append(current_wpi['Perimeter_mid_ZN_4'])

    current_obs = {
        "temp": {
            'Perimeter_mid_ZN_1': temp_mid_1, 'Perimeter_mid_ZN_2': temp_mid_2,
            'Perimeter_mid_ZN_3': temp_mid_3, 'Perimeter_mid_ZN_4': temp_mid_4,
            'Core_mid': temp_core_mid,
        },
        "light_rate": {
            'Perimeter_mid_ZN_1': light_mid_1, 'Perimeter_mid_ZN_2': light_mid_2,
            'Perimeter_mid_ZN_3': light_mid_3, 'Perimeter_mid_ZN_4': light_mid_4,
            'Core_mid': light_core_mid,
        },
        "solar": {
            'Perimeter_mid_ZN_1': solar_mid_1, 'Perimeter_mid_ZN_2': solar_mid_2,
            'Perimeter_mid_ZN_3': solar_mid_3, 'Perimeter_mid_ZN_4': solar_mid_4,
        },
        "ext_irr": {
            'Perimeter_mid_ZN_1': ext_irr_mid_1, 'Perimeter_mid_ZN_2': ext_irr_mid_2,
            'Perimeter_mid_ZN_3': ext_irr_mid_3, 'Perimeter_mid_ZN_4': ext_irr_mid_4,
        },
        "wpi": current_wpi,
    }

    raw_obs = build_observation(current_obs, tf, hvac_electricity_demand_rate)

    # ---- query the trained agent ----
    first_step = (eval_state["step"] == 0)
    result = _AGENT["get_action"](
        raw_obs,
        force_first_update=first_step,
    )
    eval_state["step"] += 1

    glz_by_zone = {WINDOW_ORDER[i]: result["glazing"][i] for i in range(len(WINDOW_ORDER))}
    light_by_zone = {ZONE_ORDER[i]: float(result["lighting_power"][i]) for i in range(len(ZONE_ORDER))}
    heat_by_zone = {ZONE_ORDER[i]: float(result["heating_sp"][i]) for i in range(len(ZONE_ORDER))}
    cool_by_zone = {ZONE_ORDER[i]: float(result["cooling_sp"][i]) for i in range(len(ZONE_ORDER))}

    # ---- actuate perimeter zones ----
    for zone, window in WINDOW_ZONES_MID.items():
        eps.actuate_cfs_state(window=window, cfs_state=glz_by_zone[zone])
        eps.actuate_lighting_power(light=zone, value=light_by_zone[zone])
        eps.actuate_heating_setpoint(zone=zone, value=heat_by_zone[zone])
        eps.actuate_cooling_setpoint(zone=zone, value=cool_by_zone[zone])

        glazing_state_data[zone].append(glz_by_zone[zone])
        lighting_power_data[zone].append(light_by_zone[zone])
        heating_setpoint_data[zone].append(heat_by_zone[zone])
        cooling_setpoint_data[zone].append(cool_by_zone[zone])

    # ---- actuate core ----
    core_zone = "Core_mid"
    eps.actuate_lighting_power(light=core_zone, value=light_by_zone[core_zone])
    eps.actuate_heating_setpoint(zone=core_zone, value=heat_by_zone[core_zone])
    eps.actuate_cooling_setpoint(zone=core_zone, value=cool_by_zone[core_zone])

    lighting_power_data[core_zone].append(light_by_zone[core_zone])
    heating_setpoint_data[core_zone].append(heat_by_zone[core_zone])
    cooling_setpoint_data[core_zone].append(cool_by_zone[core_zone])

    for zone in WINDOW_ZONES_MID:
        eval_state["glazing"][zone] = glz_by_zone[zone]

    # ---- agent log ----
    agent_data["observation"].append(raw_obs.copy())
    agent_data["selection_logits"].append(
        np.asarray(
            result["selection_logits"],
            dtype=np.float32,
        ).copy()
    )
    agent_data["selection_probability"].append(
        np.asarray(
            result["selection_probability"],
            dtype=np.float32,
        ).copy()
    )
    agent_data["selection_mask"].append(
        np.asarray(result["selection_mask"], dtype=np.float32))
    agent_data["executed_selection_mask"].append(
        np.asarray(result["executed_selection_mask"], dtype=np.float32))
    agent_data["policy_executed_action_norm"].append(
        np.asarray(result["policy_executed_action_norm"], dtype=np.float32))
    agent_data["executed_action_norm"].append(
        np.asarray(result["executed_action_norm"], dtype=np.float32))
    agent_data["override_dims"].append(
        np.asarray(result["override_dims"], dtype=np.float32))
    agent_data["deadband_applied"].append(
        np.asarray(result["deadband_applied"], dtype=np.float32))
    agent_data["lighting_safety_active"].append(bool(result["lighting_safety_active"]))
    agent_data["lighting_safety_override"].append(bool(result["lighting_safety_override"]))


# ============================================================
# POST-RUN VERIFICATION
# ============================================================

def verify_safeguard(verbose=True):
    """Check the invariant directly from the logs. Call after the run."""
    obs = np.array(agent_data["observation"])
    exe = np.array(agent_data["executed_action_norm"])
    pol = np.array(agent_data["policy_executed_action_norm"])
    unocc = obs[:, 27] < 0.5

    light_exe = exe[unocc, 4:9]
    light_pol = pol[unocc, 4:9]
    ok = np.allclose(light_exe, -1.0, atol=1e-6)

    # commanded watts from the actuation log
    lp = np.array([lighting_power_data[z] for z in ZONE_ORDER]).T
    lp_unocc = lp[unocc]
    kwh_unocc = lp_unocc.sum() * (1 / 6) / 1000.0
    kwh_total = lp.sum() * (1 / 6) / 1000.0

    # what the raw policy WOULD have done (counterfactual saving)
    would_w = 0.5 * (light_pol + 1.0)
    installed = np.array([LIGHTING_INSTALLED_W[z] for z in ZONE_ORDER])
    kwh_prevented = (would_w * installed).sum() * (1 / 6) / 1000.0

    if verbose:
        print("=== safeguard verification ===")
        print(f"  unoccupied steps                : {unocc.sum()}")
        print(f"  executed lighting == -1 (0 W)   : {ok}")
        print(f"  override fired on               : {int(np.sum(agent_data['lighting_safety_override']))} steps")
        print(f"  unoccupied lighting energy      : {kwh_unocc:,.1f} kWh   (target 0.0)")
        print(f"  total lighting energy           : {kwh_total:,.1f} kWh")
        print(f"  energy the policy WOULD have used unoccupied: {kwh_prevented:,.1f} kWh")
    return ok, kwh_unocc, kwh_prevented


In [ ]:
eps.set_callback(
    "callback_begin_system_timestep_before_predictor",
    callback_func
    
)

In [ ]:
eps.run(annual=True) # run annual simulation

In [ ]:
def verify_selector_logging(verbose=True):
    probs = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )

    logits = np.asarray(
        agent_data["selection_logits"],
        dtype=np.float32,
    )

    masks = np.asarray(
        agent_data["selection_mask"],
        dtype=np.float32,
    )

    if probs.shape != masks.shape:
        raise ValueError(
            f"Probability shape {probs.shape} "
            f"does not match mask shape {masks.shape}."
        )

    if logits.shape != masks.shape:
        raise ValueError(
            f"Logit shape {logits.shape} "
            f"does not match mask shape {masks.shape}."
        )

    if probs.ndim != 2 or probs.shape[1] != 19:
        raise ValueError(
            f"Expected selector arrays shaped (T, 19), "
            f"got {probs.shape}."
        )

    # Verify that saved probabilities correspond to saved logits.
    reconstructed_probs = 1.0 / (
        1.0 + np.exp(-logits)
    )

    if not np.allclose(
        reconstructed_probs,
        probs,
        atol=1e-6,
        rtol=0.0,
    ):
        raise ValueError(
            "Saved probabilities do not match sigmoid(logits)."
        )

    # Valid only for deterministic evaluation.
    if DETERMINISTIC:
        # Skip step 0 because force_first_update=True.
        reconstructed_masks = (
            probs[1:] >= 0.5
        ).astype(np.float32)

        if not np.array_equal(
            reconstructed_masks,
            masks[1:],
        ):
            disagreement = np.mean(
                reconstructed_masks != masks[1:]
            )

            raise ValueError(
                "Thresholded probabilities do not match "
                f"the saved masks. Disagreement={disagreement:.6%}"
            )

    if verbose:
        print("=== Selector logging verification ===")
        print(f"  logits shape:       {logits.shape}")
        print(f"  probabilities shape:{probs.shape}")
        print(f"  masks shape:        {masks.shape}")
        print(
            f"  probability range:  "
            f"[{probs.min():.6f}, {probs.max():.6f}]"
        )
        print("  sigmoid consistency: PASSED")

        if DETERMINISTIC:
            print("  deterministic mask check: PASSED")

    return logits, probs, masks

In [ ]:
for key in ("observation", "selection_mask"):
    arr = np.asarray(agent_data[key])

    first_copy = arr[0::2]
    second_copy = arr[1::2]

    if not np.allclose(first_copy, second_copy):
        raise ValueError(
            f"{key} entries are not identical duplicate pairs."
        )

    agent_data[key] = [
        row.copy() for row in first_copy
    ]

In [ ]:
# verify_safeguard()
logits, probabilities, masks = verify_selector_logging()

In [ ]:
import numpy as np
import pandas as pd


def analyze_glazing_selector(agent_data):
    obs = np.asarray(
        agent_data["observation"],
        dtype=np.float32,
    )
    probs = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )
    masks = np.asarray(
        agent_data["selection_mask"],
        dtype=np.float32,
    )

    # Observation dimensions 18:22:
    # WPI for the four perimeter zones.
    wpi = obs[:, 18:22]

    # Selector dimensions 0:4:
    # Glazing update decisions.
    glazing_probs = probs[:, 0:4]
    glazing_masks = masks[:, 0:4]

    occupied = obs[:, 27] >= 0.5
    occupied_4d = occupied[:, None]

    # Zone-specific WPI change per timestep.
    wpi_change = np.zeros_like(wpi)
    wpi_change[1:] = wpi[1:] - wpi[:-1]

    positive_changes = wpi_change[
        occupied_4d & (wpi_change > 0)
    ]

    if positive_changes.size:
        rapid_threshold = max(
            100.0,
            float(np.percentile(positive_changes, 90)),
        )
    else:
        rapid_threshold = 100.0

    conditions = {
        "WPI below 450": wpi < 450.0,
        "WPI 450-550": (
            (wpi >= 450.0) &
            (wpi <= 550.0)
        ),
        "WPI above 550": wpi > 550.0,
        "WPI above 1000": wpi > 1000.0,
        f"Rapid rise >{rapid_threshold:.0f} lux":
            wpi_change >= rapid_threshold,
    }

    zone_names = [
        "South ZN1",
        "East ZN2",
        "North ZN3",
        "West ZN4",
    ]

    rows = []

    for condition_name, condition in conditions.items():
        condition = condition & occupied_4d

        # Pooled result over all four glazing dimensions.
        selected_probs = glazing_probs[condition]
        selected_masks = glazing_masks[condition]

        if selected_probs.size:
            rows.append({
                "zone": "All zones",
                "condition": condition_name,
                "samples": selected_probs.size,
                "mean_probability": selected_probs.mean(),
                "median_probability": np.median(
                    selected_probs
                ),
                "near_threshold_%": (
                    (
                        (selected_probs >= 0.40) &
                        (selected_probs < 0.50)
                    ).mean() * 100.0
                ),
                "update_rate_%": (
                    selected_masks.mean() * 100.0
                ),
            })

        # Individual zones.
        for zone_idx, zone_name in enumerate(zone_names):
            zone_condition = condition[:, zone_idx]

            zone_probs = glazing_probs[
                zone_condition,
                zone_idx,
            ]
            zone_masks = glazing_masks[
                zone_condition,
                zone_idx,
            ]

            if not zone_probs.size:
                continue

            rows.append({
                "zone": zone_name,
                "condition": condition_name,
                "samples": zone_probs.size,
                "mean_probability": zone_probs.mean(),
                "median_probability": np.median(
                    zone_probs
                ),
                "near_threshold_%": (
                    (
                        (zone_probs >= 0.40) &
                        (zone_probs < 0.50)
                    ).mean() * 100.0
                ),
                "update_rate_%": (
                    zone_masks.mean() * 100.0
                ),
            })

    results = pd.DataFrame(rows)

    return results.sort_values(
        ["condition", "zone"]
    ).reset_index(drop=True)


selector_results = analyze_glazing_selector(agent_data)

with pd.option_context(
    "display.max_rows", None,
    "display.width", 160,
):
    print(selector_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    ))

In [ ]:
import numpy as np
import pandas as pd


def analyze_glazing_update_direction(
    agent_data,
    glazing_state_data,
):
    obs = np.asarray(
        agent_data["observation"],
        dtype=np.float32,
    )
    probs = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )[:, 0:4]
    masks = np.asarray(
        agent_data["selection_mask"],
        dtype=np.float32,
    )[:, 0:4]

    wpi = obs[:, 18:22]
    occupied = obs[:, 27] >= 0.5

    zone_names = [
        "Perimeter_mid_ZN_1",
        "Perimeter_mid_ZN_2",
        "Perimeter_mid_ZN_3",
        "Perimeter_mid_ZN_4",
    ]

    # Assumes ec01 -> ec04 means clear -> dark.
    # Reverse this list if your optical-state order is opposite.
    darkness_order = [
        "sr2_ec01",
        "sr2_ec02",
        "sr2_ec03",
        "sr2_ec04",
    ]
    darkness_id = {
        state: i
        for i, state in enumerate(darkness_order)
    }

    rows = []

    for j, zone in enumerate(zone_names):
        states = np.asarray(glazing_state_data[zone])

        if len(states) != len(obs):
            raise ValueError(
                f"{zone}: {len(states)} glazing states "
                f"but {len(obs)} observations."
            )

        tiers = np.asarray([
            darkness_id[state]
            for state in states
        ])

        tier_change = np.zeros_like(tiers)
        tier_change[1:] = tiers[1:] - tiers[:-1]

        excessive = occupied & (wpi[:, j] > 1000.0)
        selected = masks[:, j] >= 0.5
        selected_excessive = excessive & selected

        n_excessive = excessive.sum()
        n_selected = selected_excessive.sum()

        rows.append({
            "zone": zone,
            "excessive_samples": n_excessive,
            "mean_probability": (
                probs[excessive, j].mean()
                if n_excessive else np.nan
            ),
            "update_rate_%": (
                100.0 * n_selected / n_excessive
                if n_excessive else np.nan
            ),
            "darker_given_update_%": (
                100.0 * np.mean(
                    tier_change[selected_excessive] > 0
                )
                if n_selected else np.nan
            ),
            "lighter_given_update_%": (
                100.0 * np.mean(
                    tier_change[selected_excessive] < 0
                )
                if n_selected else np.nan
            ),
            "same_given_update_%": (
                100.0 * np.mean(
                    tier_change[selected_excessive] == 0
                )
                if n_selected else np.nan
            ),
        })

    return pd.DataFrame(rows)


direction_results = analyze_glazing_update_direction(
    agent_data,
    glazing_state_data,
)

print(
    direction_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TIER_OF = {'sr2_ec01': 0, 'sr2_ec02': 1, 'sr2_ec03': 2, 'sr2_ec04': 3}
ZONES = ['Perimeter_mid_ZN_1','Perimeter_mid_ZN_2','Perimeter_mid_ZN_3','Perimeter_mid_ZN_4','Core_mid']
PERIM = ZONES[:4]

# time axis (hours since start) from temporal_data
doy = np.array(temporal_data['day_of_year'], float)
hr  = np.array(temporal_data['hour'], float)
mn  = np.array(temporal_data['minute'], float)
t = (doy - doy[0]) * 24 + hr + mn / 60.0

# ---- 1. thermal: temp + setpoints, 21-24 band ----
fig, ax = plt.subplots(3, 2, figsize=(14, 9), sharex=True); ax = ax.ravel()
for i, z in enumerate(ZONES):
    a = ax[i]
    a.axhspan(21, 24, color='#f6d9b0', alpha=.6, lw=0)
    a.axhline(22, color='.4', ls=':', lw=1)
    a.plot(t, temperature_data[z], color='#c1440e', lw=1.2, label='temp')
    a.step(t, heating_setpoint_data[z], where='post', color='#e07a1f', lw=1, label='heat SP')
    a.step(t, cooling_setpoint_data[z], where='post', color='#2c6e8f', lw=1, label='cool SP')
    a.set_title(z, fontsize=9); a.set_ylabel('°C'); a.set_ylim(14, 30)
ax[0].legend(fontsize=7); ax[5].axis('off')
fig.suptitle('Thermal'); fig.tight_layout()

# ---- 2. energy: meters (J/step -> W) + hvac demand ----
STEP_S = 300.0   # <-- set to your zone timestep in seconds
fig, a = plt.subplots(figsize=(14, 4))
for k, c in [('Heating:Electricity', '#e07a1f'), ('Cooling:Electricity', '#2c6e8f'),
             ('InteriorLights:Electricity', '#d9a441')]:
    a.plot(t, np.array(meter_data[k], float) / STEP_S, lw=1, color=c, label=k)
a.plot(t, output_variable_data['HVAC_electricity_demand_rate'], lw=1.2, ls='--',
       color='#5a3a1a', label='HVAC demand (W)')
a.set_ylabel('W'); a.set_title('Energy'); a.legend(fontsize=8); fig.tight_layout()

# ---- 3. lighting power + WPI ----
fig, ax = plt.subplots(3, 2, figsize=(14, 9), sharex=True); ax = ax.ravel()
for i, z in enumerate(ZONES):
    a = ax[i]
    a.step(t, lighting_power_data[z], where='post', color='#d9a441', lw=1.2, label='power W')
    a.set_title(z, fontsize=9); a.set_ylabel('W')
    if z in PERIM:
        aw = a.twinx()
        aw.plot(t, wpi_data[z], color='#8a5a2b', lw=1)
        aw.axhline(500, color='#8a5a2b', ls=':', lw=.8); aw.set_ylabel('WPI lux')
ax[5].axis('off'); fig.suptitle('Lighting / WPI'); fig.tight_layout()

# ---- 4. glazing tier + external irradiance ----
fig, ax = plt.subplots(2, 2, figsize=(14, 7), sharex=True); ax = ax.ravel()
for i, z in enumerate(PERIM):
    a = ax[i]
    a.plot(t, ext_irr_data[z], color='#e0902f', lw=1, label='ext irr')
    a.plot(t, solar_data[z], color='#b5651d', lw=.8, alpha=.7, label='transmitted')
    a.set_title(z, fontsize=9); a.set_ylabel('W/m² · W')
    ag = a.twinx()
    tier = [TIER_OF[s] for s in glazing_state_data[z]]
    ag.step(t, tier, where='post', color='#333', lw=1.1)
    ag.set_ylim(-.3, 3.3); ag.set_yticks([0, 1, 2, 3]); ag.set_ylabel('EC tier')
ax[0].legend(fontsize=7); fig.suptitle('Daylight / glazing'); fig.tight_layout()

# ---- 5. SDAR action + mask heatmaps ----
act  = np.array(action_vector_data['action'])          # [T, 19]
mask = np.array(action_vector_data['selection_mask'])  # [T, 19]
fig, (a0, a1) = plt.subplots(2, 1, figsize=(14, 8))
im = a0.imshow(act.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1,
               extent=[t[0], t[-1], 19, 0]); a0.set_title('action a')
fig.colorbar(im, ax=a0, fraction=.02)
a1.imshow(mask.T, aspect='auto', cmap='Greys', vmin=0, vmax=1,
          extent=[t[0], t[-1], 19, 0]); a1.set_title('selection_mask b')
for e in (4, 9, 14):
    a0.axhline(e, color='k', lw=.6); a1.axhline(e, color='#e07a1f', lw=.6)
fig.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ZONES = ['Perimeter_mid_ZN_1','Perimeter_mid_ZN_2','Perimeter_mid_ZN_3','Perimeter_mid_ZN_4','Core_mid']
PERIM = ZONES[:4]
TIER_OF = {'sr2_ec01':0,'sr2_ec02':1,'sr2_ec03':2,'sr2_ec04':3}
STEP_S = 300.0        # <-- your zone timestep in seconds (300 = 5-min, 600 = 10-min)

doy = np.array(temporal_data['day_of_year'], int)
hr  = np.array(temporal_data['hour'], float)
mn  = np.array(temporal_data['minute'], float)

# ---- pick the day ----
DAY = np.unique(doy)[95]          # index into the run's days; or set DAY = 172 directly
sel = (doy == DAY)
x = (hr + mn / 60.0)[sel]
order = np.argsort(x)
x = x[order]

def g(d, z):                      # day-filtered, sorted series for zone z
    return np.array(d[z], float)[sel][order]

# ===== 1. TEMPERATURE vs SETPOINTS =====
fig, ax = plt.subplots(3, 2, figsize=(14, 9), sharex=True); ax = ax.ravel()
for i, z in enumerate(ZONES):
    a = ax[i]
    a.axvspan(8, 16, color='0.85', alpha=.5, zorder=0, lw=0)
    a.axhspan(21, 24, color='#f6d9b0', alpha=.6, lw=0, label='band 21–24')
    a.axhline(22, color='.4', ls=':', lw=1, label='override 22')
    a.plot(x, g(temperature_data, z), color='#c1440e', lw=1.6, label='temp')
    a.step(x, g(heating_setpoint_data, z), where='post', color='#e07a1f', lw=1.1, label='heat SP')
    a.step(x, g(cooling_setpoint_data, z), where='post', color='#2c6e8f', lw=1.1, label='cool SP')
    a.set_title(z, fontsize=9); a.set_ylabel('°C'); a.set_ylim(14, 30)
    a.set_xlim(0, 24); a.set_xticks(range(0, 25, 3))
ax[0].legend(fontsize=7, ncol=2); ax[5].axis('off')
for a in ax[3:5]: a.set_xlabel('hour of day')
fig.suptitle(f'Temperature vs setpoints — day {DAY}'); fig.tight_layout()

# ===== 2. LIGHTING (applied power + WPI daylight) =====
fig, ax = plt.subplots(3, 2, figsize=(14, 9), sharex=True); ax = ax.ravel()
for i, z in enumerate(ZONES):
    a = ax[i]
    a.axvspan(8, 16, color='0.85', alpha=.5, zorder=0, lw=0)
    a.step(x, g(lighting_power_data, z), where='post', color='#d9a441', lw=1.3, label='applied W')
    a.set_title(z, fontsize=9); a.set_ylabel('W')
    a.set_xlim(0, 24); a.set_xticks(range(0, 25, 3))
    if z in PERIM:
        aw = a.twinx()
        aw.plot(x, g(wpi_data, z), color='#8a5a2b', lw=1.1, label='WPI lux')
        aw.axhline(500, color='#8a5a2b', ls=':', lw=.8)
        aw.set_ylabel('WPI lux', color='#8a5a2b'); aw.tick_params(axis='y', labelcolor='#8a5a2b')
ax[0].legend(fontsize=7); ax[5].axis('off')
for a in ax[3:5]: a.set_xlabel('hour of day')
fig.suptitle(f'Lighting power / WPI — day {DAY}'); fig.tight_layout()
5
# ===== 3. DAYLIGHT (ext irradiance + transmitted solar + glazing tier) =====
fig, ax = plt.subplots(2, 2, figsize=(14, 7), sharex=True); ax = ax.ravel()
for i, z in enumerate(PERIM):
    a = ax[i]
    a.axvspan(8, 16, color='0.85', alpha=.5, zorder=0, lw=0)
    a.plot(x, g(ext_irr_data, z), color='#e0902f', lw=1.1, label='ext irr W/m²')
    a.plot(x, g(solar_data, z), color='#b5651d', lw=.9, alpha=.7, label='transmitted W')
    a.set_title(z, fontsize=9); a.set_ylabel('W/m² · W')
    a.set_xlim(0, 24); a.set_xticks(range(0, 25, 3))
    ag = a.twinx()
    tier = np.array([TIER_OF[s] for s in glazing_state_data[z]])[sel][order]
    ag.step(x, tier, where='post', color='#333', lw=1.1)
    ag.set_ylim(-.3, 3.3); ag.set_yticks([0,1,2,3])
    ag.set_yticklabels(['ec01','ec02','ec03','ec04'], fontsize=7); ag.set_ylabel('EC tier')
ax[0].legend(fontsize=7)
for a in ax[2:]: a.set_xlabel('hour of day')
fig.suptitle(f'Daylight & glazing — day {DAY}'); fig.tight_layout()

# ===== 4. ENERGY CONSUMED (meters -> W + HVAC demand) =====
def gm(key):                      # day-filtered meter (J/step -> W)
    return np.array(meter_data[key], float)[sel][order] / STEP_S

fig, a = plt.subplots(figsize=(14, 4.5))
a.axvspan(8, 16, color='0.85', alpha=.5, zorder=0, lw=0)
a.plot(x, gm('Heating:Electricity'), color='#e07a1f', lw=1.2, label='Heating (W)')
a.plot(x, gm('Cooling:Electricity'), color='#2c6e8f', lw=1.2, label='Cooling (W)')
a.plot(x, gm('InteriorLights:Electricity'), color='#d9a441', lw=1.2, label='Lights (W)')
hv = np.array(output_variable_data['HVAC_electricity_demand_rate'], float)[sel][order]
a.plot(x, hv, color='#5a3a1a', ls='--', lw=1.3, label='HVAC demand (W)')
a.set_xlim(0, 24); a.set_xticks(range(0, 25, 3))
a.set_xlabel('hour of day'); a.set_ylabel('Power (W)')
a.set_title(f'Energy — meters + HVAC demand — day {DAY}'); a.legend(fontsize=8)
fig.tight_layout()

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("agent_eval_dataset.csv")

light_cols = [
    "light_power_Perimeter_mid_ZN_1",
    "light_power_Perimeter_mid_ZN_2",
    "light_power_Perimeter_mid_ZN_3",
    "light_power_Perimeter_mid_ZN_4",
    "light_power_Core_mid",
]

df["total_light_power_w"] = df[light_cols].sum(axis=1)
df["day_index"] = np.arange(len(df)) // 144

# Find the day with the most unoccupied lighting energy
daily_unoccupied = (
    df.loc[df["is_occupied"] < 0.5]
      .groupby("day_index")["total_light_power_w"]
      .sum()
      / 6.0
      / 1000.0
)

day = int(daily_unoccupied.idxmax())
d = df[df["day_index"] == day].copy()

hours = np.arange(len(d)) / 6.0

plt.figure(figsize=(12, 4))
plt.step(hours, d["total_light_power_w"], where="post")
plt.fill_between(
    hours,
    0,
    d["total_light_power_w"],
    where=d["is_occupied"].to_numpy() < 0.5,
    color="red",
    alpha=0.35,
    label="Unoccupied lighting",
)
plt.axvspan(8, 16, color="green", alpha=0.1, label="Occupied")
plt.xlim(0, 24)
plt.ylabel("Executed lighting power [W]")
plt.xlabel("Hour")
plt.title(
    f"Day {day}: unoccupied lighting = "
    f"{daily_unoccupied.loc[day]:.2f} kWh"
)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import numpy as np

pooled = np.load("pooled_sdar_experts.npz", allow_pickle=True)
agent = np.load("agent_eval_dataset.npz", allow_pickle=True)

pooled_names = [str(x) for x in pooled["reward_component_names"]]
agent_names = [str(x) for x in agent["reward_component_names"]]

# Reorder agent columns to exactly match the pooled dataset.
agent_order = [agent_names.index(name) for name in pooled_names]

scales = pooled["reward_component_scales"].astype(np.float64)
importance = pooled["reward_component_importance"].astype(np.float64)

def shared_training_reward(raw_components):
    final_components = np.clip(
        raw_components / scales * importance,
        -10.0,
        10.0,
    )

    balanced = final_components.sum(axis=1)
    symlog = np.sign(balanced) * np.log1p(np.abs(balanced))

    return balanced, symlog, final_components

pool_balanced, pool_reward, _ = shared_training_reward(
    pooled["reward_components_raw"]
)

agent_balanced, agent_reward, _ = shared_training_reward(
    agent["reward_components_raw"][:, agent_order]
)

print("Pooled shared reward:", pool_reward.mean())
print("Agent shared reward: ", agent_reward.mean())
print("Difference:          ", agent_reward.mean() - pool_reward.mean())

In [ ]:
def score_with_thermal_clip(
    raw,
    thermal_importance,
    thermal_clip,
):
    importance = base_importance.copy()
    importance[thermal_idx] = thermal_importance

    unbounded = raw / scales * importance

    # Default clipping for all components.
    components = np.clip(unbounded, -10.0, 10.0)

    # Replace thermal column with a different clipping limit.
    if thermal_clip is None:
        components[:, thermal_idx] = unbounded[:, thermal_idx]
    else:
        components[:, thermal_idx] = np.clip(
            unbounded[:, thermal_idx],
            -thermal_clip,
            thermal_clip,
        )

    balanced = components.sum(axis=1)
    reward = np.sign(balanced) * np.log1p(
        np.abs(balanced)
    )

    thermal_values = unbounded[:, thermal_idx]
    nonzero = np.abs(thermal_values) > 1e-12

    if thermal_clip is None or not nonzero.any():
        saturation = 0.0
    else:
        saturation = np.mean(
            np.abs(thermal_values[nonzero]) >= thermal_clip
        )

    return {
        "reward": reward.mean(),
        "thermal": components[:, thermal_idx].mean(),
        "saturation": 100.0 * saturation,
    }


print(
    f"{'weight':>7s} {'clip':>7s} "
    f"{'pool reward':>13s} {'agent reward':>13s} "
    f"{'difference':>12s} "
    f"{'pool sat':>10s} {'agent sat':>10s}"
)

for thermal_weight in [2.0, 3.0, 4.0]:
    for thermal_clip in [10.0, 20.0, 50.0, None]:
        p = score_with_thermal_clip(
            pool_raw,
            thermal_weight,
            thermal_clip,
        )
        a = score_with_thermal_clip(
            agent_raw,
            thermal_weight,
            thermal_clip,
        )

        clip_name = (
            "none" if thermal_clip is None
            else f"{thermal_clip:.0f}"
        )

        print(
            f"{thermal_weight:7.1f} "
            f"{clip_name:>7s} "
            f"{p['reward']:13.6f} "
            f"{a['reward']:13.6f} "
            f"{a['reward']-p['reward']:+12.6f} "
            f"{p['saturation']:9.2f}% "
            f"{a['saturation']:9.2f}%"
        )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

LABELS = (['glz1','glz2','glz3','glz4'] +
          ['lgt1','lgt2','lgt3','lgt4','lgtC'] +
          ['heat1','heat2','heat3','heat4','heatC'] +
          ['cool1','cool2','cool3','cool4','coolC'])

doy = np.array(temporal_data['day_of_year'], int)
hr  = np.array(temporal_data['hour'], float)
mn  = np.array(temporal_data['minute'], float)

# ---- pick the day (same as before) ----
DAY = np.unique(doy)[30]           # <-- set to the day_of_year you want
sel = (doy == DAY)
x = (hr + mn / 60.0)[sel]
order = np.argsort(x)
x = x[order]

act  = np.array(action_vector_data['action'])[sel][order]          # [Nday, 19]
mask = np.array(action_vector_data['selection_mask'])[sel][order]   # [Nday, 19]

ext = [x[0], x[-1], 19, 0]   # x-extent = hour of day, y = dims 0..19

fig, (a0, a1) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

im = a0.imshow(act.T, aspect='auto', cmap='coolwarm', vmin=-1, vmax=1,
               interpolation='nearest', extent=ext)
a0.set_yticks(np.arange(19) + .5); a0.set_yticklabels(LABELS, fontsize=6)
a0.set_title(f'SDAR action a — day {DAY}')
fig.colorbar(im, ax=a0, fraction=.02, pad=.01)

a1.imshow(mask.T, aspect='auto', cmap='Greys', vmin=0, vmax=1,
          interpolation='nearest', extent=ext)
a1.set_yticks(np.arange(19) + .5); a1.set_yticklabels(LABELS, fontsize=6)
a1.set_title('selection_mask b (black = changed, white = repeat)')

for a in (a0, a1):
    for e in (4, 9, 14):                    # glaze | light | heat | cool dividers
        a.axhline(e, color='#e07a1f', lw=.7)
    a.axvspan(8, 16, color='none')          # (occupied hours already visible)
    a.set_xlim(0, 24); a.set_xticks(range(0, 25, 3))
a1.set_xlabel('hour of day')

fig.tight_layout()
plt.show()

In [ ]:
"""
Offline RL dataset generator — v8
Meter-free reward + meter-free obs + fixed comfort bands + per-zone lighting
+ per-component reward normalization (unit mean-|abs| then intentional weights)
============================================================================

Drop-in dataset builder to run AFTER an expert EnergyPlus/Sinergym callback
rollout (v12 controller) has filled the global logging dictionaries.

Expected shared globals from the expert callback:
    ZONES_MID, WINDOW_ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    action_vector_data
"""

import csv
import json
import numpy as np


# ============================================================
# CONFIG
# ============================================================

REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = None

REWARD_IMPORTANCE = {
    "energy_lighting":       0.5,
    "hvac_demand_rate":      1.0,
    "thermal_comfort":       1.0,
    "visual_under":          1.0,
    "visual_over_ctrl":      0.5,
    "visual_over_daylight":  0.25,
    "core_under":            0.5,
    "core_over":             0.5,
    "artificial_excess":     0.5,
    "solar_cooling_penalty": 0.5,
    "solar_heating_bonus":   0.25,
    "switching_penalty":     0.25,
}

TIMEOUT_STEPS = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 5e-6
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# REWARD — components + scalar
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_rate_sum = sum(lighting_data[zone][t + 1] for zone in ZONES_MID)
    e_light = -W_LIGHTING * lighting_rate_sum

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_next:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = selection_mask_t
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def normalize_reward_components(components_raw, comp_names, component_scales=None):
    C = len(comp_names)
    importance_vec = np.array([REWARD_IMPORTANCE.get(nm, 1.0) for nm in comp_names],
                              dtype=np.float64)

    if component_scales is not None:
        comp_scales = np.array([float(component_scales[nm]) for nm in comp_names],
                               dtype=np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "external (shared across experts)"
        apply = True
    elif REWARD_NORMALIZE:
        comp_scales = np.abs(components_raw).mean(axis=0).astype(np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "dataset-local mean|abs|"
        apply = True
    else:
        comp_scales = np.ones(C, dtype=np.float64)
        scales_source = "none (raw physical W_* weights)"
        apply = False

    if apply:
        components_final = (components_raw.astype(np.float64) / comp_scales) * importance_vec
    else:
        components_final = components_raw.astype(np.float64).copy()

    if REWARD_COMPONENT_CLIP is not None:
        components_final = np.clip(components_final,
                                   -REWARD_COMPONENT_CLIP, REWARD_COMPONENT_CLIP)

    return components_final.astype(np.float32), comp_scales, importance_vec, scales_source


# ============================================================
# DIAGNOSTICS
# ============================================================

def _print_reward_diagnostics(rewards_raw, rewards_tx, components_arr):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(f"  raw:         mean={rewards_raw.mean():+.4f}  std={rewards_raw.std():.4f}")
    print(f"               min={rewards_raw.min():+.4f}    max={rewards_raw.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_raw, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - rewards_raw
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - raw_total): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - raw_total): {np.max(np.abs(err)):.8f}")


def _print_normalization(comp_names, comp_scales, importance_vec, scales_source):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  clip={REWARD_COMPONENT_CLIP}  "
          f"scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'eff_weight':>10s}")
    for nm, sc, imp in zip(comp_names, comp_scales, importance_vec):
        eff = imp / sc if sc != 0 else 0.0
        print(f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  {eff:10.4f}")


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None:
        return timeouts
    if TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


# ============================================================
# MAIN BUILDER
# ============================================================

def build_offline_dataset(
    save_npz=True,
    save_csv=True,
    prefix="offline_sdar_rl_replay_dataset_new_normalized_reward",
    component_scales=None,
):
    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1

    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "sdar_action": len(action_vector_data["action"]),
        "sdar_prev_action": len(action_vector_data["prev_action"]),
        "sdar_mask": len(action_vector_data["selection_mask"]),
        "sdar_mix": len(action_vector_data["action_mix"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)

    sdar_actions = np.asarray(action_vector_data["action"], dtype=np.float32)[:T]
    sdar_prev_actions = np.asarray(action_vector_data["prev_action"], dtype=np.float32)[:T]
    sdar_masks = np.asarray(action_vector_data["selection_mask"], dtype=np.float32)[:T]
    sdar_mixes = np.asarray(action_vector_data["action_mix"], dtype=np.float32)[:T]

    for name, arr in [
        ("sdar_actions", sdar_actions),
        ("sdar_prev_actions", sdar_prev_actions),
        ("sdar_masks", sdar_masks),
        ("sdar_mixes", sdar_mixes),
    ]:
        if arr.ndim != 2 or arr.shape != (T, ACTION_DIM):
            raise ValueError(
                f"{name} has shape {arr.shape}, expected ({T}, {ACTION_DIM}). "
                "Did the callback use the 19-dim SDAR layout?"
            )

    actions = sdar_actions

    comp_names = _component_names()
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, sdar_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, comp_scales, importance_vec, scales_source = \
        normalize_reward_components(components_raw, comp_names, component_scales)

    rewards_raw = components_final.sum(axis=1).astype(np.float32)
    rewards_tx = _transform_vec(rewards_raw)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32
    )

    print("\n=== Dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (primary, SDAR-ready, per-zone lighting norm)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {sdar_prev_actions.shape}")
    print(f"  selection_masks:   {sdar_masks.shape}")
    print(f"  action_mixes:      {sdar_mixes.shape}")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced sum, pre-transform)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(f"  timeouts:          {timeouts.shape}   artificial timeouts={int(timeouts.sum())}")
    if SAVE_REWARD_COMPONENTS:
        print(f"  reward_components: {components_final.shape}  ({len(comp_names)} terms, final)")

    _print_normalization(comp_names, comp_scales, importance_vec, scales_source)
    _print_reward_diagnostics(rewards_raw, rewards_tx, components_final)
    _print_thermal_violation_rate()
    _print_switching_diagnostics(sdar_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=sdar_prev_actions,
            selection_masks=sdar_masks,
            action_mixes=sdar_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(REWARD_TRANSFORM),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            action_layout=np.array(
                {
                    "glazing": [0, 4],
                    "lighting": [4, 9],
                    "heating": [9, 14],
                    "cooling": [14, 19],
                },
                dtype=object,
            ),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)

        if "expert_id" in temporal_data:
            save_dict["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "reward_transform": REWARD_TRANSFORM,
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "reward_scales_source": scales_source,
                    "reward_energy_is_meter_free": True,
                    "obs_is_meter_free": True,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "obs_cols": obs_column_names(),
                    "act_cols": action_column_names(),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_tx_min": float(rewards_tx.min()),
                    "reward_tx_max": float(rewards_tx.max()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_importance_config": REWARD_IMPORTANCE,
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(sdar_masks.mean()),
                    "mean_selection_mask_glazing": float(sdar_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(sdar_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(sdar_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(sdar_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "timeout_steps": None if TIMEOUT_STEPS is None else int(TIMEOUT_STEPS),
                    "num_timeouts": int(timeouts.sum()),
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + sdar_masks[t].tolist()
                    + [float(rewards[t]), float(rewards_raw[t])]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    result = {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": sdar_prev_actions,
        "selection_masks": sdar_masks,
        "action_mixes": sdar_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_components": components_final,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
    }

    if "expert_id" in temporal_data:
        result["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

    return result


if __name__ == "__main__":
    dataset = build_offline_dataset(save_npz=True, save_csv=True)

In [ ]:
"""
Offline RL dataset generator — v8
Meter-free reward + meter-free obs + fixed comfort bands + per-zone lighting
+ per-component reward normalization (unit mean-|abs| then intentional weights)
============================================================================

Drop-in dataset builder to run AFTER an expert EnergyPlus/Sinergym callback
rollout (v12 controller) has filled the global logging dictionaries.

Expected shared globals from the expert callback:
    ZONES_MID, WINDOW_ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    action_vector_data
"""

import csv
import json
import numpy as np


# ============================================================
# CONFIG
# ============================================================

REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = None

REWARD_IMPORTANCE = {
    "energy_lighting":       0.5,
    "hvac_demand_rate":      1.0,
    "thermal_comfort":       1.0,
    "visual_under":          1.0,
    "visual_over_ctrl":      0.5,
    "visual_over_daylight":  0.25,
    "core_under":            0.5,
    "core_over":             0.5,
    "artificial_excess":     0.5,
    "solar_cooling_penalty": 0.5,
    "solar_heating_bonus":   0.25,
    "switching_penalty":     0.25,
}

TIMEOUT_STEPS = None

# ---- lighting energy from action-derived dimming (nominal power * duty) ----
# Reporting timestep. 6 steps/hour = 10-min steps. Set to match the run.
STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 5e-6
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM ACTION-DERIVED DIMMING
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps exactly to a
# dimming duty u = (a + 1) / 2 in [0, 1]. Energy is therefore reconstructed as
#   E = sum_z ( nominal_W[z] * u[z] ) * dt_hours / 1000   (kWh)
# using the action at time t (energy is a property of a_t in (s_t, a_t, r_t)).
# This does NOT use meters or the logged Lights Electricity Rate; those remain
# available as observations and for validation:  P_EnergyPlus ~= sum_z P_nom*u.


def compute_lighting_energy_kwh(t):
    action_t = np.asarray(
        action_vector_data["action"][t],
        dtype=np.float64,
    )

    # Convert normalized [-1, 1] actions to dimming [0, 1]
    dimming = np.clip(
        (action_t[LIGHT_SLICE] + 1.0) / 2.0,
        0.0,
        1.0,
    )

    nominal_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )

    zone_power_w = nominal_w * dimming
    total_power_w = zone_power_w.sum()

    energy_kwh = (
        total_power_w
        * TIMESTEP_HOURS
        / 1000.0
    )

    return float(energy_kwh)


# ============================================================
# REWARD — components + scalar
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    # lighting energy from action-derived dimming at time t (nominal * duty)
    lighting_energy_kwh = compute_lighting_energy_kwh(t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_next:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = selection_mask_t
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def normalize_reward_components(components_raw, comp_names, component_scales=None):
    C = len(comp_names)
    importance_vec = np.array([REWARD_IMPORTANCE.get(nm, 1.0) for nm in comp_names],
                              dtype=np.float64)

    if component_scales is not None:
        comp_scales = np.array([float(component_scales[nm]) for nm in comp_names],
                               dtype=np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "external (shared across experts)"
        apply = True
    elif REWARD_NORMALIZE:
        comp_scales = np.abs(components_raw).mean(axis=0).astype(np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "dataset-local mean|abs|"
        apply = True
    else:
        comp_scales = np.ones(C, dtype=np.float64)
        scales_source = "none (raw physical W_* weights)"
        apply = False

    if apply:
        components_final = (components_raw.astype(np.float64) / comp_scales) * importance_vec
    else:
        components_final = components_raw.astype(np.float64).copy()

    if REWARD_COMPONENT_CLIP is not None:
        components_final = np.clip(components_final,
                                   -REWARD_COMPONENT_CLIP, REWARD_COMPONENT_CLIP)

    return components_final.astype(np.float32), comp_scales, importance_vec, scales_source


# ============================================================
# DIAGNOSTICS
# ============================================================

def _print_reward_diagnostics(rewards_raw, rewards_tx, components_arr):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(f"  raw:         mean={rewards_raw.mean():+.4f}  std={rewards_raw.std():.4f}")
    print(f"               min={rewards_raw.min():+.4f}    max={rewards_raw.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_raw, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - rewards_raw
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - raw_total): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - raw_total): {np.max(np.abs(err)):.8f}")


def _print_normalization(comp_names, comp_scales, importance_vec, scales_source):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  clip={REWARD_COMPONENT_CLIP}  "
          f"scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'eff_weight':>10s}")
    for nm, sc, imp in zip(comp_names, comp_scales, importance_vec):
        eff = imp / sc if sc != 0 else 0.0
        print(f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  {eff:10.4f}")


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None:
        return timeouts
    if TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


# ============================================================
# MAIN BUILDER
# ============================================================

def build_offline_dataset(
    save_npz=True,
    save_csv=True,
    prefix="new_without_noise_ds_smooth",
    component_scales=None,
):
    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1

    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "sdar_action": len(action_vector_data["action"]),
        "sdar_prev_action": len(action_vector_data["prev_action"]),
        "sdar_mask": len(action_vector_data["selection_mask"]),
        "sdar_mix": len(action_vector_data["action_mix"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)

    sdar_actions = np.asarray(action_vector_data["action"], dtype=np.float32)[:T]
    sdar_prev_actions = np.asarray(action_vector_data["prev_action"], dtype=np.float32)[:T]
    sdar_masks = np.asarray(action_vector_data["selection_mask"], dtype=np.float32)[:T]
    sdar_mixes = np.asarray(action_vector_data["action_mix"], dtype=np.float32)[:T]

    for name, arr in [
        ("sdar_actions", sdar_actions),
        ("sdar_prev_actions", sdar_prev_actions),
        ("sdar_masks", sdar_masks),
        ("sdar_mixes", sdar_mixes),
    ]:
        if arr.ndim != 2 or arr.shape != (T, ACTION_DIM):
            raise ValueError(
                f"{name} has shape {arr.shape}, expected ({T}, {ACTION_DIM}). "
                "Did the callback use the 19-dim SDAR layout?"
            )

    actions = sdar_actions

    comp_names = _component_names()
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, sdar_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, comp_scales, importance_vec, scales_source = \
        normalize_reward_components(components_raw, comp_names, component_scales)

    rewards_raw = components_final.sum(axis=1).astype(np.float32)
    rewards_tx = _transform_vec(rewards_raw)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32
    )

    print("\n=== Dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (primary, SDAR-ready, per-zone lighting norm)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {sdar_prev_actions.shape}")
    print(f"  selection_masks:   {sdar_masks.shape}")
    print(f"  action_mixes:      {sdar_mixes.shape}")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced sum, pre-transform)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(f"  timeouts:          {timeouts.shape}   artificial timeouts={int(timeouts.sum())}")
    if SAVE_REWARD_COMPONENTS:
        print(f"  reward_components: {components_final.shape}  ({len(comp_names)} terms, final)")

    _print_normalization(comp_names, comp_scales, importance_vec, scales_source)
    _print_reward_diagnostics(rewards_raw, rewards_tx, components_final)
    _print_thermal_violation_rate()
    _print_switching_diagnostics(sdar_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=sdar_prev_actions,
            selection_masks=sdar_masks,
            action_mixes=sdar_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(REWARD_TRANSFORM),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            action_layout=np.array(
                {
                    "glazing": [0, 4],
                    "lighting": [4, 9],
                    "heating": [9, 14],
                    "cooling": [14, 19],
                },
                dtype=object,
            ),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)

        if "expert_id" in temporal_data:
            save_dict["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "reward_transform": REWARD_TRANSFORM,
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "reward_scales_source": scales_source,
                    "reward_energy_is_meter_free": True,
                    "obs_is_meter_free": True,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "obs_cols": obs_column_names(),
                    "act_cols": action_column_names(),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_tx_min": float(rewards_tx.min()),
                    "reward_tx_max": float(rewards_tx.max()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_importance_config": REWARD_IMPORTANCE,
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(sdar_masks.mean()),
                    "mean_selection_mask_glazing": float(sdar_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(sdar_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(sdar_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(sdar_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "timeout_steps": None if TIMEOUT_STEPS is None else int(TIMEOUT_STEPS),
                    "num_timeouts": int(timeouts.sum()),
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + sdar_masks[t].tolist()
                    + [float(rewards[t]), float(rewards_raw[t])]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    result = {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": sdar_prev_actions,
        "selection_masks": sdar_masks,
        "action_mixes": sdar_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_components": components_final,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
    }

    if "expert_id" in temporal_data:
        result["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

    return result


if __name__ == "__main__":
    dataset = build_offline_dataset(save_npz=True, save_csv=True)

In [ ]:
"""
Offline RL dataset generator — v8
Meter-free reward + meter-free obs + fixed comfort bands + per-zone lighting
+ per-component reward normalization (unit mean-|abs| then intentional weights)
============================================================================

Drop-in dataset builder to run AFTER an expert EnergyPlus/Sinergym callback
rollout (v12 controller) has filled the global logging dictionaries.

Expected shared globals from the expert callback:
    ZONES_MID, WINDOW_ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    action_vector_data
"""

import csv
import json
import numpy as np


# ============================================================
# CONFIG
# ============================================================

REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       2.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

# ---- lighting energy from action-derived dimming (nominal power * duty) ----
# Reporting timestep. 6 steps/hour = 10-min steps. Set to match the run.
STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

# Thermal exploration mode ids (must match the exploratory callback).
THERMAL_MODE_TO_ID = {
    "idle": 0,
    "random_inactive_band": 1,
    "active_exploration": 2,
    "expert_recovery": 3,
    "noisy_recovery": 4,
}

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM ACTION-DERIVED DIMMING
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps exactly to a
# dimming duty u = (a + 1) / 2 in [0, 1]. Energy is therefore reconstructed as
#   E = sum_z ( nominal_W[z] * u[z] ) * dt_hours / 1000   (kWh)
# using the action at time t (energy is a property of a_t in (s_t, a_t, r_t)).
# This does NOT use meters or the logged Lights Electricity Rate; those remain
# available as observations and for validation:  P_EnergyPlus ~= sum_z P_nom*u.


def compute_lighting_energy_kwh(t):
    action_t = np.asarray(
        action_vector_data["action"][t],
        dtype=np.float64,
    )

    # Convert normalized [-1, 1] actions to dimming [0, 1]
    dimming = np.clip(
        (action_t[LIGHT_SLICE] + 1.0) / 2.0,
        0.0,
        1.0,
    )

    nominal_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )

    zone_power_w = nominal_w * dimming
    total_power_w = zone_power_w.sum()

    energy_kwh = (
        total_power_w
        * TIMESTEP_HOURS
        / 1000.0
    )

    return float(energy_kwh)


# ============================================================
# REWARD — components + scalar
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    # Visual comfort is a property of the lighting ACTION at t, so gate it on
    # occupancy at t (avoids the artificial under-lighting penalty at the 08:00
    # transition where occupancy flips between t and t+1).
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    # lighting energy from action-derived dimming at time t (nominal * duty)
    lighting_energy_kwh = compute_lighting_energy_kwh(t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = selection_mask_t
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def normalize_reward_components(components_raw, comp_names, component_scales=None):
    C = len(comp_names)
    importance_vec = np.array([REWARD_IMPORTANCE.get(nm, 1.0) for nm in comp_names],
                              dtype=np.float64)

    if component_scales is not None:
        comp_scales = np.array([float(component_scales[nm]) for nm in comp_names],
                               dtype=np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "external (shared across experts)"
        apply = True
    elif REWARD_NORMALIZE:
        comp_scales = np.abs(components_raw).mean(axis=0).astype(np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "dataset-local mean|abs|"
        apply = True
    else:
        comp_scales = np.ones(C, dtype=np.float64)
        scales_source = "none (raw physical W_* weights)"
        apply = False

    if apply:
        components_final = (components_raw.astype(np.float64) / comp_scales) * importance_vec
    else:
        components_final = components_raw.astype(np.float64).copy()

    if REWARD_COMPONENT_CLIP is not None:
        components_final = np.clip(components_final,
                                   -REWARD_COMPONENT_CLIP, REWARD_COMPONENT_CLIP)

    return components_final.astype(np.float32), comp_scales, importance_vec, scales_source


# ============================================================
# DIAGNOSTICS
# ============================================================

def _print_reward_diagnostics(rewards_raw, rewards_tx, components_arr):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(f"  raw:         mean={rewards_raw.mean():+.4f}  std={rewards_raw.std():.4f}")
    print(f"               min={rewards_raw.min():+.4f}    max={rewards_raw.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_raw, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - rewards_raw
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - raw_total): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - raw_total): {np.max(np.abs(err)):.8f}")


def _print_normalization(comp_names, comp_scales, importance_vec, scales_source):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  clip={REWARD_COMPONENT_CLIP}  "
          f"scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'eff_weight':>10s}")
    for nm, sc, imp in zip(comp_names, comp_scales, importance_vec):
        eff = imp / sc if sc != 0 else 0.0
        print(f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  {eff:10.4f}")


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None:
        return timeouts
    if TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


# ============================================================
# SHARED REWARD SCALES (compute ONCE across all experts, reuse)
# ============================================================
# For pooled multi-expert offline RL, do NOT let each dataset self-normalize.
# 1) Build every expert's raw components with build_offline_dataset(..., component_scales=None)
#    and grab result["reward_components_raw"] (also saved in each NPZ).
# 2) Stack them and call compute_shared_scales() to get ONE scale dict.
# 3) Re-run every expert with build_offline_dataset(..., component_scales=SHARED).

def compute_shared_scales(components_raw_list, comp_names=None):
    """
    components_raw_list : list of (T_i, C) arrays of RAW reward components
                          (result["reward_components_raw"] from each expert).
    Returns {component_name: scale} using pooled mean-|abs| across all experts.
    """
    if comp_names is None:
        comp_names = _component_names()
    stacked = np.concatenate([np.asarray(a, dtype=np.float64)
                              for a in components_raw_list], axis=0)
    if stacked.shape[1] != len(comp_names):
        raise ValueError(
            f"components have {stacked.shape[1]} cols but {len(comp_names)} names")
    scales = np.abs(stacked).mean(axis=0)
    scales[scales < 1e-8] = 1.0
    return {nm: float(sc) for nm, sc in zip(comp_names, scales)}


# ============================================================
# MAIN BUILDER
# ============================================================

def build_offline_dataset(
    save_npz=True,
    save_csv=True,
    prefix="offline_sdar_rl_replay_dataset_new_normalized_reward",
    component_scales=None,
):
    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1

    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "sdar_action": len(action_vector_data["action"]),
        "sdar_prev_action": len(action_vector_data["prev_action"]),
        "sdar_mask": len(action_vector_data["selection_mask"]),
        "sdar_mix": len(action_vector_data["action_mix"]),
        "exploration_expert_action": len(exploration_data["expert_action"]),
        "exploration_executed_action": len(exploration_data["executed_action"]),
        "exploration_mask": len(exploration_data["exploration_mask"]),
        "exploration_glazing": len(exploration_data["glazing_explored"]),
        "exploration_lighting": len(exploration_data["lighting_explored"]),
        "exploration_thermal_mode": len(exploration_data["thermal_mode"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)

    sdar_actions = np.asarray(action_vector_data["action"], dtype=np.float32)[:T]
    sdar_prev_actions = np.asarray(action_vector_data["prev_action"], dtype=np.float32)[:T]
    sdar_masks = np.asarray(action_vector_data["selection_mask"], dtype=np.float32)[:T]
    sdar_mixes = np.asarray(action_vector_data["action_mix"], dtype=np.float32)[:T]

    for name, arr in [
        ("sdar_actions", sdar_actions),
        ("sdar_prev_actions", sdar_prev_actions),
        ("sdar_masks", sdar_masks),
        ("sdar_mixes", sdar_mixes),
    ]:
        if arr.ndim != 2 or arr.shape != (T, ACTION_DIM):
            raise ValueError(
                f"{name} has shape {arr.shape}, expected ({T}, {ACTION_DIM}). "
                "Did the callback use the 19-dim SDAR layout?"
            )

    # ---- #5: primary action is the EXECUTED action; verify it matches SDAR ----
    executed_actions = np.asarray(
        exploration_data["executed_action"], dtype=np.float32)[:T]
    if not np.allclose(executed_actions, sdar_actions, atol=1e-6):
        raise ValueError(
            "Executed exploration actions do not match SDAR actions.")
    actions = sdar_actions  # never train on expert_action

    # ---- #7: expert action + exploration mask ----
    expert_actions = np.asarray(
        exploration_data["expert_action"], dtype=np.float32)[:T]
    exploration_masks = np.asarray(
        exploration_data["exploration_mask"], dtype=np.float32)[:T]

    # per-zone explored flags -> arrays
    glazing_explored = np.array([
        [float(exploration_data["glazing_explored"][t][zone])
         for zone in WINDOW_ZONES_MID]
        for t in range(T)], dtype=np.float32)
    lighting_explored = np.array([
        [float(exploration_data["lighting_explored"][t][zone])
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.float32)

    # thermal modes -> integer ids
    thermal_mode_ids = np.array([
        [THERMAL_MODE_TO_ID[exploration_data["thermal_mode"][t][zone]]
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.int8)

    # ---- #2: physical lighting quantities from executed dimming ----
    lighting_dimming = np.array([
        np.clip((np.asarray(action_vector_data["action"][t])[LIGHT_SLICE] + 1.0) / 2.0,
                0.0, 1.0)
        for t in range(T)], dtype=np.float32)
    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0
    ).astype(np.float32)

    comp_names = _component_names()
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, sdar_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, comp_scales, importance_vec, scales_source = \
        normalize_reward_components(components_raw, comp_names, component_scales)

    rewards_raw = components_final.sum(axis=1).astype(np.float32)
    rewards_tx = _transform_vec(rewards_raw)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32
    )

    print("\n=== Dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (primary, SDAR-ready, per-zone lighting norm)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {sdar_prev_actions.shape}")
    print(f"  selection_masks:   {sdar_masks.shape}")
    print(f"  action_mixes:      {sdar_mixes.shape}")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced sum, pre-transform)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(f"  timeouts:          {timeouts.shape}   artificial timeouts={int(timeouts.sum())}")
    if SAVE_REWARD_COMPONENTS:
        print(f"  reward_components: {components_final.shape}  ({len(comp_names)} terms, final)")

    _print_normalization(comp_names, comp_scales, importance_vec, scales_source)
    _print_reward_diagnostics(rewards_raw, rewards_tx, components_final)
    _print_thermal_violation_rate()
    _print_switching_diagnostics(sdar_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=sdar_prev_actions,
            selection_masks=sdar_masks,
            action_mixes=sdar_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(REWARD_TRANSFORM),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # exploration provenance
            expert_actions=expert_actions,
            executed_actions=executed_actions,
            exploration_masks=exploration_masks,
            glazing_explored=glazing_explored,
            lighting_explored=lighting_explored,
            thermal_mode_ids=thermal_mode_ids,
            exploration_seed=np.int64(exploration_data["seed"]),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            action_layout=np.array(
                {
                    "glazing": [0, 4],
                    "lighting": [4, 9],
                    "heating": [9, 14],
                    "cooling": [14, 19],
                },
                dtype=object,
            ),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)

        if "expert_id" in temporal_data:
            save_dict["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "reward_transform": REWARD_TRANSFORM,
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "reward_scales_source": scales_source,
                    "reward_energy_is_meter_free": True,
                    "obs_is_meter_free": True,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "obs_cols": obs_column_names(),
                    "act_cols": action_column_names(),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_tx_min": float(rewards_tx.min()),
                    "reward_tx_max": float(rewards_tx.max()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_importance_config": REWARD_IMPORTANCE,
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(sdar_masks.mean()),
                    "mean_selection_mask_glazing": float(sdar_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(sdar_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(sdar_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(sdar_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "timeout_steps": None if TIMEOUT_STEPS is None else int(TIMEOUT_STEPS),
                    "num_timeouts": int(timeouts.sum()),
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + sdar_masks[t].tolist()
                    + [float(rewards[t]), float(rewards_raw[t])]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    result = {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": sdar_prev_actions,
        "selection_masks": sdar_masks,
        "action_mixes": sdar_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_components": components_final,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "expert_actions": expert_actions,
        "executed_actions": executed_actions,
        "exploration_masks": exploration_masks,
        "glazing_explored": glazing_explored,
        "lighting_explored": lighting_explored,
        "thermal_mode_ids": thermal_mode_ids,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
    }

    if "expert_id" in temporal_data:
        result["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

    return result


if __name__ == "__main__":
    dataset = build_offline_dataset(save_npz=True, save_csv=True)

In [ ]:
"""
Offline RL dataset generator — v8
Meter-free reward + meter-free obs + fixed comfort bands + per-zone lighting
+ per-component reward normalization (unit mean-|abs| then intentional weights)
============================================================================

Drop-in dataset builder to run AFTER an expert EnergyPlus/Sinergym callback
rollout (v12 controller) has filled the global logging dictionaries.

Expected shared globals from the expert callback:
    ZONES_MID, WINDOW_ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    action_vector_data
"""

import csv
import json
import numpy as np


# ============================================================
# CONFIG
# ============================================================

REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       2.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

# ---- lighting energy from action-derived dimming (nominal power * duty) ----
# Reporting timestep. 6 steps/hour = 10-min steps. Set to match the run.
STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

# Thermal exploration mode ids (must match the exploratory callback).
THERMAL_MODE_TO_ID = {
    "idle": 0,
    "random_inactive_band": 1,
    "active_exploration": 2,
    "expert_recovery": 3,
    "noisy_recovery": 4,
}

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM ACTION-DERIVED DIMMING
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps exactly to a
# dimming duty u = (a + 1) / 2 in [0, 1]. Energy is therefore reconstructed as
#   E = sum_z ( nominal_W[z] * u[z] ) * dt_hours / 1000   (kWh)
# using the action at time t (energy is a property of a_t in (s_t, a_t, r_t)).
# This does NOT use meters or the logged Lights Electricity Rate; those remain
# available as observations and for validation:  P_EnergyPlus ~= sum_z P_nom*u.


def compute_lighting_energy_kwh(t):
    action_t = np.asarray(
        action_vector_data["action"][t],
        dtype=np.float64,
    )

    # Convert normalized [-1, 1] actions to dimming [0, 1]
    dimming = np.clip(
        (action_t[LIGHT_SLICE] + 1.0) / 2.0,
        0.0,
        1.0,
    )

    nominal_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )

    zone_power_w = nominal_w * dimming
    total_power_w = zone_power_w.sum()

    energy_kwh = (
        total_power_w
        * TIMESTEP_HOURS
        / 1000.0
    )

    return float(energy_kwh)


# ============================================================
# REWARD — components + scalar
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    # Visual comfort is a property of the lighting ACTION at t, so gate it on
    # occupancy at t (avoids the artificial under-lighting penalty at the 08:00
    # transition where occupancy flips between t and t+1).
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    # lighting energy from action-derived dimming at time t (nominal * duty)
    lighting_energy_kwh = compute_lighting_energy_kwh(t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = selection_mask_t
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def normalize_reward_components(components_raw, comp_names, component_scales=None):
    C = len(comp_names)
    importance_vec = np.array([REWARD_IMPORTANCE.get(nm, 1.0) for nm in comp_names],
                              dtype=np.float64)

    if component_scales is not None:
        comp_scales = np.array([float(component_scales[nm]) for nm in comp_names],
                               dtype=np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "external (shared across experts)"
        apply = True
    elif REWARD_NORMALIZE:
        comp_scales = np.abs(components_raw).mean(axis=0).astype(np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "dataset-local mean|abs|"
        apply = True
    else:
        comp_scales = np.ones(C, dtype=np.float64)
        scales_source = "none (raw physical W_* weights)"
        apply = False

    if apply:
        components_final = (components_raw.astype(np.float64) / comp_scales) * importance_vec
    else:
        components_final = components_raw.astype(np.float64).copy()

    if REWARD_COMPONENT_CLIP is not None:
        components_final = np.clip(components_final,
                                   -REWARD_COMPONENT_CLIP, REWARD_COMPONENT_CLIP)

    return components_final.astype(np.float32), comp_scales, importance_vec, scales_source


# ============================================================
# DIAGNOSTICS
# ============================================================

def _print_reward_diagnostics(rewards_raw, rewards_tx, components_arr):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(f"  raw:         mean={rewards_raw.mean():+.4f}  std={rewards_raw.std():.4f}")
    print(f"               min={rewards_raw.min():+.4f}    max={rewards_raw.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_raw, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - rewards_raw
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - raw_total): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - raw_total): {np.max(np.abs(err)):.8f}")


def _print_normalization(comp_names, comp_scales, importance_vec, scales_source):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  clip={REWARD_COMPONENT_CLIP}  "
          f"scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'eff_weight':>10s}")
    for nm, sc, imp in zip(comp_names, comp_scales, importance_vec):
        eff = imp / sc if sc != 0 else 0.0
        print(f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  {eff:10.4f}")


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None:
        return timeouts
    if TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


# ============================================================
# SHARED REWARD SCALES (compute ONCE across all experts, reuse)
# ============================================================
# For pooled multi-expert offline RL, do NOT let each dataset self-normalize.
# 1) Build every expert's raw components with build_offline_dataset(..., component_scales=None)
#    and grab result["reward_components_raw"] (also saved in each NPZ).
# 2) Stack them and call compute_shared_scales() to get ONE scale dict.
# 3) Re-run every expert with build_offline_dataset(..., component_scales=SHARED).

def compute_shared_scales(components_raw_list, comp_names=None):
    """
    components_raw_list : list of (T_i, C) arrays of RAW reward components
                          (result["reward_components_raw"] from each expert).
    Returns {component_name: scale} using pooled mean-|abs| across all experts.
    """
    if comp_names is None:
        comp_names = _component_names()
    stacked = np.concatenate([np.asarray(a, dtype=np.float64)
                              for a in components_raw_list], axis=0)
    if stacked.shape[1] != len(comp_names):
        raise ValueError(
            f"components have {stacked.shape[1]} cols but {len(comp_names)} names")
    scales = np.abs(stacked).mean(axis=0)
    scales[scales < 1e-8] = 1.0
    return {nm: float(sc) for nm, sc in zip(comp_names, scales)}


# ============================================================
# MAIN BUILDER
# ============================================================

def build_offline_dataset(
    save_npz=True,
    save_csv=True,
    prefix="noised_smooth_exp_2",
    component_scales=None,
):
    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1

    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "sdar_action": len(action_vector_data["action"]),
        "sdar_prev_action": len(action_vector_data["prev_action"]),
        "sdar_mask": len(action_vector_data["selection_mask"]),
        "sdar_mix": len(action_vector_data["action_mix"]),
        "exploration_expert_action": len(exploration_data["expert_action"]),
        "exploration_executed_action": len(exploration_data["executed_action"]),
        "exploration_mask": len(exploration_data["exploration_mask"]),
        "exploration_glazing": len(exploration_data["glazing_explored"]),
        "exploration_lighting": len(exploration_data["lighting_explored"]),
        "exploration_thermal_mode": len(exploration_data["thermal_mode"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)

    sdar_actions = np.asarray(action_vector_data["action"], dtype=np.float32)[:T]
    sdar_prev_actions = np.asarray(action_vector_data["prev_action"], dtype=np.float32)[:T]
    sdar_masks = np.asarray(action_vector_data["selection_mask"], dtype=np.float32)[:T]
    sdar_mixes = np.asarray(action_vector_data["action_mix"], dtype=np.float32)[:T]

    for name, arr in [
        ("sdar_actions", sdar_actions),
        ("sdar_prev_actions", sdar_prev_actions),
        ("sdar_masks", sdar_masks),
        ("sdar_mixes", sdar_mixes),
    ]:
        if arr.ndim != 2 or arr.shape != (T, ACTION_DIM):
            raise ValueError(
                f"{name} has shape {arr.shape}, expected ({T}, {ACTION_DIM}). "
                "Did the callback use the 19-dim SDAR layout?"
            )

    # ---- #5: primary action is the EXECUTED action; verify it matches SDAR ----
    executed_actions = np.asarray(
        exploration_data["executed_action"], dtype=np.float32)[:T]
    if not np.allclose(executed_actions, sdar_actions, atol=1e-6):
        raise ValueError(
            "Executed exploration actions do not match SDAR actions.")
    actions = sdar_actions  # never train on expert_action

    # #2: hard guard — normalized actions must lie within [-1, 1].
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: "
            f"min={actions.min()}, max={actions.max()}")

    # ---- #7: expert action + exploration mask ----
    expert_actions = np.asarray(
        exploration_data["expert_action"], dtype=np.float32)[:T]
    exploration_masks = np.asarray(
        exploration_data["exploration_mask"], dtype=np.float32)[:T]

    # per-zone explored flags -> arrays
    glazing_explored = np.array([
        [float(exploration_data["glazing_explored"][t][zone])
         for zone in WINDOW_ZONES_MID]
        for t in range(T)], dtype=np.float32)
    lighting_explored = np.array([
        [float(exploration_data["lighting_explored"][t][zone])
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.float32)

    # thermal modes -> integer ids
    thermal_mode_ids = np.array([
        [THERMAL_MODE_TO_ID[exploration_data["thermal_mode"][t][zone]]
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.int8)

    # ---- #2: physical lighting quantities from executed dimming ----
    lighting_dimming = np.array([
        np.clip((np.asarray(action_vector_data["action"][t])[LIGHT_SLICE] + 1.0) / 2.0,
                0.0, 1.0)
        for t in range(T)], dtype=np.float32)
    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0
    ).astype(np.float32)

    comp_names = _component_names()
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, sdar_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, comp_scales, importance_vec, scales_source = \
        normalize_reward_components(components_raw, comp_names, component_scales)

    rewards_raw = components_final.sum(axis=1).astype(np.float32)
    rewards_tx = _transform_vec(rewards_raw)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32
    )

    print("\n=== Dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (primary, SDAR-ready, per-zone lighting norm)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {sdar_prev_actions.shape}")
    print(f"  selection_masks:   {sdar_masks.shape}")
    print(f"  action_mixes:      {sdar_mixes.shape}")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced sum, pre-transform)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(f"  timeouts:          {timeouts.shape}   artificial timeouts={int(timeouts.sum())}")
    if SAVE_REWARD_COMPONENTS:
        print(f"  reward_components: {components_final.shape}  ({len(comp_names)} terms, final)")

    _print_normalization(comp_names, comp_scales, importance_vec, scales_source)
    _print_reward_diagnostics(rewards_raw, rewards_tx, components_final)
    _print_thermal_violation_rate()
    _print_switching_diagnostics(sdar_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=sdar_prev_actions,
            selection_masks=sdar_masks,
            action_mixes=sdar_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(REWARD_TRANSFORM),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # exploration provenance
            expert_actions=expert_actions,
            executed_actions=executed_actions,
            exploration_masks=exploration_masks,
            glazing_explored=glazing_explored,
            lighting_explored=lighting_explored,
            thermal_mode_ids=thermal_mode_ids,
            exploration_seed=np.int64(exploration_data["seed"]),
            exploration_epsilon=np.asarray(
                exploration_data["epsilon"], dtype=np.float32)[:T],
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            action_layout=np.array(
                {
                    "glazing": [0, 4],
                    "lighting": [4, 9],
                    "heating": [9, 14],
                    "cooling": [14, 19],
                },
                dtype=object,
            ),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)

        if "expert_id" in temporal_data:
            save_dict["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "reward_transform": REWARD_TRANSFORM,
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "reward_scales_source": scales_source,
                    "lighting_and_hvac_energy_terms_are_meter_free": True,
                    "obs_is_meter_free": True,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "obs_cols": obs_column_names(),
                    "act_cols": action_column_names(),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_tx_min": float(rewards_tx.min()),
                    "reward_tx_max": float(rewards_tx.max()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_importance_config": REWARD_IMPORTANCE,
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(sdar_masks.mean()),
                    "mean_selection_mask_glazing": float(sdar_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(sdar_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(sdar_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(sdar_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "timeout_steps": None if TIMEOUT_STEPS is None else int(TIMEOUT_STEPS),
                    "num_timeouts": int(timeouts.sum()),
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + sdar_masks[t].tolist()
                    + [float(rewards[t]), float(rewards_raw[t])]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    result = {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": sdar_prev_actions,
        "selection_masks": sdar_masks,
        "action_mixes": sdar_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_components": components_final,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "expert_actions": expert_actions,
        "executed_actions": executed_actions,
        "exploration_masks": exploration_masks,
        "glazing_explored": glazing_explored,
        "lighting_explored": lighting_explored,
        "thermal_mode_ids": thermal_mode_ids,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
    }

    if "expert_id" in temporal_data:
        result["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

    return result


if __name__ == "__main__":
    dataset = build_offline_dataset(save_npz=True, save_csv=True)

In [ ]:
"""
Offline RL dataset generator — v8
Meter-free reward + meter-free obs + fixed comfort bands + per-zone lighting
+ per-component reward normalization (unit mean-|abs| then intentional weights)
============================================================================

Drop-in dataset builder to run AFTER an expert EnergyPlus/Sinergym callback
rollout (v12 controller) has filled the global logging dictionaries.

Expected shared globals from the expert callback:
    ZONES_MID, WINDOW_ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    action_vector_data
"""

import csv
import json
import numpy as np


# ============================================================
# CONFIG
# ============================================================

REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       2.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

# ---- lighting energy from action-derived dimming (nominal power * duty) ----
# Reporting timestep. 6 steps/hour = 10-min steps. Set to match the run.
STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

# Thermal exploration mode ids (must match the exploratory callback).
THERMAL_MODE_TO_ID = {
    "idle": 0,
    "random_inactive_band": 1,
    "active_exploration": 2,
    "expert_recovery": 3,
    "noisy_recovery": 4,
    "sdar_repeat": 5,
}

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM ACTION-DERIVED DIMMING
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps exactly to a
# dimming duty u = (a + 1) / 2 in [0, 1]. Energy is therefore reconstructed as
#   E = sum_z ( nominal_W[z] * u[z] ) * dt_hours / 1000   (kWh)
# using the action at time t (energy is a property of a_t in (s_t, a_t, r_t)).
# This does NOT use meters or the logged Lights Electricity Rate; those remain
# available as observations and for validation:  P_EnergyPlus ~= sum_z P_nom*u.


def compute_lighting_energy_kwh(t):
    action_t = np.asarray(
        action_vector_data["action"][t],
        dtype=np.float64,
    )

    # Convert normalized [-1, 1] actions to dimming [0, 1]
    dimming = np.clip(
        (action_t[LIGHT_SLICE] + 1.0) / 2.0,
        0.0,
        1.0,
    )

    nominal_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )

    zone_power_w = nominal_w * dimming
    total_power_w = zone_power_w.sum()

    energy_kwh = (
        total_power_w
        * TIMESTEP_HOURS
        / 1000.0
    )

    return float(energy_kwh)


# ============================================================
# REWARD — components + scalar
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    # Visual comfort is a property of the lighting ACTION at t, so gate it on
    # occupancy at t (avoids the artificial under-lighting penalty at the 08:00
    # transition where occupancy flips between t and t+1).
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    # lighting energy from action-derived dimming at time t (nominal * duty)
    lighting_energy_kwh = compute_lighting_energy_kwh(t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = selection_mask_t
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def normalize_reward_components(components_raw, comp_names, component_scales=None):
    C = len(comp_names)
    importance_vec = np.array([REWARD_IMPORTANCE.get(nm, 1.0) for nm in comp_names],
                              dtype=np.float64)

    if component_scales is not None:
        comp_scales = np.array([float(component_scales[nm]) for nm in comp_names],
                               dtype=np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "external (shared across experts)"
        apply = True
    elif REWARD_NORMALIZE:
        comp_scales = np.abs(components_raw).mean(axis=0).astype(np.float64)
        comp_scales[comp_scales < 1e-8] = 1.0
        scales_source = "dataset-local mean|abs|"
        apply = True
    else:
        comp_scales = np.ones(C, dtype=np.float64)
        scales_source = "none (raw physical W_* weights)"
        apply = False

    if apply:
        components_final = (components_raw.astype(np.float64) / comp_scales) * importance_vec
    else:
        components_final = components_raw.astype(np.float64).copy()

    if REWARD_COMPONENT_CLIP is not None:
        components_final = np.clip(components_final,
                                   -REWARD_COMPONENT_CLIP, REWARD_COMPONENT_CLIP)

    return components_final.astype(np.float32), comp_scales, importance_vec, scales_source


# ============================================================
# DIAGNOSTICS
# ============================================================

def _print_reward_diagnostics(rewards_raw, rewards_tx, components_arr):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(f"  raw:         mean={rewards_raw.mean():+.4f}  std={rewards_raw.std():.4f}")
    print(f"               min={rewards_raw.min():+.4f}    max={rewards_raw.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_raw, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - rewards_raw
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - raw_total): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - raw_total): {np.max(np.abs(err)):.8f}")


def _print_normalization(comp_names, comp_scales, importance_vec, scales_source):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  clip={REWARD_COMPONENT_CLIP}  "
          f"scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'eff_weight':>10s}")
    for nm, sc, imp in zip(comp_names, comp_scales, importance_vec):
        eff = imp / sc if sc != 0 else 0.0
        print(f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  {eff:10.4f}")


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None:
        return timeouts
    if TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


# ============================================================
# SHARED REWARD SCALES (compute ONCE across all experts, reuse)
# ============================================================
# For pooled multi-expert offline RL, do NOT let each dataset self-normalize.
# 1) Build every expert's raw components with build_offline_dataset(..., component_scales=None)
#    and grab result["reward_components_raw"] (also saved in each NPZ).
# 2) Stack them and call compute_shared_scales() to get ONE scale dict.
# 3) Re-run every expert with build_offline_dataset(..., component_scales=SHARED).

def compute_shared_scales(components_raw_list, comp_names=None):
    """
    components_raw_list : list of (T_i, C) arrays of RAW reward components
                          (result["reward_components_raw"] from each expert).
    Returns {component_name: scale} using pooled mean-|abs| across all experts.
    """
    if comp_names is None:
        comp_names = _component_names()
    stacked = np.concatenate([np.asarray(a, dtype=np.float64)
                              for a in components_raw_list], axis=0)
    if stacked.shape[1] != len(comp_names):
        raise ValueError(
            f"components have {stacked.shape[1]} cols but {len(comp_names)} names")
    scales = np.abs(stacked).mean(axis=0)
    scales[scales < 1e-8] = 1.0
    return {nm: float(sc) for nm, sc in zip(comp_names, scales)}


# ============================================================
# MAIN BUILDER
# ============================================================

def _default_prefix():
    """Unique per-rollout filename so episodes don't overwrite each other."""
    try:
        return (
            f"offline_{EXPERT_MODE}_"
            f"{exploration_data.get('noise_profile', NOISE_PROFILE)}_"
            f"episode_{exploration_data.get('episode_id', EPISODE_ID):03d}"
        )
    except Exception:
        return "offline_sdar_rl_replay_dataset"


def build_offline_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    component_scales=None,
):
    if prefix is None:
        prefix = _default_prefix()
    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1

    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "sdar_action": len(action_vector_data["action"]),
        "sdar_prev_action": len(action_vector_data["prev_action"]),
        "sdar_mask": len(action_vector_data["selection_mask"]),
        "sdar_mix": len(action_vector_data["action_mix"]),
        "exploration_expert_action": len(exploration_data["expert_action"]),
        "exploration_executed_action": len(exploration_data["executed_action"]),
        "exploration_mask": len(exploration_data["exploration_mask"]),
        "exploration_glazing": len(exploration_data["glazing_explored"]),
        "exploration_lighting": len(exploration_data["lighting_explored"]),
        "exploration_thermal_mode": len(exploration_data["thermal_mode"]),
        "exploration_epsilon": len(exploration_data["epsilon"]),
        "exploration_sdar_forced_repeat": len(exploration_data["sdar_forced_repeat"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)

    sdar_actions = np.asarray(action_vector_data["action"], dtype=np.float32)[:T]
    sdar_prev_actions = np.asarray(action_vector_data["prev_action"], dtype=np.float32)[:T]
    sdar_masks = np.asarray(action_vector_data["selection_mask"], dtype=np.float32)[:T]
    sdar_mixes = np.asarray(action_vector_data["action_mix"], dtype=np.float32)[:T]

    for name, arr in [
        ("sdar_actions", sdar_actions),
        ("sdar_prev_actions", sdar_prev_actions),
        ("sdar_masks", sdar_masks),
        ("sdar_mixes", sdar_mixes),
    ]:
        if arr.ndim != 2 or arr.shape != (T, ACTION_DIM):
            raise ValueError(
                f"{name} has shape {arr.shape}, expected ({T}, {ACTION_DIM}). "
                "Did the callback use the 19-dim SDAR layout?"
            )

    # ---- #5: primary action is the EXECUTED action; verify it matches SDAR ----
    executed_actions = np.asarray(
        exploration_data["executed_action"], dtype=np.float32)[:T]
    if not np.allclose(executed_actions, sdar_actions, atol=1e-6):
        raise ValueError(
            "Executed exploration actions do not match SDAR actions.")
    actions = sdar_actions  # never train on expert_action

    # #2: hard guard — normalized actions must lie within [-1, 1].
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: "
            f"min={actions.min()}, max={actions.max()}")

    # ---- #7: expert action + exploration mask ----
    expert_actions = np.asarray(
        exploration_data["expert_action"], dtype=np.float32)[:T]
    exploration_masks = np.asarray(
        exploration_data["exploration_mask"], dtype=np.float32)[:T]
    sdar_forced_repeat = np.asarray(
        exploration_data["sdar_forced_repeat"], dtype=np.float32)[:T]

    # per-zone explored flags -> arrays
    glazing_explored = np.array([
        [float(exploration_data["glazing_explored"][t][zone])
         for zone in WINDOW_ZONES_MID]
        for t in range(T)], dtype=np.float32)
    lighting_explored = np.array([
        [float(exploration_data["lighting_explored"][t][zone])
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.float32)

    # thermal modes -> integer ids
    thermal_mode_ids = np.array([
        [THERMAL_MODE_TO_ID[exploration_data["thermal_mode"][t][zone]]
         for zone in ZONES_MID]
        for t in range(T)], dtype=np.int8)

    # ---- #2: physical lighting quantities from executed dimming ----
    lighting_dimming = np.array([
        np.clip((np.asarray(action_vector_data["action"][t])[LIGHT_SLICE] + 1.0) / 2.0,
                0.0, 1.0)
        for t in range(T)], dtype=np.float32)
    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0
    ).astype(np.float32)

    comp_names = _component_names()
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, sdar_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, comp_scales, importance_vec, scales_source = \
        normalize_reward_components(components_raw, comp_names, component_scales)

    rewards_raw = components_final.sum(axis=1).astype(np.float32)
    rewards_tx = _transform_vec(rewards_raw)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32
    )

    print("\n=== Dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (primary, SDAR-ready, per-zone lighting norm)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {sdar_prev_actions.shape}")
    print(f"  selection_masks:   {sdar_masks.shape}")
    print(f"  action_mixes:      {sdar_mixes.shape}")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced sum, pre-transform)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(f"  timeouts:          {timeouts.shape}   artificial timeouts={int(timeouts.sum())}")
    if SAVE_REWARD_COMPONENTS:
        print(f"  reward_components: {components_final.shape}  ({len(comp_names)} terms, final)")

    _print_normalization(comp_names, comp_scales, importance_vec, scales_source)
    _print_reward_diagnostics(rewards_raw, rewards_tx, components_final)
    _print_thermal_violation_rate()
    _print_switching_diagnostics(sdar_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=sdar_prev_actions,
            selection_masks=sdar_masks,
            action_mixes=sdar_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(REWARD_TRANSFORM),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # exploration provenance
            expert_actions=expert_actions,
            executed_actions=executed_actions,
            exploration_masks=exploration_masks,
            glazing_explored=glazing_explored,
            lighting_explored=lighting_explored,
            thermal_mode_ids=thermal_mode_ids,
            exploration_seed=np.int64(exploration_data["seed"]),
            exploration_epsilon=np.asarray(
                exploration_data["epsilon"], dtype=np.float32)[:T],
            sdar_forced_repeat=sdar_forced_repeat,
            episode_id=np.int64(exploration_data.get("episode_id", -1)),
            noise_profile=np.array(str(exploration_data.get("noise_profile", "unknown"))),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            action_layout=np.array(
                {
                    "glazing": [0, 4],
                    "lighting": [4, 9],
                    "heating": [9, 14],
                    "cooling": [14, 19],
                },
                dtype=object,
            ),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)

        if "expert_id" in temporal_data:
            save_dict["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "reward_transform": REWARD_TRANSFORM,
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "reward_scales_source": scales_source,
                    "lighting_and_hvac_energy_terms_are_meter_free": True,
                    "obs_is_meter_free": True,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "obs_cols": obs_column_names(),
                    "act_cols": action_column_names(),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_tx_min": float(rewards_tx.min()),
                    "reward_tx_max": float(rewards_tx.max()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_importance_config": REWARD_IMPORTANCE,
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(sdar_masks.mean()),
                    "mean_selection_mask_glazing": float(sdar_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(sdar_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(sdar_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(sdar_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "timeout_steps": None if TIMEOUT_STEPS is None else int(TIMEOUT_STEPS),
                    "num_timeouts": int(timeouts.sum()),
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + sdar_masks[t].tolist()
                    + [float(rewards[t]), float(rewards_raw[t])]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    result = {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": sdar_prev_actions,
        "selection_masks": sdar_masks,
        "action_mixes": sdar_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_components": components_final,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "expert_actions": expert_actions,
        "executed_actions": executed_actions,
        "exploration_masks": exploration_masks,
        "sdar_forced_repeat": sdar_forced_repeat,
        "glazing_explored": glazing_explored,
        "lighting_explored": lighting_explored,
        "thermal_mode_ids": thermal_mode_ids,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
    }

    if "expert_id" in temporal_data:
        result["expert_ids"] = np.asarray(temporal_data["expert_id"], dtype=np.int64)[:T]

    return result


if __name__ == "__main__":
    dataset = build_offline_dataset(save_npz=True, save_csv=True)

In [ ]:
from pathlib import Path
import re
import numpy as np


INPUT_PATTERN = "offline_smooth_*_episode_*.npz"
OUTPUT_PATH = "pooled_sdar_experts.npz"


def episode_number(path):
    match = re.search(r"episode_(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Cannot extract episode number from {path.name}")
    return int(match.group(1))


paths = sorted(Path(".").glob(INPUT_PATTERN), key=episode_number)

if not paths:
    raise FileNotFoundError(f"No files matched {INPUT_PATTERN!r}")

print("Episodes:")
for path in paths:
    print(" ", path.name)


all_arrays = {}
episode_lengths = []
episode_ids = []
episode_seeds = []

for path in paths:
    with np.load(path, allow_pickle=True) as data:
        episode = {key: np.array(data[key], copy=True) for key in data.files}

    n = len(episode["observations"])

    # Essential shape checks
    required = [
        "observations",
        "actions",
        "actions_raw",
        "prev_actions",
        "selection_masks",
        "action_mixes",
        "rewards",
        "rewards_raw",
        "reward_components_raw",
        "next_observations",
        "terminals",
    ]

    for key in required:
        if key not in episode:
            raise KeyError(f"{path.name} is missing {key!r}")

    for key in required:
        if episode[key].shape[0] != n:
            raise ValueError(
                f"{path.name}: {key} has length "
                f"{episode[key].shape[0]}, expected {n}"
            )

    terminals = episode["terminals"].reshape(-1)

    if int(terminals.sum()) != 1 or terminals[-1] != 1:
        raise ValueError(
            f"{path.name}: expected exactly one terminal at the final step"
        )

    if episode["actions"].shape[1] != 19:
        raise ValueError(f"{path.name}: actions must have 19 dimensions")

    if np.max(np.abs(episode["actions"])) > 1.0001:
        raise ValueError(f"{path.name}: normalized actions exceed [-1,1]")

    # Automatically retain every transition-aligned field
    transition_keys = {
        key
        for key, value in episode.items()
        if value.ndim >= 1 and value.shape[0] == n
    }

    for key in transition_keys:
        all_arrays.setdefault(key, []).append(episode[key])

    ep_id = episode_number(path)
    episode_ids.append(np.full(n, ep_id, dtype=np.int32))
    episode_lengths.append(n)

    seed = int(np.asarray(episode.get("exploration_seed", -1)).item())
    episode_seeds.append(seed)


# Concatenate complete episodes without interleaving them
pooled = {
    key: np.concatenate(values, axis=0)
    for key, values in all_arrays.items()
}

pooled["episode_ids"] = np.concatenate(episode_ids)
pooled["episode_lengths"] = np.asarray(episode_lengths, dtype=np.int32)
pooled["episode_seeds"] = np.asarray(episode_seeds, dtype=np.int64)
pooled["source_files"] = np.asarray([p.name for p in paths])


# ============================================================
# Recompute shared reward-component scales across all episodes
# ============================================================

components_raw = pooled["reward_components_raw"].astype(np.float64)

shared_scales = np.mean(np.abs(components_raw), axis=0)
shared_scales[shared_scales < 1e-8] = 1.0

with np.load(paths[0], allow_pickle=True) as first:
    importance = np.asarray(
        first["reward_component_importance"],
        dtype=np.float64,
    )
    component_names = np.array(first["reward_component_names"], copy=True)
    component_clip = float(first["reward_component_clip"])

components = (
    components_raw / shared_scales[None, :]
) * importance[None, :]

if component_clip >= 0:
    components = np.clip(
        components,
        -component_clip,
        component_clip,
    )

rewards_raw = components.sum(axis=1)
rewards = np.sign(rewards_raw) * np.log1p(np.abs(rewards_raw))

pooled["reward_components"] = components.astype(np.float32)
pooled["reward_component_scales"] = shared_scales.astype(np.float32)
pooled["reward_component_importance"] = importance.astype(np.float32)
pooled["reward_component_names"] = component_names
pooled["rewards_raw"] = rewards_raw.astype(np.float32)
pooled["rewards"] = rewards.astype(np.float32)


# Recompute pooled statistics
pooled["obs_mean"] = pooled["observations"].mean(axis=0).astype(np.float32)
pooled["obs_std"] = (
    pooled["observations"].std(axis=0) + 1e-6
).astype(np.float32)

pooled["act_mean"] = pooled["actions"].mean(axis=0).astype(np.float32)
pooled["act_std"] = (
    pooled["actions"].std(axis=0) + 1e-6
).astype(np.float32)


# Final checks
expected_transitions = sum(episode_lengths)

assert len(pooled["observations"]) == expected_transitions
assert int(pooled["terminals"].sum()) == len(paths)
assert pooled["terminals"][-1] == 1

np.savez_compressed(OUTPUT_PATH, **pooled)

print("\nSaved:", OUTPUT_PATH)
print("Episodes:", len(paths))
print("Transitions:", expected_transitions)
print("Terminal flags:", int(pooled["terminals"].sum()))
print("Observations:", pooled["observations"].shape)
print("Actions:", pooled["actions"].shape)

In [ ]:
"""
Agent-rollout dataset builder — evaluation counterpart of the v8 expert builder
================================================================================

Runs AFTER the evaluation callback (`evaluate_agent_callback.py`) has rolled the
TRAINED SDAR-IQL agent through EnergyPlus and filled the in-memory logging dicts.
It reconstructs the same offline-RL dataset format from the agent's trajectory so
you can score the policy (return, per-component reward, thermal violations,
switching, energy) and, if you want, re-pool the agent rollout with the training
data.

WHAT'S DIFFERENT FROM THE v8 EXPERT BUILDER
-------------------------------------------
The reward math, observation layout, and diagnostics are IDENTICAL (reward is a
property of states/actions, not of which policy produced them). Only the action
source changes:

  expert builder                     agent builder (this file)
  ------------------------------     --------------------------------------
  action_vector_data["action"]   ->  agent_data["executed_action_norm"]
  action_vector_data["...mask"]  ->  agent_data["selection_mask"]
  prev_action / action_mix logged -> RECONSTRUCTED from the agent trajectory
  exploration_data (expert vs        (removed — no exploration during eval;
    executed, thermal modes, ...)     selector sampling is logged directly)

Required in-memory globals (filled by the eval callback rollout):
    ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    agent_data  (observation, selection_logits, selection_probability,
                 selection_mask, executed_action_norm)

REWARD COMPARABILITY (important)
--------------------------------
This version REQUIRES the pooled decomposed training NPZ as a reward reference.
It loads the exact:

    reward_component_names
    reward_component_scales
    reward_component_importance
    reward_component_clips
    reward_transform

used by training. For the current decomposed run that means thermal importance
3.0, thermal clip 50.0, and clip 10.0 for all other components. The evaluated
rollout never defines its own normalization scales, so rewards remain comparable
across checkpoints and against the experts.

`rewards_raw` is kept as the training-compatible name for the balanced,
post-normalization/post-importance/post-component-clip sum before symlog.
`reward_physical_raw` is additionally saved for the sum of physical W_* reward
components before normalization.
"""

import csv
import hashlib
import json
import re
from pathlib import Path

import numpy as np


# ============================================================
# ROLLOUT LOGS — provided by the eval callback
# ============================================================
# If this modupace),
# bind the logging globals from the eval callback module. They are the same
# mutable objects the callback appended to, so they carry the rollout dle is imported standalone (not exec'd in the rollout namesata.
try:
    agent_data  # noqa: F821  (present when run in the rollout namespace)
except NameError:
    from evaluate_agent_callback import (  # noqa: F401
        ZONES_MID,
        AVAILABLE_GLAZING_STATES,
        temperature_data,
        lighting_data,
        solar_data,
        ext_irr_data,
        wpi_data,
        meter_data,
        output_variable_data,
        temporal_data,
        glazing_state_data,
        lighting_power_data,
        heating_setpoint_data,
        cooling_setpoint_data,
        agent_data,
        SELECTOR_MODE,
        PROPOSAL_DETERMINISTIC,
        SELECTOR_SEED,
    )


# ============================================================
# CONFIG
# ============================================================

# These values are validated against the reference NPZ. They are not used as an
# independent reward definition.
REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0  # legacy/default clip; per-component clips come from NPZ

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR

POLICY_TAG = "sdar_iql_decomposed_agent"

# ---------------------------------------------------------------------------
# EDIT THESE FOR EACH CHECKPOINT EVALUATION.
# Use the decomposed pooled dataset that trained the agent. If it is elsewhere,
# provide the absolute path.
# ---------------------------------------------------------------------------
REFERENCE_REWARD_DATASET = (
    "pooled_sdar_experts_thermal3_tclip50_decomposed.npz"
)
EVAL_CHECKPOINT_PATH = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_checkpoint_epoch100.pt"   # e.g. "/.../checkpoint_epoch150.pt"
EVAL_CHECKPOINT_EPOCH = None  # optional; inferred from the checkpoint filename
EVAL_SEED = 123
EVAL_SELECTOR_MODE = str(globals().get("SELECTOR_MODE", "sample")).lower()
EVAL_PROPOSAL_DETERMINISTIC = bool(
    globals().get("PROPOSAL_DETERMINISTIC", True)
)
EVAL_SELECTOR_SEED = int(globals().get("SELECTOR_SEED", 20260728))
EVAL_SAFEGUARD_ENABLED = True
EVAL_WEATHER_FILE = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}

HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
ACTION_VALIDATION_ATOL = 0.5


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    """Physical (actuated) action, read from the eval callback's logs."""
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM THE AGENT'S NORMALIZED ACTION
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps to a dimming duty
#   u = (a + 1) / 2 in [0, 1], and E = sum_z nominal_W[z]*u[z] * dt / 1000 (kWh).
# Meter-free: uses the executed action, not the logged Lights Electricity Rate.

def compute_lighting_energy_kwh(action_norm_t):
    a = np.asarray(action_norm_t, dtype=np.float64)
    dimming = np.clip((a[LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
    nominal_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float64)
    total_power_w = float((nominal_w * dimming).sum())
    return total_power_w * TIMESTEP_HOURS / 1000.0


# ============================================================
# REWARD — components + scalar (identical logic to the v8 builder)
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, action_norm_t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_energy_kwh = compute_lighting_energy_kwh(action_norm_t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = np.asarray(selection_mask_t)
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _npz_scalar(data, key, default=None):
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar NPZ field {key!r}, got shape {value.shape}.")
    return value.reshape(()).item()


def load_reward_reference(reference_dataset_path, comp_names):
    """
    Load the immutable reward ruler from the pooled decomposed training NPZ.

    There is deliberately no rollout-local fallback. A missing or incompatible
    reference dataset is an error because otherwise checkpoint rewards would be
    calculated on different scales.
    """
    path = Path(reference_dataset_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "Reward reference dataset not found: "
            f"{path}. Set REFERENCE_REWARD_DATASET to the exact pooled NPZ used "
            "to train the decomposed agent."
        )

    required = {
        "reward_component_names",
        "reward_component_scales",
        "reward_component_importance",
    }
    with np.load(path, allow_pickle=False) as data:
        missing = sorted(required.difference(data.files))
        if missing:
            raise KeyError(
                f"Reference NPZ {path} is missing required reward metadata: {missing}"
            )

        ref_names = [str(x) for x in np.asarray(data["reward_component_names"]).tolist()]
        if ref_names != list(comp_names):
            raise ValueError(
                "Reward component order differs from the training dataset.\n"
                f"  builder:   {list(comp_names)}\n"
                f"  reference: {ref_names}"
            )

        scales = np.asarray(data["reward_component_scales"], dtype=np.float64)
        importance = np.asarray(
            data["reward_component_importance"], dtype=np.float64
        )
        if scales.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_scales shape {scales.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if importance.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_importance shape {importance.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if np.any(~np.isfinite(scales)) or np.any(scales <= 0.0):
            raise ValueError("Reference reward scales must all be finite and positive.")
        if np.any(~np.isfinite(importance)):
            raise ValueError("Reference reward importance values must all be finite.")

        if "reward_component_clips" in data.files:
            clips = np.asarray(data["reward_component_clips"], dtype=np.float64)
        else:
            legacy_clip = float(
                _npz_scalar(data, "reward_component_clip", REWARD_COMPONENT_CLIP)
            )
            clips = np.full(len(comp_names), legacy_clip, dtype=np.float64)
            if "thermal_reward_component_clip" in data.files:
                thermal_i = comp_names.index("thermal_comfort")
                clips[thermal_i] = float(
                    _npz_scalar(data, "thermal_reward_component_clip")
                )

        if clips.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_clips shape {clips.shape}; "
                f"expected {(len(comp_names),)}"
            )
        # A negative or non-finite stored clip means unbounded for that component.
        clips = np.where(np.isfinite(clips) & (clips >= 0.0), clips, np.inf)

        transform = str(
            _npz_scalar(data, "reward_transform", REWARD_TRANSFORM)
        )
        reweighting_tag = str(
            _npz_scalar(data, "reward_reweighting_tag", "")
        )
        reference_mask_value = float(
            _npz_scalar(data, "action_mask_value", ACTION_MASK_VALUE)
        )

    configured_importance = np.array(
        [REWARD_IMPORTANCE.get(name, 1.0) for name in comp_names],
        dtype=np.float64,
    )
    if not np.allclose(configured_importance, importance, atol=1e-7, rtol=0.0):
        raise ValueError(
            "Builder REWARD_IMPORTANCE differs from the training NPZ. "
            "Do not evaluate until the reward definitions match.\n"
            f"  builder:   {configured_importance.tolist()}\n"
            f"  reference: {importance.tolist()}"
        )
    if transform != REWARD_TRANSFORM:
        raise ValueError(
            f"Builder transform {REWARD_TRANSFORM!r} differs from reference "
            f"transform {transform!r}."
        )
    if not np.isclose(reference_mask_value, ACTION_MASK_VALUE):
        raise ValueError(
            f"Reference action_mask_value={reference_mask_value} differs from "
            f"builder ACTION_MASK_VALUE={ACTION_MASK_VALUE}."
        )

    return {
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
        "names": list(comp_names),
        "scales": scales,
        "importance": importance,
        "clips": clips,
        "transform": transform,
        "reweighting_tag": reweighting_tag,
    }


def apply_reward_reference(components_raw, reward_reference):
    """Apply training scales, importance, and per-component clips exactly."""
    scales = reward_reference["scales"]
    importance = reward_reference["importance"]
    clips = reward_reference["clips"]

    unbounded = (
        components_raw.astype(np.float64)
        / scales[None, :]
        * importance[None, :]
    )
    components_final = np.clip(
        unbounded,
        -clips[None, :],
        clips[None, :],
    )
    return components_final.astype(np.float32), unbounded.astype(np.float32)


def _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch):
    if checkpoint_epoch is not None:
        epoch = int(checkpoint_epoch)
        if epoch < 0:
            raise ValueError("checkpoint_epoch must be non-negative.")
        return epoch
    if checkpoint_path:
        match = re.search(r"epoch[_-]?(\d+)", Path(checkpoint_path).name, re.IGNORECASE)
        if match:
            return int(match.group(1))
    raise ValueError(
        "Checkpoint epoch is required for unambiguous output naming. Set "
        "EVAL_CHECKPOINT_EPOCH or use a checkpoint filename containing "
        "'epoch<number>'."
    )


def _checkpoint_provenance(checkpoint_path, checkpoint_epoch):
    epoch = _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch)
    if checkpoint_path is None:
        return {
            "epoch": epoch,
            "path": "",
            "sha256": "",
        }

    path = Path(checkpoint_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return {
        "epoch": epoch,
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
    }


# ============================================================
# DIAGNOSTICS (identical to the v8 builder)
# ============================================================

def _print_reward_diagnostics(
    reward_physical_raw,
    reward_balanced,
    rewards_tx,
    components_arr,
):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(
        f"  physical raw: mean={reward_physical_raw.mean():+.4f}  "
        f"std={reward_physical_raw.std():.4f}"
    )
    print(
        f"                min={reward_physical_raw.min():+.4f}    "
        f"max={reward_physical_raw.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_physical_raw, qs)]}")
    print(
        f"  balanced:     mean={reward_balanced.mean():+.4f}  "
        f"std={reward_balanced.std():.4f}"
    )
    print(
        f"                min={reward_balanced.min():+.4f}    "
        f"max={reward_balanced.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_balanced, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - reward_balanced
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - balanced): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - balanced): {np.max(np.abs(err)):.8f}")


def _print_normalization(
    comp_names,
    comp_scales,
    importance_vec,
    component_clips,
    scales_source,
):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'clip':>8s}  {'eff_weight':>10s}")
    for nm, sc, imp, clip in zip(
        comp_names, comp_scales, importance_vec, component_clips
    ):
        eff = imp / sc if sc != 0 else 0.0
        clip_name = "none" if not np.isfinite(clip) else f"{clip:.1f}"
        print(
            f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  "
            f"{clip_name:>8s}  {eff:10.4f}"
        )


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics (agent selector) ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_selector_probability_diagnostics(
    selection_probabilities,
    selection_masks,
    selector_mode,
):
    # The callback forces all dimensions to update on the first step, so
    # exclude that step when checking the learned selector realization.
    probabilities = selection_probabilities[1:]
    masks = selection_masks[1:]

    groups = {
        "glazing": GLAZE_SLICE,
        "lighting": LIGHT_SLICE,
        "heating": HEAT_SLICE,
        "cooling": COOL_SLICE,
        "overall": slice(0, ACTION_DIM),
    }

    print(
        "\n=== Selector probability diagnostics "
        f"(mode={selector_mode}) ==="
    )
    stats = {}
    for name, slc in groups.items():
        p = probabilities[:, slc]
        m = masks[:, slc]
        mean_probability = float(p.mean())
        sampled_rate = float(m.mean())
        difference = sampled_rate - mean_probability
        stats[name] = {
            "mean_probability": mean_probability,
            "sampled_rate": sampled_rate,
            "difference": difference,
        }
        print(
            f"  {name:10s} "
            f"mean probability={mean_probability:.4f}  "
            f"sampled rate={sampled_rate:.4f}  "
            f"difference={difference:+.4f}"
        )

    threshold_masks = (probabilities >= 0.5).astype(np.float32)
    stats["threshold_agreement"] = float(
        np.mean(masks == threshold_masks)
    )
    print(
        "  agreement with p>=0.5 threshold: "
        f"{100.0 * stats['threshold_agreement']:.2f}%"
    )
    return stats


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None or TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


def _denormalize_linear(values, physical_range):
    low, high = physical_range
    values = np.asarray(values, dtype=np.float64)
    return low + 0.5 * (values + 1.0) * (high - low)


def _validate_actuated_actions(actions, actions_raw):
    """Verify all 19 normalized executed actions against physical actuation logs."""
    n_glazing = len(AVAILABLE_GLAZING_STATES)
    glz_ids_from_norm = np.clip(
        np.round((actions[:, GLAZE_SLICE] + 1.0) * 0.5 * (n_glazing - 1)),
        0,
        n_glazing - 1,
    ).astype(np.int64)
    glz_ids_logged = actions_raw[:, GLAZE_SLICE].astype(np.int64)
    if not np.array_equal(glz_ids_from_norm, glz_ids_logged):
        mismatch = np.argwhere(glz_ids_from_norm != glz_ids_logged)[0]
        t, dim = (int(mismatch[0]), int(mismatch[1]))
        raise ValueError(
            "Glazing decoded from executed_action_norm does not match the "
            f"actuation log at transition {t}, glazing dim {dim}: "
            f"decoded={glz_ids_from_norm[t, dim]}, logged={glz_ids_logged[t, dim]}."
        )

    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )
    light_decoded = (
        np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
        * nominal_power_w[None, :]
    )
    heat_decoded = _denormalize_linear(
        actions[:, HEAT_SLICE], HEATING_SETPOINT_RANGE
    )
    cool_decoded = _denormalize_linear(
        actions[:, COOL_SLICE], COOLING_SETPOINT_RANGE
    )

    checks = [
        (
            "lighting",
            light_decoded,
            actions_raw[:, LIGHT_SLICE],
            "W",
        ),
        (
            "heating setpoint",
            heat_decoded,
            actions_raw[:, HEAT_SLICE],
            "degC",
        ),
        (
            "cooling setpoint",
            cool_decoded,
            actions_raw[:, COOL_SLICE],
            "degC",
        ),
    ]
    for label, decoded, logged, unit in checks:
        if not np.allclose(
            decoded,
            logged,
            atol=ACTION_VALIDATION_ATOL,
            rtol=1e-4,
        ):
            abs_error = np.abs(decoded - logged)
            t, dim = np.unravel_index(np.argmax(abs_error), abs_error.shape)
            raise ValueError(
                f"{label} decoded from executed_action_norm does not match the "
                f"actuation log. Largest error at transition {t}, local dim {dim}: "
                f"decoded={decoded[t, dim]:.6g} {unit}, "
                f"logged={logged[t, dim]:.6g} {unit}, "
                f"abs_error={abs_error[t, dim]:.6g} {unit}."
            )


def _reconstruct_sdar_context(actions, selection_masks):
    previous_keys = (
        "previous_action_norm",
        "prev_action_norm",
        "previous_executed_action_norm",
    )
    previous_key = next((k for k in previous_keys if k in agent_data), None)

    if previous_key is not None:
        logged_prev = np.asarray(agent_data[previous_key], dtype=np.float32)
        if logged_prev.ndim != 2 or logged_prev.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{previous_key!r}] has shape {logged_prev.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_prev.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{previous_key!r}] has only {logged_prev.shape[0]} "
                f"rows for {len(actions)} transitions."
            )
        prev_actions = logged_prev[:len(actions)].copy()
    else:
        if not np.all(selection_masks[0] == 1.0):
            raise ValueError(
                "The first selection mask is not a full update, but the callback "
                "did not log previous_action_norm. The initial SDAR context cannot "
                "be reconstructed without leaking action[0] into prev_action[0]."
            )
        prev_actions = actions.copy()
        prev_actions[1:] = actions[:-1]

    if len(actions) > 1 and not np.allclose(
        prev_actions[1:],
        actions[:-1],
        atol=1e-5,
        rtol=0.0,
    ):
        max_error = float(
            np.max(np.abs(prev_actions[1:] - actions[:-1]))
        )
        raise ValueError(
            "Logged/reconstructed previous actions do not equal the preceding "
            f"executed actions (max abs error {max_error:.6g})."
        )

    action_mixes = (
        (1.0 - selection_masks) * prev_actions
        + selection_masks * ACTION_MASK_VALUE
    ).astype(np.float32)

    for key in ("action_mix", "action_mixes"):
        if key not in agent_data:
            continue
        logged_mix = np.asarray(agent_data[key], dtype=np.float32)
        if logged_mix.ndim != 2 or logged_mix.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{key!r}] has shape {logged_mix.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_mix.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{key!r}] has only {logged_mix.shape[0]} rows for "
                f"{len(actions)} transitions."
            )
        if not np.allclose(
            action_mixes,
            logged_mix[:len(actions)],
            atol=1e-5,
            rtol=0.0,
        ):
            max_error = float(
                np.max(np.abs(action_mixes - logged_mix[:len(actions)]))
            )
            raise ValueError(
                f"Reconstructed action_mix differs from agent_data[{key!r}] "
                f"(max abs error {max_error:.6g})."
            )
        break

    return prev_actions.astype(np.float32), action_mixes


# ============================================================
# MAIN BUILDER
# ============================================================

def build_agent_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    reference_dataset_path=REFERENCE_REWARD_DATASET,
    checkpoint_path=EVAL_CHECKPOINT_PATH,
    checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
    eval_seed=EVAL_SEED,
    selector_mode=EVAL_SELECTOR_MODE,
    proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
    selector_seed=EVAL_SELECTOR_SEED,
    safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
    weather_file=EVAL_WEATHER_FILE,
):
    selector_mode = str(selector_mode).lower()
    if selector_mode not in {"threshold", "sample"}:
        raise ValueError(
            "selector_mode must be 'threshold' or 'sample', "
            f"got {selector_mode!r}."
        )
    proposal_deterministic = bool(proposal_deterministic)
    selector_seed = int(selector_seed)

    comp_names = _component_names()
    reward_reference = load_reward_reference(
        reference_dataset_path,
        comp_names,
    )
    checkpoint = _checkpoint_provenance(
        checkpoint_path,
        checkpoint_epoch,
    )
    if prefix is None:
        proposal_tag = (
            "propdet" if proposal_deterministic else "propstoch"
        )
        prefix = (
            f"agent_eval_dataset_decomposed_epoch{checkpoint['epoch']}"
            "_thermal3_tclip50"
            f"_selector-{selector_mode}_{proposal_tag}"
            f"_s{selector_seed}"
        )
    prefix = str(Path(prefix).expanduser())
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)

    print("=== Evaluation provenance ===")
    print(f"  checkpoint epoch: {checkpoint['epoch']}")
    print(f"  checkpoint path:  {checkpoint['path'] or '(not supplied)'}")
    print(f"  reference NPZ:    {reward_reference['path']}")
    print(
        "  reward tag:       "
        f"{reward_reference['reweighting_tag'] or '(not stored)'}"
    )
    print(f"  selector mode:    {selector_mode}")
    print(
        "  proposal deterministic: "
        f"{proposal_deterministic}"
    )
    print(f"  selector seed:    {selector_seed}")
    print(f"  safeguard:        {bool(safeguard_enabled)}")
    print(f"  eval seed:        {eval_seed}")
    print(f"  weather file:     {weather_file}")

    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1
    if T < 1:
        raise ValueError(f"Not enough logged steps to build transitions (n={n}).")

    # ---- length sanity check (agent logs only) ----
    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "agent_observation": len(agent_data["observation"]),
        "agent_selection_logits": len(agent_data["selection_logits"]),
        "agent_selection_probability": len(
            agent_data["selection_probability"]
        ),
        "agent_selection_mask": len(agent_data["selection_mask"]),
        "agent_executed_action_norm": len(agent_data["executed_action_norm"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    # ---- observations ----
    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    # Integrity: the obs we rebuild here must equal what the agent actually saw.
    agent_obs = np.asarray(agent_data["observation"], dtype=np.float32)[:T]
    if not np.allclose(observations, agent_obs, atol=1e-4):
        max_err = float(np.max(np.abs(observations - agent_obs)))
        raise ValueError(
            "Rebuilt observations differ from what the agent saw "
            f"(max abs err {max_err:.6g}). The dataset obs layout has drifted "
            "from the eval callback's build_observation.")

    # ---- actions: agent's executed normalized action is the primary action ----
    agent_actions = np.asarray(agent_data["executed_action_norm"], dtype=np.float32)
    agent_logits = np.asarray(
        agent_data["selection_logits"],
        dtype=np.float32,
    )
    agent_probabilities = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )
    agent_masks = np.asarray(agent_data["selection_mask"], dtype=np.float32)

    if agent_actions.shape != (n, ACTION_DIM):
        raise ValueError(f"executed_action_norm has shape {agent_actions.shape}, expected ({n}, {ACTION_DIM}).")
    if agent_logits.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_logits has shape "
            f"{agent_logits.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_probabilities.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_probability has shape "
            f"{agent_probabilities.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_masks.shape != (n, ACTION_DIM):
        raise ValueError(f"selection_mask has shape {agent_masks.shape}, expected ({n}, {ACTION_DIM}).")

    actions = agent_actions[:T]
    selection_logits = agent_logits[:T]
    selection_probabilities = agent_probabilities[:T]
    selection_masks = agent_masks[:T]

    if np.any(~np.isfinite(selection_logits)):
        raise ValueError("selection_logits contains NaN or infinite values.")
    if np.any(~np.isfinite(selection_probabilities)):
        raise ValueError(
            "selection_probability contains NaN or infinite values."
        )
    if (
        np.any(selection_probabilities < -1e-7)
        or np.any(selection_probabilities > 1.0 + 1e-7)
    ):
        raise ValueError(
            "selection_probability must lie in [0,1]: "
            f"min={selection_probabilities.min()}, "
            f"max={selection_probabilities.max()}."
        )

    reconstructed_probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(selection_logits, -80.0, 80.0))
    )
    if not np.allclose(
        selection_probabilities,
        reconstructed_probabilities,
        atol=1e-6,
        rtol=1e-6,
    ):
        max_err = float(
            np.max(
                np.abs(
                    selection_probabilities
                    - reconstructed_probabilities
                )
            )
        )
        raise ValueError(
            "selection_probability is inconsistent with sigmoid(logits): "
            f"max abs error={max_err:.6g}."
        )

    # Normalized executed actions must be finite and lie within [-1, 1].
    if np.any(~np.isfinite(actions)):
        raise ValueError("executed_action_norm contains NaN or infinite values.")
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: min={actions.min()}, max={actions.max()}")

    # Selector masks must be exactly binary; round only after checking tolerance.
    if np.any(~np.isfinite(selection_masks)):
        raise ValueError("selection_mask contains NaN or infinite values.")
    rounded_masks = np.rint(selection_masks)
    if not np.allclose(selection_masks, rounded_masks, atol=1e-6, rtol=0.0):
        bad = np.argwhere(np.abs(selection_masks - rounded_masks) > 1e-6)[0]
        t, dim = (int(bad[0]), int(bad[1]))
        raise ValueError(
            "selection_mask must be binary. "
            f"Found {selection_masks[t, dim]} at transition {t}, dim {dim}."
        )
    selection_masks = rounded_masks.astype(np.float32)

    if selector_mode == "threshold":
        expected_masks = (
            selection_probabilities[1:] >= 0.5
        ).astype(np.float32)
        if not np.array_equal(selection_masks[1:], expected_masks):
            raise ValueError(
                "selector_mode='threshold', but masks after the forced "
                "first step do not equal probability >= 0.5."
            )

    # Validate all physical actuations against the executed normalized action.
    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)
    _validate_actuated_actions(actions, actions_raw)

    # ---- reconstruct SDAR prev_action / action_mix from the agent trajectory ----
    prev_actions, action_mixes = _reconstruct_sdar_context(
        actions,
        selection_masks,
    )

    # ---- physical lighting quantities from the executed dimming ----
    lighting_dimming = np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)
    nominal_power_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0).astype(np.float32)

    # ---- rewards ----
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, actions[t], selection_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, components_unbounded = apply_reward_reference(
        components_raw,
        reward_reference,
    )
    comp_scales = reward_reference["scales"]
    importance_vec = reward_reference["importance"]
    component_clips = reward_reference["clips"]
    scales_source = f"training reference: {reward_reference['path']}"

    reward_physical_raw = components_raw.sum(axis=1).astype(np.float32)
    reward_balanced = components_final.sum(axis=1).astype(np.float32)
    # Legacy training-compatible key: `rewards_raw` means balanced pre-symlog.
    rewards_raw = reward_balanced
    rewards_tx = _transform_vec(reward_balanced)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32)

    # ---- episode return (undiscounted) for quick scoring ----
    ep_return_physical_raw = float(reward_physical_raw.sum())
    ep_return_balanced = float(reward_balanced.sum())
    # Legacy training-compatible alias.
    ep_return_raw = ep_return_balanced
    ep_return_tx = float(rewards.sum())

    print("\n=== Agent evaluation dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (agent executed, SDAR space)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {prev_actions.shape}  (reconstructed)")
    print(f"  selection_logits:  {selection_logits.shape}")
    print(
        "  selection_probs:   "
        f"{selection_probabilities.shape}"
    )
    print(f"  selection_masks:   {selection_masks.shape}  (agent selector)")
    print(f"  action_mixes:      {action_mixes.shape}  (reconstructed)")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced pre-transform)")
    print(f"  physical raw:      {reward_physical_raw.shape}  (sum of physical W_* terms)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(
        "  episode return:    "
        f"physical={ep_return_physical_raw:+.4f}  "
        f"balanced={ep_return_balanced:+.4f}  "
        f"transformed={ep_return_tx:+.4f}"
    )

    _print_normalization(
        comp_names,
        comp_scales,
        importance_vec,
        component_clips,
        scales_source,
    )
    _print_reward_diagnostics(
        reward_physical_raw,
        reward_balanced,
        rewards_tx,
        components_final,
    )
    _print_thermal_violation_rate()
    _print_switching_diagnostics(selection_masks)
    selector_diagnostics = _print_selector_probability_diagnostics(
        selection_probabilities,
        selection_masks,
        selector_mode,
    )
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=prev_actions,
            selection_logits=selection_logits,
            selection_probabilities=selection_probabilities,
            selection_masks=selection_masks,
            action_mixes=action_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            reward_balanced=reward_balanced,
            reward_physical_raw=reward_physical_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(reward_reference["transform"]),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            thermal_reward_component_clip=np.float32(
                component_clips[comp_names.index("thermal_comfort")]
            ),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            # evaluation provenance / quick scores
            policy=np.array(POLICY_TAG),
            dataset_kind=np.array("agent_evaluation"),
            checkpoint_epoch=np.int64(checkpoint["epoch"]),
            checkpoint_path=np.array(checkpoint["path"]),
            checkpoint_sha256=np.array(checkpoint["sha256"]),
            eval_seed=np.int64(-1 if eval_seed is None else int(eval_seed)),
            eval_selector_mode=np.array(selector_mode),
            eval_proposal_deterministic=np.array(
                proposal_deterministic
            ),
            eval_selector_seed=np.int64(selector_seed),
            eval_safeguard_enabled=np.array(bool(safeguard_enabled)),
            eval_weather_file=np.array("" if weather_file is None else str(weather_file)),
            reward_reference_dataset=np.array(reward_reference["path"]),
            reward_reference_sha256=np.array(reward_reference["sha256"]),
            reward_reweighting_tag=np.array(reward_reference["reweighting_tag"]),
            episode_return_physical_raw=np.float32(ep_return_physical_raw),
            episode_return_balanced=np.float32(ep_return_balanced),
            episode_return_raw=np.float32(ep_return_raw),
            episode_return_transformed=np.float32(ep_return_tx),
            selector_diagnostic_groups=np.array(
                ["glazing", "lighting", "heating", "cooling", "overall"]
            ),
            selector_mean_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["mean_probability"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sampled_rate_by_group=np.array(
                [
                    selector_diagnostics[name]["sampled_rate"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sample_minus_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["difference"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_threshold_agreement=np.float32(
                selector_diagnostics["threshold_agreement"]
            ),
            action_layout=np.array(
                {"glazing": [0, 4], "lighting": [4, 9], "heating": [9, 14], "cooling": [14, 19]},
                dtype=object),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_unbounded"] = components_unbounded
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)
            save_dict["reward_component_clips"] = component_clips.astype(np.float32)

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "policy": POLICY_TAG,
                    "dataset_kind": "agent_evaluation",
                    "checkpoint": checkpoint,
                    "evaluation": {
                        "seed": None if eval_seed is None else int(eval_seed),
                        "selector_mode": selector_mode,
                        "proposal_deterministic": proposal_deterministic,
                        "selector_seed": selector_seed,
                        "safeguard_enabled": bool(safeguard_enabled),
                        "weather_file": None if weather_file is None else str(weather_file),
                    },
                    "reward_reference": {
                        "dataset": reward_reference["path"],
                        "sha256": reward_reference["sha256"],
                        "reweighting_tag": reward_reference["reweighting_tag"],
                    },
                    "reward_transform": reward_reference["transform"],
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "thermal_reward_component_clip": float(
                        component_clips[comp_names.index("thermal_comfort")]
                    ),
                    "reward_scales_source": scales_source,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "episode_return_physical_raw": ep_return_physical_raw,
                    "episode_return_balanced": ep_return_balanced,
                    "episode_return_raw": ep_return_raw,
                    "episode_return_transformed": ep_return_tx,
                    "reward_physical_raw_mean": float(reward_physical_raw.mean()),
                    "reward_physical_raw_std": float(reward_physical_raw.std()),
                    "reward_balanced_mean": float(reward_balanced.mean()),
                    "reward_balanced_std": float(reward_balanced.std()),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_component_clips": component_clips.tolist(),
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(selection_masks.mean()),
                    "mean_selection_mask_glazing": float(selection_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(selection_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(selection_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(selection_masks[:, COOL_SLICE].mean()),
                    "selector_probability_diagnostics": selector_diagnostics,
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        logit_cols = [f"selector_logit_{c}" for c in act_cols]
        probability_cols = [
            f"selector_probability_{c}"
            for c in act_cols
        ]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + logit_cols
            + probability_cols
            + ["reward", "reward_raw", "reward_physical_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + selection_masks[t].tolist()
                    + selection_logits[t].tolist()
                    + selection_probabilities[t].tolist()
                    + [
                        float(rewards[t]),
                        float(rewards_raw[t]),
                        float(reward_physical_raw[t]),
                    ]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    return {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": prev_actions,
        "selection_logits": selection_logits,
        "selection_probabilities": selection_probabilities,
        "selection_masks": selection_masks,
        "action_mixes": action_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_balanced": reward_balanced,
        "reward_physical_raw": reward_physical_raw,
        "reward_components": components_final,
        "reward_components_unbounded": components_unbounded,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "reward_component_clips": component_clips,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
        "checkpoint": checkpoint,
        "evaluation": {
            "seed": None if eval_seed is None else int(eval_seed),
            "selector_mode": selector_mode,
            "proposal_deterministic": proposal_deterministic,
            "selector_seed": selector_seed,
            "safeguard_enabled": bool(safeguard_enabled),
            "weather_file": (
                None if weather_file is None else str(weather_file)
            ),
        },
        "selector_probability_diagnostics": selector_diagnostics,
        "reward_reference": reward_reference,
        "episode_return_physical_raw": ep_return_physical_raw,
        "episode_return_balanced": ep_return_balanced,
        "episode_return_raw": ep_return_raw,
        "episode_return_transformed": ep_return_tx,
    }


if __name__ == "__main__":
    dataset = build_agent_dataset(
        save_npz=True,
        save_csv=True,
        reference_dataset_path=REFERENCE_REWARD_DATASET,
        checkpoint_path=EVAL_CHECKPOINT_PATH,
        checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
        eval_seed=EVAL_SEED,
        selector_mode=EVAL_SELECTOR_MODE,
        proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
        selector_seed=EVAL_SELECTOR_SEED,
        safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
        weather_file=EVAL_WEATHER_FILE,
    )

In [ ]:
"""
Agent-rollout dataset builder — evaluation counterpart of the v8 expert builder
================================================================================

Runs AFTER the evaluation callback (`evaluate_agent_callback.py`) has rolled the
TRAINED SDAR-IQL agent through EnergyPlus and filled the in-memory logging dicts.
It reconstructs the same offline-RL dataset format from the agent's trajectory so
you can score the policy (return, per-component reward, thermal violations,
switching, energy) and, if you want, re-pool the agent rollout with the training
data.

WHAT'S DIFFERENT FROM THE v8 EXPERT BUILDER
-------------------------------------------
The reward math, observation layout, and diagnostics are IDENTICAL (reward is a
property of states/actions, not of which policy produced them). Only the action
source changes:

  expert builder                     agent builder (this file)
  ------------------------------     --------------------------------------
  action_vector_data["action"]   ->  agent_data["executed_action_norm"]
  action_vector_data["...mask"]  ->  agent_data["selection_mask"]
  prev_action / action_mix logged -> RECONSTRUCTED from the agent trajectory
  exploration_data (expert vs        (removed — no exploration during eval;
    executed, thermal modes, ...)     the agent is deterministic/greedy)

Required in-memory globals (filled by the eval callback rollout):
    ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    agent_data  (observation, selection_mask, executed_action_norm)

REWARD COMPARABILITY (important)
--------------------------------
This version REQUIRES the pooled decomposed training NPZ as a reward reference.
It loads the exact:

    reward_component_names
    reward_component_scales
    reward_component_importance
    reward_component_clips
    reward_transform

used by training. For the current decomposed run that means thermal importance
3.0, thermal clip 50.0, and clip 10.0 for all other components. The evaluated
rollout never defines its own normalization scales, so rewards remain comparable
across checkpoints and against the experts.

`rewards_raw` is kept as the training-compatible name for the balanced,
post-normalization/post-importance/post-component-clip sum before symlog.
`reward_physical_raw` is additionally saved for the sum of physical W_* reward
components before normalization.
"""

import csv
import hashlib
import json
import re
from pathlib import Path

import numpy as np


# ============================================================
# ROLLOUT LOGS — provided by the eval callback
# ============================================================
# If this modupace),
# bind the logging globals from the eval callback module. They are the same
# mutable objects the callback appended to, so they carry the rollout dle is imported standalone (not exec'd in the rollout namesata.
try:
    agent_data  # noqa: F821  (present when run in the rollout namespace)
except NameError:
    from evaluate_agent_callback import (  # noqa: F401
        ZONES_MID,
        AVAILABLE_GLAZING_STATES,
        temperature_data,
        lighting_data,
        solar_data,
        ext_irr_data,
        wpi_data,
        meter_data,
        output_variable_data,
        temporal_data,
        glazing_state_data,
        lighting_power_data,
        heating_setpoint_data,
        cooling_setpoint_data,
        agent_data,
    )


# ============================================================
# CONFIG
# ============================================================

# These values are validated against the reference NPZ. They are not used as an
# independent reward definition.
REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0  # legacy/default clip; per-component clips come from NPZ

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR

POLICY_TAG = "sdar_iql_decomposed_agent"

# ---------------------------------------------------------------------------
# EDIT THESE FOR EACH CHECKPOINT EVALUATION.
# Use the decomposed pooled dataset that trained the agent. If it is elsewhere,
# provide the absolute path.
# ---------------------------------------------------------------------------
REFERENCE_REWARD_DATASET = (
    "pooled_sdar_experts_thermal3_tclip50_decomposed.npz"
)
EVAL_CHECKPOINT_PATH = "checkpoints/sdar_iql_v11_dualcritic_preserve_old_total_thermal3_tclip50_gru2_seq36_decomnposed/sdar_iql_sdar_iql_v11_dualcritic_preserve_old_total_thermal3_tclip50_gru2_seq36_decomnposed_checkpoint_epoch250.pt"   # e.g. "/.../checkpoint_epoch150.pt"
EVAL_CHECKPOINT_EPOCH = None  # optional; inferred from the checkpoint filename
EVAL_SEED = 123
EVAL_DETERMINISTIC = True
EVAL_SAFEGUARD_ENABLED = True
EVAL_WEATHER_FILE = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}

HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
ACTION_VALIDATION_ATOL = 0.5


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    """Physical (actuated) action, read from the eval callback's logs."""
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM THE AGENT'S NORMALIZED ACTION
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps to a dimming duty
#   u = (a + 1) / 2 in [0, 1], and E = sum_z nominal_W[z]*u[z] * dt / 1000 (kWh).
# Meter-free: uses the executed action, not the logged Lights Electricity Rate.

def compute_lighting_energy_kwh(action_norm_t):
    a = np.asarray(action_norm_t, dtype=np.float64)
    dimming = np.clip((a[LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
    nominal_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float64)
    total_power_w = float((nominal_w * dimming).sum())
    return total_power_w * TIMESTEP_HOURS / 1000.0


# ============================================================
# REWARD — components + scalar (identical logic to the v8 builder)
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, action_norm_t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_energy_kwh = compute_lighting_energy_kwh(action_norm_t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = np.asarray(selection_mask_t)
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _npz_scalar(data, key, default=None):
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar NPZ field {key!r}, got shape {value.shape}.")
    return value.reshape(()).item()


def load_reward_reference(reference_dataset_path, comp_names):
    """
    Load the immutable reward ruler from the pooled decomposed training NPZ.

    There is deliberately no rollout-local fallback. A missing or incompatible
    reference dataset is an error because otherwise checkpoint rewards would be
    calculated on different scales.
    """
    path = Path(reference_dataset_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "Reward reference dataset not found: "
            f"{path}. Set REFERENCE_REWARD_DATASET to the exact pooled NPZ used "
            "to train the decomposed agent."
        )

    required = {
        "reward_component_names",
        "reward_component_scales",
        "reward_component_importance",
    }
    with np.load(path, allow_pickle=False) as data:
        missing = sorted(required.difference(data.files))
        if missing:
            raise KeyError(
                f"Reference NPZ {path} is missing required reward metadata: {missing}"
            )

        ref_names = [str(x) for x in np.asarray(data["reward_component_names"]).tolist()]
        if ref_names != list(comp_names):
            raise ValueError(
                "Reward component order differs from the training dataset.\n"
                f"  builder:   {list(comp_names)}\n"
                f"  reference: {ref_names}"
            )

        scales = np.asarray(data["reward_component_scales"], dtype=np.float64)
        importance = np.asarray(
            data["reward_component_importance"], dtype=np.float64
        )
        if scales.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_scales shape {scales.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if importance.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_importance shape {importance.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if np.any(~np.isfinite(scales)) or np.any(scales <= 0.0):
            raise ValueError("Reference reward scales must all be finite and positive.")
        if np.any(~np.isfinite(importance)):
            raise ValueError("Reference reward importance values must all be finite.")

        if "reward_component_clips" in data.files:
            clips = np.asarray(data["reward_component_clips"], dtype=np.float64)
        else:
            legacy_clip = float(
                _npz_scalar(data, "reward_component_clip", REWARD_COMPONENT_CLIP)
            )
            clips = np.full(len(comp_names), legacy_clip, dtype=np.float64)
            if "thermal_reward_component_clip" in data.files:
                thermal_i = comp_names.index("thermal_comfort")
                clips[thermal_i] = float(
                    _npz_scalar(data, "thermal_reward_component_clip")
                )

        if clips.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_clips shape {clips.shape}; "
                f"expected {(len(comp_names),)}"
            )
        # A negative or non-finite stored clip means unbounded for that component.
        clips = np.where(np.isfinite(clips) & (clips >= 0.0), clips, np.inf)

        transform = str(
            _npz_scalar(data, "reward_transform", REWARD_TRANSFORM)
        )
        reweighting_tag = str(
            _npz_scalar(data, "reward_reweighting_tag", "")
        )
        reference_mask_value = float(
            _npz_scalar(data, "action_mask_value", ACTION_MASK_VALUE)
        )

    configured_importance = np.array(
        [REWARD_IMPORTANCE.get(name, 1.0) for name in comp_names],
        dtype=np.float64,
    )
    if not np.allclose(configured_importance, importance, atol=1e-7, rtol=0.0):
        raise ValueError(
            "Builder REWARD_IMPORTANCE differs from the training NPZ. "
            "Do not evaluate until the reward definitions match.\n"
            f"  builder:   {configured_importance.tolist()}\n"
            f"  reference: {importance.tolist()}"
        )
    if transform != REWARD_TRANSFORM:
        raise ValueError(
            f"Builder transform {REWARD_TRANSFORM!r} differs from reference "
            f"transform {transform!r}."
        )
    if not np.isclose(reference_mask_value, ACTION_MASK_VALUE):
        raise ValueError(
            f"Reference action_mask_value={reference_mask_value} differs from "
            f"builder ACTION_MASK_VALUE={ACTION_MASK_VALUE}."
        )

    return {
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
        "names": list(comp_names),
        "scales": scales,
        "importance": importance,
        "clips": clips,
        "transform": transform,
        "reweighting_tag": reweighting_tag,
    }


def apply_reward_reference(components_raw, reward_reference):
    """Apply training scales, importance, and per-component clips exactly."""
    scales = reward_reference["scales"]
    importance = reward_reference["importance"]
    clips = reward_reference["clips"]

    unbounded = (
        components_raw.astype(np.float64)
        / scales[None, :]
        * importance[None, :]
    )
    components_final = np.clip(
        unbounded,
        -clips[None, :],
        clips[None, :],
    )
    return components_final.astype(np.float32), unbounded.astype(np.float32)


def _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch):
    if checkpoint_epoch is not None:
        epoch = int(checkpoint_epoch)
        if epoch < 0:
            raise ValueError("checkpoint_epoch must be non-negative.")
        return epoch
    if checkpoint_path:
        match = re.search(r"epoch[_-]?(\d+)", Path(checkpoint_path).name, re.IGNORECASE)
        if match:
            return int(match.group(1))
    raise ValueError(
        "Checkpoint epoch is required for unambiguous output naming. Set "
        "EVAL_CHECKPOINT_EPOCH or use a checkpoint filename containing "
        "'epoch<number>'."
    )


def _checkpoint_provenance(checkpoint_path, checkpoint_epoch):
    epoch = _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch)
    if checkpoint_path is None:
        return {
            "epoch": epoch,
            "path": "",
            "sha256": "",
        }

    path = Path(checkpoint_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return {
        "epoch": epoch,
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
    }


# ============================================================
# DIAGNOSTICS (identical to the v8 builder)
# ============================================================

def _print_reward_diagnostics(
    reward_physical_raw,
    reward_balanced,
    rewards_tx,
    components_arr,
):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(
        f"  physical raw: mean={reward_physical_raw.mean():+.4f}  "
        f"std={reward_physical_raw.std():.4f}"
    )
    print(
        f"                min={reward_physical_raw.min():+.4f}    "
        f"max={reward_physical_raw.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_physical_raw, qs)]}")
    print(
        f"  balanced:     mean={reward_balanced.mean():+.4f}  "
        f"std={reward_balanced.std():.4f}"
    )
    print(
        f"                min={reward_balanced.min():+.4f}    "
        f"max={reward_balanced.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_balanced, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - reward_balanced
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - balanced): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - balanced): {np.max(np.abs(err)):.8f}")


def _print_normalization(
    comp_names,
    comp_scales,
    importance_vec,
    component_clips,
    scales_source,
):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'clip':>8s}  {'eff_weight':>10s}")
    for nm, sc, imp, clip in zip(
        comp_names, comp_scales, importance_vec, component_clips
    ):
        eff = imp / sc if sc != 0 else 0.0
        clip_name = "none" if not np.isfinite(clip) else f"{clip:.1f}"
        print(
            f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  "
            f"{clip_name:>8s}  {eff:10.4f}"
        )


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics (agent selector) ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None or TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


def _denormalize_linear(values, physical_range):
    low, high = physical_range
    values = np.asarray(values, dtype=np.float64)
    return low + 0.5 * (values + 1.0) * (high - low)


def _validate_actuated_actions(actions, actions_raw):
    """Verify all 19 normalized executed actions against physical actuation logs."""
    n_glazing = len(AVAILABLE_GLAZING_STATES)
    glz_ids_from_norm = np.clip(
        np.round((actions[:, GLAZE_SLICE] + 1.0) * 0.5 * (n_glazing - 1)),
        0,
        n_glazing - 1,
    ).astype(np.int64)
    glz_ids_logged = actions_raw[:, GLAZE_SLICE].astype(np.int64)
    if not np.array_equal(glz_ids_from_norm, glz_ids_logged):
        mismatch = np.argwhere(glz_ids_from_norm != glz_ids_logged)[0]
        t, dim = (int(mismatch[0]), int(mismatch[1]))
        raise ValueError(
            "Glazing decoded from executed_action_norm does not match the "
            f"actuation log at transition {t}, glazing dim {dim}: "
            f"decoded={glz_ids_from_norm[t, dim]}, logged={glz_ids_logged[t, dim]}."
        )

    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )
    light_decoded = (
        np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
        * nominal_power_w[None, :]
    )
    heat_decoded = _denormalize_linear(
        actions[:, HEAT_SLICE], HEATING_SETPOINT_RANGE
    )
    cool_decoded = _denormalize_linear(
        actions[:, COOL_SLICE], COOLING_SETPOINT_RANGE
    )

    checks = [
        (
            "lighting",
            light_decoded,
            actions_raw[:, LIGHT_SLICE],
            "W",
        ),
        (
            "heating setpoint",
            heat_decoded,
            actions_raw[:, HEAT_SLICE],
            "degC",
        ),
        (
            "cooling setpoint",
            cool_decoded,
            actions_raw[:, COOL_SLICE],
            "degC",
        ),
    ]
    for label, decoded, logged, unit in checks:
        if not np.allclose(
            decoded,
            logged,
            atol=ACTION_VALIDATION_ATOL,
            rtol=1e-4,
        ):
            abs_error = np.abs(decoded - logged)
            t, dim = np.unravel_index(np.argmax(abs_error), abs_error.shape)
            raise ValueError(
                f"{label} decoded from executed_action_norm does not match the "
                f"actuation log. Largest error at transition {t}, local dim {dim}: "
                f"decoded={decoded[t, dim]:.6g} {unit}, "
                f"logged={logged[t, dim]:.6g} {unit}, "
                f"abs_error={abs_error[t, dim]:.6g} {unit}."
            )


def _reconstruct_sdar_context(actions, selection_masks):
    previous_keys = (
        "previous_action_norm",
        "prev_action_norm",
        "previous_executed_action_norm",
    )
    previous_key = next((k for k in previous_keys if k in agent_data), None)

    if previous_key is not None:
        logged_prev = np.asarray(agent_data[previous_key], dtype=np.float32)
        if logged_prev.ndim != 2 or logged_prev.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{previous_key!r}] has shape {logged_prev.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_prev.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{previous_key!r}] has only {logged_prev.shape[0]} "
                f"rows for {len(actions)} transitions."
            )
        prev_actions = logged_prev[:len(actions)].copy()
    else:
        if not np.all(selection_masks[0] == 1.0):
            raise ValueError(
                "The first selection mask is not a full update, but the callback "
                "did not log previous_action_norm. The initial SDAR context cannot "
                "be reconstructed without leaking action[0] into prev_action[0]."
            )
        prev_actions = actions.copy()
        prev_actions[1:] = actions[:-1]

    if len(actions) > 1 and not np.allclose(
        prev_actions[1:],
        actions[:-1],
        atol=1e-5,
        rtol=0.0,
    ):
        max_error = float(
            np.max(np.abs(prev_actions[1:] - actions[:-1]))
        )
        raise ValueError(
            "Logged/reconstructed previous actions do not equal the preceding "
            f"executed actions (max abs error {max_error:.6g})."
        )

    action_mixes = (
        (1.0 - selection_masks) * prev_actions
        + selection_masks * ACTION_MASK_VALUE
    ).astype(np.float32)

    for key in ("action_mix", "action_mixes"):
        if key not in agent_data:
            continue
        logged_mix = np.asarray(agent_data[key], dtype=np.float32)
        if logged_mix.ndim != 2 or logged_mix.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{key!r}] has shape {logged_mix.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_mix.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{key!r}] has only {logged_mix.shape[0]} rows for "
                f"{len(actions)} transitions."
            )
        if not np.allclose(
            action_mixes,
            logged_mix[:len(actions)],
            atol=1e-5,
            rtol=0.0,
        ):
            max_error = float(
                np.max(np.abs(action_mixes - logged_mix[:len(actions)]))
            )
            raise ValueError(
                f"Reconstructed action_mix differs from agent_data[{key!r}] "
                f"(max abs error {max_error:.6g})."
            )
        break

    return prev_actions.astype(np.float32), action_mixes


# ============================================================
# MAIN BUILDER
# ============================================================

def build_agent_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    reference_dataset_path=REFERENCE_REWARD_DATASET,
    checkpoint_path=EVAL_CHECKPOINT_PATH,
    checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
    eval_seed=EVAL_SEED,
    deterministic=EVAL_DETERMINISTIC,
    safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
    weather_file=EVAL_WEATHER_FILE,
):
    comp_names = _component_names()
    reward_reference = load_reward_reference(
        reference_dataset_path,
        comp_names,
    )
    checkpoint = _checkpoint_provenance(
        checkpoint_path,
        checkpoint_epoch,
    )
    if prefix is None:
        prefix = (
            f"agent_eval_dataset_decomposed_epoch{checkpoint['epoch']}"
            "_thermal3_tclip50"
        )
    prefix = str(Path(prefix).expanduser())
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)

    print("=== Evaluation provenance ===")
    print(f"  checkpoint epoch: {checkpoint['epoch']}")
    print(f"  checkpoint path:  {checkpoint['path'] or '(not supplied)'}")
    print(f"  reference NPZ:    {reward_reference['path']}")
    print(
        "  reward tag:       "
        f"{reward_reference['reweighting_tag'] or '(not stored)'}"
    )
    print(f"  deterministic:    {bool(deterministic)}")
    print(f"  safeguard:        {bool(safeguard_enabled)}")
    print(f"  eval seed:        {eval_seed}")
    print(f"  weather file:     {weather_file}")

    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1
    if T < 1:
        raise ValueError(f"Not enough logged steps to build transitions (n={n}).")

    # ---- length sanity check (agent logs only) ----
    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "agent_observation": len(agent_data["observation"]),
        "agent_selection_mask": len(agent_data["selection_mask"]),
        "agent_executed_action_norm": len(agent_data["executed_action_norm"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    # ---- observations ----
    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    # Integrity: the obs we rebuild here must equal what the agent actually saw.
    agent_obs = np.asarray(agent_data["observation"], dtype=np.float32)[:T]
    if not np.allclose(observations, agent_obs, atol=1e-4):
        max_err = float(np.max(np.abs(observations - agent_obs)))
        raise ValueError(
            "Rebuilt observations differ from what the agent saw "
            f"(max abs err {max_err:.6g}). The dataset obs layout has drifted "
            "from the eval callback's build_observation.")

    # ---- actions: agent's executed normalized action is the primary action ----
    agent_actions = np.asarray(agent_data["executed_action_norm"], dtype=np.float32)
    agent_masks = np.asarray(agent_data["selection_mask"], dtype=np.float32)

    if agent_actions.shape != (n, ACTION_DIM):
        raise ValueError(f"executed_action_norm has shape {agent_actions.shape}, expected ({n}, {ACTION_DIM}).")
    if agent_masks.shape != (n, ACTION_DIM):
        raise ValueError(f"selection_mask has shape {agent_masks.shape}, expected ({n}, {ACTION_DIM}).")

    actions = agent_actions[:T]
    selection_masks = agent_masks[:T]

    # Normalized executed actions must be finite and lie within [-1, 1].
    if np.any(~np.isfinite(actions)):
        raise ValueError("executed_action_norm contains NaN or infinite values.")
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: min={actions.min()}, max={actions.max()}")

    # Selector masks must be exactly binary; round only after checking tolerance.
    if np.any(~np.isfinite(selection_masks)):
        raise ValueError("selection_mask contains NaN or infinite values.")
    rounded_masks = np.rint(selection_masks)
    if not np.allclose(selection_masks, rounded_masks, atol=1e-6, rtol=0.0):
        bad = np.argwhere(np.abs(selection_masks - rounded_masks) > 1e-6)[0]
        t, dim = (int(bad[0]), int(bad[1]))
        raise ValueError(
            "selection_mask must be binary. "
            f"Found {selection_masks[t, dim]} at transition {t}, dim {dim}."
        )
    selection_masks = rounded_masks.astype(np.float32)

    # Validate all physical actuations against the executed normalized action.
    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)
    _validate_actuated_actions(actions, actions_raw)

    # ---- reconstruct SDAR prev_action / action_mix from the agent trajectory ----
    prev_actions, action_mixes = _reconstruct_sdar_context(
        actions,
        selection_masks,
    )

    # ---- physical lighting quantities from the executed dimming ----
    lighting_dimming = np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)
    nominal_power_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0).astype(np.float32)

    # ---- rewards ----
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, actions[t], selection_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, components_unbounded = apply_reward_reference(
        components_raw,
        reward_reference,
    )
    comp_scales = reward_reference["scales"]
    importance_vec = reward_reference["importance"]
    component_clips = reward_reference["clips"]
    scales_source = f"training reference: {reward_reference['path']}"

    reward_physical_raw = components_raw.sum(axis=1).astype(np.float32)
    reward_balanced = components_final.sum(axis=1).astype(np.float32)
    # Legacy training-compatible key: `rewards_raw` means balanced pre-symlog.
    rewards_raw = reward_balanced
    rewards_tx = _transform_vec(reward_balanced)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32)

    # ---- episode return (undiscounted) for quick scoring ----
    ep_return_physical_raw = float(reward_physical_raw.sum())
    ep_return_balanced = float(reward_balanced.sum())
    # Legacy training-compatible alias.
    ep_return_raw = ep_return_balanced
    ep_return_tx = float(rewards.sum())

    print("\n=== Agent evaluation dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (agent executed, SDAR space)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {prev_actions.shape}  (reconstructed)")
    print(f"  selection_masks:   {selection_masks.shape}  (agent selector)")
    print(f"  action_mixes:      {action_mixes.shape}  (reconstructed)")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced pre-transform)")
    print(f"  physical raw:      {reward_physical_raw.shape}  (sum of physical W_* terms)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(
        "  episode return:    "
        f"physical={ep_return_physical_raw:+.4f}  "
        f"balanced={ep_return_balanced:+.4f}  "
        f"transformed={ep_return_tx:+.4f}"
    )

    _print_normalization(
        comp_names,
        comp_scales,
        importance_vec,
        component_clips,
        scales_source,
    )
    _print_reward_diagnostics(
        reward_physical_raw,
        reward_balanced,
        rewards_tx,
        components_final,
    )
    _print_thermal_violation_rate()
    _print_switching_diagnostics(selection_masks)
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=prev_actions,
            selection_masks=selection_masks,
            action_mixes=action_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            reward_balanced=reward_balanced,
            reward_physical_raw=reward_physical_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(reward_reference["transform"]),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            thermal_reward_component_clip=np.float32(
                component_clips[comp_names.index("thermal_comfort")]
            ),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            # evaluation provenance / quick scores
            policy=np.array(POLICY_TAG),
            dataset_kind=np.array("agent_evaluation"),
            checkpoint_epoch=np.int64(checkpoint["epoch"]),
            checkpoint_path=np.array(checkpoint["path"]),
            checkpoint_sha256=np.array(checkpoint["sha256"]),
            eval_seed=np.int64(-1 if eval_seed is None else int(eval_seed)),
            eval_deterministic=np.array(bool(deterministic)),
            eval_safeguard_enabled=np.array(bool(safeguard_enabled)),
            eval_weather_file=np.array("" if weather_file is None else str(weather_file)),
            reward_reference_dataset=np.array(reward_reference["path"]),
            reward_reference_sha256=np.array(reward_reference["sha256"]),
            reward_reweighting_tag=np.array(reward_reference["reweighting_tag"]),
            episode_return_physical_raw=np.float32(ep_return_physical_raw),
            episode_return_balanced=np.float32(ep_return_balanced),
            episode_return_raw=np.float32(ep_return_raw),
            episode_return_transformed=np.float32(ep_return_tx),
            action_layout=np.array(
                {"glazing": [0, 4], "lighting": [4, 9], "heating": [9, 14], "cooling": [14, 19]},
                dtype=object),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_unbounded"] = components_unbounded
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)
            save_dict["reward_component_clips"] = component_clips.astype(np.float32)

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "policy": POLICY_TAG,
                    "dataset_kind": "agent_evaluation",
                    "checkpoint": checkpoint,
                    "evaluation": {
                        "seed": None if eval_seed is None else int(eval_seed),
                        "deterministic": bool(deterministic),
                        "safeguard_enabled": bool(safeguard_enabled),
                        "weather_file": None if weather_file is None else str(weather_file),
                    },
                    "reward_reference": {
                        "dataset": reward_reference["path"],
                        "sha256": reward_reference["sha256"],
                        "reweighting_tag": reward_reference["reweighting_tag"],
                    },
                    "reward_transform": reward_reference["transform"],
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "thermal_reward_component_clip": float(
                        component_clips[comp_names.index("thermal_comfort")]
                    ),
                    "reward_scales_source": scales_source,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "episode_return_physical_raw": ep_return_physical_raw,
                    "episode_return_balanced": ep_return_balanced,
                    "episode_return_raw": ep_return_raw,
                    "episode_return_transformed": ep_return_tx,
                    "reward_physical_raw_mean": float(reward_physical_raw.mean()),
                    "reward_physical_raw_std": float(reward_physical_raw.std()),
                    "reward_balanced_mean": float(reward_balanced.mean()),
                    "reward_balanced_std": float(reward_balanced.std()),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_component_clips": component_clips.tolist(),
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(selection_masks.mean()),
                    "mean_selection_mask_glazing": float(selection_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(selection_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(selection_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(selection_masks[:, COOL_SLICE].mean()),
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + ["reward", "reward_raw", "reward_physical_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + selection_masks[t].tolist()
                    + [
                        float(rewards[t]),
                        float(rewards_raw[t]),
                        float(reward_physical_raw[t]),
                    ]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    return {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": prev_actions,
        "selection_masks": selection_masks,
        "action_mixes": action_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_balanced": reward_balanced,
        "reward_physical_raw": reward_physical_raw,
        "reward_components": components_final,
        "reward_components_unbounded": components_unbounded,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "reward_component_clips": component_clips,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
        "checkpoint": checkpoint,
        "reward_reference": reward_reference,
        "episode_return_physical_raw": ep_return_physical_raw,
        "episode_return_balanced": ep_return_balanced,
        "episode_return_raw": ep_return_raw,
        "episode_return_transformed": ep_return_tx,
    }


if __name__ == "__main__":
    dataset = build_agent_dataset(
        save_npz=True,
        save_csv=True,
        reference_dataset_path=REFERENCE_REWARD_DATASET,
        checkpoint_path=EVAL_CHECKPOINT_PATH,
        checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
        eval_seed=EVAL_SEED,
        deterministic=EVAL_DETERMINISTIC,
        safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
        weather_file=EVAL_WEATHER_FILE,
    )

In [ ]:
probs = np.asarray(
    agent_data["selection_probability"],
    dtype=np.float32,
)[1:]

masks = np.asarray(
    agent_data["selection_mask"],
    dtype=np.float32,
)[1:]

groups = {
    "glazing": slice(0, 4),
    "lighting": slice(4, 9),
    "heating": slice(9, 14),
    "cooling": slice(14, 19),
}

for name, slc in groups.items():
    p = probs[:, slc]
    m = masks[:, slc]

    print(
        f"{name:10s} "
        f"mean probability={p.mean():.4f}  "
        f"sampled rate={m.mean():.4f}  "
        f"difference={m.mean()-p.mean():+.4f}"
    )

assert not np.array_equal(
    masks,
    (probs >= 0.5).astype(np.float32),
)

In [ ]:
"""
Agent-rollout dataset builder â€” evaluation counterpart of the v8 expert builder
================================================================================

Runs AFTER the evaluation callback (`evaluate_agent_callback.py`) has rolled the
TRAINED SDAR-IQL agent through EnergyPlus and filled the in-memory logging dicts.
It reconstructs the same offline-RL dataset format from the agent's trajectory so
you can score the policy (return, per-component reward, thermal violations,
switching, energy) and, if you want, re-pool the agent rollout with the training
data.

WHAT'S DIFFERENT FROM THE v8 EXPERT BUILDER
-------------------------------------------
The reward math, observation layout, and diagnostics are IDENTICAL (reward is a
property of states/actions, not of which policy produced them). Only the action
source changes:

  expert builder                     agent builder (this file)
  ------------------------------     --------------------------------------
  action_vector_data["action"]   ->  agent_data["executed_action_norm"]
  action_vector_data["...mask"]  ->  agent_data["selection_mask"]
  prev_action / action_mix logged -> RECONSTRUCTED from the agent trajectory
  exploration_data (expert vs        (removed â€” no exploration during eval;
    executed, thermal modes, ...)     selector sampling is logged directly)

Required in-memory globals (filled by the eval callback rollout):
    ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    agent_data  (observation, selection_logits, selection_probability,
                 selection_mask, executed_action_norm)

REWARD COMPARABILITY (important)
--------------------------------
This version REQUIRES the pooled decomposed training NPZ as a reward reference.
It loads the exact:

    reward_component_names
    reward_component_scales
    reward_component_importance
    reward_component_clips
    reward_transform

used by training. For the current decomposed run that means thermal importance
3.0, thermal clip 50.0, and clip 10.0 for all other components. The evaluated
rollout never defines its own normalization scales, so rewards remain comparable
across checkpoints and against the experts.

`rewards_raw` is kept as the training-compatible name for the balanced,
post-normalization/post-importance/post-component-clip sum before symlog.
`reward_physical_raw` is additionally saved for the sum of physical W_* reward
components before normalization.
"""

import csv
import hashlib
import json
import re
from pathlib import Path

import numpy as np


# ============================================================
# ROLLOUT LOGS â€” provided by the eval callback
# ============================================================
# If this modupace),
# bind the logging globals from the eval callback module. They are the same
# mutable objects the callback appended to, so they carry the rollout dle is imported standalone (not exec'd in the rollout namesata.
try:
    agent_data  # noqa: F821  (present when run in the rollout namespace)
except NameError:
    from evaluate_agent_callback import (  # noqa: F401
        ZONES_MID,
        AVAILABLE_GLAZING_STATES,
        temperature_data,
        lighting_data,
        solar_data,
        ext_irr_data,
        wpi_data,
        meter_data,
        output_variable_data,
        temporal_data,
        glazing_state_data,
        lighting_power_data,
        heating_setpoint_data,
        cooling_setpoint_data,
        agent_data,
        SELECTOR_MODE,
        PROPOSAL_DETERMINISTIC,
        SELECTOR_SEED,
    )


# ============================================================
# CONFIG
# ============================================================

# These values are validated against the reference NPZ. They are not used as an
# independent reward definition.
REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0  # legacy/default clip; per-component clips come from NPZ

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR

POLICY_TAG = "sdar_iql_decomposed_agent"

# ---------------------------------------------------------------------------
# EDIT THESE FOR EACH CHECKPOINT EVALUATION.
# Use the decomposed pooled dataset that trained the agent. If it is elsewhere,
# provide the absolute path.
# ---------------------------------------------------------------------------
REFERENCE_REWARD_DATASET = (
    "pooled_sdar_experts_thermal3_tclip50_decomposed.npz"
)
EVAL_CHECKPOINT_PATH = "checkpoints/sdar_iql_v11_dualcritic_preserve_old_total_thermal3_tclip50_gru2_seq36_decomnposed/sdar_iql_sdar_iql_v11_dualcritic_preserve_old_total_thermal3_tclip50_gru2_seq36_decomnposed_checkpoint_epoch250.pt"   # e.g. "/.../checkpoint_epoch150.pt"
EVAL_CHECKPOINT_EPOCH = None  # optional; inferred from the checkpoint filename
EVAL_SEED = 123
EVAL_SELECTOR_MODE = str(globals().get("SELECTOR_MODE", "sample")).lower()
EVAL_PROPOSAL_DETERMINISTIC = bool(
    globals().get("PROPOSAL_DETERMINISTIC", True)
)
EVAL_SELECTOR_SEED = int(globals().get("SELECTOR_SEED", 20260728))
EVAL_SAFEGUARD_ENABLED = True
EVAL_WEATHER_FILE = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}

HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
ACTION_VALIDATION_ATOL = 0.5


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    """Physical (actuated) action, read from the eval callback's logs."""
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM THE AGENT'S NORMALIZED ACTION
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps to a dimming duty
#   u = (a + 1) / 2 in [0, 1], and E = sum_z nominal_W[z]*u[z] * dt / 1000 (kWh).
# Meter-free: uses the executed action, not the logged Lights Electricity Rate.

def compute_lighting_energy_kwh(action_norm_t):
    a = np.asarray(action_norm_t, dtype=np.float64)
    dimming = np.clip((a[LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
    nominal_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float64)
    total_power_w = float((nominal_w * dimming).sum())
    return total_power_w * TIMESTEP_HOURS / 1000.0


# ============================================================
# REWARD â€” components + scalar (identical logic to the v8 builder)
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, action_norm_t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_energy_kwh = compute_lighting_energy_kwh(action_norm_t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = np.asarray(selection_mask_t)
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _npz_scalar(data, key, default=None):
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar NPZ field {key!r}, got shape {value.shape}.")
    return value.reshape(()).item()


def load_reward_reference(reference_dataset_path, comp_names):
    """
    Load the immutable reward ruler from the pooled decomposed training NPZ.

    There is deliberately no rollout-local fallback. A missing or incompatible
    reference dataset is an error because otherwise checkpoint rewards would be
    calculated on different scales.
    """
    path = Path(reference_dataset_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "Reward reference dataset not found: "
            f"{path}. Set REFERENCE_REWARD_DATASET to the exact pooled NPZ used "
            "to train the decomposed agent."
        )

    required = {
        "reward_component_names",
        "reward_component_scales",
        "reward_component_importance",
    }
    with np.load(path, allow_pickle=False) as data:
        missing = sorted(required.difference(data.files))
        if missing:
            raise KeyError(
                f"Reference NPZ {path} is missing required reward metadata: {missing}"
            )

        ref_names = [str(x) for x in np.asarray(data["reward_component_names"]).tolist()]
        if ref_names != list(comp_names):
            raise ValueError(
                "Reward component order differs from the training dataset.\n"
                f"  builder:   {list(comp_names)}\n"
                f"  reference: {ref_names}"
            )

        scales = np.asarray(data["reward_component_scales"], dtype=np.float64)
        importance = np.asarray(
            data["reward_component_importance"], dtype=np.float64
        )
        if scales.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_scales shape {scales.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if importance.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_importance shape {importance.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if np.any(~np.isfinite(scales)) or np.any(scales <= 0.0):
            raise ValueError("Reference reward scales must all be finite and positive.")
        if np.any(~np.isfinite(importance)):
            raise ValueError("Reference reward importance values must all be finite.")

        if "reward_component_clips" in data.files:
            clips = np.asarray(data["reward_component_clips"], dtype=np.float64)
        else:
            legacy_clip = float(
                _npz_scalar(data, "reward_component_clip", REWARD_COMPONENT_CLIP)
            )
            clips = np.full(len(comp_names), legacy_clip, dtype=np.float64)
            if "thermal_reward_component_clip" in data.files:
                thermal_i = comp_names.index("thermal_comfort")
                clips[thermal_i] = float(
                    _npz_scalar(data, "thermal_reward_component_clip")
                )

        if clips.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_clips shape {clips.shape}; "
                f"expected {(len(comp_names),)}"
            )
        # A negative or non-finite stored clip means unbounded for that component.
        clips = np.where(np.isfinite(clips) & (clips >= 0.0), clips, np.inf)

        transform = str(
            _npz_scalar(data, "reward_transform", REWARD_TRANSFORM)
        )
        reweighting_tag = str(
            _npz_scalar(data, "reward_reweighting_tag", "")
        )
        reference_mask_value = float(
            _npz_scalar(data, "action_mask_value", ACTION_MASK_VALUE)
        )

    configured_importance = np.array(
        [REWARD_IMPORTANCE.get(name, 1.0) for name in comp_names],
        dtype=np.float64,
    )
    if not np.allclose(configured_importance, importance, atol=1e-7, rtol=0.0):
        raise ValueError(
            "Builder REWARD_IMPORTANCE differs from the training NPZ. "
            "Do not evaluate until the reward definitions match.\n"
            f"  builder:   {configured_importance.tolist()}\n"
            f"  reference: {importance.tolist()}"
        )
    if transform != REWARD_TRANSFORM:
        raise ValueError(
            f"Builder transform {REWARD_TRANSFORM!r} differs from reference "
            f"transform {transform!r}."
        )
    if not np.isclose(reference_mask_value, ACTION_MASK_VALUE):
        raise ValueError(
            f"Reference action_mask_value={reference_mask_value} differs from "
            f"builder ACTION_MASK_VALUE={ACTION_MASK_VALUE}."
        )

    return {
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
        "names": list(comp_names),
        "scales": scales,
        "importance": importance,
        "clips": clips,
        "transform": transform,
        "reweighting_tag": reweighting_tag,
    }


def apply_reward_reference(components_raw, reward_reference):
    """Apply training scales, importance, and per-component clips exactly."""
    scales = reward_reference["scales"]
    importance = reward_reference["importance"]
    clips = reward_reference["clips"]

    unbounded = (
        components_raw.astype(np.float64)
        / scales[None, :]
        * importance[None, :]
    )
    components_final = np.clip(
        unbounded,
        -clips[None, :],
        clips[None, :],
    )
    return components_final.astype(np.float32), unbounded.astype(np.float32)


def _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch):
    if checkpoint_epoch is not None:
        epoch = int(checkpoint_epoch)
        if epoch < 0:
            raise ValueError("checkpoint_epoch must be non-negative.")
        return epoch
    if checkpoint_path:
        match = re.search(r"epoch[_-]?(\d+)", Path(checkpoint_path).name, re.IGNORECASE)
        if match:
            return int(match.group(1))
    raise ValueError(
        "Checkpoint epoch is required for unambiguous output naming. Set "
        "EVAL_CHECKPOINT_EPOCH or use a checkpoint filename containing "
        "'epoch<number>'."
    )


def _checkpoint_provenance(checkpoint_path, checkpoint_epoch):
    epoch = _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch)
    if checkpoint_path is None:
        return {
            "epoch": epoch,
            "path": "",
            "sha256": "",
        }

    path = Path(checkpoint_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return {
        "epoch": epoch,
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
    }


# ============================================================
# DIAGNOSTICS (identical to the v8 builder)
# ============================================================

def _print_reward_diagnostics(
    reward_physical_raw,
    reward_balanced,
    rewards_tx,
    components_arr,
):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(
        f"  physical raw: mean={reward_physical_raw.mean():+.4f}  "
        f"std={reward_physical_raw.std():.4f}"
    )
    print(
        f"                min={reward_physical_raw.min():+.4f}    "
        f"max={reward_physical_raw.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_physical_raw, qs)]}")
    print(
        f"  balanced:     mean={reward_balanced.mean():+.4f}  "
        f"std={reward_balanced.std():.4f}"
    )
    print(
        f"                min={reward_balanced.min():+.4f}    "
        f"max={reward_balanced.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_balanced, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - reward_balanced
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - balanced): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - balanced): {np.max(np.abs(err)):.8f}")


def _print_normalization(
    comp_names,
    comp_scales,
    importance_vec,
    component_clips,
    scales_source,
):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'clip':>8s}  {'eff_weight':>10s}")
    for nm, sc, imp, clip in zip(
        comp_names, comp_scales, importance_vec, component_clips
    ):
        eff = imp / sc if sc != 0 else 0.0
        clip_name = "none" if not np.isfinite(clip) else f"{clip:.1f}"
        print(
            f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  "
            f"{clip_name:>8s}  {eff:10.4f}"
        )


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}Â°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics (agent selector) ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_selector_probability_diagnostics(
    selection_probabilities,
    selection_masks,
    selector_mode,
):
    # The callback forces all dimensions to update on the first step, so
    # exclude that step when checking the learned selector realization.
    probabilities = selection_probabilities[1:]
    masks = selection_masks[1:]

    groups = {
        "glazing": GLAZE_SLICE,
        "lighting": LIGHT_SLICE,
        "heating": HEAT_SLICE,
        "cooling": COOL_SLICE,
        "overall": slice(0, ACTION_DIM),
    }

    print(
        "\n=== Selector probability diagnostics "
        f"(mode={selector_mode}) ==="
    )
    stats = {}
    for name, slc in groups.items():
        p = probabilities[:, slc]
        m = masks[:, slc]
        mean_probability = float(p.mean())
        sampled_rate = float(m.mean())
        difference = sampled_rate - mean_probability
        stats[name] = {
            "mean_probability": mean_probability,
            "sampled_rate": sampled_rate,
            "difference": difference,
        }
        print(
            f"  {name:10s} "
            f"mean probability={mean_probability:.4f}  "
            f"sampled rate={sampled_rate:.4f}  "
            f"difference={difference:+.4f}"
        )

    threshold_masks = (probabilities >= 0.5).astype(np.float32)
    stats["threshold_agreement"] = float(
        np.mean(masks == threshold_masks)
    )
    print(
        "  agreement with p>=0.5 threshold: "
        f"{100.0 * stats['threshold_agreement']:.2f}%"
    )
    return stats


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None or TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


def _denormalize_linear(values, physical_range):
    low, high = physical_range
    values = np.asarray(values, dtype=np.float64)
    return low + 0.5 * (values + 1.0) * (high - low)


def _validate_actuated_actions(actions, actions_raw):
    """Verify all 19 normalized executed actions against physical actuation logs."""
    n_glazing = len(AVAILABLE_GLAZING_STATES)
    glz_ids_from_norm = np.clip(
        np.round((actions[:, GLAZE_SLICE] + 1.0) * 0.5 * (n_glazing - 1)),
        0,
        n_glazing - 1,
    ).astype(np.int64)
    glz_ids_logged = actions_raw[:, GLAZE_SLICE].astype(np.int64)
    if not np.array_equal(glz_ids_from_norm, glz_ids_logged):
        mismatch = np.argwhere(glz_ids_from_norm != glz_ids_logged)[0]
        t, dim = (int(mismatch[0]), int(mismatch[1]))
        raise ValueError(
            "Glazing decoded from executed_action_norm does not match the "
            f"actuation log at transition {t}, glazing dim {dim}: "
            f"decoded={glz_ids_from_norm[t, dim]}, logged={glz_ids_logged[t, dim]}."
        )

    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )
    light_decoded = (
        np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
        * nominal_power_w[None, :]
    )
    heat_decoded = _denormalize_linear(
        actions[:, HEAT_SLICE], HEATING_SETPOINT_RANGE
    )
    cool_decoded = _denormalize_linear(
        actions[:, COOL_SLICE], COOLING_SETPOINT_RANGE
    )

    checks = [
        (
            "lighting",
            light_decoded,
            actions_raw[:, LIGHT_SLICE],
            "W",
        ),
        (
            "heating setpoint",
            heat_decoded,
            actions_raw[:, HEAT_SLICE],
            "degC",
        ),
        (
            "cooling setpoint",
            cool_decoded,
            actions_raw[:, COOL_SLICE],
            "degC",
        ),
    ]
    for label, decoded, logged, unit in checks:
        if not np.allclose(
            decoded,
            logged,
            atol=ACTION_VALIDATION_ATOL,
            rtol=1e-4,
        ):
            abs_error = np.abs(decoded - logged)
            t, dim = np.unravel_index(np.argmax(abs_error), abs_error.shape)
            raise ValueError(
                f"{label} decoded from executed_action_norm does not match the "
                f"actuation log. Largest error at transition {t}, local dim {dim}: "
                f"decoded={decoded[t, dim]:.6g} {unit}, "
                f"logged={logged[t, dim]:.6g} {unit}, "
                f"abs_error={abs_error[t, dim]:.6g} {unit}."
            )


def _reconstruct_sdar_context(actions, selection_masks):
    previous_keys = (
        "previous_action_norm",
        "prev_action_norm",
        "previous_executed_action_norm",
    )
    previous_key = next((k for k in previous_keys if k in agent_data), None)

    if previous_key is not None:
        logged_prev = np.asarray(agent_data[previous_key], dtype=np.float32)
        if logged_prev.ndim != 2 or logged_prev.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{previous_key!r}] has shape {logged_prev.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_prev.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{previous_key!r}] has only {logged_prev.shape[0]} "
                f"rows for {len(actions)} transitions."
            )
        prev_actions = logged_prev[:len(actions)].copy()
    else:
        if not np.all(selection_masks[0] == 1.0):
            raise ValueError(
                "The first selection mask is not a full update, but the callback "
                "did not log previous_action_norm. The initial SDAR context cannot "
                "be reconstructed without leaking action[0] into prev_action[0]."
            )
        prev_actions = actions.copy()
        prev_actions[1:] = actions[:-1]

    if len(actions) > 1 and not np.allclose(
        prev_actions[1:],
        actions[:-1],
        atol=1e-5,
        rtol=0.0,
    ):
        max_error = float(
            np.max(np.abs(prev_actions[1:] - actions[:-1]))
        )
        raise ValueError(
            "Logged/reconstructed previous actions do not equal the preceding "
            f"executed actions (max abs error {max_error:.6g})."
        )

    action_mixes = (
        (1.0 - selection_masks) * prev_actions
        + selection_masks * ACTION_MASK_VALUE
    ).astype(np.float32)

    for key in ("action_mix", "action_mixes"):
        if key not in agent_data:
            continue
        logged_mix = np.asarray(agent_data[key], dtype=np.float32)
        if logged_mix.ndim != 2 or logged_mix.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{key!r}] has shape {logged_mix.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_mix.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{key!r}] has only {logged_mix.shape[0]} rows for "
                f"{len(actions)} transitions."
            )
        if not np.allclose(
            action_mixes,
            logged_mix[:len(actions)],
            atol=1e-5,
            rtol=0.0,
        ):
            max_error = float(
                np.max(np.abs(action_mixes - logged_mix[:len(actions)]))
            )
            raise ValueError(
                f"Reconstructed action_mix differs from agent_data[{key!r}] "
                f"(max abs error {max_error:.6g})."
            )
        break

    return prev_actions.astype(np.float32), action_mixes


# ============================================================
# MAIN BUILDER
# ============================================================

def build_agent_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    reference_dataset_path=REFERENCE_REWARD_DATASET,
    checkpoint_path=EVAL_CHECKPOINT_PATH,
    checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
    eval_seed=EVAL_SEED,
    selector_mode=EVAL_SELECTOR_MODE,
    proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
    selector_seed=EVAL_SELECTOR_SEED,
    safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
    weather_file=EVAL_WEATHER_FILE,
):
    selector_mode = str(selector_mode).lower()
    if selector_mode not in {"threshold", "sample"}:
        raise ValueError(
            "selector_mode must be 'threshold' or 'sample', "
            f"got {selector_mode!r}."
        )
    proposal_deterministic = bool(proposal_deterministic)
    selector_seed = int(selector_seed)

    comp_names = _component_names()
    reward_reference = load_reward_reference(
        reference_dataset_path,
        comp_names,
    )
    checkpoint = _checkpoint_provenance(
        checkpoint_path,
        checkpoint_epoch,
    )
    if prefix is None:
        proposal_tag = (
            "propdet" if proposal_deterministic else "propstoch"
        )
        prefix = (
            f"agent_eval_dataset_decomposed_epoch{checkpoint['epoch']}"
            "_thermal3_tclip50"
            f"_selector-{selector_mode}_{proposal_tag}"
            f"_s{selector_seed}"
        )
    prefix = str(Path(prefix).expanduser())
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)

    print("=== Evaluation provenance ===")
    print(f"  checkpoint epoch: {checkpoint['epoch']}")
    print(f"  checkpoint path:  {checkpoint['path'] or '(not supplied)'}")
    print(f"  reference NPZ:    {reward_reference['path']}")
    print(
        "  reward tag:       "
        f"{reward_reference['reweighting_tag'] or '(not stored)'}"
    )
    print(f"  selector mode:    {selector_mode}")
    print(
        "  proposal deterministic: "
        f"{proposal_deterministic}"
    )
    print(f"  selector seed:    {selector_seed}")
    print(f"  safeguard:        {bool(safeguard_enabled)}")
    print(f"  eval seed:        {eval_seed}")
    print(f"  weather file:     {weather_file}")

    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1
    if T < 1:
        raise ValueError(f"Not enough logged steps to build transitions (n={n}).")

    # ---- length sanity check (agent logs only) ----
    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "agent_observation": len(agent_data["observation"]),
        "agent_selection_logits": len(agent_data["selection_logits"]),
        "agent_selection_probability": len(
            agent_data["selection_probability"]
        ),
        "agent_selection_mask": len(agent_data["selection_mask"]),
        "agent_executed_action_norm": len(agent_data["executed_action_norm"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    # ---- observations ----
    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    # Integrity: the obs we rebuild here must equal what the agent actually saw.
    agent_obs = np.asarray(agent_data["observation"], dtype=np.float32)[:T]
    if not np.allclose(observations, agent_obs, atol=1e-4):
        max_err = float(np.max(np.abs(observations - agent_obs)))
        raise ValueError(
            "Rebuilt observations differ from what the agent saw "
            f"(max abs err {max_err:.6g}). The dataset obs layout has drifted "
            "from the eval callback's build_observation.")

    # ---- actions: agent's executed normalized action is the primary action ----
    agent_actions = np.asarray(agent_data["executed_action_norm"], dtype=np.float32)
    agent_logits = np.asarray(
        agent_data["selection_logits"],
        dtype=np.float32,
    )
    agent_probabilities = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )
    agent_masks = np.asarray(agent_data["selection_mask"], dtype=np.float32)

    if agent_actions.shape != (n, ACTION_DIM):
        raise ValueError(f"executed_action_norm has shape {agent_actions.shape}, expected ({n}, {ACTION_DIM}).")
    if agent_logits.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_logits has shape "
            f"{agent_logits.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_probabilities.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_probability has shape "
            f"{agent_probabilities.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_masks.shape != (n, ACTION_DIM):
        raise ValueError(f"selection_mask has shape {agent_masks.shape}, expected ({n}, {ACTION_DIM}).")

    actions = agent_actions[:T]
    selection_logits = agent_logits[:T]
    selection_probabilities = agent_probabilities[:T]
    selection_masks = agent_masks[:T]

    if np.any(~np.isfinite(selection_logits)):
        raise ValueError("selection_logits contains NaN or infinite values.")
    if np.any(~np.isfinite(selection_probabilities)):
        raise ValueError(
            "selection_probability contains NaN or infinite values."
        )
    if (
        np.any(selection_probabilities < -1e-7)
        or np.any(selection_probabilities > 1.0 + 1e-7)
    ):
        raise ValueError(
            "selection_probability must lie in [0,1]: "
            f"min={selection_probabilities.min()}, "
            f"max={selection_probabilities.max()}."
        )

    reconstructed_probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(selection_logits, -80.0, 80.0))
    )
    if not np.allclose(
        selection_probabilities,
        reconstructed_probabilities,
        atol=1e-6,
        rtol=1e-6,
    ):
        max_err = float(
            np.max(
                np.abs(
                    selection_probabilities
                    - reconstructed_probabilities
                )
            )
        )
        raise ValueError(
            "selection_probability is inconsistent with sigmoid(logits): "
            f"max abs error={max_err:.6g}."
        )

    # Normalized executed actions must be finite and lie within [-1, 1].
    if np.any(~np.isfinite(actions)):
        raise ValueError("executed_action_norm contains NaN or infinite values.")
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: min={actions.min()}, max={actions.max()}")

    # Selector masks must be exactly binary; round only after checking tolerance.
    if np.any(~np.isfinite(selection_masks)):
        raise ValueError("selection_mask contains NaN or infinite values.")
    rounded_masks = np.rint(selection_masks)
    if not np.allclose(selection_masks, rounded_masks, atol=1e-6, rtol=0.0):
        bad = np.argwhere(np.abs(selection_masks - rounded_masks) > 1e-6)[0]
        t, dim = (int(bad[0]), int(bad[1]))
        raise ValueError(
            "selection_mask must be binary. "
            f"Found {selection_masks[t, dim]} at transition {t}, dim {dim}."
        )
    selection_masks = rounded_masks.astype(np.float32)

    if selector_mode == "threshold":
        expected_masks = (
            selection_probabilities[1:] >= 0.5
        ).astype(np.float32)
        if not np.array_equal(selection_masks[1:], expected_masks):
            raise ValueError(
                "selector_mode='threshold', but masks after the forced "
                "first step do not equal probability >= 0.5."
            )

    # Validate all physical actuations against the executed normalized action.
    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)
    _validate_actuated_actions(actions, actions_raw)

    # ---- reconstruct SDAR prev_action / action_mix from the agent trajectory ----
    prev_actions, action_mixes = _reconstruct_sdar_context(
        actions,
        selection_masks,
    )

    # ---- physical lighting quantities from the executed dimming ----
    lighting_dimming = np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)
    nominal_power_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0).astype(np.float32)

    # ---- rewards ----
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, actions[t], selection_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, components_unbounded = apply_reward_reference(
        components_raw,
        reward_reference,
    )
    comp_scales = reward_reference["scales"]
    importance_vec = reward_reference["importance"]
    component_clips = reward_reference["clips"]
    scales_source = f"training reference: {reward_reference['path']}"

    reward_physical_raw = components_raw.sum(axis=1).astype(np.float32)
    reward_balanced = components_final.sum(axis=1).astype(np.float32)
    # Legacy training-compatible key: `rewards_raw` means balanced pre-symlog.
    rewards_raw = reward_balanced
    rewards_tx = _transform_vec(reward_balanced)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32)

    # ---- episode return (undiscounted) for quick scoring ----
    ep_return_physical_raw = float(reward_physical_raw.sum())
    ep_return_balanced = float(reward_balanced.sum())
    # Legacy training-compatible alias.
    ep_return_raw = ep_return_balanced
    ep_return_tx = float(rewards.sum())

    print("\n=== Agent evaluation dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (agent executed, SDAR space)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {prev_actions.shape}  (reconstructed)")
    print(f"  selection_logits:  {selection_logits.shape}")
    print(
        "  selection_probs:   "
        f"{selection_probabilities.shape}"
    )
    print(f"  selection_masks:   {selection_masks.shape}  (agent selector)")
    print(f"  action_mixes:      {action_mixes.shape}  (reconstructed)")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced pre-transform)")
    print(f"  physical raw:      {reward_physical_raw.shape}  (sum of physical W_* terms)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(
        "  episode return:    "
        f"physical={ep_return_physical_raw:+.4f}  "
        f"balanced={ep_return_balanced:+.4f}  "
        f"transformed={ep_return_tx:+.4f}"
    )

    _print_normalization(
        comp_names,
        comp_scales,
        importance_vec,
        component_clips,
        scales_source,
    )
    _print_reward_diagnostics(
        reward_physical_raw,
        reward_balanced,
        rewards_tx,
        components_final,
    )
    _print_thermal_violation_rate()
    _print_switching_diagnostics(selection_masks)
    selector_diagnostics = _print_selector_probability_diagnostics(
        selection_probabilities,
        selection_masks,
        selector_mode,
    )
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=prev_actions,
            selection_logits=selection_logits,
            selection_probabilities=selection_probabilities,
            selection_masks=selection_masks,
            action_mixes=action_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            reward_balanced=reward_balanced,
            reward_physical_raw=reward_physical_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(reward_reference["transform"]),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            thermal_reward_component_clip=np.float32(
                component_clips[comp_names.index("thermal_comfort")]
            ),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            # evaluation provenance / quick scores
            policy=np.array(POLICY_TAG),
            dataset_kind=np.array("agent_evaluation"),
            checkpoint_epoch=np.int64(checkpoint["epoch"]),
            checkpoint_path=np.array(checkpoint["path"]),
            checkpoint_sha256=np.array(checkpoint["sha256"]),
            eval_seed=np.int64(-1 if eval_seed is None else int(eval_seed)),
            eval_selector_mode=np.array(selector_mode),
            eval_proposal_deterministic=np.array(
                proposal_deterministic
            ),
            eval_selector_seed=np.int64(selector_seed),
            eval_safeguard_enabled=np.array(bool(safeguard_enabled)),
            eval_weather_file=np.array("" if weather_file is None else str(weather_file)),
            reward_reference_dataset=np.array(reward_reference["path"]),
            reward_reference_sha256=np.array(reward_reference["sha256"]),
            reward_reweighting_tag=np.array(reward_reference["reweighting_tag"]),
            episode_return_physical_raw=np.float32(ep_return_physical_raw),
            episode_return_balanced=np.float32(ep_return_balanced),
            episode_return_raw=np.float32(ep_return_raw),
            episode_return_transformed=np.float32(ep_return_tx),
            selector_diagnostic_groups=np.array(
                ["glazing", "lighting", "heating", "cooling", "overall"]
            ),
            selector_mean_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["mean_probability"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sampled_rate_by_group=np.array(
                [
                    selector_diagnostics[name]["sampled_rate"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sample_minus_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["difference"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_threshold_agreement=np.float32(
                selector_diagnostics["threshold_agreement"]
            ),
            action_layout=np.array(
                {"glazing": [0, 4], "lighting": [4, 9], "heating": [9, 14], "cooling": [14, 19]},
                dtype=object),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_unbounded"] = components_unbounded
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)
            save_dict["reward_component_clips"] = component_clips.astype(np.float32)

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "policy": POLICY_TAG,
                    "dataset_kind": "agent_evaluation",
                    "checkpoint": checkpoint,
                    "evaluation": {
                        "seed": None if eval_seed is None else int(eval_seed),
                        "selector_mode": selector_mode,
                        "proposal_deterministic": proposal_deterministic,
                        "selector_seed": selector_seed,
                        "safeguard_enabled": bool(safeguard_enabled),
                        "weather_file": None if weather_file is None else str(weather_file),
                    },
                    "reward_reference": {
                        "dataset": reward_reference["path"],
                        "sha256": reward_reference["sha256"],
                        "reweighting_tag": reward_reference["reweighting_tag"],
                    },
                    "reward_transform": reward_reference["transform"],
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "thermal_reward_component_clip": float(
                        component_clips[comp_names.index("thermal_comfort")]
                    ),
                    "reward_scales_source": scales_source,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "episode_return_physical_raw": ep_return_physical_raw,
                    "episode_return_balanced": ep_return_balanced,
                    "episode_return_raw": ep_return_raw,
                    "episode_return_transformed": ep_return_tx,
                    "reward_physical_raw_mean": float(reward_physical_raw.mean()),
                    "reward_physical_raw_std": float(reward_physical_raw.std()),
                    "reward_balanced_mean": float(reward_balanced.mean()),
                    "reward_balanced_std": float(reward_balanced.std()),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_component_clips": component_clips.tolist(),
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(selection_masks.mean()),
                    "mean_selection_mask_glazing": float(selection_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(selection_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(selection_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(selection_masks[:, COOL_SLICE].mean()),
                    "selector_probability_diagnostics": selector_diagnostics,
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        logit_cols = [f"selector_logit_{c}" for c in act_cols]
        probability_cols = [
            f"selector_probability_{c}"
            for c in act_cols
        ]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + logit_cols
            + probability_cols
            + ["reward", "reward_raw", "reward_physical_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + selection_masks[t].tolist()
                    + selection_logits[t].tolist()
                    + selection_probabilities[t].tolist()
                    + [
                        float(rewards[t]),
                        float(rewards_raw[t]),
                        float(reward_physical_raw[t]),
                    ]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    return {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": prev_actions,
        "selection_logits": selection_logits,
        "selection_probabilities": selection_probabilities,
        "selection_masks": selection_masks,
        "action_mixes": action_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_balanced": reward_balanced,
        "reward_physical_raw": reward_physical_raw,
        "reward_components": components_final,
        "reward_components_unbounded": components_unbounded,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "reward_component_clips": component_clips,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
        "checkpoint": checkpoint,
        "evaluation": {
            "seed": None if eval_seed is None else int(eval_seed),
            "selector_mode": selector_mode,
            "proposal_deterministic": proposal_deterministic,
            "selector_seed": selector_seed,
            "safeguard_enabled": bool(safeguard_enabled),
            "weather_file": (
                None if weather_file is None else str(weather_file)
            ),
        },
        "selector_probability_diagnostics": selector_diagnostics,
        "reward_reference": reward_reference,
        "episode_return_physical_raw": ep_return_physical_raw,
        "episode_return_balanced": ep_return_balanced,
        "episode_return_raw": ep_return_raw,
        "episode_return_transformed": ep_return_tx,
    }


if __name__ == "__main__":
    dataset = build_agent_dataset(
        save_npz=True,
        save_csv=True,
        reference_dataset_path=REFERENCE_REWARD_DATASET,
        checkpoint_path=EVAL_CHECKPOINT_PATH,
        checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
        eval_seed=EVAL_SEED,
        selector_mode=EVAL_SELECTOR_MODE,
        proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
        selector_seed=EVAL_SELECTOR_SEED,
        safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
        weather_file=EVAL_WEATHER_FILE,
    )


In [ ]:
"""
Compare a trained-agent evaluation rollout against the pooled expert dataset
============================================================================

Both inputs are NPZs produced by the offline dataset builders (v8 expert builder
for the pooled set, build_agent_eval_dataset for the agent). This does a FAIR
comparison, which for this setup means:

1. Reward on a common footing.
   - PRIMARY: physical per-component reward `reward_components_raw` (pre-
     normalization). Its per-component and summed values are comparable no matter
     what reward scales each dataset used.
   - SECONDARY: shared-normalized reward (`reward_components` / `rewards`), which
     is only comparable if both NPZs were built with the SAME
     `reward_component_scales`. The tool checks this and only reports it if so.

2. The right baseline. The pooled set is a mixture of experts (some deliberately
   weak) plus exploration noise, so its MEAN is a weak bar. We break the pool out
   BY EXPERT (via `expert_ids`) and highlight the best expert as the reference,
   and also report the pooled top-decile of per-step reward.

3. Weight-free physical KPIs (don't depend on reward tuning): thermal-violation
   rate, HVAC energy, lighting energy, visual-band occupancy, switching activity.

4. Uncertainty. Day-block bootstrap 95% CIs (blocks respect episode boundaries),
   so temporal autocorrelation doesn't inflate confidence.

CONFOUND WARNING the tool raises: if the agent and pooled sets cover different
day-of-year ranges (read from the doy_sin/doy_cos obs), energy/comfort deltas are
partly weather, not policy. For a clean head-to-head, evaluate the agent on the
SAME period as a reference expert.
"""

import argparse
import json

import numpy as np


STEPS_PER_HOUR = 6
STEPS_PER_DAY = STEPS_PER_HOUR * 24          # 144
DT_HOURS = 1.0 / STEPS_PER_HOUR

COMFORT_LOW = 21.0
COMFORT_HIGH = 24.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0
LIGHTING_TO_LUX = 0.4

GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)


# ============================================================
# LOADING / COLUMN RESOLUTION
# ============================================================

def _load(path):
    d = np.load(path, allow_pickle=True)
    return {k: d[k] for k in d.files}


def _obs_cols(d):
    if "obs_cols" not in d:
        raise KeyError(f"NPZ missing 'obs_cols'; cannot resolve observation columns.")
    return [str(c) for c in d["obs_cols"]]


def _col_idx(cols, name):
    return cols.index(name)


def _cols_prefixed(cols, prefix):
    return [i for i, c in enumerate(cols) if c.startswith(prefix)]


# ============================================================
# EPISODE / DAY BLOCKS  (for block bootstrap)
# ============================================================

def _episode_bounds(terminals):
    """Return list of (start, end_exclusive) per episode using terminal flags."""
    term = np.where(np.asarray(terminals).reshape(-1) > 0.5)[0]
    bounds = []
    start = 0
    for e in term:
        bounds.append((start, int(e) + 1))
        start = int(e) + 1
    if start < len(terminals):
        bounds.append((start, len(terminals)))
    return bounds


def _day_blocks(terminals):
    """Contiguous day-length blocks that never cross an episode boundary."""
    blocks = []
    for s, e in _episode_bounds(terminals):
        i = s
        while i < e:
            j = min(i + STEPS_PER_DAY, e)
            blocks.append(np.arange(i, j))
            i = j
    return blocks


def _block_bootstrap_ratio(numer, denom, blocks, n_boot=1000, seed=0):
    """95% CI for sum(numer)/sum(denom) via resampling day-blocks with replacement.
    denom may be an all-ones array (=> a plain mean of numer)."""
    rng = np.random.default_rng(seed)
    numer = np.asarray(numer, dtype=np.float64)
    denom = np.asarray(denom, dtype=np.float64)
    point = numer.sum() / max(denom.sum(), 1e-12)
    if len(blocks) == 0:
        return point, point, point
    nb = len(blocks)
    vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, nb, size=nb)
        idx = np.concatenate([blocks[p] for p in pick])
        vals[b] = numer[idx].sum() / max(denom[idx].sum(), 1e-12)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(point), float(lo), float(hi)


# ============================================================
# PER-TRANSITION METRIC ARRAYS
# ============================================================

def _metrics(d):
    """Return per-transition arrays used by every comparison, plus block list."""
    cols = _obs_cols(d)
    obs = d["observations"].astype(np.float64)
    T = obs.shape[0]

    temp_idx = _cols_prefixed(cols, "temp_")
    light_rate_idx = _cols_prefixed(cols, "light_rate_")
    wpi_idx = _cols_prefixed(cols, "wpi_")
    occ_idx = _col_idx(cols, "is_occupied")
    hvac_idx = _col_idx(cols, "hvac_electricity_demand_rate")
    doy_sin_idx = _col_idx(cols, "doy_sin")
    doy_cos_idx = _col_idx(cols, "doy_cos")

    occupied = obs[:, occ_idx] > 0.5

    # thermal: per-transition count of zones out of band (all 5 zones)
    temps = obs[:, temp_idx]
    out_of_band = ((temps < COMFORT_LOW) | (temps > COMFORT_HIGH)).sum(axis=1).astype(np.float64)
    n_zones = float(len(temp_idx))

    # hvac demand (W)
    hvac_w = obs[:, hvac_idx]

    # lighting energy (kWh) per transition
    if "lighting_energy_kwh" in d:
        light_kwh = d["lighting_energy_kwh"].astype(np.float64).reshape(-1)[:T]
    elif "lighting_power_from_action_w" in d:
        light_kwh = d["lighting_power_from_action_w"].astype(np.float64).sum(axis=1)[:T] * DT_HOURS / 1000.0
    else:
        light_kwh = np.full(T, np.nan)

    # visual band occupancy on the 4 perimeter window zones (needs perimeter light power)
    visual_in_band = np.full(T, np.nan)
    visual_denom = np.zeros(T)
    if "lighting_power_from_action_w" in d and len(wpi_idx) == len(light_rate_idx[:4]):
        # perimeter light power = first 4 of the 5-zone action light power (zone order)
        lp = d["lighting_power_from_action_w"].astype(np.float64)[:T, :4]
        wpi = obs[:, wpi_idx]                      # 4 window zones
        total_lux = wpi + lp * LIGHTING_TO_LUX      # (T,4)
        in_band = ((total_lux >= ILLUM_LOW) & (total_lux <= ILLUM_HIGH)).astype(np.float64)
        # count only occupied steps
        visual_in_band = (in_band.sum(axis=1)) * occupied
        visual_denom = occupied.astype(np.float64) * in_band.shape[1]

    # switching activity from the SDAR mask
    if "selection_masks" in d:
        masks = d["selection_masks"].astype(np.float64)[:T]
        switch_overall = masks.mean(axis=1)
        switch_glaze = masks[:, GLAZE_SLICE].mean(axis=1)
        switch_light = masks[:, LIGHT_SLICE].mean(axis=1)
        switch_hvac = np.concatenate([masks[:, HEAT_SLICE], masks[:, COOL_SLICE]], axis=1).mean(axis=1)
    else:
        switch_overall = switch_glaze = switch_light = switch_hvac = np.full(T, np.nan)

    # physical per-component reward (scale-free)
    if "reward_components_raw" in d:
        comps_raw = d["reward_components_raw"].astype(np.float64)
        comp_names = [str(c) for c in d["reward_component_names"]] if "reward_component_names" in d else \
            [f"c{i}" for i in range(comps_raw.shape[1])]
        phys_total = comps_raw.sum(axis=1)
    else:
        comps_raw = None
        comp_names = []
        phys_total = np.asarray(d.get("rewards_raw", np.full(T, np.nan))).reshape(-1)[:T]

    terminals = d["terminals"].reshape(-1)[:T]
    blocks = _day_blocks(terminals)
    n_days = max(1, T / STEPS_PER_DAY)

    return {
        "T": T,
        "occupied": occupied,
        "out_of_band": out_of_band,
        "n_zones": n_zones,
        "hvac_w": hvac_w,
        "light_kwh": light_kwh,
        "visual_in_band": visual_in_band,
        "visual_denom": visual_denom,
        "switch_overall": switch_overall,
        "switch_glaze": switch_glaze,
        "switch_light": switch_light,
        "switch_hvac": switch_hvac,
        "comps_raw": comps_raw,
        "comp_names": comp_names,
        "phys_total": phys_total,
        "blocks": blocks,
        "n_days": n_days,
        "doy_sin": obs[:, doy_sin_idx],
        "doy_cos": obs[:, doy_cos_idx],
        "scales": d.get("reward_component_scales"),
        "expert_ids": d.get("expert_ids"),
    }


def _restrict(m, mask):
    """Restrict all per-transition arrays (and rebuild blocks) to a boolean mask.
    Used for per-expert slices of the pooled set. Arrays are COMPACTED to the
    kept transitions, and blocks are rebuilt in that compacted index space."""
    idx = np.where(mask)[0]
    out = dict(m)
    for key in ["occupied", "out_of_band", "hvac_w", "light_kwh", "visual_in_band",
                "visual_denom", "switch_overall", "switch_glaze", "switch_light",
                "switch_hvac", "phys_total", "doy_sin", "doy_cos"]:
        out[key] = m[key][idx]
    if m["comps_raw"] is not None:
        out["comps_raw"] = m["comps_raw"][idx]
    out["T"] = len(idx)

    # Blocks are positions 0..len(idx)-1: group originally-contiguous kept
    # indices, then split each run into day-length blocks.
    blocks = []
    run_start = 0
    for k in range(1, len(idx) + 1):
        if k == len(idx) or idx[k] != idx[k - 1] + 1:
            run = np.arange(run_start, k)
            for j in range(0, len(run), STEPS_PER_DAY):
                blocks.append(run[j:j + STEPS_PER_DAY])
            run_start = k
    out["blocks"] = blocks
    out["n_days"] = max(1, len(idx) / STEPS_PER_DAY)
    return out


# ============================================================
# KPI TABLE FOR ONE DATASET SLICE
# ============================================================

def _kpis(m, n_boot, seed):
    ones = np.ones(m["T"])
    occ = m["occupied"].astype(np.float64)

    def ci(numer, denom):
        return _block_bootstrap_ratio(numer, denom, m["blocks"], n_boot, seed)

    thermal = ci(m["out_of_band"] * occ, occ * m["n_zones"])       # frac occupied zone-steps out of band
    hvac_kwh_day = ci(m["hvac_w"] * DT_HOURS / 1000.0, ones / m["n_days"])  # kWh per day (sum/day)
    light_kwh_day = ci(np.nan_to_num(m["light_kwh"]), ones / m["n_days"])
    reward = ci(m["phys_total"], ones)                              # per-step physical reward
    switch = ci(m["switch_overall"], ones)

    out = {
        "n_transitions": m["T"],
        "reward_per_step_physical": thermal_fmt(reward),
        "thermal_violation_rate_pct": thermal_fmt(thermal, scale=100.0),
        "hvac_kwh_per_day": thermal_fmt(hvac_kwh_day),
        "lighting_kwh_per_day": thermal_fmt(light_kwh_day),
        "switch_frac_per_step": thermal_fmt(switch),
    }
    if not np.all(np.isnan(m["visual_in_band"])) and m["visual_denom"].sum() > 0:
        vis = ci(m["visual_in_band"], m["visual_denom"])
        out["visual_in_band_pct"] = thermal_fmt(vis, scale=100.0)
    return out


def thermal_fmt(ci_tuple, scale=1.0):
    p, lo, hi = ci_tuple
    return {"point": p * scale, "ci95": [lo * scale, hi * scale]}


# ============================================================
# MAIN COMPARISON
# ============================================================

def compare(pooled_path, agent_path, n_boot=1000, seed=0, out_json=None):
    pooled = _load(pooled_path)
    agent = _load(agent_path)

    mp = _metrics(pooled)
    ma = _metrics(agent)

    print("=" * 74)
    print("AGENT vs POOLED — comparison")
    print("=" * 74)
    print(f"  pooled: {mp['T']} transitions (~{mp['n_days']:.1f} days), "
          f"{len(_episode_bounds(pooled['terminals']))} episodes")
    print(f"  agent : {ma['T']} transitions (~{ma['n_days']:.1f} days), "
          f"{len(_episode_bounds(agent['terminals']))} episodes")

    # --- reward-scale consistency (does shared-normalized reward compare?) ---
    scales_match = False
    if mp["scales"] is not None and ma["scales"] is not None:
        sp = np.asarray(mp["scales"], dtype=np.float64)
        sa = np.asarray(ma["scales"], dtype=np.float64)
        if sp.shape == sa.shape:
            scales_match = bool(np.allclose(sp, sa, rtol=1e-3, atol=1e-8))
    print(f"  reward_component_scales identical: {scales_match}  "
          f"({'shared-normalized reward is comparable' if scales_match else 'use PHYSICAL reward only'})")

    # --- period / weather confound check via day-of-year coverage ---
    def doy_range(m):
        ang = np.arctan2(m["doy_sin"], m["doy_cos"])
        doy = (ang / (2 * np.pi)) % 1.0 * 365.0
        return doy.min(), doy.max()
    pmin, pmax = doy_range(mp)
    amin, amax = doy_range(ma)
    overlap = not (amax < pmin or amin > pmax)
    print(f"  day-of-year coverage  pooled=[{pmin:.0f},{pmax:.0f}]  agent=[{amin:.0f},{amax:.0f}]  "
          f"{'OVERLAP' if overlap else 'DISJOINT -> energy/comfort deltas are partly weather!'}")

    # --- component-wise physical reward (per-step mean) ---
    print("\n--- Physical per-component reward (per-step mean; scale-free) ---")
    if mp["comps_raw"] is not None and ma["comps_raw"] is not None and mp["comp_names"] == ma["comp_names"]:
        print(f"  {'component':24s}  {'pooled(all)':>12s}  {'agent':>12s}  {'agent-pooled':>13s}")
        for i, name in enumerate(mp["comp_names"]):
            pp = mp["comps_raw"][:, i].mean()
            aa = ma["comps_raw"][:, i].mean()
            print(f"  {name:24s}  {pp:12.4f}  {aa:12.4f}  {aa - pp:+13.4f}")
        pt = mp["phys_total"].mean(); at = ma["phys_total"].mean()
        print(f"  {'TOTAL':24s}  {pt:12.4f}  {at:12.4f}  {at - pt:+13.4f}")
    else:
        print("  (component names/arrays differ between datasets; skipping component table)")

    # --- per-expert breakdown of the pooled set + best-expert reference ---
    expert_summ = {}
    best_expert = None
    if mp["expert_ids"] is not None:
        eids = np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]]
        print("\n--- Pooled per-expert reference (per-step physical reward) ---")
        best_val = -np.inf
        for e in np.unique(eids):
            me = _restrict(mp, eids == e)
            r = me["phys_total"].mean()
            expert_summ[int(e)] = {"n": me["T"], "reward_per_step_physical": float(r)}
            print(f"  expert {int(e):2d}:  n={me['T']:6d}  reward/step={r:+.4f}")
            if r > best_val:
                best_val = r; best_expert = int(e)
        print(f"  -> best expert = {best_expert} (reward/step {best_val:+.4f}) used as the reference")

    # --- KPI tables with bootstrap CIs ---
    print(f"\n--- KPIs (point [95% block-bootstrap CI], n_boot={n_boot}) ---")
    kp_pool = _kpis(mp, n_boot, seed)
    kp_agent = _kpis(ma, n_boot, seed)
    kp_best = None
    if best_expert is not None:
        kp_best = _kpis(_restrict(mp, np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]] == best_expert),
                        n_boot, seed)

    metrics = [k for k in kp_agent.keys() if k != "n_transitions"]
    hdr = f"  {'metric':28s}  {'pooled(all)':>22s}  "
    hdr += f"{'best expert':>22s}  " if kp_best else ""
    hdr += f"{'agent':>22s}"
    print(hdr)
    for k in metrics:
        def cell(kp):
            if k not in kp:
                return f"{'--':>22s}"
            p = kp[k]["point"]; lo, hi = kp[k]["ci95"]
            return f"{p:8.3f} [{lo:7.3f},{hi:7.3f}]"
        row = f"  {k:28s}  {cell(kp_pool):>22s}  "
        row += f"{cell(kp_best):>22s}  " if kp_best else ""
        row += f"{cell(kp_agent):>22s}"
        print(row)

    # --- headline difference: agent vs best expert, per-step physical reward ---
    if best_expert is not None:
        me = _restrict(mp, np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]] == best_expert)
        rng = np.random.default_rng(seed)
        diffs = np.empty(n_boot)
        for b in range(n_boot):
            ia = np.concatenate([ma["blocks"][p] for p in rng.integers(0, len(ma["blocks"]), len(ma["blocks"]))])
            ie = np.concatenate([me["blocks"][p] for p in rng.integers(0, len(me["blocks"]), len(me["blocks"]))])
            diffs[b] = ma["phys_total"][ia].mean() - me["phys_total"][ie].mean()
        dlo, dhi = np.percentile(diffs, [2.5, 97.5])
        dpt = ma["phys_total"].mean() - me["phys_total"].mean()
        verdict = "AGENT better" if dlo > 0 else ("EXPERT better" if dhi < 0 else "no significant diff")
        print(f"\n  HEADLINE  agent - best_expert (reward/step) = {dpt:+.4f}  "
              f"95% CI [{dlo:+.4f}, {dhi:+.4f}]  ->  {verdict}")

    result = {
        "pooled_transitions": mp["T"],
        "agent_transitions": ma["T"],
        "reward_scales_match": scales_match,
        "doy_overlap": overlap,
        "kpis_pooled_all": kp_pool,
        "kpis_agent": kp_agent,
        "kpis_best_expert": kp_best,
        "best_expert_id": best_expert,
        "expert_reward_per_step": expert_summ,
    }
    if out_json:
        with open(out_json, "w") as f:
            json.dump(result, f, indent=2)
        print(f"\nSaved: {out_json}")
    return result


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("pooled", help="pooled expert dataset .npz (shared reward)")
    ap.add_argument("agent", help="agent evaluation dataset .npz")
    ap.add_argument("--n-boot", type=int, default=1000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out-json", default=None)
    args = ap.parse_args()
    compare(args.pooled, args.agent, n_boot=args.n_boot, seed=args.seed, out_json=args.out_json)

In [ ]:
"""
Compare a trained-agent evaluation rollout against the pooled expert dataset
============================================================================

Both inputs are NPZs produced by the offline dataset builders (v8 expert builder
for the pooled set, build_agent_eval_dataset for the agent). This does a FAIR
comparison, which for this setup means:

1. Reward on a common footing.
   - PRIMARY: physical per-component reward `reward_components_raw` (pre-
     normalization). Its per-component and summed values are comparable no matter
     what reward scales each dataset used.
   - SECONDARY: shared-normalized reward (`reward_components` / `rewards`), which
     is only comparable if both NPZs were built with the SAME
     `reward_component_scales`. The tool checks this and only reports it if so.

2. The right baseline. The pooled set is a mixture of experts (some deliberately
   weak) plus exploration noise, so its MEAN is a weak bar. We break the pool out
   BY EXPERT (via `expert_ids`) and highlight the best expert as the reference,
   and also report the pooled top-decile of per-step reward.

3. Weight-free physical KPIs (don't depend on reward tuning): thermal-violation
   rate, HVAC energy, lighting energy, visual-band occupancy, switching activity.

4. Uncertainty. Day-block bootstrap 95% CIs (blocks respect episode boundaries),
   so temporal autocorrelation doesn't inflate confidence.

CONFOUND WARNING the tool raises: if the agent and pooled sets cover different
day-of-year ranges (read from the doy_sin/doy_cos obs), energy/comfort deltas are
partly weather, not policy. For a clean head-to-head, evaluate the agent on the
SAME period as a reference expert.
"""

import argparse
import json

import numpy as np


STEPS_PER_HOUR = 6
STEPS_PER_DAY = STEPS_PER_HOUR * 24          # 144
DT_HOURS = 1.0 / STEPS_PER_HOUR

COMFORT_LOW = 21.0
COMFORT_HIGH = 24.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0
LIGHTING_TO_LUX = 0.4

GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)


# ============================================================
# LOADING / COLUMN RESOLUTION
# ============================================================

def _load(path):
    d = np.load(path, allow_pickle=True)
    return {k: d[k] for k in d.files}


def _obs_cols(d):
    if "obs_cols" not in d:
        raise KeyError(f"NPZ missing 'obs_cols'; cannot resolve observation columns.")
    return [str(c) for c in d["obs_cols"]]


def _col_idx(cols, name):
    return cols.index(name)


def _cols_prefixed(cols, prefix):
    return [i for i, c in enumerate(cols) if c.startswith(prefix)]


# ============================================================
# EPISODE / DAY BLOCKS  (for block bootstrap)
# ============================================================

def _episode_bounds(terminals):
    """Return list of (start, end_exclusive) per episode using terminal flags."""
    term = np.where(np.asarray(terminals).reshape(-1) > 0.5)[0]
    bounds = []
    start = 0
    for e in term:
        bounds.append((start, int(e) + 1))
        start = int(e) + 1
    if start < len(terminals):
        bounds.append((start, len(terminals)))
    return bounds


def _day_blocks(terminals):
    """Contiguous day-length blocks that never cross an episode boundary."""
    blocks = []
    for s, e in _episode_bounds(terminals):
        i = s
        while i < e:
            j = min(i + STEPS_PER_DAY, e)
            blocks.append(np.arange(i, j))
            i = j
    return blocks


def _block_bootstrap_ratio(numer, denom, blocks, n_boot=1000, seed=0):
    """95% CI for sum(numer)/sum(denom) via resampling day-blocks with replacement.
    denom may be an all-ones array (=> a plain mean of numer)."""
    rng = np.random.default_rng(seed)
    numer = np.asarray(numer, dtype=np.float64)
    denom = np.asarray(denom, dtype=np.float64)
    point = numer.sum() / max(denom.sum(), 1e-12)
    if len(blocks) == 0:
        return point, point, point
    nb = len(blocks)
    vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, nb, size=nb)
        idx = np.concatenate([blocks[p] for p in pick])
        vals[b] = numer[idx].sum() / max(denom[idx].sum(), 1e-12)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(point), float(lo), float(hi)


# ============================================================
# PER-TRANSITION METRIC ARRAYS
# ============================================================

def _metrics(d):
    """Return per-transition arrays used by every comparison, plus block list."""
    cols = _obs_cols(d)
    obs = d["observations"].astype(np.float64)
    T = obs.shape[0]

    temp_idx = _cols_prefixed(cols, "temp_")
    light_rate_idx = _cols_prefixed(cols, "light_rate_")
    wpi_idx = _cols_prefixed(cols, "wpi_")
    occ_idx = _col_idx(cols, "is_occupied")
    hvac_idx = _col_idx(cols, "hvac_electricity_demand_rate")
    doy_sin_idx = _col_idx(cols, "doy_sin")
    doy_cos_idx = _col_idx(cols, "doy_cos")

    occupied = obs[:, occ_idx] > 0.5

    # thermal: per-transition count of zones out of band (all 5 zones)
    temps = obs[:, temp_idx]
    out_of_band = ((temps < COMFORT_LOW) | (temps > COMFORT_HIGH)).sum(axis=1).astype(np.float64)
    n_zones = float(len(temp_idx))

    # hvac demand (W)
    hvac_w = obs[:, hvac_idx]

    # lighting energy (kWh) per transition
    if "lighting_energy_kwh" in d:
        light_kwh = d["lighting_energy_kwh"].astype(np.float64).reshape(-1)[:T]
    elif "lighting_power_from_action_w" in d:
        light_kwh = d["lighting_power_from_action_w"].astype(np.float64).sum(axis=1)[:T] * DT_HOURS / 1000.0
    else:
        light_kwh = np.full(T, np.nan)

    # visual band occupancy on the 4 perimeter window zones (needs perimeter light power)
    visual_in_band = np.full(T, np.nan)
    visual_denom = np.zeros(T)
    if "lighting_power_from_action_w" in d and len(wpi_idx) == len(light_rate_idx[:4]):
        # perimeter light power = first 4 of the 5-zone action light power (zone order)
        lp = d["lighting_power_from_action_w"].astype(np.float64)[:T, :4]
        wpi = obs[:, wpi_idx]                      # 4 window zones
        total_lux = wpi + lp * LIGHTING_TO_LUX      # (T,4)
        in_band = ((total_lux >= ILLUM_LOW) & (total_lux <= ILLUM_HIGH)).astype(np.float64)
        # count only occupied steps
        visual_in_band = (in_band.sum(axis=1)) * occupied
        visual_denom = occupied.astype(np.float64) * in_band.shape[1]

    # switching activity from the SDAR mask
    if "selection_masks" in d:
        masks = d["selection_masks"].astype(np.float64)[:T]
        switch_overall = masks.mean(axis=1)
        switch_glaze = masks[:, GLAZE_SLICE].mean(axis=1)
        switch_light = masks[:, LIGHT_SLICE].mean(axis=1)
        switch_hvac = np.concatenate([masks[:, HEAT_SLICE], masks[:, COOL_SLICE]], axis=1).mean(axis=1)
    else:
        switch_overall = switch_glaze = switch_light = switch_hvac = np.full(T, np.nan)

    # physical per-component reward (scale-free)
    if "reward_components_raw" in d:
        comps_raw = d["reward_components_raw"].astype(np.float64)
        comp_names = [str(c) for c in d["reward_component_names"]] if "reward_component_names" in d else \
            [f"c{i}" for i in range(comps_raw.shape[1])]
        phys_total = comps_raw.sum(axis=1)
    else:
        comps_raw = None
        comp_names = []
        phys_total = np.asarray(d.get("rewards_raw", np.full(T, np.nan))).reshape(-1)[:T]

    terminals = d["terminals"].reshape(-1)[:T]
    blocks = _day_blocks(terminals)
    n_days = max(1, T / STEPS_PER_DAY)

    return {
        "T": T,
        "occupied": occupied,
        "out_of_band": out_of_band,
        "n_zones": n_zones,
        "hvac_w": hvac_w,
        "light_kwh": light_kwh,
        "visual_in_band": visual_in_band,
        "visual_denom": visual_denom,
        "switch_overall": switch_overall,
        "switch_glaze": switch_glaze,
        "switch_light": switch_light,
        "switch_hvac": switch_hvac,
        "comps_raw": comps_raw,
        "comp_names": comp_names,
        "phys_total": phys_total,
        "blocks": blocks,
        "n_days": n_days,
        "doy_sin": obs[:, doy_sin_idx],
        "doy_cos": obs[:, doy_cos_idx],
        "scales": d.get("reward_component_scales"),
        "expert_ids": d.get("expert_ids"),
    }


def _restrict(m, mask):
    """Restrict all per-transition arrays (and rebuild blocks) to a boolean mask.
    Used for per-expert slices of the pooled set. Arrays are COMPACTED to the
    kept transitions, and blocks are rebuilt in that compacted index space."""
    idx = np.where(mask)[0]
    out = dict(m)
    for key in ["occupied", "out_of_band", "hvac_w", "light_kwh", "visual_in_band",
                "visual_denom", "switch_overall", "switch_glaze", "switch_light",
                "switch_hvac", "phys_total", "doy_sin", "doy_cos"]:
        out[key] = m[key][idx]
    if m["comps_raw"] is not None:
        out["comps_raw"] = m["comps_raw"][idx]
    out["T"] = len(idx)

    # Blocks are positions 0..len(idx)-1: group originally-contiguous kept
    # indices, then split each run into day-length blocks.
    blocks = []
    run_start = 0
    for k in range(1, len(idx) + 1):
        if k == len(idx) or idx[k] != idx[k - 1] + 1:
            run = np.arange(run_start, k)
            for j in range(0, len(run), STEPS_PER_DAY):
                blocks.append(run[j:j + STEPS_PER_DAY])
            run_start = k
    out["blocks"] = blocks
    out["n_days"] = max(1, len(idx) / STEPS_PER_DAY)
    return out


# ============================================================
# KPI TABLE FOR ONE DATASET SLICE
# ============================================================

def _kpis(m, n_boot, seed):
    ones = np.ones(m["T"])
    occ = m["occupied"].astype(np.float64)

    def ci(numer, denom):
        return _block_bootstrap_ratio(numer, denom, m["blocks"], n_boot, seed)

    thermal = ci(m["out_of_band"] * occ, occ * m["n_zones"])       # frac occupied zone-steps out of band
    hvac_kwh_day = ci(m["hvac_w"] * DT_HOURS / 1000.0, ones / m["n_days"])  # kWh per day (sum/day)
    light_kwh_day = ci(np.nan_to_num(m["light_kwh"]), ones / m["n_days"])
    reward = ci(m["phys_total"], ones)                              # per-step physical reward
    switch = ci(m["switch_overall"], ones)

    out = {
        "n_transitions": m["T"],
        "reward_per_step_physical": thermal_fmt(reward),
        "thermal_violation_rate_pct": thermal_fmt(thermal, scale=100.0),
        "hvac_kwh_per_day": thermal_fmt(hvac_kwh_day),
        "lighting_kwh_per_day": thermal_fmt(light_kwh_day),
        "switch_frac_per_step": thermal_fmt(switch),
    }
    if not np.all(np.isnan(m["visual_in_band"])) and m["visual_denom"].sum() > 0:
        vis = ci(m["visual_in_band"], m["visual_denom"])
        out["visual_in_band_pct"] = thermal_fmt(vis, scale=100.0)
    return out


def thermal_fmt(ci_tuple, scale=1.0):
    p, lo, hi = ci_tuple
    return {"point": p * scale, "ci95": [lo * scale, hi * scale]}


# ============================================================
# MAIN COMPARISON
# ============================================================

def compare(pooled_path, agent_path, n_boot=1000, seed=0, out_json=None):
    pooled = _load(pooled_path)
    agent = _load(agent_path)

    mp = _metrics(pooled)
    ma = _metrics(agent)

    print("=" * 74)
    print("AGENT vs POOLED — comparison")
    print("=" * 74)
    print(f"  pooled: {mp['T']} transitions (~{mp['n_days']:.1f} days), "
          f"{len(_episode_bounds(pooled['terminals']))} episodes")
    print(f"  agent : {ma['T']} transitions (~{ma['n_days']:.1f} days), "
          f"{len(_episode_bounds(agent['terminals']))} episodes")

    # --- reward-scale consistency (does shared-normalized reward compare?) ---
    scales_match = False
    if mp["scales"] is not None and ma["scales"] is not None:
        sp = np.asarray(mp["scales"], dtype=np.float64)
        sa = np.asarray(ma["scales"], dtype=np.float64)
        if sp.shape == sa.shape:
            scales_match = bool(np.allclose(sp, sa, rtol=1e-3, atol=1e-8))
    print(f"  reward_component_scales identical: {scales_match}  "
          f"({'shared-normalized reward is comparable' if scales_match else 'use PHYSICAL reward only'})")

    # --- period / weather confound check via day-of-year coverage ---
    def doy_range(m):
        ang = np.arctan2(m["doy_sin"], m["doy_cos"])
        doy = (ang / (2 * np.pi)) % 1.0 * 365.0
        return doy.min(), doy.max()
    pmin, pmax = doy_range(mp)
    amin, amax = doy_range(ma)
    overlap = not (amax < pmin or amin > pmax)
    print(f"  day-of-year coverage  pooled=[{pmin:.0f},{pmax:.0f}]  agent=[{amin:.0f},{amax:.0f}]  "
          f"{'OVERLAP' if overlap else 'DISJOINT -> energy/comfort deltas are partly weather!'}")

    # --- component-wise physical reward (per-step mean) ---
    print("\n--- Physical per-component reward (per-step mean; scale-free) ---")
    if mp["comps_raw"] is not None and ma["comps_raw"] is not None and mp["comp_names"] == ma["comp_names"]:
        print(f"  {'component':24s}  {'pooled(all)':>12s}  {'agent':>12s}  {'agent-pooled':>13s}")
        for i, name in enumerate(mp["comp_names"]):
            pp = mp["comps_raw"][:, i].mean()
            aa = ma["comps_raw"][:, i].mean()
            print(f"  {name:24s}  {pp:12.4f}  {aa:12.4f}  {aa - pp:+13.4f}")
        pt = mp["phys_total"].mean(); at = ma["phys_total"].mean()
        print(f"  {'TOTAL':24s}  {pt:12.4f}  {at:12.4f}  {at - pt:+13.4f}")
    else:
        print("  (component names/arrays differ between datasets; skipping component table)")

    # --- per-expert breakdown of the pooled set + best-expert reference ---
    expert_summ = {}
    best_expert = None
    if mp["expert_ids"] is not None:
        eids = np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]]
        print("\n--- Pooled per-expert reference (per-step physical reward) ---")
        best_val = -np.inf
        for e in np.unique(eids):
            me = _restrict(mp, eids == e)
            r = me["phys_total"].mean()
            expert_summ[int(e)] = {"n": me["T"], "reward_per_step_physical": float(r)}
            print(f"  expert {int(e):2d}:  n={me['T']:6d}  reward/step={r:+.4f}")
            if r > best_val:
                best_val = r; best_expert = int(e)
        print(f"  -> best expert = {best_expert} (reward/step {best_val:+.4f}) used as the reference")

    # --- KPI tables with bootstrap CIs ---
    print(f"\n--- KPIs (point [95% block-bootstrap CI], n_boot={n_boot}) ---")
    kp_pool = _kpis(mp, n_boot, seed)
    kp_agent = _kpis(ma, n_boot, seed)
    kp_best = None
    if best_expert is not None:
        kp_best = _kpis(_restrict(mp, np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]] == best_expert),
                        n_boot, seed)

    metrics = [k for k in kp_agent.keys() if k != "n_transitions"]
    hdr = f"  {'metric':28s}  {'pooled(all)':>22s}  "
    hdr += f"{'best expert':>22s}  " if kp_best else ""
    hdr += f"{'agent':>22s}"
    print(hdr)
    for k in metrics:
        def cell(kp):
            if k not in kp:
                return f"{'--':>22s}"
            p = kp[k]["point"]; lo, hi = kp[k]["ci95"]
            return f"{p:8.3f} [{lo:7.3f},{hi:7.3f}]"
        row = f"  {k:28s}  {cell(kp_pool):>22s}  "
        row += f"{cell(kp_best):>22s}  " if kp_best else ""
        row += f"{cell(kp_agent):>22s}"
        print(row)

    # --- headline difference: agent vs best expert, per-step physical reward ---
    if best_expert is not None:
        me = _restrict(mp, np.asarray(mp["expert_ids"]).reshape(-1)[:mp["T"]] == best_expert)
        rng = np.random.default_rng(seed)
        diffs = np.empty(n_boot)
        for b in range(n_boot):
            ia = np.concatenate([ma["blocks"][p] for p in rng.integers(0, len(ma["blocks"]), len(ma["blocks"]))])
            ie = np.concatenate([me["blocks"][p] for p in rng.integers(0, len(me["blocks"]), len(me["blocks"]))])
            diffs[b] = ma["phys_total"][ia].mean() - me["phys_total"][ie].mean()
        dlo, dhi = np.percentile(diffs, [2.5, 97.5])
        dpt = ma["phys_total"].mean() - me["phys_total"].mean()
        verdict = "AGENT better" if dlo > 0 else ("EXPERT better" if dhi < 0 else "no significant diff")
        print(f"\n  HEADLINE  agent - best_expert (reward/step) = {dpt:+.4f}  "
              f"95% CI [{dlo:+.4f}, {dhi:+.4f}]  ->  {verdict}")

    result = {
        "pooled_transitions": mp["T"],
        "agent_transitions": ma["T"],
        "reward_scales_match": scales_match,
        "doy_overlap": overlap,
        "kpis_pooled_all": kp_pool,
        "kpis_agent": kp_agent,
        "kpis_best_expert": kp_best,
        "best_expert_id": best_expert,
        "expert_reward_per_step": expert_summ,
    }
    if out_json:
        with open(out_json, "w") as f:
            json.dump(result, f, indent=2)
        print(f"\nSaved: {out_json}")
    return result


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("pooled", help="pooled expert dataset .npz (shared reward)")
    ap.add_argument("agent", help="agent evaluation dataset .npz")
    ap.add_argument("--n-boot", type=int, default=1000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out-json", default=None)
    args = ap.parse_args()
    compare(args.pooled, args.agent, n_boot=args.n_boot, seed=args.seed, out_json=args.out_json)

In [ ]:
"""
Reweight an existing pooled SDAR dataset without rerunning EnergyPlus.

Changes:
    thermal_comfort importance: 2.0 -> 3.0
    thermal_comfort clip:       10.0 -> 50.0
    all other component clips:  10.0

The following arrays are rebuilt:
    reward_components
    reward_component_importance
    reward_component_clips
    rewards_raw
    rewards

The physical reward_components_raw and shared normalization scales are
preserved unchanged.
"""

from pathlib import Path

import numpy as np


# ============================================================
# PATHS
# ============================================================

INPUT_PATH = Path("pooled_sdar_experts.npz")

OUTPUT_PATH = Path(
    "pooled_sdar_experts_thermal3_tclip50.npz"
)


# ============================================================
# REWARD CONFIGURATION
# ============================================================

REWARD_TRANSFORM = "symlog"

DEFAULT_COMPONENT_CLIP = 10.0
THERMAL_COMPONENT_CLIP = 50.0

NEW_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}


# ============================================================
# REWARD TRANSFORM
# ============================================================

def symlog(x):
    """Signed logarithm used by the original dataset builder."""
    x = np.asarray(x, dtype=np.float64)

    return (
        np.sign(x) * np.log1p(np.abs(x))
    )


# ============================================================
# MAIN
# ============================================================

def main():
    if not INPUT_PATH.exists():
        raise FileNotFoundError(
            f"Input dataset does not exist: {INPUT_PATH.resolve()}"
        )

    if INPUT_PATH.resolve() == OUTPUT_PATH.resolve():
        raise ValueError(
            "INPUT_PATH and OUTPUT_PATH must be different. "
            "Do not overwrite the original dataset."
        )

    if OUTPUT_PATH.exists():
        raise FileExistsError(
            f"Output already exists: {OUTPUT_PATH.resolve()}\n"
            "Rename OUTPUT_PATH or remove the existing output "
            "after verifying that it is no longer needed."
        )

    print("=" * 72)
    print("REWEIGHT POOLED SDAR DATASET")
    print("=" * 72)
    print("Input: ", INPUT_PATH.resolve())
    print("Output:", OUTPUT_PATH.resolve())

    with np.load(INPUT_PATH, allow_pickle=True) as source:
        required_keys = [
            "reward_components_raw",
            "reward_component_names",
            "reward_component_scales",
            "reward_component_importance",
            "rewards_raw",
            "rewards",
            "observations",
            "next_observations",
            "terminals",
        ]

        missing_keys = [
            key for key in required_keys
            if key not in source.files
        ]

        if missing_keys:
            raise KeyError(
                "Input dataset is missing required arrays: "
                f"{missing_keys}"
            )

        # ----------------------------------------------------
        # Load reward metadata
        # ----------------------------------------------------

        component_names = [
            str(name)
            for name in source["reward_component_names"]
        ]

        raw_components = source[
            "reward_components_raw"
        ].astype(np.float64)

        shared_scales = source[
            "reward_component_scales"
        ].astype(np.float64).reshape(-1)

        old_importance = source[
            "reward_component_importance"
        ].astype(np.float64).reshape(-1)

        old_rewards = source[
            "rewards"
        ].astype(np.float64).reshape(-1)

        old_rewards_raw = source[
            "rewards_raw"
        ].astype(np.float64).reshape(-1)

        n_transitions, n_components = raw_components.shape

        # ----------------------------------------------------
        # Validate shapes and names
        # ----------------------------------------------------

        if len(component_names) != n_components:
            raise ValueError(
                f"Found {len(component_names)} component names, "
                f"but reward_components_raw has "
                f"{n_components} columns."
            )

        if shared_scales.shape[0] != n_components:
            raise ValueError(
                "reward_component_scales has incompatible shape: "
                f"{shared_scales.shape}"
            )

        if old_importance.shape[0] != n_components:
            raise ValueError(
                "reward_component_importance has incompatible shape: "
                f"{old_importance.shape}"
            )

        if old_rewards.shape[0] != n_transitions:
            raise ValueError(
                "rewards length does not match "
                "reward_components_raw."
            )

        missing_importance = [
            name for name in component_names
            if name not in NEW_IMPORTANCE
        ]

        extra_importance = [
            name for name in NEW_IMPORTANCE
            if name not in component_names
        ]

        if missing_importance:
            raise KeyError(
                "NEW_IMPORTANCE is missing these components: "
                f"{missing_importance}"
            )

        if extra_importance:
            raise KeyError(
                "NEW_IMPORTANCE contains unknown components: "
                f"{extra_importance}"
            )

        if np.any(~np.isfinite(raw_components)):
            raise ValueError(
                "reward_components_raw contains non-finite values."
            )

        if np.any(~np.isfinite(shared_scales)):
            raise ValueError(
                "reward_component_scales contains non-finite values."
            )

        if np.any(shared_scales <= 0.0):
            bad = [
                component_names[i]
                for i in np.where(shared_scales <= 0.0)[0]
            ]

            raise ValueError(
                "Reward component scales must be positive. "
                f"Invalid components: {bad}"
            )

        # ----------------------------------------------------
        # Construct new importance and clipping vectors
        # ----------------------------------------------------

        new_importance = np.array(
            [
                NEW_IMPORTANCE[name]
                for name in component_names
            ],
            dtype=np.float64,
        )

        component_clips = np.full(
            n_components,
            DEFAULT_COMPONENT_CLIP,
            dtype=np.float64,
        )

        thermal_idx = component_names.index(
            "thermal_comfort"
        )

        component_clips[
            thermal_idx
        ] = THERMAL_COMPONENT_CLIP

        # ----------------------------------------------------
        # Recompute normalized reward components
        # ----------------------------------------------------

        # Keep the original pooled shared scales. They describe
        # the physical dataset and should not be recomputed merely
        # because component priorities have changed.
        normalized_components = (
            raw_components
            / shared_scales[None, :]
            * new_importance[None, :]
        )

        reward_components = np.clip(
            normalized_components,
            -component_clips[None, :],
            component_clips[None, :],
        ).astype(np.float32)

        # Balanced normalized component sum before symlog.
        rewards_raw = reward_components.sum(
            axis=1
        ).astype(np.float32)

        if REWARD_TRANSFORM == "symlog":
            rewards = symlog(
                rewards_raw
            ).astype(np.float32)
        else:
            raise ValueError(
                f"Unsupported REWARD_TRANSFORM: "
                f"{REWARD_TRANSFORM}"
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        if reward_components.shape != raw_components.shape:
            raise ValueError(
                "New reward component shape is incorrect."
            )

        if rewards_raw.shape != old_rewards_raw.shape:
            raise ValueError(
                "New rewards_raw shape differs from the original."
            )

        if rewards.shape != old_rewards.shape:
            raise ValueError(
                "New rewards shape differs from the original."
            )

        if not np.all(np.isfinite(reward_components)):
            raise ValueError(
                "New reward_components contains non-finite values."
            )

        if not np.all(np.isfinite(rewards_raw)):
            raise ValueError(
                "New rewards_raw contains non-finite values."
            )

        if not np.all(np.isfinite(rewards)):
            raise ValueError(
                "New rewards contains non-finite values."
            )

        if not np.allclose(
            reward_components.sum(axis=1),
            rewards_raw,
            rtol=1e-5,
            atol=1e-5,
        ):
            raise ValueError(
                "rewards_raw does not equal the component sum."
            )

        expected_rewards = symlog(
            rewards_raw
        ).astype(np.float32)

        if not np.allclose(
            expected_rewards,
            rewards,
            rtol=1e-6,
            atol=1e-6,
        ):
            raise ValueError(
                "Symlog reward verification failed."
            )

        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------

        print("\nDataset:")
        print(f"  transitions: {n_transitions}")
        print(f"  components:  {n_components}")
        print(
            f"  terminals:   "
            f"{int(np.asarray(source['terminals']).sum())}"
        )

        print("\nImportance and clipping:")
        print(
            f"  {'component':25s}"
            f"{'old imp':>10s}"
            f"{'new imp':>10s}"
            f"{'clip':>10s}"
        )

        for i, name in enumerate(component_names):
            print(
                f"  {name:25s}"
                f"{old_importance[i]:10.2f}"
                f"{new_importance[i]:10.2f}"
                f"{component_clips[i]:10.1f}"
            )

        thermal_unbounded = normalized_components[
            :, thermal_idx
        ]

        thermal_nonzero = (
            np.abs(thermal_unbounded) > 1e-12
        )

        if thermal_nonzero.any():
            thermal_saturation = np.mean(
                np.abs(
                    thermal_unbounded[thermal_nonzero]
                )
                >= THERMAL_COMPONENT_CLIP
            )
        else:
            thermal_saturation = 0.0

        print("\nThermal diagnostics:")
        print(
            f"  thermal component mean before clipping: "
            f"{thermal_unbounded.mean():+.6f}"
        )
        print(
            f"  thermal component mean after clipping:  "
            f"{reward_components[:, thermal_idx].mean():+.6f}"
        )
        print(
            f"  nonzero thermal values clipped:         "
            f"{100.0 * thermal_saturation:.2f}%"
        )

        print("\nReward change:")
        print(
            f"  old balanced mean:    "
            f"{old_rewards_raw.mean():+.6f}"
        )
        print(
            f"  new balanced mean:    "
            f"{rewards_raw.mean():+.6f}"
        )
        print(
            f"  old transformed mean: "
            f"{old_rewards.mean():+.6f}"
        )
        print(
            f"  new transformed mean: "
            f"{rewards.mean():+.6f}"
        )
        print(
            f"  old transformed std:  "
            f"{old_rewards.std():.6f}"
        )
        print(
            f"  new transformed std:  "
            f"{rewards.std():.6f}"
        )
        print(
            f"  new transformed min:  "
            f"{rewards.min():+.6f}"
        )
        print(
            f"  new transformed max:  "
            f"{rewards.max():+.6f}"
        )

        # ----------------------------------------------------
        # Preserve original arrays and replace reward-derived
        # arrays only.
        # ----------------------------------------------------

        output = {
            key: source[key]
            for key in source.files
        }

        output.update({
            "reward_components": reward_components,
            "reward_components_raw": (
                raw_components.astype(np.float32)
            ),
            "reward_component_names": np.array(
                component_names
            ),
            "reward_component_scales": (
                shared_scales.astype(np.float32)
            ),
            "reward_component_importance": (
                new_importance.astype(np.float32)
            ),
            "reward_component_clips": (
                component_clips.astype(np.float32)
            ),
            # Preserve the default/global clip for compatibility
            # with code expecting a scalar value.
            "reward_component_clip": np.float32(
                DEFAULT_COMPONENT_CLIP
            ),
            "thermal_reward_component_clip": np.float32(
                THERMAL_COMPONENT_CLIP
            ),
            "rewards_raw": rewards_raw,
            "rewards": rewards,
            "reward_transform": np.array(
                REWARD_TRANSFORM
            ),
            "reward_reweighting_tag": np.array(
                "thermal_importance_3_clip_50"
            ),
        })

        print("\nSaving compressed dataset...")
        np.savez_compressed(
            OUTPUT_PATH,
            **output,
        )

    # ========================================================
    # Reopen saved dataset and verify it independently
    # ========================================================

    print("\nVerifying saved dataset...")

    with np.load(
        OUTPUT_PATH,
        allow_pickle=True,
    ) as check:
        if check["observations"].shape[0] != n_transitions:
            raise ValueError(
                "Saved transition count changed."
            )

        if check["reward_components"].shape != (
            n_transitions,
            n_components,
        ):
            raise ValueError(
                "Saved reward component shape is incorrect."
            )

        if check["rewards"].shape != (
            n_transitions,
        ):
            raise ValueError(
                "Saved reward shape is incorrect."
            )

        if not np.all(np.isfinite(check["rewards"])):
            raise ValueError(
                "Saved rewards contain non-finite values."
            )

        saved_terminal_sum = int(
            np.asarray(check["terminals"]).sum()
        )

        original_terminal_sum = int(
            np.asarray(output["terminals"]).sum()
        )

        if saved_terminal_sum != original_terminal_sum:
            raise ValueError(
                "Terminal flags changed while saving."
            )

        saved_names = [
            str(name)
            for name in check["reward_component_names"]
        ]

        saved_thermal_idx = saved_names.index(
            "thermal_comfort"
        )

        saved_importance = check[
            "reward_component_importance"
        ]

        saved_clips = check[
            "reward_component_clips"
        ]

        if not np.isclose(
            saved_importance[saved_thermal_idx],
            3.0,
        ):
            raise ValueError(
                "Saved thermal importance is incorrect."
            )

        if not np.isclose(
            saved_clips[saved_thermal_idx],
            50.0,
        ):
            raise ValueError(
                "Saved thermal clipping limit is incorrect."
            )

        print("Verification passed.")
        print(
            f"Saved transitions: {check['rewards'].shape[0]}"
        )
        print(
            f"Terminal flags:    {saved_terminal_sum}"
        )
        print(
            f"Thermal importance:"
            f" {saved_importance[saved_thermal_idx]:.1f}"
        )
        print(
            f"Thermal clip:      "
            f" {saved_clips[saved_thermal_idx]:.1f}"
        )

    print("\nDone.")
    print("New dataset:", OUTPUT_PATH.resolve())
    print(
        "\nTrain from scratch with:\n"
        f'DATASET_PATH = "{OUTPUT_PATH.name}"'
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Build the pooled SDAR dataset required by sdar_iql_dualcritic_training.py.

The builder:
  * concatenates complete expert episodes without interleaving them;
  * validates transition fields and reward metadata across every source file;
  * recomputes shared component scales over the complete pool;
  * separates the transformed pre-symlog reward into:
        task_rewards_raw + sdar_rewards_raw == rewards_raw
  * stores expert_ids for the held-out fingerprinting diagnostic;
  * preserves the existing scalar reward as:
        rewards == symlog(rewards_raw)

No new EnergyPlus rollout is required. This script only rebuilds the pooled NPZ.
"""

from pathlib import Path
import re

import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_PATTERN = "offline_smooth_*_episode_*.npz"

# Matches DATASET_PATH in the dual-critic training script.
OUTPUT_PATH = "pooled_sdar_experts_thermal3_tclip50_decomposed.npz"

# Exact name is preferred. Aliases are used only if the exact name is absent.
SDAR_COMPONENT_NAME = "switching_penalty"
SDAR_COMPONENT_ALIASES = (
    "switching",
    "sdar_switching",
    "action_switching",
    "action_update",
    "action_update_cost",
    "switching_cost",
)

REWARD_CLOSURE_ATOL = 1e-5
METADATA_ATOL = 1e-8

# Explicit target reward configuration for the new dataset. Values stored in
# individual episode files may differ because reward_components_raw is saved
# before importance weighting and clipping. These overrides therefore require
# no new simulation rollout.
TARGET_COMPONENT_IMPORTANCE = {
    "thermal_comfort": 3.0,
}
TARGET_COMPONENT_CLIPS = {
    "thermal_comfort": 50.0,
}

REQUIRED_TRANSITION_KEYS = (
    "observations",
    "actions",
    "actions_raw",
    "prev_actions",
    "selection_masks",
    "action_mixes",
    "rewards",
    "rewards_raw",
    "reward_components_raw",
    "next_observations",
    "terminals",
)

EXPERT_METADATA_KEYS = (
    "expert_label",
    "expert_name",
    "expert_id",
    "policy_id",
    "source_expert",
)


# ============================================================
# HELPERS
# ============================================================

def episode_number(path: Path) -> int:
    match = re.search(r"episode_(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Cannot extract episode number from {path.name}")
    return int(match.group(1))


def filename_expert_label(path: Path) -> str:
    """Return the text between `offline_smooth_` and `_episode_<n>`."""
    match = re.match(
        r"offline_smooth_(.+)_episode_\d+$",
        path.stem,
    )
    if match is None:
        raise ValueError(
            f"Cannot extract expert label from filename {path.name!r}. "
            "Expected offline_smooth_<expert>_episode_<number>.npz"
        )
    return match.group(1)


def _python_scalar(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.generic):
        return value.item()
    return value


def expert_label(path: Path, episode: dict[str, np.ndarray]) -> str:
    """Prefer a constant expert label stored in the episode; otherwise use filename."""
    for key in EXPERT_METADATA_KEYS:
        if key not in episode:
            continue

        values = np.asarray(episode[key]).reshape(-1)
        if len(values) == 0:
            continue

        unique = np.unique(values)
        if len(unique) == 1:
            return str(_python_scalar(unique[0]))

    return filename_expert_label(path)


def normalize_component_names(values: np.ndarray) -> np.ndarray:
    names = []
    for value in np.asarray(values).reshape(-1):
        value = _python_scalar(value)
        names.append(str(value))
    return np.asarray(names, dtype=str)


def read_component_clips(
    episode: dict[str, np.ndarray],
    n_components: int,
    source_name: str,
) -> tuple[np.ndarray, float | None]:
    """Return per-component clips and an optional legacy scalar clip."""
    if "reward_component_clips" in episode:
        clips = np.asarray(
            episode["reward_component_clips"],
            dtype=np.float64,
        ).reshape(-1)
        if len(clips) != n_components:
            raise ValueError(
                f"{source_name}: reward_component_clips has {len(clips)} "
                f"entries, expected {n_components}"
            )
        legacy_scalar = (
            float(clips[0])
            if np.allclose(clips, clips[0], atol=METADATA_ATOL, rtol=0.0)
            else None
        )
        return clips, legacy_scalar

    if "reward_component_clip" in episode:
        clip_value = np.asarray(
            episode["reward_component_clip"],
            dtype=np.float64,
        ).reshape(-1)

        if len(clip_value) == 1:
            scalar = float(clip_value[0])
            return np.full(n_components, scalar, dtype=np.float64), scalar

        if len(clip_value) == n_components:
            clips = clip_value
            legacy_scalar = (
                float(clips[0])
                if np.allclose(clips, clips[0], atol=METADATA_ATOL, rtol=0.0)
                else None
            )
            return clips, legacy_scalar

        raise ValueError(
            f"{source_name}: reward_component_clip has {len(clip_value)} "
            f"entries, expected 1 or {n_components}"
        )

    raise KeyError(
        f"{source_name} is missing reward_component_clips "
        "and reward_component_clip"
    )


def read_reward_metadata(
    episode: dict[str, np.ndarray],
    source_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float | None]:
    required = (
        "reward_component_names",
        "reward_component_importance",
    )
    for key in required:
        if key not in episode:
            raise KeyError(f"{source_name} is missing {key!r}")

    names = normalize_component_names(
        episode["reward_component_names"]
    )
    importance = np.asarray(
        episode["reward_component_importance"],
        dtype=np.float64,
    ).reshape(-1)

    if len(importance) != len(names):
        raise ValueError(
            f"{source_name}: importance has {len(importance)} entries, "
            f"but component names has {len(names)}"
        )

    clips, legacy_scalar = read_component_clips(
        episode,
        len(names),
        source_name,
    )

    return names, importance, clips, legacy_scalar


def apply_reward_targets(
    component_names: np.ndarray,
    importance: np.ndarray,
    clips: np.ndarray,
    *,
    verbose: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply explicit pooling-time reward targets to raw component metadata."""
    normalized_names = [
        name.strip().lower()
        for name in component_names.tolist()
    ]
    target_importance = {
        name.strip().lower(): float(value)
        for name, value in TARGET_COMPONENT_IMPORTANCE.items()
    }
    target_clips = {
        name.strip().lower(): float(value)
        for name, value in TARGET_COMPONENT_CLIPS.items()
    }

    unknown_targets = (
        set(target_importance)
        | set(target_clips)
    ) - set(normalized_names)
    if unknown_targets:
        raise ValueError(
            "Reward target configuration references missing components: "
            f"{sorted(unknown_targets)}. "
            f"Available components: {normalized_names}"
        )

    importance_out = np.asarray(
        importance,
        dtype=np.float64,
    ).copy()
    clips_out = np.asarray(
        clips,
        dtype=np.float64,
    ).copy()

    for name, target in target_importance.items():
        index = normalized_names.index(name)
        source = float(importance_out[index])
        importance_out[index] = target
        if verbose:
            print(
                f"  reward override: importance[{name}] "
                f"{source:g} -> {target:g}"
            )

    for name, target in target_clips.items():
        index = normalized_names.index(name)
        source = float(clips_out[index])
        clips_out[index] = target
        if verbose:
            print(
                f"  reward override: clip[{name}] "
                f"{source:g} -> {target:g}"
            )

    return importance_out, clips_out


def find_sdar_component(component_names: np.ndarray) -> int:
    normalized = [name.strip().lower() for name in component_names.tolist()]
    candidates = [
        SDAR_COMPONENT_NAME.strip().lower(),
        *[name.strip().lower() for name in SDAR_COMPONENT_ALIASES],
    ]

    matches = [
        i
        for i, name in enumerate(normalized)
        if name in candidates
    ]

    if len(matches) != 1:
        raise ValueError(
            "Expected exactly one SDAR/switching reward component. "
            f"Found indices {matches}; component names are {normalized}. "
            "Set SDAR_COMPONENT_NAME to the exact component used by your reward."
        )

    return matches[0]


def symlog(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    return np.sign(x) * np.log1p(np.abs(x))


def arrays_match(
    left: np.ndarray,
    right: np.ndarray,
    atol: float = METADATA_ATOL,
) -> bool:
    return np.array_equal(left, right) or np.allclose(
        left,
        right,
        atol=atol,
        rtol=0.0,
    )


# ============================================================
# BUILDER
# ============================================================

def main():
    paths = sorted(
        Path(".").glob(INPUT_PATTERN),
        key=lambda path: (
            filename_expert_label(path),
            episode_number(path),
            path.name,
        ),
    )

    if not paths:
        raise FileNotFoundError(
            f"No files matched {INPUT_PATTERN!r}"
        )

    print("Episodes:")
    for path in paths:
        print(" ", path.name)

    all_arrays: dict[str, list[np.ndarray]] = {}
    transition_keys_reference: set[str] | None = None

    episode_lengths: list[int] = []
    pooled_episode_ids: list[np.ndarray] = []
    source_episode_numbers: list[int] = []
    episode_seeds: list[int] = []
    source_files: list[str] = []

    expert_ids: list[np.ndarray] = []
    episode_expert_ids: list[int] = []
    expert_label_to_id: dict[str, int] = {}

    component_names_reference: np.ndarray | None = None
    importance_reference: np.ndarray | None = None
    clips_reference: np.ndarray | None = None
    source_importance_reference: np.ndarray | None = None
    source_clips_reference: np.ndarray | None = None
    legacy_component_clip: float | None = None

    for pooled_episode_id, path in enumerate(paths):
        with np.load(path, allow_pickle=True) as data:
            episode = {
                key: np.array(data[key], copy=True)
                for key in data.files
            }

        for key in REQUIRED_TRANSITION_KEYS:
            if key not in episode:
                raise KeyError(
                    f"{path.name} is missing {key!r}"
                )

        observations = np.asarray(episode["observations"])
        if observations.ndim != 2:
            raise ValueError(
                f"{path.name}: observations must be 2-D, "
                f"got {observations.shape}"
            )
        n = len(observations)

        for key in REQUIRED_TRANSITION_KEYS:
            value = np.asarray(episode[key])
            if value.ndim < 1 or value.shape[0] != n:
                raise ValueError(
                    f"{path.name}: {key} has shape {value.shape}; "
                    f"first dimension must equal {n}"
                )

        terminals = np.asarray(
            episode["terminals"]
        ).reshape(-1)
        if int(terminals.sum()) != 1 or terminals[-1] != 1:
            raise ValueError(
                f"{path.name}: expected exactly one terminal "
                "at the final transition"
            )

        actions = np.asarray(episode["actions"])
        if actions.ndim != 2 or actions.shape[1] != 19:
            raise ValueError(
                f"{path.name}: actions must have shape (T, 19), "
                f"got {actions.shape}"
            )
        if np.max(np.abs(actions)) > 1.0001:
            raise ValueError(
                f"{path.name}: normalized actions exceed [-1, 1]"
            )

        components_raw = np.asarray(
            episode["reward_components_raw"]
        )
        if components_raw.ndim != 2:
            raise ValueError(
                f"{path.name}: reward_components_raw must be 2-D, "
                f"got {components_raw.shape}"
            )

        (
            component_names,
            importance,
            component_clips,
            _source_legacy_clip,
        ) = read_reward_metadata(episode, path.name)

        source_importance = importance.copy()
        source_component_clips = component_clips.copy()
        importance, component_clips = apply_reward_targets(
            component_names,
            importance,
            component_clips,
            verbose=component_names_reference is None,
        )

        if components_raw.shape[1] != len(component_names):
            raise ValueError(
                f"{path.name}: reward_components_raw has "
                f"{components_raw.shape[1]} columns, but metadata contains "
                f"{len(component_names)} component names"
            )

        if component_names_reference is None:
            component_names_reference = component_names
            importance_reference = importance
            clips_reference = component_clips
            source_importance_reference = source_importance
            source_clips_reference = source_component_clips
            legacy_component_clip = (
                float(component_clips[0])
                if np.allclose(
                    component_clips,
                    component_clips[0],
                    atol=METADATA_ATOL,
                    rtol=0.0,
                )
                else None
            )
        else:
            if not np.array_equal(
                component_names,
                component_names_reference,
            ):
                raise ValueError(
                    f"{path.name}: reward component names/order differs. "
                    f"Expected {component_names_reference.tolist()}, "
                    f"got {component_names.tolist()}"
                )
            if not arrays_match(
                importance,
                importance_reference,
            ):
                raise ValueError(
                    f"{path.name}: reward component importance differs"
                )
            if not arrays_match(
                component_clips,
                clips_reference,
            ):
                raise ValueError(
                    f"{path.name}: reward component clips differ"
                )

        transition_keys = {
            key
            for key, value in episode.items()
            if np.asarray(value).ndim >= 1
            and np.asarray(value).shape[0] == n
        }

        if transition_keys_reference is None:
            transition_keys_reference = transition_keys
        elif transition_keys != transition_keys_reference:
            missing = sorted(
                transition_keys_reference - transition_keys
            )
            extra = sorted(
                transition_keys - transition_keys_reference
            )
            raise ValueError(
                f"{path.name}: transition-aligned fields differ from the "
                f"first episode. Missing={missing}, extra={extra}"
            )

        for key in sorted(transition_keys):
            all_arrays.setdefault(key, []).append(
                np.asarray(episode[key])
            )

        label = expert_label(path, episode)
        if label not in expert_label_to_id:
            expert_label_to_id[label] = len(expert_label_to_id)
        expert_id = expert_label_to_id[label]

        pooled_episode_ids.append(
            np.full(n, pooled_episode_id, dtype=np.int32)
        )
        expert_ids.append(
            np.full(n, expert_id, dtype=np.int32)
        )
        episode_expert_ids.append(expert_id)

        source_episode_numbers.append(
            episode_number(path)
        )
        episode_lengths.append(n)
        source_files.append(path.name)

        seed_value = np.asarray(
            episode.get("exploration_seed", -1)
        ).reshape(-1)
        if len(seed_value) != 1:
            raise ValueError(
                f"{path.name}: exploration_seed must be scalar"
            )
        episode_seeds.append(int(seed_value[0]))

    if (
        component_names_reference is None
        or importance_reference is None
        or clips_reference is None
    ):
        raise RuntimeError("Reward metadata was not initialized")

    if (
        source_importance_reference is None
        or source_clips_reference is None
    ):
        raise RuntimeError(
            "Source reward metadata was not initialized"
        )

    pooled = {
        key: np.concatenate(values, axis=0)
        for key, values in all_arrays.items()
    }

    pooled["episode_ids"] = np.concatenate(
        pooled_episode_ids
    )
    pooled["expert_ids"] = np.concatenate(
        expert_ids
    )
    pooled["episode_lengths"] = np.asarray(
        episode_lengths,
        dtype=np.int32,
    )
    pooled["episode_expert_ids"] = np.asarray(
        episode_expert_ids,
        dtype=np.int32,
    )
    pooled["source_episode_numbers"] = np.asarray(
        source_episode_numbers,
        dtype=np.int32,
    )
    pooled["episode_seeds"] = np.asarray(
        episode_seeds,
        dtype=np.int64,
    )
    pooled["source_files"] = np.asarray(
        source_files,
        dtype=str,
    )

    expert_labels = np.empty(
        len(expert_label_to_id),
        dtype=object,
    )
    for label, expert_id in expert_label_to_id.items():
        expert_labels[expert_id] = label
    pooled["expert_labels"] = expert_labels.astype(str)

    # ========================================================
    # Recompute shared reward-component scales
    # ========================================================

    components_raw = np.asarray(
        pooled["reward_components_raw"],
        dtype=np.float64,
    )

    shared_scales = np.mean(
        np.abs(components_raw),
        axis=0,
    )
    shared_scales[shared_scales < 1e-8] = 1.0

    components = (
        components_raw
        / shared_scales[None, :]
    ) * importance_reference[None, :]

    lower_bounds = np.where(
        clips_reference >= 0.0,
        -clips_reference,
        -np.inf,
    )
    upper_bounds = np.where(
        clips_reference >= 0.0,
        clips_reference,
        np.inf,
    )
    components = np.clip(
        components,
        lower_bounds[None, :],
        upper_bounds[None, :],
    )

    sdar_component_index = find_sdar_component(
        component_names_reference
    )

    sdar_rewards_raw = components[
        :, sdar_component_index
    ].copy()
    rewards_raw = components.sum(axis=1)
    task_rewards_raw = (
        rewards_raw - sdar_rewards_raw
    )
    rewards = symlog(rewards_raw)

    # ========================================================
    # Reward integrity checks
    # ========================================================

    if not np.isfinite(task_rewards_raw).all():
        raise ValueError(
            "task_rewards_raw contains NaN or infinity"
        )
    if not np.isfinite(sdar_rewards_raw).all():
        raise ValueError(
            "sdar_rewards_raw contains NaN or infinity"
        )

    positive_sdar = sdar_rewards_raw > 1e-8
    if positive_sdar.any():
        raise ValueError(
            "The selected SDAR component is not a non-positive reward. "
            f"Found {int(positive_sdar.sum())} positive transitions; "
            f"maximum={float(sdar_rewards_raw.max()):.6g}. "
            "Check the switching component sign and SDAR_COMPONENT_NAME."
        )

    reconstructed_raw = (
        task_rewards_raw + sdar_rewards_raw
    )
    raw_closure_error = float(
        np.max(np.abs(reconstructed_raw - rewards_raw))
    )
    if raw_closure_error > REWARD_CLOSURE_ATOL:
        raise AssertionError(
            "Raw reward decomposition failed: "
            f"max error={raw_closure_error:.3e}"
        )

    symlog_closure_error = float(
        np.max(
            np.abs(
                symlog(reconstructed_raw)
                - rewards
            )
        )
    )
    if symlog_closure_error > REWARD_CLOSURE_ATOL:
        raise AssertionError(
            "Symlog reward closure failed: "
            f"max error={symlog_closure_error:.3e}"
        )

    pooled["reward_components"] = components.astype(
        np.float32
    )
    pooled["reward_component_scales"] = shared_scales.astype(
        np.float32
    )
    pooled["reward_component_importance"] = (
        importance_reference.astype(np.float32)
    )
    pooled["source_reward_component_importance"] = (
        source_importance_reference.astype(np.float32)
    )
    pooled["reward_component_names"] = (
        component_names_reference.astype(str)
    )
    pooled["reward_component_clips"] = (
        clips_reference.astype(np.float32)
    )
    pooled["source_reward_component_clips"] = (
        source_clips_reference.astype(np.float32)
    )
    if legacy_component_clip is not None:
        pooled["reward_component_clip"] = np.asarray(
            legacy_component_clip,
            dtype=np.float32,
        )

    # These are the exact fields required by the dual-critic trainer.
    # "raw" here means after shared scaling/importance/clipping but before
    # the final total symlog.
    pooled["task_rewards_raw"] = task_rewards_raw.astype(
        np.float32
    )
    pooled["sdar_rewards_raw"] = sdar_rewards_raw.astype(
        np.float32
    )
    pooled["rewards_raw"] = rewards_raw.astype(
        np.float32
    )
    pooled["rewards"] = rewards.astype(
        np.float32
    )

    pooled["reward_parts_are_raw"] = np.asarray(True)
    pooled["reward_split_version"] = np.asarray(
        "task_plus_sdar_pre_symlog_v1"
    )
    pooled["reward_reweighting_tag"] = np.asarray(
        "shared_component_scales_dualcritic_v1"
    )
    pooled["sdar_component_name"] = np.asarray(
        component_names_reference[
            sdar_component_index
        ]
    )
    pooled["sdar_component_index"] = np.asarray(
        sdar_component_index,
        dtype=np.int32,
    )

    # ========================================================
    # Recompute pooled statistics
    # ========================================================

    pooled["obs_mean"] = pooled[
        "observations"
    ].mean(axis=0).astype(np.float32)
    pooled["obs_std"] = (
        pooled["observations"].std(axis=0)
        + 1e-6
    ).astype(np.float32)

    pooled["act_mean"] = pooled[
        "actions"
    ].mean(axis=0).astype(np.float32)
    pooled["act_std"] = (
        pooled["actions"].std(axis=0)
        + 1e-6
    ).astype(np.float32)

    # ========================================================
    # Final structural checks
    # ========================================================

    expected_transitions = int(
        sum(episode_lengths)
    )
    if len(pooled["observations"]) != expected_transitions:
        raise AssertionError(
            "Unexpected pooled transition count"
        )

    terminal_positions = np.flatnonzero(
        np.asarray(
            pooled["terminals"]
        ).reshape(-1)
        > 0.5
    )
    expected_terminal_positions = (
        np.cumsum(episode_lengths) - 1
    )
    if not np.array_equal(
        terminal_positions,
        expected_terminal_positions,
    ):
        raise AssertionError(
            "Terminal positions do not match concatenated episode boundaries"
        )

    if int(
        np.asarray(pooled["terminals"]).sum()
    ) != len(paths):
        raise AssertionError(
            "Expected one terminal flag per source episode"
        )

    # Check closure again after the float32 storage cast, exactly as training
    # will read it.
    task32 = pooled["task_rewards_raw"]
    sdar32 = pooled["sdar_rewards_raw"]
    rewards32 = pooled["rewards"]
    closure32 = symlog(
        task32.astype(np.float64)
        + sdar32.astype(np.float64)
    ).astype(np.float32)
    float32_error = float(
        np.max(np.abs(closure32 - rewards32))
    )
    if float32_error > REWARD_CLOSURE_ATOL:
        raise AssertionError(
            "Float32 stored reward closure failed: "
            f"max error={float32_error:.3e}"
        )

    np.savez_compressed(
        OUTPUT_PATH,
        **pooled,
    )

    # ========================================================
    # Summary
    # ========================================================

    print("\nSaved:", OUTPUT_PATH)
    print("Episodes:", len(paths))
    print("Experts:", len(expert_label_to_id))
    print(
        "Expert mapping:",
        {
            expert_id: label
            for label, expert_id
            in expert_label_to_id.items()
        },
    )
    print("Transitions:", expected_transitions)
    print(
        "Terminal flags:",
        int(np.asarray(pooled["terminals"]).sum()),
    )
    print(
        "Observations:",
        pooled["observations"].shape,
    )
    print(
        "Actions:",
        pooled["actions"].shape,
    )
    print(
        "SDAR component:",
        component_names_reference[
            sdar_component_index
        ],
        f"(index {sdar_component_index})",
    )
    print(
        "task_rewards_raw:",
        f"mean={task_rewards_raw.mean():.6f}, "
        f"std={task_rewards_raw.std():.6f}",
    )
    print(
        "sdar_rewards_raw:",
        f"mean={sdar_rewards_raw.mean():.6f}, "
        f"std={sdar_rewards_raw.std():.6f}, "
        f"max={sdar_rewards_raw.max():.6f}",
    )
    print(
        "Raw closure max error:",
        f"{raw_closure_error:.3e}",
    )
    print(
        "Stored float32 closure max error:",
        f"{float32_error:.3e}",
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Agent-rollout dataset builder — evaluation counterpart of the v8 expert builder
================================================================================

Runs AFTER the evaluation callback (`evaluate_agent_callback.py`) has rolled the
TRAINED SDAR-IQL agent through EnergyPlus and filled the in-memory logging dicts.
It reconstructs the same offline-RL dataset format from the agent's trajectory so
you can score the policy (return, per-component reward, thermal violations,
switching, energy) and, if you want, re-pool the agent rollout with the training
data.

WHAT'S DIFFERENT FROM THE v8 EXPERT BUILDER
-------------------------------------------
The reward math, observation layout, and diagnostics are IDENTICAL (reward is a
property of states/actions, not of which policy produced them). Only the action
source changes:

  expert builder                     agent builder (this file)
  ------------------------------     --------------------------------------
  action_vector_data["action"]   ->  agent_data["executed_action_norm"]
  action_vector_data["...mask"]  ->  agent_data["selection_mask"]
  prev_action / action_mix logged -> RECONSTRUCTED from the agent trajectory
  exploration_data (expert vs        (removed — no exploration during eval;
    executed, thermal modes, ...)     selector sampling is logged directly)

Required in-memory globals (filled by the eval callback rollout):
    ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    agent_data  (observation, selection_logits, selection_probability,
                 selection_mask, executed_action_norm)

REWARD COMPARABILITY (important)
--------------------------------
This version REQUIRES the pooled decomposed training NPZ as a reward reference.
It loads the exact:

    reward_component_names
    reward_component_scales
    reward_component_importance
    reward_component_clips
    reward_transform

used by training. For the current decomposed run that means thermal importance
3.0, thermal clip 50.0, and clip 10.0 for all other components. The evaluated
rollout never defines its own normalization scales, so rewards remain comparable
across checkpoints and against the experts.

`rewards_raw` is kept as the training-compatible name for the balanced,
post-normalization/post-importance/post-component-clip sum before symlog.
`reward_physical_raw` is additionally saved for the sum of physical W_* reward
components before normalization.
"""

import csv
import hashlib
import json
import re
from pathlib import Path

import numpy as np


# ============================================================
# ROLLOUT LOGS — provided by the eval callback
# ============================================================
# If this modupace),
# bind the logging globals from the eval callback module. They are the same
# mutable objects the callback appended to, so they carry the rollout dle is imported standalone (not exec'd in the rollout namesata.
try:
    agent_data  # noqa: F821  (present when run in the rollout namespace)
except NameError:
    from evaluate_agent_callback import (  # noqa: F401
        ZONES_MID,
        AVAILABLE_GLAZING_STATES,
        temperature_data,
        lighting_data,
        solar_data,
        ext_irr_data,
        wpi_data,
        meter_data,
        output_variable_data,
        temporal_data,
        glazing_state_data,
        lighting_power_data,
        heating_setpoint_data,
        cooling_setpoint_data,
        agent_data,
        SELECTOR_MODE,
        PROPOSAL_DETERMINISTIC,
        SELECTOR_SEED,
    )


# ============================================================
# CONFIG
# ============================================================

# These values are validated against the reference NPZ. They are not used as an
# independent reward definition.
REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0  # legacy/default clip; per-component clips come from NPZ

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR

POLICY_TAG = "sdar_iql_augmented_value_standard_iql_"

# ---------------------------------------------------------------------------
# EDIT THESE FOR EACH CHECKPOINT EVALUATION.
# Use the decomposed pooled dataset that trained the agent. If it is elsewhere,
# provide the absolute path.
# ---------------------------------------------------------------------------
REFERENCE_REWARD_DATASET = (
    "pooled_sdar_experts_thermal3_tclip50.npz"
)
EVAL_CHECKPOINT_PATH = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_checkpoint_epoch100.pt"   # e.g. "/.../checkpoint_epoch150.pt"
EVAL_CHECKPOINT_EPOCH = None  # optional; inferred from the checkpoint filename
EVAL_SEED = 123
EVAL_SELECTOR_MODE = str(globals().get("SELECTOR_MODE", "sample")).lower()
EVAL_PROPOSAL_DETERMINISTIC = bool(
    globals().get("PROPOSAL_DETERMINISTIC", True)
)
EVAL_SELECTOR_SEED = int(globals().get("SELECTOR_SEED", 20260728))
EVAL_SAFEGUARD_ENABLED = True
EVAL_WEATHER_FILE = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}

HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
ACTION_VALIDATION_ATOL = 0.5


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    """Physical (actuated) action, read from the eval callback's logs."""
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM THE AGENT'S NORMALIZED ACTION
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps to a dimming duty
#   u = (a + 1) / 2 in [0, 1], and E = sum_z nominal_W[z]*u[z] * dt / 1000 (kWh).
# Meter-free: uses the executed action, not the logged Lights Electricity Rate.

def compute_lighting_energy_kwh(action_norm_t):
    a = np.asarray(action_norm_t, dtype=np.float64)
    dimming = np.clip((a[LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
    nominal_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float64)
    total_power_w = float((nominal_w * dimming).sum())
    return total_power_w * TIMESTEP_HOURS / 1000.0


# ============================================================
# REWARD — components + scalar (identical logic to the v8 builder)
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, action_norm_t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_energy_kwh = compute_lighting_energy_kwh(action_norm_t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = np.asarray(selection_mask_t)
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _npz_scalar(data, key, default=None):
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar NPZ field {key!r}, got shape {value.shape}.")
    return value.reshape(()).item()


def load_reward_reference(reference_dataset_path, comp_names):
    """
    Load the immutable reward ruler from the pooled decomposed training NPZ.

    There is deliberately no rollout-local fallback. A missing or incompatible
    reference dataset is an error because otherwise checkpoint rewards would be
    calculated on different scales.
    """
    path = Path(reference_dataset_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "Reward reference dataset not found: "
            f"{path}. Set REFERENCE_REWARD_DATASET to the exact pooled NPZ used "
            "to train the decomposed agent."
        )

    required = {
        "reward_component_names",
        "reward_component_scales",
        "reward_component_importance",
    }
    with np.load(path, allow_pickle=False) as data:
        missing = sorted(required.difference(data.files))
        if missing:
            raise KeyError(
                f"Reference NPZ {path} is missing required reward metadata: {missing}"
            )

        ref_names = [str(x) for x in np.asarray(data["reward_component_names"]).tolist()]
        if ref_names != list(comp_names):
            raise ValueError(
                "Reward component order differs from the training dataset.\n"
                f"  builder:   {list(comp_names)}\n"
                f"  reference: {ref_names}"
            )

        scales = np.asarray(data["reward_component_scales"], dtype=np.float64)
        importance = np.asarray(
            data["reward_component_importance"], dtype=np.float64
        )
        if scales.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_scales shape {scales.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if importance.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_importance shape {importance.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if np.any(~np.isfinite(scales)) or np.any(scales <= 0.0):
            raise ValueError("Reference reward scales must all be finite and positive.")
        if np.any(~np.isfinite(importance)):
            raise ValueError("Reference reward importance values must all be finite.")

        if "reward_component_clips" in data.files:
            clips = np.asarray(data["reward_component_clips"], dtype=np.float64)
        else:
            legacy_clip = float(
                _npz_scalar(data, "reward_component_clip", REWARD_COMPONENT_CLIP)
            )
            clips = np.full(len(comp_names), legacy_clip, dtype=np.float64)
            if "thermal_reward_component_clip" in data.files:
                thermal_i = comp_names.index("thermal_comfort")
                clips[thermal_i] = float(
                    _npz_scalar(data, "thermal_reward_component_clip")
                )

        if clips.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_clips shape {clips.shape}; "
                f"expected {(len(comp_names),)}"
            )
        # A negative or non-finite stored clip means unbounded for that component.
        clips = np.where(np.isfinite(clips) & (clips >= 0.0), clips, np.inf)

        transform = str(
            _npz_scalar(data, "reward_transform", REWARD_TRANSFORM)
        )
        reweighting_tag = str(
            _npz_scalar(data, "reward_reweighting_tag", "")
        )
        reference_mask_value = float(
            _npz_scalar(data, "action_mask_value", ACTION_MASK_VALUE)
        )

    configured_importance = np.array(
        [REWARD_IMPORTANCE.get(name, 1.0) for name in comp_names],
        dtype=np.float64,
    )
    if not np.allclose(configured_importance, importance, atol=1e-7, rtol=0.0):
        raise ValueError(
            "Builder REWARD_IMPORTANCE differs from the training NPZ. "
            "Do not evaluate until the reward definitions match.\n"
            f"  builder:   {configured_importance.tolist()}\n"
            f"  reference: {importance.tolist()}"
        )
    if transform != REWARD_TRANSFORM:
        raise ValueError(
            f"Builder transform {REWARD_TRANSFORM!r} differs from reference "
            f"transform {transform!r}."
        )
    if not np.isclose(reference_mask_value, ACTION_MASK_VALUE):
        raise ValueError(
            f"Reference action_mask_value={reference_mask_value} differs from "
            f"builder ACTION_MASK_VALUE={ACTION_MASK_VALUE}."
        )

    return {
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
        "names": list(comp_names),
        "scales": scales,
        "importance": importance,
        "clips": clips,
        "transform": transform,
        "reweighting_tag": reweighting_tag,
    }


def apply_reward_reference(components_raw, reward_reference):
    """Apply training scales, importance, and per-component clips exactly."""
    scales = reward_reference["scales"]
    importance = reward_reference["importance"]
    clips = reward_reference["clips"]

    unbounded = (
        components_raw.astype(np.float64)
        / scales[None, :]
        * importance[None, :]
    )
    components_final = np.clip(
        unbounded,
        -clips[None, :],
        clips[None, :],
    )
    return components_final.astype(np.float32), unbounded.astype(np.float32)


def _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch):
    if checkpoint_epoch is not None:
        epoch = int(checkpoint_epoch)
        if epoch < 0:
            raise ValueError("checkpoint_epoch must be non-negative.")
        return epoch
    if checkpoint_path:
        match = re.search(r"epoch[_-]?(\d+)", Path(checkpoint_path).name, re.IGNORECASE)
        if match:
            return int(match.group(1))
    raise ValueError(
        "Checkpoint epoch is required for unambiguous output naming. Set "
        "EVAL_CHECKPOINT_EPOCH or use a checkpoint filename containing "
        "'epoch<number>'."
    )


def _checkpoint_provenance(checkpoint_path, checkpoint_epoch):
    epoch = _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch)
    if checkpoint_path is None:
        return {
            "epoch": epoch,
            "path": "",
            "sha256": "",
        }

    path = Path(checkpoint_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return {
        "epoch": epoch,
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
    }


# ============================================================
# DIAGNOSTICS (identical to the v8 builder)
# ============================================================

def _print_reward_diagnostics(
    reward_physical_raw,
    reward_balanced,
    rewards_tx,
    components_arr,
):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(
        f"  physical raw: mean={reward_physical_raw.mean():+.4f}  "
        f"std={reward_physical_raw.std():.4f}"
    )
    print(
        f"                min={reward_physical_raw.min():+.4f}    "
        f"max={reward_physical_raw.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_physical_raw, qs)]}")
    print(
        f"  balanced:     mean={reward_balanced.mean():+.4f}  "
        f"std={reward_balanced.std():.4f}"
    )
    print(
        f"                min={reward_balanced.min():+.4f}    "
        f"max={reward_balanced.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_balanced, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - reward_balanced
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - balanced): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - balanced): {np.max(np.abs(err)):.8f}")


def _print_normalization(
    comp_names,
    comp_scales,
    importance_vec,
    component_clips,
    scales_source,
):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'clip':>8s}  {'eff_weight':>10s}")
    for nm, sc, imp, clip in zip(
        comp_names, comp_scales, importance_vec, component_clips
    ):
        eff = imp / sc if sc != 0 else 0.0
        clip_name = "none" if not np.isfinite(clip) else f"{clip:.1f}"
        print(
            f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  "
            f"{clip_name:>8s}  {eff:10.4f}"
        )


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics (agent selector) ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_selector_probability_diagnostics(
    selection_probabilities,
    selection_masks,
    selector_mode,
):
    # The callback forces all dimensions to update on the first step, so
    # exclude that step when checking the learned selector realization.
    probabilities = selection_probabilities[1:]
    masks = selection_masks[1:]

    groups = {
        "glazing": GLAZE_SLICE,
        "lighting": LIGHT_SLICE,
        "heating": HEAT_SLICE,
        "cooling": COOL_SLICE,
        "overall": slice(0, ACTION_DIM),
    }

    print(
        "\n=== Selector probability diagnostics "
        f"(mode={selector_mode}) ==="
    )
    stats = {}
    for name, slc in groups.items():
        p = probabilities[:, slc]
        m = masks[:, slc]
        mean_probability = float(p.mean())
        sampled_rate = float(m.mean())
        difference = sampled_rate - mean_probability
        stats[name] = {
            "mean_probability": mean_probability,
            "sampled_rate": sampled_rate,
            "difference": difference,
        }
        print(
            f"  {name:10s} "
            f"mean probability={mean_probability:.4f}  "
            f"sampled rate={sampled_rate:.4f}  "
            f"difference={difference:+.4f}"
        )

    threshold_masks = (probabilities >= 0.5).astype(np.float32)
    stats["threshold_agreement"] = float(
        np.mean(masks == threshold_masks)
    )
    print(
        "  agreement with p>=0.5 threshold: "
        f"{100.0 * stats['threshold_agreement']:.2f}%"
    )
    return stats


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None or TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


def _denormalize_linear(values, physical_range):
    low, high = physical_range
    values = np.asarray(values, dtype=np.float64)
    return low + 0.5 * (values + 1.0) * (high - low)


def _validate_actuated_actions(actions, actions_raw):
    """Verify all 19 normalized executed actions against physical actuation logs."""
    n_glazing = len(AVAILABLE_GLAZING_STATES)
    glz_ids_from_norm = np.clip(
        np.round((actions[:, GLAZE_SLICE] + 1.0) * 0.5 * (n_glazing - 1)),
        0,
        n_glazing - 1,
    ).astype(np.int64)
    glz_ids_logged = actions_raw[:, GLAZE_SLICE].astype(np.int64)
    if not np.array_equal(glz_ids_from_norm, glz_ids_logged):
        mismatch = np.argwhere(glz_ids_from_norm != glz_ids_logged)[0]
        t, dim = (int(mismatch[0]), int(mismatch[1]))
        raise ValueError(
            "Glazing decoded from executed_action_norm does not match the "
            f"actuation log at transition {t}, glazing dim {dim}: "
            f"decoded={glz_ids_from_norm[t, dim]}, logged={glz_ids_logged[t, dim]}."
        )

    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )
    light_decoded = (
        np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
        * nominal_power_w[None, :]
    )
    heat_decoded = _denormalize_linear(
        actions[:, HEAT_SLICE], HEATING_SETPOINT_RANGE
    )
    cool_decoded = _denormalize_linear(
        actions[:, COOL_SLICE], COOLING_SETPOINT_RANGE
    )

    checks = [
        (
            "lighting",
            light_decoded,
            actions_raw[:, LIGHT_SLICE],
            "W",
        ),
        (
            "heating setpoint",
            heat_decoded,
            actions_raw[:, HEAT_SLICE],
            "degC",
        ),
        (
            "cooling setpoint",
            cool_decoded,
            actions_raw[:, COOL_SLICE],
            "degC",
        ),
    ]
    for label, decoded, logged, unit in checks:
        if not np.allclose(
            decoded,
            logged,
            atol=ACTION_VALIDATION_ATOL,
            rtol=1e-4,
        ):
            abs_error = np.abs(decoded - logged)
            t, dim = np.unravel_index(np.argmax(abs_error), abs_error.shape)
            raise ValueError(
                f"{label} decoded from executed_action_norm does not match the "
                f"actuation log. Largest error at transition {t}, local dim {dim}: "
                f"decoded={decoded[t, dim]:.6g} {unit}, "
                f"logged={logged[t, dim]:.6g} {unit}, "
                f"abs_error={abs_error[t, dim]:.6g} {unit}."
            )


def _reconstruct_sdar_context(actions, selection_masks):
    previous_keys = (
        "previous_action_norm",
        "prev_action_norm",
        "previous_executed_action_norm",
    )
    previous_key = next((k for k in previous_keys if k in agent_data), None)

    if previous_key is not None:
        logged_prev = np.asarray(agent_data[previous_key], dtype=np.float32)
        if logged_prev.ndim != 2 or logged_prev.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{previous_key!r}] has shape {logged_prev.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_prev.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{previous_key!r}] has only {logged_prev.shape[0]} "
                f"rows for {len(actions)} transitions."
            )
        prev_actions = logged_prev[:len(actions)].copy()
    else:
        if not np.all(selection_masks[0] == 1.0):
            raise ValueError(
                "The first selection mask is not a full update, but the callback "
                "did not log previous_action_norm. The initial SDAR context cannot "
                "be reconstructed without leaking action[0] into prev_action[0]."
            )
        prev_actions = actions.copy()
        prev_actions[1:] = actions[:-1]

    if len(actions) > 1 and not np.allclose(
        prev_actions[1:],
        actions[:-1],
        atol=1e-5,
        rtol=0.0,
    ):
        max_error = float(
            np.max(np.abs(prev_actions[1:] - actions[:-1]))
        )
        raise ValueError(
            "Logged/reconstructed previous actions do not equal the preceding "
            f"executed actions (max abs error {max_error:.6g})."
        )

    action_mixes = (
        (1.0 - selection_masks) * prev_actions
        + selection_masks * ACTION_MASK_VALUE
    ).astype(np.float32)

    for key in ("action_mix", "action_mixes"):
        if key not in agent_data:
            continue
        logged_mix = np.asarray(agent_data[key], dtype=np.float32)
        if logged_mix.ndim != 2 or logged_mix.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{key!r}] has shape {logged_mix.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_mix.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{key!r}] has only {logged_mix.shape[0]} rows for "
                f"{len(actions)} transitions."
            )
        if not np.allclose(
            action_mixes,
            logged_mix[:len(actions)],
            atol=1e-5,
            rtol=0.0,
        ):
            max_error = float(
                np.max(np.abs(action_mixes - logged_mix[:len(actions)]))
            )
            raise ValueError(
                f"Reconstructed action_mix differs from agent_data[{key!r}] "
                f"(max abs error {max_error:.6g})."
            )
        break

    return prev_actions.astype(np.float32), action_mixes


# ============================================================
# MAIN BUILDER
# ============================================================

def build_agent_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    reference_dataset_path=REFERENCE_REWARD_DATASET,
    checkpoint_path=EVAL_CHECKPOINT_PATH,
    checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
    eval_seed=EVAL_SEED,
    selector_mode=EVAL_SELECTOR_MODE,
    proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
    selector_seed=EVAL_SELECTOR_SEED,
    safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
    weather_file=EVAL_WEATHER_FILE,
):
    selector_mode = str(selector_mode).lower()
    if selector_mode not in {"threshold", "sample"}:
        raise ValueError(
            "selector_mode must be 'threshold' or 'sample', "
            f"got {selector_mode!r}."
        )
    proposal_deterministic = bool(proposal_deterministic)
    selector_seed = int(selector_seed)

    comp_names = _component_names()
    reward_reference = load_reward_reference(
        reference_dataset_path,
        comp_names,
    )
    checkpoint = _checkpoint_provenance(
        checkpoint_path,
        checkpoint_epoch,
    )
    if prefix is None:
        proposal_tag = (
            "propdet" if proposal_deterministic else "propstoch"
        )
        prefix = (
            f"agent_eval_dataset_decomposed_epoch{checkpoint['epoch']}"
            "_thermal3_tclip50"
            f"_selector-{selector_mode}_{proposal_tag}"
            f"_s{selector_seed}"
        )
    prefix = str(Path(prefix).expanduser())
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)

    print("=== Evaluation provenance ===")
    print(f"  checkpoint epoch: {checkpoint['epoch']}")
    print(f"  checkpoint path:  {checkpoint['path'] or '(not supplied)'}")
    print(f"  reference NPZ:    {reward_reference['path']}")
    print(
        "  reward tag:       "
        f"{reward_reference['reweighting_tag'] or '(not stored)'}"
    )
    print(f"  selector mode:    {selector_mode}")
    print(
        "  proposal deterministic: "
        f"{proposal_deterministic}"
    )
    print(f"  selector seed:    {selector_seed}")
    print(f"  safeguard:        {bool(safeguard_enabled)}")
    print(f"  eval seed:        {eval_seed}")
    print(f"  weather file:     {weather_file}")

    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1
    if T < 1:
        raise ValueError(f"Not enough logged steps to build transitions (n={n}).")

    # ---- length sanity check (agent logs only) ----
    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "agent_observation": len(agent_data["observation"]),
        "agent_selection_logits": len(agent_data["selection_logits"]),
        "agent_selection_probability": len(
            agent_data["selection_probability"]
        ),
        "agent_selection_mask": len(agent_data["selection_mask"]),
        "agent_executed_action_norm": len(agent_data["executed_action_norm"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    # ---- observations ----
    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    # Integrity: the obs we rebuild here must equal what the agent actually saw.
    agent_obs = np.asarray(agent_data["observation"], dtype=np.float32)[:T]
    if not np.allclose(observations, agent_obs, atol=1e-4):
        max_err = float(np.max(np.abs(observations - agent_obs)))
        raise ValueError(
            "Rebuilt observations differ from what the agent saw "
            f"(max abs err {max_err:.6g}). The dataset obs layout has drifted "
            "from the eval callback's build_observation.")

    # ---- actions: agent's executed normalized action is the primary action ----
    agent_actions = np.asarray(agent_data["executed_action_norm"], dtype=np.float32)
    agent_logits = np.asarray(
        agent_data["selection_logits"],
        dtype=np.float32,
    )
    agent_probabilities = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )
    agent_masks = np.asarray(agent_data["selection_mask"], dtype=np.float32)

    if agent_actions.shape != (n, ACTION_DIM):
        raise ValueError(f"executed_action_norm has shape {agent_actions.shape}, expected ({n}, {ACTION_DIM}).")
    if agent_logits.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_logits has shape "
            f"{agent_logits.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_probabilities.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_probability has shape "
            f"{agent_probabilities.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_masks.shape != (n, ACTION_DIM):
        raise ValueError(f"selection_mask has shape {agent_masks.shape}, expected ({n}, {ACTION_DIM}).")

    actions = agent_actions[:T]
    selection_logits = agent_logits[:T]
    selection_probabilities = agent_probabilities[:T]
    selection_masks = agent_masks[:T]

    if np.any(~np.isfinite(selection_logits)):
        raise ValueError("selection_logits contains NaN or infinite values.")
    if np.any(~np.isfinite(selection_probabilities)):
        raise ValueError(
            "selection_probability contains NaN or infinite values."
        )
    if (
        np.any(selection_probabilities < -1e-7)
        or np.any(selection_probabilities > 1.0 + 1e-7)
    ):
        raise ValueError(
            "selection_probability must lie in [0,1]: "
            f"min={selection_probabilities.min()}, "
            f"max={selection_probabilities.max()}."
        )

    reconstructed_probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(selection_logits, -80.0, 80.0))
    )
    if not np.allclose(
        selection_probabilities,
        reconstructed_probabilities,
        atol=1e-6,
        rtol=1e-6,
    ):
        max_err = float(
            np.max(
                np.abs(
                    selection_probabilities
                    - reconstructed_probabilities
                )
            )
        )
        raise ValueError(
            "selection_probability is inconsistent with sigmoid(logits): "
            f"max abs error={max_err:.6g}."
        )

    # Normalized executed actions must be finite and lie within [-1, 1].
    if np.any(~np.isfinite(actions)):
        raise ValueError("executed_action_norm contains NaN or infinite values.")
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: min={actions.min()}, max={actions.max()}")

    # Selector masks must be exactly binary; round only after checking tolerance.
    if np.any(~np.isfinite(selection_masks)):
        raise ValueError("selection_mask contains NaN or infinite values.")
    rounded_masks = np.rint(selection_masks)
    if not np.allclose(selection_masks, rounded_masks, atol=1e-6, rtol=0.0):
        bad = np.argwhere(np.abs(selection_masks - rounded_masks) > 1e-6)[0]
        t, dim = (int(bad[0]), int(bad[1]))
        raise ValueError(
            "selection_mask must be binary. "
            f"Found {selection_masks[t, dim]} at transition {t}, dim {dim}."
        )
    selection_masks = rounded_masks.astype(np.float32)

    if selector_mode == "threshold":
        expected_masks = (
            selection_probabilities[1:] >= 0.5
        ).astype(np.float32)
        if not np.array_equal(selection_masks[1:], expected_masks):
            raise ValueError(
                "selector_mode='threshold', but masks after the forced "
                "first step do not equal probability >= 0.5."
            )

    # Validate all physical actuations against the executed normalized action.
    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)
    _validate_actuated_actions(actions, actions_raw)

    # ---- reconstruct SDAR prev_action / action_mix from the agent trajectory ----
    prev_actions, action_mixes = _reconstruct_sdar_context(
        actions,
        selection_masks,
    )

    # ---- physical lighting quantities from the executed dimming ----
    lighting_dimming = np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)
    nominal_power_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0).astype(np.float32)

    # ---- rewards ----
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, actions[t], selection_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, components_unbounded = apply_reward_reference(
        components_raw,
        reward_reference,
    )
    comp_scales = reward_reference["scales"]
    importance_vec = reward_reference["importance"]
    component_clips = reward_reference["clips"]
    scales_source = f"training reference: {reward_reference['path']}"

    reward_physical_raw = components_raw.sum(axis=1).astype(np.float32)
    reward_balanced = components_final.sum(axis=1).astype(np.float32)
    # Legacy training-compatible key: `rewards_raw` means balanced pre-symlog.
    rewards_raw = reward_balanced
    rewards_tx = _transform_vec(reward_balanced)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32)

    # ---- episode return (undiscounted) for quick scoring ----
    ep_return_physical_raw = float(reward_physical_raw.sum())
    ep_return_balanced = float(reward_balanced.sum())
    # Legacy training-compatible alias.
    ep_return_raw = ep_return_balanced
    ep_return_tx = float(rewards.sum())

    print("\n=== Agent evaluation dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (agent executed, SDAR space)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {prev_actions.shape}  (reconstructed)")
    print(f"  selection_logits:  {selection_logits.shape}")
    print(
        "  selection_probs:   "
        f"{selection_probabilities.shape}"
    )
    print(f"  selection_masks:   {selection_masks.shape}  (agent selector)")
    print(f"  action_mixes:      {action_mixes.shape}  (reconstructed)")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced pre-transform)")
    print(f"  physical raw:      {reward_physical_raw.shape}  (sum of physical W_* terms)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(
        "  episode return:    "
        f"physical={ep_return_physical_raw:+.4f}  "
        f"balanced={ep_return_balanced:+.4f}  "
        f"transformed={ep_return_tx:+.4f}"
    )

    _print_normalization(
        comp_names,
        comp_scales,
        importance_vec,
        component_clips,
        scales_source,
    )
    _print_reward_diagnostics(
        reward_physical_raw,
        reward_balanced,
        rewards_tx,
        components_final,
    )
    _print_thermal_violation_rate()
    _print_switching_diagnostics(selection_masks)
    selector_diagnostics = _print_selector_probability_diagnostics(
        selection_probabilities,
        selection_masks,
        selector_mode,
    )
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=prev_actions,
            selection_logits=selection_logits,
            selection_probabilities=selection_probabilities,
            selection_masks=selection_masks,
            action_mixes=action_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            reward_balanced=reward_balanced,
            reward_physical_raw=reward_physical_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(reward_reference["transform"]),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            thermal_reward_component_clip=np.float32(
                component_clips[comp_names.index("thermal_comfort")]
            ),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            # evaluation provenance / quick scores
            policy=np.array(POLICY_TAG),
            dataset_kind=np.array("agent_evaluation"),
            checkpoint_epoch=np.int64(checkpoint["epoch"]),
            checkpoint_path=np.array(checkpoint["path"]),
            checkpoint_sha256=np.array(checkpoint["sha256"]),
            eval_seed=np.int64(-1 if eval_seed is None else int(eval_seed)),
            eval_selector_mode=np.array(selector_mode),
            eval_proposal_deterministic=np.array(
                proposal_deterministic
            ),
            eval_selector_seed=np.int64(selector_seed),
            eval_safeguard_enabled=np.array(bool(safeguard_enabled)),
            eval_weather_file=np.array("" if weather_file is None else str(weather_file)),
            reward_reference_dataset=np.array(reward_reference["path"]),
            reward_reference_sha256=np.array(reward_reference["sha256"]),
            reward_reweighting_tag=np.array(reward_reference["reweighting_tag"]),
            episode_return_physical_raw=np.float32(ep_return_physical_raw),
            episode_return_balanced=np.float32(ep_return_balanced),
            episode_return_raw=np.float32(ep_return_raw),
            episode_return_transformed=np.float32(ep_return_tx),
            selector_diagnostic_groups=np.array(
                ["glazing", "lighting", "heating", "cooling", "overall"]
            ),
            selector_mean_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["mean_probability"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sampled_rate_by_group=np.array(
                [
                    selector_diagnostics[name]["sampled_rate"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sample_minus_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["difference"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_threshold_agreement=np.float32(
                selector_diagnostics["threshold_agreement"]
            ),
            action_layout=np.array(
                {"glazing": [0, 4], "lighting": [4, 9], "heating": [9, 14], "cooling": [14, 19]},
                dtype=object),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_unbounded"] = components_unbounded
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)
            save_dict["reward_component_clips"] = component_clips.astype(np.float32)

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "policy": POLICY_TAG,
                    "dataset_kind": "agent_evaluation",
                    "checkpoint": checkpoint,
                    "evaluation": {
                        "seed": None if eval_seed is None else int(eval_seed),
                        "selector_mode": selector_mode,
                        "proposal_deterministic": proposal_deterministic,
                        "selector_seed": selector_seed,
                        "safeguard_enabled": bool(safeguard_enabled),
                        "weather_file": None if weather_file is None else str(weather_file),
                    },
                    "reward_reference": {
                        "dataset": reward_reference["path"],
                        "sha256": reward_reference["sha256"],
                        "reweighting_tag": reward_reference["reweighting_tag"],
                    },
                    "reward_transform": reward_reference["transform"],
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "thermal_reward_component_clip": float(
                        component_clips[comp_names.index("thermal_comfort")]
                    ),
                    "reward_scales_source": scales_source,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "episode_return_physical_raw": ep_return_physical_raw,
                    "episode_return_balanced": ep_return_balanced,
                    "episode_return_raw": ep_return_raw,
                    "episode_return_transformed": ep_return_tx,
                    "reward_physical_raw_mean": float(reward_physical_raw.mean()),
                    "reward_physical_raw_std": float(reward_physical_raw.std()),
                    "reward_balanced_mean": float(reward_balanced.mean()),
                    "reward_balanced_std": float(reward_balanced.std()),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_component_clips": component_clips.tolist(),
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(selection_masks.mean()),
                    "mean_selection_mask_glazing": float(selection_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(selection_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(selection_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(selection_masks[:, COOL_SLICE].mean()),
                    "selector_probability_diagnostics": selector_diagnostics,
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        logit_cols = [f"selector_logit_{c}" for c in act_cols]
        probability_cols = [
            f"selector_probability_{c}"
            for c in act_cols
        ]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + logit_cols
            + probability_cols
            + ["reward", "reward_raw", "reward_physical_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + selection_masks[t].tolist()
                    + selection_logits[t].tolist()
                    + selection_probabilities[t].tolist()
                    + [
                        float(rewards[t]),
                        float(rewards_raw[t]),
                        float(reward_physical_raw[t]),
                    ]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    return {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": prev_actions,
        "selection_logits": selection_logits,
        "selection_probabilities": selection_probabilities,
        "selection_masks": selection_masks,
        "action_mixes": action_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_balanced": reward_balanced,
        "reward_physical_raw": reward_physical_raw,
        "reward_components": components_final,
        "reward_components_unbounded": components_unbounded,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "reward_component_clips": component_clips,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
        "checkpoint": checkpoint,
        "evaluation": {
            "seed": None if eval_seed is None else int(eval_seed),
            "selector_mode": selector_mode,
            "proposal_deterministic": proposal_deterministic,
            "selector_seed": selector_seed,
            "safeguard_enabled": bool(safeguard_enabled),
            "weather_file": (
                None if weather_file is None else str(weather_file)
            ),
        },
        "selector_probability_diagnostics": selector_diagnostics,
        "reward_reference": reward_reference,
        "episode_return_physical_raw": ep_return_physical_raw,
        "episode_return_balanced": ep_return_balanced,
        "episode_return_raw": ep_return_raw,
        "episode_return_transformed": ep_return_tx,
    }


if __name__ == "__main__":
    dataset = build_agent_dataset(
        save_npz=True,
        save_csv=True,
        reference_dataset_path=REFERENCE_REWARD_DATASET,
        checkpoint_path=EVAL_CHECKPOINT_PATH,
        checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
        eval_seed=EVAL_SEED,
        selector_mode=EVAL_SELECTOR_MODE,
        proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
        selector_seed=EVAL_SELECTOR_SEED,
        safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
        weather_file=EVAL_WEATHER_FILE,
    )

In [ ]:
"""
Agent-rollout dataset builder — evaluation counterpart of the v8 expert builder
================================================================================

Runs AFTER the evaluation callback (`evaluate_agent_callback.py`) has rolled the
TRAINED SDAR-IQL agent through EnergyPlus and filled the in-memory logging dicts.
It reconstructs the same offline-RL dataset format from the agent's trajectory so
you can score the policy (return, per-component reward, thermal violations,
switching, energy) and, if you want, re-pool the agent rollout with the training
data.

WHAT'S DIFFERENT FROM THE v8 EXPERT BUILDER
-------------------------------------------
The reward math, observation layout, and diagnostics are IDENTICAL (reward is a
property of states/actions, not of which policy produced them). Only the action
source changes:

  expert builder                     agent builder (this file)
  ------------------------------     --------------------------------------
  action_vector_data["action"]   ->  agent_data["executed_action_norm"]
  action_vector_data["...mask"]  ->  agent_data["selection_mask"]
  prev_action / action_mix logged -> RECONSTRUCTED from the agent trajectory
  exploration_data (expert vs        (removed — no exploration during eval;
    executed, thermal modes, ...)     selector sampling is logged directly)

Required in-memory globals (filled by the eval callback rollout):
    ZONES_MID, AVAILABLE_GLAZING_STATES
    temperature_data, lighting_data, solar_data, ext_irr_data, wpi_data
    meter_data, output_variable_data, temporal_data
    glazing_state_data, lighting_power_data,
    heating_setpoint_data, cooling_setpoint_data
    agent_data  (observation, selection_logits, selection_probability,
                 selection_mask, executed_action_norm)

REWARD COMPARABILITY (important)
--------------------------------
This version REQUIRES the pooled decomposed training NPZ as a reward reference.
It loads the exact:

    reward_component_names
    reward_component_scales
    reward_component_importance
    reward_component_clips
    reward_transform

used by training. For the current decomposed run that means thermal importance
3.0, thermal clip 50.0, and clip 10.0 for all other components. The evaluated
rollout never defines its own normalization scales, so rewards remain comparable
across checkpoints and against the experts.

`rewards_raw` is kept as the training-compatible name for the balanced,
post-normalization/post-importance/post-component-clip sum before symlog.
`reward_physical_raw` is additionally saved for the sum of physical W_* reward
components before normalization.
"""

import csv
import hashlib
import json
import re
from pathlib import Path

import numpy as np


# ============================================================
# ROLLOUT LOGS — provided by the eval callback
# ============================================================
# If this modupace),
# bind the logging globals from the eval callback module. They are the same
# mutable objects the callback appended to, so they carry the rollout dle is imported standalone (not exec'd in the rollout namesata.
try:
    agent_data  # noqa: F821  (present when run in the rollout namespace)
except NameError:
    from evaluate_agent_callback import (  # noqa: F401
        ZONES_MID,
        AVAILABLE_GLAZING_STATES,
        temperature_data,
        lighting_data,
        solar_data,
        ext_irr_data,
        wpi_data,
        meter_data,
        output_variable_data,
        temporal_data,
        glazing_state_data,
        lighting_power_data,
        heating_setpoint_data,
        cooling_setpoint_data,
        agent_data,
        SELECTOR_MODE,
        PROPOSAL_DETERMINISTIC,
        SELECTOR_SEED,
    )


# ============================================================
# CONFIG
# ============================================================

# These values are validated against the reference NPZ. They are not used as an
# independent reward definition.
REWARD_TRANSFORM = "symlog"     # "symlog" | "clip" | "none"
REWARD_CLIP_LIMIT = 50.0
SAVE_REWARD_COMPONENTS = True

REWARD_NORMALIZE = True
REWARD_COMPONENT_CLIP = 10.0  # legacy/default clip; per-component clips come from NPZ

REWARD_IMPORTANCE = {
    "energy_lighting":       0.40,
    "hvac_demand_rate":      1.00,
    "thermal_comfort":       3.00,
    "visual_under":          1.50,
    "visual_over_ctrl":      0.40,
    "visual_over_daylight":  0.20,
    "core_under":            1.00,
    "core_over":             0.40,
    "artificial_excess":     0.25,
    "solar_cooling_penalty": 0.20,
    "solar_heating_bonus":   0.10,
    "switching_penalty":     0.10,
}

TIMEOUT_STEPS = None

STEPS_PER_HOUR = 6
TIMESTEP_HOURS = 1.0 / STEPS_PER_HOUR

POLICY_TAG = "sdar_iql_augmented_value_standard_iql_beta3"
OUTPUT_TAG = "beta3_augv"

# ---------------------------------------------------------------------------
# EDIT THESE FOR EACH CHECKPOINT EVALUATION.
# Use the decomposed pooled dataset that trained the agent. If it is elsewhere,
# provide the absolute path.
# ---------------------------------------------------------------------------
REFERENCE_REWARD_DATASET = (
    "pooled_sdar_experts_thermal3_tclip50.npz"
)
EVAL_CHECKPOINT_PATH = "checkpoints/sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3/sdar_iql_sdar_iql_v10_thermal3_tclip50_14episodes_gru2_seq36_augmented_value_standard_iql_beta3_checkpoint_epoch500.pt"   # e.g. "/.../checkpoint_epoch150.pt"
EVAL_CHECKPOINT_EPOCH = None  # optional; inferred from the checkpoint filename
EVAL_SEED = 123
EVAL_SELECTOR_MODE = str(globals().get("SELECTOR_MODE", "sample")).lower()
EVAL_PROPOSAL_DETERMINISTIC = bool(
    globals().get("PROPOSAL_DETERMINISTIC", True)
)
EVAL_SELECTOR_SEED = int(globals().get("SELECTOR_SEED", 20260728))
EVAL_SAFEGUARD_ENABLED = True
EVAL_WEATHER_FILE = None


# ============================================================
# WINDOW MAP
# ============================================================

WINDOW_ZONES_MID = {
    "Perimeter_mid_ZN_1": "Perimeter_mid_ZN_1_Wall_South_Window",
    "Perimeter_mid_ZN_2": "Perimeter_mid_ZN_2_Wall_East_Window",
    "Perimeter_mid_ZN_3": "Perimeter_mid_ZN_3_Wall_North_Window",
    "Perimeter_mid_ZN_4": "Perimeter_mid_ZN_4_Wall_West_Window",
}

GLAZING_TO_ID = {name: i for i, name in enumerate(AVAILABLE_GLAZING_STATES)}


# ============================================================
# ACTION VECTOR LAYOUT
# ============================================================

ACTION_DIM = 19
GLAZE_SLICE = slice(0, 4)
LIGHT_SLICE = slice(4, 9)
HEAT_SLICE = slice(9, 14)
COOL_SLICE = slice(14, 19)

ACTION_MASK_VALUE = -2.0

LIGHTING_INSTALLED_W = {
    "Perimeter_mid_ZN_1": 2231.0,
    "Perimeter_mid_ZN_2": 2231.0,
    "Perimeter_mid_ZN_3": 1412.0,
    "Perimeter_mid_ZN_4": 1412.0,
    "Core_mid":          10586.0,
}

HEATING_SETPOINT_RANGE = (16.0, 24.0)
COOLING_SETPOINT_RANGE = (22.0, 28.0)
ACTION_VALIDATION_ATOL = 0.5


# ============================================================
# REWARD WEIGHTS  (physical; cancel out when REWARD_NORMALIZE=True)
# ============================================================

LIGHTING_TO_LUX = 0.4

ILLUM_TARGET = 500.0
ILLUM_LOW = 450.0
ILLUM_HIGH = 550.0

CORE_TARGET = 500.0
CORE_LOW = 450.0
CORE_HIGH = 550.0

W_LIGHTING = 1.0
W_HVAC_DEMAND = 0.03

W_THERMAL = 10.0
COMFORT_TEMP_LOW = 21.0
COMFORT_TEMP_HIGH = 24.0
COMFORT_OCCUPIED_ONLY = True

W_VISUAL_UNDER = 0.03
W_VISUAL_OVER_CTRL = 0.015
W_VISUAL_OVER_DAYLIGHT = 0.002
W_CORE_UNDER = 0.03
W_CORE_OVER = 0.015

W_ARTIFICIAL_EXCESS = 0.02

W_SOLAR_COOLING = 1e-4
W_SOLAR_HEATING_BONUS = 5e-5

COOLING_ACTIVE_THRESH = 1.0
HEATING_ACTIVE_THRESH = 1.0
SOLAR_IGNORE_THRESH = 50.0
SOLAR_HEAT_CAP = 400.0

W_SWITCH_GLAZING = 0.05
W_SWITCH_LIGHTING = 0.01
W_SWITCH_HVAC = 0.002


# ============================================================
# COLUMN NAMES
# ============================================================

def obs_column_names():
    cols = []
    for zone in ZONES_MID:
        cols.append(f"temp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_rate_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"solar_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"ext_irr_{zone}")
    for zone in WINDOW_ZONES_MID:
        cols.append(f"wpi_{zone}")
    cols += [
        "hvac_electricity_demand_rate",
        "hour_sin",
        "hour_cos",
        "doy_sin",
        "doy_cos",
        "is_occupied",
        "is_preconditioning",
    ]
    return cols


def action_column_names():
    cols = []
    for zone in WINDOW_ZONES_MID:
        cols.append(f"glazing_{zone}")
    for zone in ZONES_MID:
        cols.append(f"light_power_{zone}")
    for zone in ZONES_MID:
        cols.append(f"heat_sp_{zone}")
    for zone in ZONES_MID:
        cols.append(f"cool_sp_{zone}")
    return cols


REWARD_COMPONENT_NAMES = [
    "energy_lighting",
    "hvac_demand_rate",
    "thermal_comfort",
    "visual_under",
    "visual_over_ctrl",
    "visual_over_daylight",
    "core_under",
    "core_over",
    "artificial_excess",
    "solar_cooling_penalty",
    "solar_heating_bonus",
    "switching_penalty",
    "raw_total",
    "transformed_total",
]


def _component_names():
    return [n for n in REWARD_COMPONENT_NAMES if n not in ("raw_total", "transformed_total")]


# ============================================================
# BUILDERS
# ============================================================

def build_observation(t):
    obs = []
    for zone in ZONES_MID:
        obs.append(temperature_data[zone][t])
    for zone in ZONES_MID:
        obs.append(lighting_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(solar_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(ext_irr_data[zone][t])
    for zone in WINDOW_ZONES_MID:
        obs.append(wpi_data[zone][t])
    obs.append(output_variable_data["HVAC_electricity_demand_rate"][t])
    obs.append(temporal_data["hour_sin"][t])
    obs.append(temporal_data["hour_cos"][t])
    obs.append(temporal_data["doy_sin"][t])
    obs.append(temporal_data["doy_cos"][t])
    obs.append(float(temporal_data["is_occupied"][t]))
    obs.append(float(temporal_data["is_preconditioning"][t]))
    if len(obs) != len(obs_column_names()):
        raise ValueError(f"Observation length {len(obs)} != columns {len(obs_column_names())}")
    return obs


def build_action_raw(t):
    """Physical (actuated) action, read from the eval callback's logs."""
    act = []
    for zone in WINDOW_ZONES_MID:
        act.append(GLAZING_TO_ID[glazing_state_data[zone][t]])
    for zone in ZONES_MID:
        act.append(lighting_power_data[zone][t])
    for zone in ZONES_MID:
        act.append(heating_setpoint_data[zone][t])
    for zone in ZONES_MID:
        act.append(cooling_setpoint_data[zone][t])
    if len(act) != ACTION_DIM:
        raise ValueError(f"Action length {len(act)} != ACTION_DIM {ACTION_DIM}")
    return act


# ============================================================
# LIGHTING ENERGY FROM THE AGENT'S NORMALIZED ACTION
# ============================================================
# The per-zone normalized lighting action a in [-1, 1] maps to a dimming duty
#   u = (a + 1) / 2 in [0, 1], and E = sum_z nominal_W[z]*u[z] * dt / 1000 (kWh).
# Meter-free: uses the executed action, not the logged Lights Electricity Rate.

def compute_lighting_energy_kwh(action_norm_t):
    a = np.asarray(action_norm_t, dtype=np.float64)
    dimming = np.clip((a[LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
    nominal_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float64)
    total_power_w = float((nominal_w * dimming).sum())
    return total_power_w * TIMESTEP_HOURS / 1000.0


# ============================================================
# REWARD — components + scalar (identical logic to the v8 builder)
# ============================================================

def _symlog(x):
    return float(np.sign(x) * np.log1p(np.abs(x)))


def _transform(r):
    if REWARD_TRANSFORM == "symlog":
        return _symlog(r)
    if REWARD_TRANSFORM == "clip":
        return float(np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT))
    if REWARD_TRANSFORM == "none":
        return float(r)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def _transform_vec(r):
    r = np.asarray(r, dtype=np.float64)
    if REWARD_TRANSFORM == "symlog":
        return (np.sign(r) * np.log1p(np.abs(r))).astype(np.float32)
    if REWARD_TRANSFORM == "clip":
        return np.clip(r, -REWARD_CLIP_LIMIT, REWARD_CLIP_LIMIT).astype(np.float32)
    if REWARD_TRANSFORM == "none":
        return r.astype(np.float32)
    raise ValueError(f"Unknown REWARD_TRANSFORM={REWARD_TRANSFORM!r}")


def compute_reward_components(t, action_norm_t, selection_mask_t):
    occupied_next = bool(temporal_data["is_occupied"][t + 1])
    occupied_lighting = bool(temporal_data["is_occupied"][t])

    hvac_demand_w = output_variable_data["HVAC_electricity_demand_rate"][t + 1]
    r_hvac_demand = -W_HVAC_DEMAND * (hvac_demand_w / 1000.0)

    lighting_energy_kwh = compute_lighting_energy_kwh(action_norm_t)
    e_light = -W_LIGHTING * lighting_energy_kwh

    thermal_penalty = 0.0
    if (not COMFORT_OCCUPIED_ONLY) or occupied_next:
        for zone in ZONES_MID:
            temp_next = temperature_data[zone][t + 1]
            if temp_next < COMFORT_TEMP_LOW:
                thermal_penalty += COMFORT_TEMP_LOW - temp_next
            elif temp_next > COMFORT_TEMP_HIGH:
                thermal_penalty += temp_next - COMFORT_TEMP_HIGH
    r_thermal = -W_THERMAL * thermal_penalty

    visual_under_p = 0.0
    visual_over_ctrl_p = 0.0
    visual_over_daylight_p = 0.0
    artificial_excess_p = 0.0
    core_under_p = 0.0
    core_over_p = 0.0

    if occupied_lighting:
        for zone in WINDOW_ZONES_MID:
            daylight_lux = wpi_data[zone][t + 1]
            light_power = lighting_power_data[zone][t]
            artificial_lux = light_power * LIGHTING_TO_LUX
            total_lux = daylight_lux + artificial_lux

            if total_lux < ILLUM_LOW:
                visual_under_p += ILLUM_LOW - total_lux
            elif total_lux > ILLUM_HIGH:
                if daylight_lux >= ILLUM_HIGH:
                    visual_over_daylight_p += np.log1p(daylight_lux - ILLUM_HIGH)
                    visual_over_ctrl_p += artificial_lux
                else:
                    visual_over_ctrl_p += total_lux - ILLUM_HIGH

            needed_art = max(0.0, ILLUM_TARGET - min(daylight_lux, ILLUM_TARGET))
            artificial_excess_p += max(0.0, artificial_lux - needed_art)

        core_power = lighting_power_data["Core_mid"][t]
        core_lux = core_power * LIGHTING_TO_LUX
        core_under_p = max(0.0, CORE_LOW - core_lux)
        core_over_p = max(0.0, core_lux - CORE_HIGH)
        artificial_excess_p += max(0.0, core_lux - CORE_TARGET)

    r_visual_under = -W_VISUAL_UNDER * visual_under_p
    r_visual_over_ctrl = -W_VISUAL_OVER_CTRL * visual_over_ctrl_p
    r_visual_over_daylight = -W_VISUAL_OVER_DAYLIGHT * visual_over_daylight_p
    r_core_under = -W_CORE_UNDER * core_under_p
    r_core_over = -W_CORE_OVER * core_over_p
    r_artificial_excess = -W_ARTIFICIAL_EXCESS * artificial_excess_p

    heating_flag = meter_data["Heating:Electricity"][t + 1] > HEATING_ACTIVE_THRESH
    cooling_flag = meter_data["Cooling:Electricity"][t + 1] > COOLING_ACTIVE_THRESH

    solar_cooling_p = 0.0
    solar_heating_b = 0.0
    for zone in WINDOW_ZONES_MID:
        solar_next = solar_data[zone][t + 1]
        if cooling_flag:
            if solar_next > SOLAR_IGNORE_THRESH:
                solar_cooling_p += solar_next - SOLAR_IGNORE_THRESH
        elif heating_flag:
            solar_heating_b += min(solar_next, SOLAR_HEAT_CAP)

    r_solar_cool = -W_SOLAR_COOLING * solar_cooling_p
    r_solar_heat = +W_SOLAR_HEATING_BONUS * solar_heating_b

    if t == 0:
        r_switch = 0.0
    else:
        b = np.asarray(selection_mask_t)
        n_glaze = float(b[GLAZE_SLICE].sum())
        n_light = float(b[LIGHT_SLICE].sum())
        n_hvac = float(b[HEAT_SLICE].sum() + b[COOL_SLICE].sum())
        r_switch = -(
            W_SWITCH_GLAZING * n_glaze
            + W_SWITCH_LIGHTING * n_light
            + W_SWITCH_HVAC * n_hvac
        )

    raw_total = (
        e_light + r_hvac_demand + r_thermal
        + r_visual_under + r_visual_over_ctrl + r_visual_over_daylight
        + r_core_under + r_core_over + r_artificial_excess
        + r_solar_cool + r_solar_heat + r_switch
    )

    return {
        "energy_lighting": e_light,
        "hvac_demand_rate": r_hvac_demand,
        "thermal_comfort": r_thermal,
        "visual_under": r_visual_under,
        "visual_over_ctrl": r_visual_over_ctrl,
        "visual_over_daylight": r_visual_over_daylight,
        "core_under": r_core_under,
        "core_over": r_core_over,
        "artificial_excess": r_artificial_excess,
        "solar_cooling_penalty": r_solar_cool,
        "solar_heating_bonus": r_solar_heat,
        "switching_penalty": r_switch,
        "raw_total": raw_total,
        "transformed_total": _transform(raw_total),
    }


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _npz_scalar(data, key, default=None):
    if key not in data.files:
        return default
    value = np.asarray(data[key])
    if value.size != 1:
        raise ValueError(f"Expected scalar NPZ field {key!r}, got shape {value.shape}.")
    return value.reshape(()).item()


def load_reward_reference(reference_dataset_path, comp_names):
    """
    Load the immutable reward ruler from the pooled decomposed training NPZ.

    There is deliberately no rollout-local fallback. A missing or incompatible
    reference dataset is an error because otherwise checkpoint rewards would be
    calculated on different scales.
    """
    path = Path(reference_dataset_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "Reward reference dataset not found: "
            f"{path}. Set REFERENCE_REWARD_DATASET to the exact pooled NPZ used "
            "to train the decomposed agent."
        )

    required = {
        "reward_component_names",
        "reward_component_scales",
        "reward_component_importance",
    }
    with np.load(path, allow_pickle=False) as data:
        missing = sorted(required.difference(data.files))
        if missing:
            raise KeyError(
                f"Reference NPZ {path} is missing required reward metadata: {missing}"
            )

        ref_names = [str(x) for x in np.asarray(data["reward_component_names"]).tolist()]
        if ref_names != list(comp_names):
            raise ValueError(
                "Reward component order differs from the training dataset.\n"
                f"  builder:   {list(comp_names)}\n"
                f"  reference: {ref_names}"
            )

        scales = np.asarray(data["reward_component_scales"], dtype=np.float64)
        importance = np.asarray(
            data["reward_component_importance"], dtype=np.float64
        )
        if scales.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_scales shape {scales.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if importance.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_importance shape {importance.shape}; "
                f"expected {(len(comp_names),)}"
            )
        if np.any(~np.isfinite(scales)) or np.any(scales <= 0.0):
            raise ValueError("Reference reward scales must all be finite and positive.")
        if np.any(~np.isfinite(importance)):
            raise ValueError("Reference reward importance values must all be finite.")

        if "reward_component_clips" in data.files:
            clips = np.asarray(data["reward_component_clips"], dtype=np.float64)
        else:
            legacy_clip = float(
                _npz_scalar(data, "reward_component_clip", REWARD_COMPONENT_CLIP)
            )
            clips = np.full(len(comp_names), legacy_clip, dtype=np.float64)
            if "thermal_reward_component_clip" in data.files:
                thermal_i = comp_names.index("thermal_comfort")
                clips[thermal_i] = float(
                    _npz_scalar(data, "thermal_reward_component_clip")
                )

        if clips.shape != (len(comp_names),):
            raise ValueError(
                f"reward_component_clips shape {clips.shape}; "
                f"expected {(len(comp_names),)}"
            )
        # A negative or non-finite stored clip means unbounded for that component.
        clips = np.where(np.isfinite(clips) & (clips >= 0.0), clips, np.inf)

        transform = str(
            _npz_scalar(data, "reward_transform", REWARD_TRANSFORM)
        )
        reweighting_tag = str(
            _npz_scalar(data, "reward_reweighting_tag", "")
        )
        reference_mask_value = float(
            _npz_scalar(data, "action_mask_value", ACTION_MASK_VALUE)
        )

    configured_importance = np.array(
        [REWARD_IMPORTANCE.get(name, 1.0) for name in comp_names],
        dtype=np.float64,
    )
    if not np.allclose(configured_importance, importance, atol=1e-7, rtol=0.0):
        raise ValueError(
            "Builder REWARD_IMPORTANCE differs from the training NPZ. "
            "Do not evaluate until the reward definitions match.\n"
            f"  builder:   {configured_importance.tolist()}\n"
            f"  reference: {importance.tolist()}"
        )
    if transform != REWARD_TRANSFORM:
        raise ValueError(
            f"Builder transform {REWARD_TRANSFORM!r} differs from reference "
            f"transform {transform!r}."
        )
    if not np.isclose(reference_mask_value, ACTION_MASK_VALUE):
        raise ValueError(
            f"Reference action_mask_value={reference_mask_value} differs from "
            f"builder ACTION_MASK_VALUE={ACTION_MASK_VALUE}."
        )

    return {
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
        "names": list(comp_names),
        "scales": scales,
        "importance": importance,
        "clips": clips,
        "transform": transform,
        "reweighting_tag": reweighting_tag,
    }


def apply_reward_reference(components_raw, reward_reference):
    """Apply training scales, importance, and per-component clips exactly."""
    scales = reward_reference["scales"]
    importance = reward_reference["importance"]
    clips = reward_reference["clips"]

    unbounded = (
        components_raw.astype(np.float64)
        / scales[None, :]
        * importance[None, :]
    )
    components_final = np.clip(
        unbounded,
        -clips[None, :],
        clips[None, :],
    )
    return components_final.astype(np.float32), unbounded.astype(np.float32)


def _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch):
    if checkpoint_epoch is not None:
        epoch = int(checkpoint_epoch)
        if epoch < 0:
            raise ValueError("checkpoint_epoch must be non-negative.")
        return epoch
    if checkpoint_path:
        match = re.search(r"epoch[_-]?(\d+)", Path(checkpoint_path).name, re.IGNORECASE)
        if match:
            return int(match.group(1))
    raise ValueError(
        "Checkpoint epoch is required for unambiguous output naming. Set "
        "EVAL_CHECKPOINT_EPOCH or use a checkpoint filename containing "
        "'epoch<number>'."
    )


def _checkpoint_provenance(checkpoint_path, checkpoint_epoch):
    epoch = _infer_checkpoint_epoch(checkpoint_path, checkpoint_epoch)
    if checkpoint_path is None:
        return {
            "epoch": epoch,
            "path": "",
            "sha256": "",
        }

    path = Path(checkpoint_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return {
        "epoch": epoch,
        "path": str(path.resolve()),
        "sha256": _sha256_file(path),
    }


# ============================================================
# DIAGNOSTICS (identical to the v8 builder)
# ============================================================

def _print_reward_diagnostics(
    reward_physical_raw,
    reward_balanced,
    rewards_tx,
    components_arr,
):
    print("\n=== Reward diagnostics ===")
    qs = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    print(
        f"  physical raw: mean={reward_physical_raw.mean():+.4f}  "
        f"std={reward_physical_raw.std():.4f}"
    )
    print(
        f"                min={reward_physical_raw.min():+.4f}    "
        f"max={reward_physical_raw.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_physical_raw, qs)]}")
    print(
        f"  balanced:     mean={reward_balanced.mean():+.4f}  "
        f"std={reward_balanced.std():.4f}"
    )
    print(
        f"                min={reward_balanced.min():+.4f}    "
        f"max={reward_balanced.max():+.4f}"
    )
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(reward_balanced, qs)]}")
    print(f"  transformed: mean={rewards_tx.mean():+.4f}  std={rewards_tx.std():.4f}")
    print(f"               min={rewards_tx.min():+.4f}    max={rewards_tx.max():+.4f}")
    print(f"               quantiles {qs} = "
          f"{[round(float(q), 3) for q in np.quantile(rewards_tx, qs)]}")
    print("\n=== Per-component mean |contribution| (final, post-normalization) ===")
    comp_names = _component_names()
    for i, name in enumerate(comp_names):
        col = components_arr[:, i]
        print(f"  {name:24s}  mean={col.mean():+.6f}  mean|.|={np.abs(col).mean():.6f}  "
              f"min={col.min():+.4f}  max={col.max():+.4f}")
    comp_sum = components_arr.sum(axis=1)
    err = comp_sum - reward_balanced
    print("\n=== Component sum check ===")
    print(f"  mean abs(sum_components - balanced): {np.mean(np.abs(err)):.8f}")
    print(f"  max  abs(sum_components - balanced): {np.max(np.abs(err)):.8f}")


def _print_normalization(
    comp_names,
    comp_scales,
    importance_vec,
    component_clips,
    scales_source,
):
    print("\n=== Reward component normalization ===")
    print(f"  REWARD_NORMALIZE={REWARD_NORMALIZE}  scales={scales_source}")
    print(f"  {'component':24s}  {'scale(mean|abs|)':>16s}  {'importance':>10s}  "
          f"{'clip':>8s}  {'eff_weight':>10s}")
    for nm, sc, imp, clip in zip(
        comp_names, comp_scales, importance_vec, component_clips
    ):
        eff = imp / sc if sc != 0 else 0.0
        clip_name = "none" if not np.isfinite(clip) else f"{clip:.1f}"
        print(
            f"  {nm:24s}  {sc:16.6f}  {imp:10.3f}  "
            f"{clip_name:>8s}  {eff:10.4f}"
        )


def _print_thermal_violation_rate():
    n_steps = len(temperature_data[ZONES_MID[0]])
    violations = 0
    total = 0
    for t in range(n_steps):
        occ = bool(temporal_data["is_occupied"][t])
        if COMFORT_OCCUPIED_ONLY and not occ:
            continue
        for zone in ZONES_MID:
            tmp = temperature_data[zone][t]
            total += 1
            if tmp < COMFORT_TEMP_LOW or tmp > COMFORT_TEMP_HIGH:
                violations += 1
    rate = 100.0 * violations / max(total, 1)
    scope = "occupied zone-steps" if COMFORT_OCCUPIED_ONLY else "all zone-steps"
    print("\n=== Thermal violation rate (fixed band "
          f"{COMFORT_TEMP_LOW}-{COMFORT_TEMP_HIGH}°C) ===")
    print(f"  {violations}/{total} {scope} outside band ({rate:.2f}%)")


def _print_switching_diagnostics(selection_masks):
    print("\n=== SDAR selection mask diagnostics (agent selector) ===")
    print("  b=1 means dim acted/changed, b=0 means dim repeated")
    groups = {
        "glazing  (dims 0:4)": selection_masks[:, GLAZE_SLICE],
        "lighting (dims 4:9)": selection_masks[:, LIGHT_SLICE],
        "heating  (dims 9:14)": selection_masks[:, HEAT_SLICE],
        "cooling  (dims 14:19)": selection_masks[:, COOL_SLICE],
        "overall  (all 19 dims)": selection_masks,
    }
    for name, m in groups.items():
        mean_b = float(m.mean())
        apr = 1.0 / max(mean_b, 1e-9)
        acted_dims = float(m.sum(axis=1).mean())
        print(f"  {name:24s}  mean(b)={mean_b:.4f}  APR={apr:7.2f}  acted_dims/step={acted_dims:.3f}")


def _print_selector_probability_diagnostics(
    selection_probabilities,
    selection_masks,
    selector_mode,
):
    # The callback forces all dimensions to update on the first step, so
    # exclude that step when checking the learned selector realization.
    probabilities = selection_probabilities[1:]
    masks = selection_masks[1:]

    groups = {
        "glazing": GLAZE_SLICE,
        "lighting": LIGHT_SLICE,
        "heating": HEAT_SLICE,
        "cooling": COOL_SLICE,
        "overall": slice(0, ACTION_DIM),
    }

    print(
        "\n=== Selector probability diagnostics "
        f"(mode={selector_mode}) ==="
    )
    stats = {}
    for name, slc in groups.items():
        p = probabilities[:, slc]
        m = masks[:, slc]
        mean_probability = float(p.mean())
        sampled_rate = float(m.mean())
        difference = sampled_rate - mean_probability
        stats[name] = {
            "mean_probability": mean_probability,
            "sampled_rate": sampled_rate,
            "difference": difference,
        }
        print(
            f"  {name:10s} "
            f"mean probability={mean_probability:.4f}  "
            f"sampled rate={sampled_rate:.4f}  "
            f"difference={difference:+.4f}"
        )

    threshold_masks = (probabilities >= 0.5).astype(np.float32)
    stats["threshold_agreement"] = float(
        np.mean(masks == threshold_masks)
    )
    print(
        "  agreement with p>=0.5 threshold: "
        f"{100.0 * stats['threshold_agreement']:.2f}%"
    )
    return stats


def _print_scale_check(observations, actions_raw, actions_norm):
    print("\n=== Observation column ranges ===")
    cols = obs_column_names()
    for i, name in enumerate(cols):
        col = observations[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (raw / physical units) ===")
    a_cols = action_column_names()
    for i, name in enumerate(a_cols):
        col = actions_raw[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")
    print("\n=== Action column ranges (normalized [-1, 1], SDAR space) ===")
    for i, name in enumerate(a_cols):
        col = actions_norm[:, i]
        print(f"  [{i:2d}] {name:40s} mean={col.mean():+.3f}  std={col.std():.3f}  "
              f"min={col.min():+.2f}  max={col.max():+.2f}")


def _print_hvac_demand_diagnostics(observations):
    cols = obs_column_names()
    hvac_idx = cols.index("hvac_electricity_demand_rate")
    occ_idx = cols.index("is_occupied")
    pre_idx = cols.index("is_preconditioning")
    hvac = observations[:, hvac_idx]
    occupied = observations[:, occ_idx] > 0.5
    pre = observations[:, pre_idx] > 0.5
    setback = (~occupied) & (~pre)
    print("\n=== HVAC demand-rate diagnostics ===")
    for name, mask in {
        "all": np.ones(len(hvac), dtype=bool),
        "occupied": occupied,
        "preconditioning": pre,
        "setback": setback,
    }.items():
        if mask.sum() == 0:
            print(f"  {name:16s}: no samples")
            continue
        x = hvac[mask]
        print(f"  {name:16s}: mean={x.mean():10.2f} W  p95={np.quantile(x, 0.95):10.2f} W  "
              f"max={x.max():10.2f} W")


def _make_timeouts(T):
    timeouts = np.zeros(T, dtype=np.float32)
    if TIMEOUT_STEPS is None or TIMEOUT_STEPS <= 0:
        return timeouts
    for i in range(T):
        if (i + 1) % int(TIMEOUT_STEPS) == 0:
            timeouts[i] = 1.0
    return timeouts


def _denormalize_linear(values, physical_range):
    low, high = physical_range
    values = np.asarray(values, dtype=np.float64)
    return low + 0.5 * (values + 1.0) * (high - low)


def _validate_actuated_actions(actions, actions_raw):
    """Verify all 19 normalized executed actions against physical actuation logs."""
    n_glazing = len(AVAILABLE_GLAZING_STATES)
    glz_ids_from_norm = np.clip(
        np.round((actions[:, GLAZE_SLICE] + 1.0) * 0.5 * (n_glazing - 1)),
        0,
        n_glazing - 1,
    ).astype(np.int64)
    glz_ids_logged = actions_raw[:, GLAZE_SLICE].astype(np.int64)
    if not np.array_equal(glz_ids_from_norm, glz_ids_logged):
        mismatch = np.argwhere(glz_ids_from_norm != glz_ids_logged)[0]
        t, dim = (int(mismatch[0]), int(mismatch[1]))
        raise ValueError(
            "Glazing decoded from executed_action_norm does not match the "
            f"actuation log at transition {t}, glazing dim {dim}: "
            f"decoded={glz_ids_from_norm[t, dim]}, logged={glz_ids_logged[t, dim]}."
        )

    nominal_power_w = np.array(
        [LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID],
        dtype=np.float64,
    )
    light_decoded = (
        np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0)
        * nominal_power_w[None, :]
    )
    heat_decoded = _denormalize_linear(
        actions[:, HEAT_SLICE], HEATING_SETPOINT_RANGE
    )
    cool_decoded = _denormalize_linear(
        actions[:, COOL_SLICE], COOLING_SETPOINT_RANGE
    )

    checks = [
        (
            "lighting",
            light_decoded,
            actions_raw[:, LIGHT_SLICE],
            "W",
        ),
        (
            "heating setpoint",
            heat_decoded,
            actions_raw[:, HEAT_SLICE],
            "degC",
        ),
        (
            "cooling setpoint",
            cool_decoded,
            actions_raw[:, COOL_SLICE],
            "degC",
        ),
    ]
    for label, decoded, logged, unit in checks:
        if not np.allclose(
            decoded,
            logged,
            atol=ACTION_VALIDATION_ATOL,
            rtol=1e-4,
        ):
            abs_error = np.abs(decoded - logged)
            t, dim = np.unravel_index(np.argmax(abs_error), abs_error.shape)
            raise ValueError(
                f"{label} decoded from executed_action_norm does not match the "
                f"actuation log. Largest error at transition {t}, local dim {dim}: "
                f"decoded={decoded[t, dim]:.6g} {unit}, "
                f"logged={logged[t, dim]:.6g} {unit}, "
                f"abs_error={abs_error[t, dim]:.6g} {unit}."
            )


def _reconstruct_sdar_context(actions, selection_masks):
    previous_keys = (
        "previous_action_norm",
        "prev_action_norm",
        "previous_executed_action_norm",
    )
    previous_key = next((k for k in previous_keys if k in agent_data), None)

    if previous_key is not None:
        logged_prev = np.asarray(agent_data[previous_key], dtype=np.float32)
        if logged_prev.ndim != 2 or logged_prev.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{previous_key!r}] has shape {logged_prev.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_prev.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{previous_key!r}] has only {logged_prev.shape[0]} "
                f"rows for {len(actions)} transitions."
            )
        prev_actions = logged_prev[:len(actions)].copy()
    else:
        if not np.all(selection_masks[0] == 1.0):
            raise ValueError(
                "The first selection mask is not a full update, but the callback "
                "did not log previous_action_norm. The initial SDAR context cannot "
                "be reconstructed without leaking action[0] into prev_action[0]."
            )
        prev_actions = actions.copy()
        prev_actions[1:] = actions[:-1]

    if len(actions) > 1 and not np.allclose(
        prev_actions[1:],
        actions[:-1],
        atol=1e-5,
        rtol=0.0,
    ):
        max_error = float(
            np.max(np.abs(prev_actions[1:] - actions[:-1]))
        )
        raise ValueError(
            "Logged/reconstructed previous actions do not equal the preceding "
            f"executed actions (max abs error {max_error:.6g})."
        )

    action_mixes = (
        (1.0 - selection_masks) * prev_actions
        + selection_masks * ACTION_MASK_VALUE
    ).astype(np.float32)

    for key in ("action_mix", "action_mixes"):
        if key not in agent_data:
            continue
        logged_mix = np.asarray(agent_data[key], dtype=np.float32)
        if logged_mix.ndim != 2 or logged_mix.shape[1] != ACTION_DIM:
            raise ValueError(
                f"agent_data[{key!r}] has shape {logged_mix.shape}; "
                f"expected (*, {ACTION_DIM})."
            )
        if logged_mix.shape[0] < len(actions):
            raise ValueError(
                f"agent_data[{key!r}] has only {logged_mix.shape[0]} rows for "
                f"{len(actions)} transitions."
            )
        if not np.allclose(
            action_mixes,
            logged_mix[:len(actions)],
            atol=1e-5,
            rtol=0.0,
        ):
            max_error = float(
                np.max(np.abs(action_mixes - logged_mix[:len(actions)]))
            )
            raise ValueError(
                f"Reconstructed action_mix differs from agent_data[{key!r}] "
                f"(max abs error {max_error:.6g})."
            )
        break

    return prev_actions.astype(np.float32), action_mixes


# ============================================================
# MAIN BUILDER
# ============================================================

def build_agent_dataset(
    save_npz=True,
    save_csv=True,
    prefix=None,
    reference_dataset_path=REFERENCE_REWARD_DATASET,
    checkpoint_path=EVAL_CHECKPOINT_PATH,
    checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
    eval_seed=EVAL_SEED,
    selector_mode=EVAL_SELECTOR_MODE,
    proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
    selector_seed=EVAL_SELECTOR_SEED,
    safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
    weather_file=EVAL_WEATHER_FILE,
):
    selector_mode = str(selector_mode).lower()
    if selector_mode not in {"threshold", "sample"}:
        raise ValueError(
            "selector_mode must be 'threshold' or 'sample', "
            f"got {selector_mode!r}."
        )
    proposal_deterministic = bool(proposal_deterministic)
    selector_seed = int(selector_seed)

    comp_names = _component_names()
    reward_reference = load_reward_reference(
        reference_dataset_path,
        comp_names,
    )
    checkpoint = _checkpoint_provenance(
        checkpoint_path,
        checkpoint_epoch,
    )
    if prefix is None:
        proposal_tag = (
            "propdet" if proposal_deterministic else "propstoch"
        )
        prefix = (
            f"agent_eval_dataset_decomposed_epoch{checkpoint['epoch']}"
            "_thermal3_tclip50"
            f"_selector-{selector_mode}_{proposal_tag}"
            f"_s{selector_seed}"
        )
    prefix = str(Path(prefix).expanduser())
    Path(prefix).parent.mkdir(parents=True, exist_ok=True)

    print("=== Evaluation provenance ===")
    print(f"  checkpoint epoch: {checkpoint['epoch']}")
    print(f"  checkpoint path:  {checkpoint['path'] or '(not supplied)'}")
    print(f"  reference NPZ:    {reward_reference['path']}")
    print(
        "  reward tag:       "
        f"{reward_reference['reweighting_tag'] or '(not stored)'}"
    )
    print(f"  selector mode:    {selector_mode}")
    print(
        "  proposal deterministic: "
        f"{proposal_deterministic}"
    )
    print(f"  selector seed:    {selector_seed}")
    print(f"  safeguard:        {bool(safeguard_enabled)}")
    print(f"  eval seed:        {eval_seed}")
    print(f"  weather file:     {weather_file}")

    n = len(temperature_data[ZONES_MID[0]])
    T = n - 1
    if T < 1:
        raise ValueError(f"Not enough logged steps to build transitions (n={n}).")

    # ---- length sanity check (agent logs only) ----
    expected_lengths = {
        "temperature_data": len(temperature_data[ZONES_MID[0]]),
        "lighting_data": len(lighting_data[ZONES_MID[0]]),
        "solar_data": len(solar_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "ext_irr_data": len(ext_irr_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "wpi_data": len(wpi_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "meter_heating": len(meter_data["Heating:Electricity"]),
        "meter_cooling": len(meter_data["Cooling:Electricity"]),
        "hvac_demand_rate": len(output_variable_data["HVAC_electricity_demand_rate"]),
        "glazing_state": len(glazing_state_data[list(WINDOW_ZONES_MID.keys())[0]]),
        "lighting_power": len(lighting_power_data[ZONES_MID[0]]),
        "heating_setpoint": len(heating_setpoint_data[ZONES_MID[0]]),
        "cooling_setpoint": len(cooling_setpoint_data[ZONES_MID[0]]),
        "temporal_hour_sin": len(temporal_data["hour_sin"]),
        "temporal_is_occupied": len(temporal_data["is_occupied"]),
        "agent_observation": len(agent_data["observation"]),
        "agent_selection_logits": len(agent_data["selection_logits"]),
        "agent_selection_probability": len(
            agent_data["selection_probability"]
        ),
        "agent_selection_mask": len(agent_data["selection_mask"]),
        "agent_executed_action_norm": len(agent_data["executed_action_norm"]),
    }

    print("=== Length sanity check ===")
    bad_lengths = []
    for k, v in expected_lengths.items():
        status = "OK" if v == n else f"MISMATCH (expected {n})"
        print(f"  {k}: {v}  {status}")
        if v != n:
            bad_lengths.append((k, v))
    if bad_lengths:
        raise ValueError(f"Length mismatch in logged arrays: {bad_lengths}")

    # ---- observations ----
    observations = np.array([build_observation(t) for t in range(T)], dtype=np.float32)
    next_observations = np.array([build_observation(t + 1) for t in range(T)], dtype=np.float32)

    obs_dim = len(obs_column_names())
    if observations.shape[1] != obs_dim:
        raise ValueError(f"Expected {obs_dim}-dim observations, got {observations.shape}")

    # Integrity: the obs we rebuild here must equal what the agent actually saw.
    agent_obs = np.asarray(agent_data["observation"], dtype=np.float32)[:T]
    if not np.allclose(observations, agent_obs, atol=1e-4):
        max_err = float(np.max(np.abs(observations - agent_obs)))
        raise ValueError(
            "Rebuilt observations differ from what the agent saw "
            f"(max abs err {max_err:.6g}). The dataset obs layout has drifted "
            "from the eval callback's build_observation.")

    # ---- actions: agent's executed normalized action is the primary action ----
    agent_actions = np.asarray(agent_data["executed_action_norm"], dtype=np.float32)
    agent_logits = np.asarray(
        agent_data["selection_logits"],
        dtype=np.float32,
    )
    agent_probabilities = np.asarray(
        agent_data["selection_probability"],
        dtype=np.float32,
    )
    agent_masks = np.asarray(agent_data["selection_mask"], dtype=np.float32)

    if agent_actions.shape != (n, ACTION_DIM):
        raise ValueError(f"executed_action_norm has shape {agent_actions.shape}, expected ({n}, {ACTION_DIM}).")
    if agent_logits.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_logits has shape "
            f"{agent_logits.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_probabilities.shape != (n, ACTION_DIM):
        raise ValueError(
            "selection_probability has shape "
            f"{agent_probabilities.shape}, expected ({n}, {ACTION_DIM})."
        )
    if agent_masks.shape != (n, ACTION_DIM):
        raise ValueError(f"selection_mask has shape {agent_masks.shape}, expected ({n}, {ACTION_DIM}).")

    actions = agent_actions[:T]
    selection_logits = agent_logits[:T]
    selection_probabilities = agent_probabilities[:T]
    selection_masks = agent_masks[:T]

    if np.any(~np.isfinite(selection_logits)):
        raise ValueError("selection_logits contains NaN or infinite values.")
    if np.any(~np.isfinite(selection_probabilities)):
        raise ValueError(
            "selection_probability contains NaN or infinite values."
        )
    if (
        np.any(selection_probabilities < -1e-7)
        or np.any(selection_probabilities > 1.0 + 1e-7)
    ):
        raise ValueError(
            "selection_probability must lie in [0,1]: "
            f"min={selection_probabilities.min()}, "
            f"max={selection_probabilities.max()}."
        )

    reconstructed_probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(selection_logits, -80.0, 80.0))
    )
    if not np.allclose(
        selection_probabilities,
        reconstructed_probabilities,
        atol=1e-6,
        rtol=1e-6,
    ):
        max_err = float(
            np.max(
                np.abs(
                    selection_probabilities
                    - reconstructed_probabilities
                )
            )
        )
        raise ValueError(
            "selection_probability is inconsistent with sigmoid(logits): "
            f"max abs error={max_err:.6g}."
        )

    # Normalized executed actions must be finite and lie within [-1, 1].
    if np.any(~np.isfinite(actions)):
        raise ValueError("executed_action_norm contains NaN or infinite values.")
    if np.any(actions < -1.0001) or np.any(actions > 1.0001):
        raise ValueError(
            f"Normalized actions outside [-1,1]: min={actions.min()}, max={actions.max()}")

    # Selector masks must be exactly binary; round only after checking tolerance.
    if np.any(~np.isfinite(selection_masks)):
        raise ValueError("selection_mask contains NaN or infinite values.")
    rounded_masks = np.rint(selection_masks)
    if not np.allclose(selection_masks, rounded_masks, atol=1e-6, rtol=0.0):
        bad = np.argwhere(np.abs(selection_masks - rounded_masks) > 1e-6)[0]
        t, dim = (int(bad[0]), int(bad[1]))
        raise ValueError(
            "selection_mask must be binary. "
            f"Found {selection_masks[t, dim]} at transition {t}, dim {dim}."
        )
    selection_masks = rounded_masks.astype(np.float32)

    if selector_mode == "threshold":
        expected_masks = (
            selection_probabilities[1:] >= 0.5
        ).astype(np.float32)
        if not np.array_equal(selection_masks[1:], expected_masks):
            raise ValueError(
                "selector_mode='threshold', but masks after the forced "
                "first step do not equal probability >= 0.5."
            )

    # Validate all physical actuations against the executed normalized action.
    actions_raw = np.array([build_action_raw(t) for t in range(T)], dtype=np.float32)
    _validate_actuated_actions(actions, actions_raw)

    # ---- reconstruct SDAR prev_action / action_mix from the agent trajectory ----
    prev_actions, action_mixes = _reconstruct_sdar_context(
        actions,
        selection_masks,
    )

    # ---- physical lighting quantities from the executed dimming ----
    lighting_dimming = np.clip((actions[:, LIGHT_SLICE] + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)
    nominal_power_w = np.array([LIGHTING_INSTALLED_W[zone] for zone in ZONES_MID], dtype=np.float32)
    lighting_power_from_action_w = (lighting_dimming * nominal_power_w[None, :]).astype(np.float32)
    lighting_energy_kwh = (
        lighting_power_from_action_w.sum(axis=1) * TIMESTEP_HOURS / 1000.0).astype(np.float32)

    # ---- rewards ----
    components_raw = np.zeros((T, len(comp_names)), dtype=np.float32)
    for t in range(T):
        comps = compute_reward_components(t, actions[t], selection_masks[t])
        for i, name in enumerate(comp_names):
            components_raw[t, i] = comps[name]

    components_final, components_unbounded = apply_reward_reference(
        components_raw,
        reward_reference,
    )
    comp_scales = reward_reference["scales"]
    importance_vec = reward_reference["importance"]
    component_clips = reward_reference["clips"]
    scales_source = f"training reference: {reward_reference['path']}"

    reward_physical_raw = components_raw.sum(axis=1).astype(np.float32)
    reward_balanced = components_final.sum(axis=1).astype(np.float32)
    # Legacy training-compatible key: `rewards_raw` means balanced pre-symlog.
    rewards_raw = reward_balanced
    rewards_tx = _transform_vec(reward_balanced)
    rewards = rewards_tx.astype(np.float32)

    terminals = np.zeros(T, dtype=np.float32)
    terminals[-1] = 1.0
    timeouts = _make_timeouts(T)

    obs_mean = observations.mean(axis=0).astype(np.float32)
    obs_std = observations.std(axis=0).astype(np.float32) + 1e-6
    act_mean = actions.mean(axis=0).astype(np.float32)
    act_std = actions.std(axis=0).astype(np.float32) + 1e-6

    light_ranges = np.array(
        [[0.0, LIGHTING_INSTALLED_W[zone]] for zone in ZONES_MID], dtype=np.float32)

    # ---- episode return (undiscounted) for quick scoring ----
    ep_return_physical_raw = float(reward_physical_raw.sum())
    ep_return_balanced = float(reward_balanced.sum())
    # Legacy training-compatible alias.
    ep_return_raw = ep_return_balanced
    ep_return_tx = float(rewards.sum())

    print("\n=== Agent evaluation dataset shapes ===")
    print(f"  observations:      {observations.shape}  (29-dim, meter-free)")
    print(f"  actions  (norm):   {actions.shape}      (agent executed, SDAR space)")
    print(f"  actions_raw:       {actions_raw.shape}  (physical units)")
    print(f"  prev_actions:      {prev_actions.shape}  (reconstructed)")
    print(f"  selection_logits:  {selection_logits.shape}")
    print(
        "  selection_probs:   "
        f"{selection_probabilities.shape}"
    )
    print(f"  selection_masks:   {selection_masks.shape}  (agent selector)")
    print(f"  action_mixes:      {action_mixes.shape}  (reconstructed)")
    print(f"  rewards:           {rewards.shape}  (transform={REWARD_TRANSFORM}, normalize={REWARD_NORMALIZE})")
    print(f"  rewards_raw:       {rewards_raw.shape}  (balanced pre-transform)")
    print(f"  physical raw:      {reward_physical_raw.shape}  (sum of physical W_* terms)")
    print(f"  next_observations: {next_observations.shape}")
    print(f"  terminals:         {terminals.shape}  true terminals={int(terminals.sum())}")
    print(
        "  episode return:    "
        f"physical={ep_return_physical_raw:+.4f}  "
        f"balanced={ep_return_balanced:+.4f}  "
        f"transformed={ep_return_tx:+.4f}"
    )

    _print_normalization(
        comp_names,
        comp_scales,
        importance_vec,
        component_clips,
        scales_source,
    )
    _print_reward_diagnostics(
        reward_physical_raw,
        reward_balanced,
        rewards_tx,
        components_final,
    )
    _print_thermal_violation_rate()
    _print_switching_diagnostics(selection_masks)
    selector_diagnostics = _print_selector_probability_diagnostics(
        selection_probabilities,
        selection_masks,
        selector_mode,
    )
    _print_hvac_demand_diagnostics(observations)
    _print_scale_check(observations, actions_raw, actions)

    if save_npz:
        npz_path = f"{prefix}.npz"
        hvac_idx = obs_column_names().index("hvac_electricity_demand_rate")
        hvac_demand_rate = observations[:, hvac_idx].astype(np.float32)
        next_hvac_demand_rate = next_observations[:, hvac_idx].astype(np.float32)

        save_dict = dict(
            observations=observations,
            actions=actions,
            actions_raw=actions_raw,
            prev_actions=prev_actions,
            selection_logits=selection_logits,
            selection_probabilities=selection_probabilities,
            selection_masks=selection_masks,
            action_mixes=action_mixes,
            rewards=rewards,
            rewards_raw=rewards_raw,
            reward_balanced=reward_balanced,
            reward_physical_raw=reward_physical_raw,
            next_observations=next_observations,
            terminals=terminals,
            timeouts=timeouts,
            hvac_demand_rate=hvac_demand_rate,
            next_hvac_demand_rate=next_hvac_demand_rate,
            obs_mean=obs_mean,
            obs_std=obs_std,
            act_mean=act_mean,
            act_std=act_std,
            obs_cols=np.array(obs_column_names()),
            act_cols=np.array(action_column_names()),
            reward_transform=np.array(reward_reference["transform"]),
            reward_normalize=np.array(REWARD_NORMALIZE),
            reward_component_clip=np.array(
                -1.0 if REWARD_COMPONENT_CLIP is None else REWARD_COMPONENT_CLIP,
                dtype=np.float32),
            thermal_reward_component_clip=np.float32(
                component_clips[comp_names.index("thermal_comfort")]
            ),
            action_mask_value=np.float32(ACTION_MASK_VALUE),
            # physical lighting quantities (action-derived)
            lighting_dimming=lighting_dimming,
            lighting_power_from_action_w=lighting_power_from_action_w,
            lighting_energy_kwh=lighting_energy_kwh,
            light_ranges_per_zone=light_ranges,
            light_range_zone_order=np.array(list(ZONES_MID)),
            # evaluation provenance / quick scores
            policy=np.array(POLICY_TAG),
            dataset_kind=np.array("agent_evaluation"),
            checkpoint_epoch=np.int64(checkpoint["epoch"]),
            checkpoint_path=np.array(checkpoint["path"]),
            checkpoint_sha256=np.array(checkpoint["sha256"]),
            eval_seed=np.int64(-1 if eval_seed is None else int(eval_seed)),
            eval_selector_mode=np.array(selector_mode),
            eval_proposal_deterministic=np.array(
                proposal_deterministic
            ),
            eval_selector_seed=np.int64(selector_seed),
            eval_safeguard_enabled=np.array(bool(safeguard_enabled)),
            eval_weather_file=np.array("" if weather_file is None else str(weather_file)),
            reward_reference_dataset=np.array(reward_reference["path"]),
            reward_reference_sha256=np.array(reward_reference["sha256"]),
            reward_reweighting_tag=np.array(reward_reference["reweighting_tag"]),
            episode_return_physical_raw=np.float32(ep_return_physical_raw),
            episode_return_balanced=np.float32(ep_return_balanced),
            episode_return_raw=np.float32(ep_return_raw),
            episode_return_transformed=np.float32(ep_return_tx),
            selector_diagnostic_groups=np.array(
                ["glazing", "lighting", "heating", "cooling", "overall"]
            ),
            selector_mean_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["mean_probability"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sampled_rate_by_group=np.array(
                [
                    selector_diagnostics[name]["sampled_rate"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_sample_minus_probability_by_group=np.array(
                [
                    selector_diagnostics[name]["difference"]
                    for name in (
                        "glazing",
                        "lighting",
                        "heating",
                        "cooling",
                        "overall",
                    )
                ],
                dtype=np.float32,
            ),
            selector_threshold_agreement=np.float32(
                selector_diagnostics["threshold_agreement"]
            ),
            action_layout=np.array(
                {"glazing": [0, 4], "lighting": [4, 9], "heating": [9, 14], "cooling": [14, 19]},
                dtype=object),
        )

        if SAVE_REWARD_COMPONENTS:
            save_dict["reward_components"] = components_final
            save_dict["reward_components_unbounded"] = components_unbounded
            save_dict["reward_components_raw"] = components_raw
            save_dict["reward_component_names"] = np.array(comp_names)
            save_dict["reward_component_scales"] = comp_scales.astype(np.float32)
            save_dict["reward_component_importance"] = importance_vec.astype(np.float32)
            save_dict["reward_component_clips"] = component_clips.astype(np.float32)

        np.savez_compressed(npz_path, **save_dict)
        print(f"\nSaved NPZ: {npz_path}")

        stats_path = f"{prefix}_stats.json"
        with open(stats_path, "w") as f:
            json.dump(
                {
                    "policy": POLICY_TAG,
                    "dataset_kind": "agent_evaluation",
                    "checkpoint": checkpoint,
                    "evaluation": {
                        "seed": None if eval_seed is None else int(eval_seed),
                        "selector_mode": selector_mode,
                        "proposal_deterministic": proposal_deterministic,
                        "selector_seed": selector_seed,
                        "safeguard_enabled": bool(safeguard_enabled),
                        "weather_file": None if weather_file is None else str(weather_file),
                    },
                    "reward_reference": {
                        "dataset": reward_reference["path"],
                        "sha256": reward_reference["sha256"],
                        "reweighting_tag": reward_reference["reweighting_tag"],
                    },
                    "reward_transform": reward_reference["transform"],
                    "reward_normalize": bool(REWARD_NORMALIZE),
                    "reward_component_clip": REWARD_COMPONENT_CLIP,
                    "thermal_reward_component_clip": float(
                        component_clips[comp_names.index("thermal_comfort")]
                    ),
                    "reward_scales_source": scales_source,
                    "n_transitions": int(T),
                    "obs_dim": int(observations.shape[1]),
                    "act_dim": int(actions.shape[1]),
                    "episode_return_physical_raw": ep_return_physical_raw,
                    "episode_return_balanced": ep_return_balanced,
                    "episode_return_raw": ep_return_raw,
                    "episode_return_transformed": ep_return_tx,
                    "reward_physical_raw_mean": float(reward_physical_raw.mean()),
                    "reward_physical_raw_std": float(reward_physical_raw.std()),
                    "reward_balanced_mean": float(reward_balanced.mean()),
                    "reward_balanced_std": float(reward_balanced.std()),
                    "reward_raw_mean": float(rewards_raw.mean()),
                    "reward_raw_std": float(rewards_raw.std()),
                    "reward_raw_min": float(rewards_raw.min()),
                    "reward_raw_max": float(rewards_raw.max()),
                    "reward_tx_mean": float(rewards_tx.mean()),
                    "reward_tx_std": float(rewards_tx.std()),
                    "reward_component_names": comp_names,
                    "reward_component_scales": comp_scales.tolist(),
                    "reward_component_importance": importance_vec.tolist(),
                    "reward_component_clips": component_clips.tolist(),
                    "comfort_band": {
                        "low_c": COMFORT_TEMP_LOW,
                        "high_c": COMFORT_TEMP_HIGH,
                        "occupied_only": COMFORT_OCCUPIED_ONLY,
                    },
                    "mean_selection_mask_overall": float(selection_masks.mean()),
                    "mean_selection_mask_glazing": float(selection_masks[:, GLAZE_SLICE].mean()),
                    "mean_selection_mask_lighting": float(selection_masks[:, LIGHT_SLICE].mean()),
                    "mean_selection_mask_heating": float(selection_masks[:, HEAT_SLICE].mean()),
                    "mean_selection_mask_cooling": float(selection_masks[:, COOL_SLICE].mean()),
                    "selector_probability_diagnostics": selector_diagnostics,
                    "hvac_demand_rate_mean_w": float(hvac_demand_rate.mean()),
                    "hvac_demand_rate_p95_w": float(np.quantile(hvac_demand_rate, 0.95)),
                    "hvac_demand_rate_max_w": float(hvac_demand_rate.max()),
                    "light_installed_w": {z: LIGHTING_INSTALLED_W[z] for z in ZONES_MID},
                    "obs_mean": obs_mean.tolist(),
                    "obs_std": obs_std.tolist(),
                    "act_mean": act_mean.tolist(),
                    "act_std": act_std.tolist(),
                },
                f,
                indent=2,
            )
        print(f"Saved stats: {stats_path}")

    if save_csv:
        csv_path = f"{prefix}.csv"
        obs_cols = obs_column_names()
        act_cols = action_column_names()
        mask_cols = [f"mask_{c}" for c in act_cols]
        logit_cols = [f"selector_logit_{c}" for c in act_cols]
        probability_cols = [
            f"selector_probability_{c}"
            for c in act_cols
        ]
        next_obs_cols = [f"next_{c}" for c in obs_cols]

        header = (
            obs_cols
            + act_cols
            + mask_cols
            + logit_cols
            + probability_cols
            + ["reward", "reward_raw", "reward_physical_raw"]
            + (comp_names if SAVE_REWARD_COMPONENTS else [])
            + next_obs_cols
            + ["terminal", "timeout"]
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)
            for t in range(T):
                row = (
                    observations[t].tolist()
                    + actions_raw[t].tolist()
                    + selection_masks[t].tolist()
                    + selection_logits[t].tolist()
                    + selection_probabilities[t].tolist()
                    + [
                        float(rewards[t]),
                        float(rewards_raw[t]),
                        float(reward_physical_raw[t]),
                    ]
                    + (components_final[t].tolist() if SAVE_REWARD_COMPONENTS else [])
                    + next_observations[t].tolist()
                    + [float(terminals[t]), float(timeouts[t])]
                )
                writer.writerow(row)
        print(f"Saved CSV: {csv_path}  ({T} rows, {len(header)} cols)")

    return {
        "observations": observations,
        "actions": actions,
        "actions_raw": actions_raw,
        "prev_actions": prev_actions,
        "selection_logits": selection_logits,
        "selection_probabilities": selection_probabilities,
        "selection_masks": selection_masks,
        "action_mixes": action_mixes,
        "rewards": rewards,
        "rewards_raw": rewards_raw,
        "reward_balanced": reward_balanced,
        "reward_physical_raw": reward_physical_raw,
        "reward_components": components_final,
        "reward_components_unbounded": components_unbounded,
        "reward_components_raw": components_raw,
        "reward_component_names": comp_names,
        "reward_component_scales": comp_scales,
        "reward_component_importance": importance_vec,
        "reward_component_clips": component_clips,
        "next_observations": next_observations,
        "terminals": terminals,
        "timeouts": timeouts,
        "obs_mean": obs_mean,
        "obs_std": obs_std,
        "act_mean": act_mean,
        "act_std": act_std,
        "light_ranges_per_zone": light_ranges,
        "lighting_dimming": lighting_dimming,
        "lighting_power_from_action_w": lighting_power_from_action_w,
        "lighting_energy_kwh": lighting_energy_kwh,
        "checkpoint": checkpoint,
        "evaluation": {
            "seed": None if eval_seed is None else int(eval_seed),
            "selector_mode": selector_mode,
            "proposal_deterministic": proposal_deterministic,
            "selector_seed": selector_seed,
            "safeguard_enabled": bool(safeguard_enabled),
            "weather_file": (
                None if weather_file is None else str(weather_file)
            ),
        },
        "selector_probability_diagnostics": selector_diagnostics,
        "reward_reference": reward_reference,
        "episode_return_physical_raw": ep_return_physical_raw,
        "episode_return_balanced": ep_return_balanced,
        "episode_return_raw": ep_return_raw,
        "episode_return_transformed": ep_return_tx,
    }


if __name__ == "__main__":
    dataset = build_agent_dataset(
        save_npz=True,
        save_csv=True,
        reference_dataset_path=REFERENCE_REWARD_DATASET,
        checkpoint_path=EVAL_CHECKPOINT_PATH,
        checkpoint_epoch=EVAL_CHECKPOINT_EPOCH,
        eval_seed=EVAL_SEED,
        selector_mode=EVAL_SELECTOR_MODE,
        proposal_deterministic=EVAL_PROPOSAL_DETERMINISTIC,
        selector_seed=EVAL_SELECTOR_SEED,
        safeguard_enabled=EVAL_SAFEGUARD_ENABLED,
        weather_file=EVAL_WEATHER_FILE,
    )